# NIHONGA - Phase 7: Adaptive Architecture Pruning

Phase 6 accepted a 31-block student: block 5 replaced by its pretrained rank-256 bridge ("B"). Phase 7 looks for
more blocks to remove. The target is **four removed blocks**, non-adjacent where possible.

This notebook records every Phase 7 step in order: what was run, the exact code (each worker is embedded exactly
as it executed, taken from its saved request), the results and what they mean. Re-running it **does not use the
GPU**: every step loads its saved result. A step only launches Modal work when its result is missing and
`ALLOW_GPU = True`.

Shared infrastructure stays in `tools/`: Modal account selection (`modal_accounts.py`), runtime pinning
(`experiment_runtime.py`), progress sync (`experiment_status.py`), archive/recovery tools and the site publisher
(`publish_site_scan.py`). Large tensors and images live in `.archive/` and on the Modal volume.

Velocity error below always means the relative L2 distance between the student's and the teacher's velocity
prediction on the same saved noisy input, averaged over 60 development prompts (10 "check" + 50 "wider") at early,
middle and late denoising moments: 180 comparisons. 0% means identical to the teacher. It measures fidelity to
the teacher, not image quality; images decide.

In [ ]:
import json
import sys
from pathlib import Path
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
assert (ROOT / 'PLAN.md').exists(), 'Run this notebook from the repository root.'
sys.path.insert(0, str(ROOT / 'tools'))
for stream in (sys.stdout, sys.stderr):      # Modal prints non-ASCII progress symbols
    if hasattr(stream, 'reconfigure'):
        stream.reconfigure(encoding='utf-8')
ALLOW_GPU = False          # set True only to launch a step whose saved result is missing
SOURCES = {}               # worker sources, embedded per step exactly as executed


def load_summary(run):
    path = ROOT / run / 'summary.json'
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else None


def reuse_or_launch(run, launch):
    summary = load_summary(run)
    if summary is not None:
        print('Saved result:', run, '| status', summary.get('status'))
        return summary
    if (ROOT / run).parent.joinpath('aborted.json').exists():
        print('Run stopped by request; nothing to reuse.')
        return None
    if not ALLOW_GPU:
        raise RuntimeError(f'No saved result for {run}; set ALLOW_GPU = True to launch it on Modal.')
    launch()
    return load_summary(run)


def pct(value, digits=2):
    return f'{100 * value:.{digits}f}%'


def velocity_row(label, block):
    o = block['overall']['teacher_relative_l2']
    p = {k: block['panels'][k]['teacher_relative_l2']['mean'] for k in block['panels']}
    s = {k: block['stages'][k]['teacher_relative_l2']['mean'] for k in block['stages']}
    return (f"| {label} | {pct(o['mean'])} | {pct(p['gate10'])} | {pct(p['wider50'])} | "
            f"{pct(s['early'], 1)} / {pct(s['middle'], 1)} / {pct(s['late'], 1)} | {pct(o['p90'], 1)} | {pct(o['maximum'], 1)} |")


def velocity_table(rows):
    head = '| Model | Overall | Check (10) | Wider (50) | Early / middle / late | p90 | Worst |\n|---|---|---|---|---|---|---|\n'
    display(Markdown(head + '\n'.join(rows)))


def show_variants(summary):
    rows = [velocity_row(f"{name} (step {v['fit']['selected_step']}, {v['parameters'] / 1e6:.1f}M params)", v)
            for name, v in summary['variants'].items()]
    velocity_table(rows)


def show_images(folder, names):
    for name in names:
        display(Image(filename=str(ROOT / folder / name), width=1000))

## 1. Where Phase 7 started (recovered history)

Phase 7 began on 4 October 2026 and its first steps were recovered from Modal after the account migration.

* **Four blocks at once (2-5 removed, 28 blocks)** failed even after 480 healing updates: 13.2% velocity error
  against 5.4% for B, with wrong subjects and broken text.
* **Two blocks (4-5 removed, candidate A)**, one rank-512 region bridge and 60 healing updates: 7.41% on the 50
  wider prompts and 7.97% on the 10 check prompts. That fails the predeclared gate (B's 5.52% on the check prompts
  plus one point: 6.52%), but a small unblinded 30-prompt image screen looked close to the teacher.
* The saved plan was to treat A as a provisional base and recalculate block importance inside it.

In [ ]:
published = json.loads((ROOT / 'data/published/2026-10-04/results.json').read_bytes())
{key: published[key] for key in ('phase7_four_blocks', 'phase7_healing_28_blocks', 'phase7_blocks_4_5', 'phase7_candidate_images',
                                 'phase7_adaptive_pruning_plan') if key in published}

Candidate A itself: the recovered healing run (blocks 4-5, rank-512 region bridge, 60 updates). The worker source in its request must match the verbatim copy preserved under `data/recovered/sources`.

In [ ]:
import hashlib
run = ROOT / 'data/recovered/modal/phase7/blocks45_healing/95fcd2cfae35cb7f124b8af2865c30f928cbd681a0d015df5e56e8ae5ba4fd78'
a_request = json.loads((run / 'request.json').read_bytes())['request']
a_summary = json.loads((run / 'summary.json').read_bytes())
source_hash = hashlib.sha256(a_request['worker_source'].encode()).hexdigest()
assert (ROOT / 'data/recovered/sources' / (source_hash + '.py')).read_bytes() == a_request['worker_source'].encode()
print('Recovered worker:', source_hash)
{key: a_summary[key] for key in ('status', 'selected_step', 'gate_passed_steps', 'surviving_blocks')}

## 2. Candidate A, blinded image test

Before building on A, a fresh blinded panel decides whether it keeps image quality: 159 new validation prompts (35 counting, 34 spatial, 30 colour, 30 multiple-object, 30 text), one seed each, teacher and A at 40 steps. The protocol, prompts and gates were committed before any image was generated (`data/phase7/blind_quality_v1/protocol.json`).

In [ ]:
# Candidate A, blinded image test: reuses the saved run; launches GPU work only when its result is missing and ALLOW_GPU is True.

RUN = 'data/phase7/blind_quality/c2956d3683056c2850fccf4540ec288e86052150e89bff7444e13d04e1e5ce26'

LAUNCH_ARGS = []

SOURCES['blind_quality_worker.py'] = r'''def generate_blind_quality(request, key):
    """Matched 40-step images for the Phase 7 candidates. family 'blocks45': teacher, C (bridge only), A and B (healed);
    family 'blocks25': D (28-block healed). Same prompt, CPU noise seed, size, scheduler and 40 steps for every model."""
    import gc
    import hashlib
    import inspect
    import json
    import os
    import time
    from pathlib import Path

    import torch
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache, QwenImage21KVLayerCache

    directory = Path('/phase3/phase7/blind_quality') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete image attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def table_hash(value):
        return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

    variants = ['teacher','A_healed_u60']
    report = {'request_key': key, 'status': 'running', 'rows': [], 'artifacts': [],
              'calls': {v: 0 for v in variants}, 'optimizer_updates': 0}
    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('Expected H100.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
                                                   dtype=torch.bfloat16, local_files_only=True).to('cuda')
        pipe.set_progress_bar_config(disable=True)
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Transformer source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--')) /
                       'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        # Derived position/time tables: use the audited values on every host (as in steps 7-11).
        derived = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
        derived['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
        expected = request['expected_derived_tables']
        mismatched = sorted(n for n, v in derived.items() if table_hash(v) != expected[n])
        if mismatched:
            canonical = torch.load(Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt'),
                                   weights_only=True, map_location='cpu')
            if any(table_hash(canonical[n]) != expected[n] for n in expected):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [canonical[f'pos_embed.freqs.{a}'].to(device=v.device, dtype=v.dtype) for a, v in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(canonical['time_text_embed.time_proj.freqs'])
            if any(table_hash(v) != expected[n] for n, v in
                   list({f'pos_embed.freqs.{a}': v for a, v in enumerate(model.pos_embed.freqs)}.items()) +
                   [('time_text_embed.time_proj.freqs', model.time_text_embed.time_proj.freqs)]):
                raise RuntimeError('Derived tables still differ after loading the audited copy.')
            report['canonical_tables_applied'] = mismatched

        def generate(row, variant):
            path = directory / (row['source_id'] + '-' + variant + '.png')
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'],
                             num_inference_steps=40, true_cfg_scale=teacher['true_cfg_scale'],
                             generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=True, output_type='pil').images[0]
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            temporary = path.with_suffix('.tmp')
            image.save(temporary, format='PNG')
            temporary.replace(path)
            results_volume.commit()
            report['calls'][variant] += 1
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            return {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size, 'seconds': elapsed}

        namespace = {}
        exec(compile(request['core_source'], 'bridge_definition.py', 'exec'), namespace)
        exec(compile(request['wrapper_source'], 'bridge_block_definition.py', 'exec'), namespace)
        exec(compile(request['components_source'], 'healing_components.py', 'exec'), namespace)

        def assemble(student):
            region = student['region_bridge']
            path = Path('/phase3/phase7/region_bridge_fit') / region['fit_key'] / region['checkpoint']
            if digest(path) != region['sha256']:
                raise RuntimeError('Region bridge checkpoint differs.')
            value = torch.load(path, weights_only=True, map_location='cpu')
            if value['request_key'] != region['fit_key'] or value['region'] != student['slots'] or value['rank'] != region['rank']:
                raise RuntimeError('Region bridge provenance differs.')
            bridge = namespace['MasterBridge'](4096, region['rank'])
            bridge.load_state_dict(value['state_dict'])
            removed, _ = namespace['insert_bridge'](model, student['slots'][0], bridge, QwenImage21KVCache)
            bridge.float()
            bridge.load_state_dict(value['state_dict'])
            del removed
            for slot in student['slots'][1:]:
                passthrough = namespace['ResidualBottleneckBridge'](4096, 256)
                with torch.no_grad():
                    passthrough.up.weight.zero_()
                removed, _ = namespace['insert_bridge'](model, slot, passthrough, QwenImage21KVCache)
                del removed
            surviving = sum(not hasattr(b, 'bridge') for b in model.transformer_blocks)
            if surviving != 32 - len(student['slots']):
                raise RuntimeError('Student assembly differs.')
            return surviving

        def wrap(student):
            for slot, block in enumerate(model.transformer_blocks):
                if slot not in student['slots']:
                    model.transformer_blocks[slot] = namespace['FunctionalCheckpointBlock'](block, QwenImage21KVLayerCache)
            frozen = {f'transformer_blocks.{s}.bridge.{t}.weight' for s in student['slots'][1:] for t in ('down', 'up')}
            parameters = {n: p for n, p in model.named_parameters() if n not in frozen}
            if sum(p.numel() for p in parameters.values()) != student['trainable_parameters']:
                raise RuntimeError('Trained parameter structure differs.')
            return parameters

        def load_trained(parameters, checkpoint):
            path = Path(checkpoint['remote_path'])
            if path.stat().st_size != checkpoint['bytes'] or digest(path) != checkpoint['sha256']:
                raise RuntimeError('Healed checkpoint differs: ' + checkpoint['remote_path'])
            state = torch.load(path, weights_only=True, map_location='cpu')
            if state['request_key'] != checkpoint['request_key'] or state['step'] != checkpoint['step'] or \
                    set(parameters) != set(state['master_state']):
                raise RuntimeError('Healed checkpoint provenance or parameter set differs.')
            with torch.no_grad():
                for name, parameter in parameters.items():
                    parameter.copy_(state['master_state'][name].to(device=parameter.device, dtype=parameter.dtype))
            del state
            gc.collect()
            model.eval()

        outputs = {v: {} for v in variants}
        students = request['students']
        for row in request['rows']:
            outputs['teacher'][row['source_id']] = generate(row,'teacher')
        report['surviving_blocks'] = assemble(students['blocks45'])
        parameters = wrap(students['blocks45'])
        load_trained(parameters,request['checkpoints']['A_healed_u60'])
        for row in request['rows']:
            outputs['A_healed_u60'][row['source_id']] = generate(row,'A_healed_u60')
        for row in request['rows']:
            report['rows'].append({**row, **{v: outputs[v][row['source_id']] for v in variants}})
            report['artifacts'].extend(outputs[v][row['source_id']] for v in variants)
        report['status'] = 'passed'
    except Exception as exc:
        report['status'] = 'failed'
        report['error'] = type(exc).__name__ + ': ' + str(exc)
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=True) + '\n')
    results_volume.commit()
    return report
'''


"""Generate the preregistered fresh blinded quality panel; reuse completed remote results."""
import argparse
import hashlib
import json
import os
import sys
from pathlib import Path


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--old-account', action='store_true')
    args = parser.parse_args(LAUNCH_ARGS)
    if args.old_account:
        parser.error('Old account is download-only; GPU execution must use the new account.')
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    root = ROOT
    recovered = root / 'data/recovered/modal'
    images_key = 'dda8c184e657dfb604c35fb261d83371c108c39fac033a0f57db66f7686c30ad'
    request = json.loads((recovered / 'phase7/candidate_images' / images_key / 'request.json').read_bytes())['request']
    source = SOURCES['blind_quality_worker.py']
    protocol_path = root / 'data/phase7/blind_quality_v1/protocol.json'
    prompts_path = root / 'data/phase7/blind_quality_v1/prompts.json'
    protocol = json.loads(protocol_path.read_bytes())
    if hashlib.sha256(prompts_path.read_bytes()).hexdigest() != protocol['prompt_file_sha256']:
        raise RuntimeError('Preregistered prompt list changed')
    request.update(version='phase7-blind-quality-v1', worker_source=source, execution_account='new',
                   rows=json.loads(prompts_path.read_bytes()), quality_protocol=protocol)
    from experiment_runtime import pin_reference_runtime
    pin_reference_runtime(request,recovered)
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase7/blind_quality' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2)+'\n',encoding='utf-8')
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase7/blind_quality/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        try:
            b''.join(results_volume.read_file(remote + '/started.json'))
        except FileNotFoundError:
            pass
        else:
            raise RuntimeError('Run already started; inspect its progress. No duplicate GPU scheduled.')
        app = modal.App('nihonga-phase7-blind-quality')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(source, 'blind_quality_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8,
                              memory=98304, max_containers=1, retries=0,
                              volumes={'/phase3':results_volume,
                                       '/root/.cache/huggingface':weights.with_mount_options(read_only=True)})(namespace['generate_blind_quality'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'),indent=2)+'\n')
            print('Request:',request_key,'Call:',call.object_id,flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2)+'\n',encoding='utf-8')
    print(json.dumps({k:report.get(k) for k in ('status','error','calls')}),flush=True)
    if report.get('status') != 'passed':
        raise SystemExit(1)

summary = reuse_or_launch(RUN, main)

print('images rendered:', summary['calls'])

### What the results mean

All 318 images were generated (159 teacher, 159 A). Grading follows below.

### Build the model-blind review sheets

Left/right positions are randomized per prompt; the identity key stays in `.archive/` and only its hash is public until all scores are committed.

In [ ]:
# Build the model-blind review sheets: local work only (no GPU).
LAUNCH_ARGS = []

"""Fetch available quality pairs and build model-blind review sheets.

The per-prompt randomized identity key stays in ignored .archive until scores
are locked. Sheets contain only prompt IDs, questions and neutral image labels.
"""
import argparse
import hashlib
import json
import os
from pathlib import Path
import random
import textwrap


def main():
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--old-account',action='store_true')
    args=parser.parse_args(LAUNCH_ARGS)
    from modal_accounts import set_modal_account
    set_modal_account('old' if args.old_account else 'current', purpose='download')
    import modal
    from PIL import Image,ImageDraw,ImageFont
    root=ROOT
    public=root/'data/phase7/blind_quality_v1'
    private=root/'.archive/blind_quality_v1'
    private.mkdir(parents=True,exist_ok=True)
    prompts=json.loads((public/'prompts.json').read_bytes())
    key_path=private/'private_labels.json'
    if not key_path.exists():
        labels={}
        for row in prompts:
            variants=['teacher','A_healed_u60']
            random.SystemRandom().shuffle(variants)
            labels[row['source_id']]=dict(zip(('L','R'),variants))
        key_path.write_text(json.dumps(labels,indent=2)+'\n')
    labels=json.loads(key_path.read_bytes())
    (public/'blind_key_commitment.json').write_text(json.dumps(dict(
        sha256=hashlib.sha256(key_path.read_bytes()).hexdigest(),
        policy='Reveal this identity key only after all visual scores are locked.'),indent=2)+'\n')
    volume=modal.Volume.from_name('nihonga-phase3-artifacts')
    calls=list((root/'data/phase7/blind_quality').glob('*/call.json'))
    if len(calls)!=1:raise RuntimeError('Expected exactly one registered quality run')
    request_key=json.loads(calls[0].read_bytes())['request_key']
    remote='/phase7/blind_quality/'+request_key
    for filename in ('summary.json','progress.json'):
        try:report=json.loads(b''.join(volume.read_file(remote+'/'+filename)))
        except FileNotFoundError:continue
        break
    else:raise RuntimeError('Image job has not reported progress yet')
    available=min(report['calls'].get('teacher',0),report['calls'].get('A_healed_u60',0))
    font_path='C:/Windows/Fonts/arial.ttf'
    for candidate in ('C:/Windows/Fonts/msyh.ttc','C:/Windows/Fonts/simsun.ttc'):
        if Path(candidate).exists():font_path=candidate;break
    font=ImageFont.truetype(font_path,19)
    title_font=ImageFont.truetype(font_path,24)
    sheet_dir=public/'review_sheets'
    sheet_dir.mkdir(exist_ok=True)
    manifest=[]
    for index,row in enumerate(prompts[:available]):
        paths={}
        for label,variant in labels[row['source_id']].items():
            name=row['source_id']+'-'+variant+'.png'
            path=private/name
            if not path.exists():
                temporary=path.with_suffix('.partial')
                with temporary.open('wb') as stream:volume._read_file_into_fileobj(remote+'/'+name,stream,concurrency=2)
                temporary.replace(path)
            if report.get('status')=='passed':
                artifact=next(a for a in report['artifacts'] if a['name']==name)
                if hashlib.sha256(path.read_bytes()).hexdigest()!=artifact['sha256']:
                    raise RuntimeError('Quality image hash mismatch')
            paths[label]=path
        sheet=Image.new('RGB',(2048,1280),'white')
        draw=ImageDraw.Draw(sheet)
        draw.text((12,6),f"Q{index+1:03d} | {row['source_id']} | {row['subdimension']}",font=title_font,fill='black')
        prompt_lines=textwrap.wrap(row['prompt'],width=150)
        for line_number,line in enumerate(prompt_lines[:6]):draw.text((12,43+line_number*25),line,font=font,fill='black')
        draw.text((12,200),row['question'],font=font,fill='black')
        for x,label in ((0,'L'),(1024,'R')):
            draw.text((x+12,235),label,font=title_font,fill='black')
            with Image.open(paths[label]) as image:sheet.paste(image.convert('RGB'),(x,256))
        target=sheet_dir/f'Q{index+1:03d}.jpg'
        if not target.exists():sheet.save(target,quality=95,subsampling=0)
        manifest.append(dict(question_id=f'Q{index+1:03d}',source_id=row['source_id'],
                             prompt=row['prompt'],question=row['question'],group=row['subdimension'],
                             sheet=str(target.relative_to(root)).replace('\\','/'),
                             sha256=hashlib.sha256(target.read_bytes()).hexdigest()))
    (public/'review_manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
    contact_dir=public/'review_contacts'
    contact_dir.mkdir(exist_ok=True)
    for offset in range(0,len(manifest)-2,3):
        target=contact_dir/f'panel_{offset//3+1:02d}.jpg'
        if target.exists():continue
        canvas=Image.new('RGB',(1024,1680),'white')
        draw=ImageDraw.Draw(canvas)
        for position,entry in enumerate(manifest[offset:offset+3]):
            y=position*560
            draw.text((10,y+5),entry['question_id']+'  L | R  '+entry['group'],font=title_font,fill='black')
            with Image.open(root/entry['sheet']) as image:
                crop=image.crop((0,256,2048,1280)).resize((1024,512),Image.Resampling.LANCZOS)
                canvas.paste(crop,(0,y+40))
        canvas.save(target,quality=95,subsampling=0)
    print(f'{available}/{len(prompts)} blinded pairs available; {len(manifest)} sheets ready.')

if not (ROOT / 'data/phase7/blind_quality_v1/review_manifest.json').exists():
    main()
else:
    print("Saved output present; not rebuilt.")

### Grading

Each image was scored pass / fail / uncertain against one fixed question by model-blind AI reviewers (Q001-Q064 in
one pass, Q065-Q159 by three parallel reviewers that never saw the identity key). One grade was corrected before
locking (Q129 L: every requested object was present; colours are outside that question). The complete 318 scores
were committed (`data/phase7/blind_quality_v1/scores.json`) before the key was revealed.

### Unblind the locked scores and apply the gates

Refuses to run unless the complete scores are committed; the lock file prevents any later revision.

In [ ]:
# Unblind the locked scores and apply the gates: local work only (no GPU).
LAUNCH_ARGS = []

"""Unblind committed, locked visual scores and apply the preregistered gates."""
import argparse
from collections import Counter
import hashlib
import json
from pathlib import Path
import random
import shutil
import subprocess


def main():
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--scores',type=Path,default=Path('data/phase7/blind_quality_v1/scores.json'))
    args=parser.parse_args(LAUNCH_ARGS)
    root=ROOT
    source=args.scores.resolve()
    if not source.is_relative_to(root):raise ValueError('Scores must be in the repository')
    relative=source.relative_to(root).as_posix()
    raw=source.read_bytes()
    committed=subprocess.run(['git','show','HEAD:'+relative],cwd=root,capture_output=True,check=True).stdout
    if raw!=committed:raise RuntimeError('Commit the complete blinded scores before revealing identities')
    public=root/'data/phase7/blind_quality_v1'
    protocol=json.loads((public/'protocol.json').read_bytes())
    prompts=json.loads((public/'prompts.json').read_bytes())
    scores=json.loads(raw)
    expected={(f'Q{i+1:03d}',label) for i in range(len(prompts)) for label in ('L','R')}
    identities=[(r['question_id'],r['label']) for r in scores]
    if len(scores)!=318 or len(set(identities))!=318 or set(identities)!=expected:
        raise RuntimeError('Require exactly 318 unique blinded image scores')
    if any(r['score'] not in ('pass','fail','uncertain') or not r.get('note') for r in scores):
        raise RuntimeError('Every score requires a valid status and an observation')
    for row in scores:
        index=int(row['question_id'][1:])-1
        if prompts[index]['subdimension']=='text_rendering' and not isinstance(row.get('requested_phrase_absent'),bool):
            raise RuntimeError('Text reviews must record whether any requested phrase is entirely absent')
    lock=dict(scores_sha256=hashlib.sha256(raw).hexdigest(),images=len(scores),
              commit=subprocess.check_output(['git','rev-parse','HEAD'],cwd=root,text=True).strip())
    lock_path=public/'score_lock.json'
    if lock_path.exists():
        if json.loads(lock_path.read_bytes())['scores_sha256']!=lock['scores_sha256']:
            raise RuntimeError('Scores changed after locking; do not revise after unblinding')
    else:lock_path.write_text(json.dumps(lock,indent=2)+'\n')
    key_path=root/'.archive/blind_quality_v1/private_labels.json'
    commitment=json.loads((public/'blind_key_commitment.json').read_bytes())['sha256']
    if hashlib.sha256(key_path.read_bytes()).hexdigest()!=commitment:
        raise RuntimeError('Blind key commitment differs')
    labels=json.loads(key_path.read_bytes())
    by_id={(r['question_id'],r['label']):r for r in scores}
    pairs=[]
    for index,prompt in enumerate(prompts):
        qid=f'Q{index+1:03d}'
        models={labels[prompt['source_id']][label]:by_id[(qid,label)] for label in ('L','R')}
        a=models['A_healed_u60'];teacher=models['teacher']
        pairs.append(dict(question_id=qid,source_id=prompt['source_id'],group=prompt['subdimension'],
                          teacher=teacher['score'],candidate_A=a['score'],
                          clear_pass_delta=int(a['score']=='pass')-int(teacher['score']=='pass'),
                          missing_phrase_loss=prompt['subdimension']=='text_rendering' and
                            teacher['score']=='pass' and a['score']!='pass' and a.get('requested_phrase_absent',False)))
    adherence=[r for r in pairs if r['group']!='text_rendering']
    text=[r for r in pairs if r['group']=='text_rendering']
    if len(adherence)!=129 or len(text)!=30:raise RuntimeError('Panel counts differ')
    rng=random.Random(protocol['bootstrap_seed'])
    deltas=[r['clear_pass_delta'] for r in adherence]
    bootstrap=sorted(sum(rng.choices(deltas,k=129))/129 for _ in range(protocol['bootstrap_replicates']))
    ci=[bootstrap[499],bootstrap[19499]]
    counts={panel:{model:dict(Counter(r[model] for r in rows)) for model in ('teacher','candidate_A')}
            for panel,rows in (('adherence',adherence),('text',text))}
    adherence_pass=ci[0]>=-.05
    text_pass=(sum(r['candidate_A']=='pass' for r in text)>=sum(r['teacher']=='pass' for r in text)-1
               and not any(r['missing_phrase_loss'] for r in text))
    benchmark=None
    for path in (root/'data/phase7/latency_benchmark').glob('*/summary.json'):
        report=json.loads(path.read_bytes())
        if report.get('status')=='passed':benchmark=report
    speed_pass=None if benchmark is None else benchmark['paired_speedup']['bootstrap_95pct'][0]>1
    report=dict(status='scored',protocol_sha256=hashlib.sha256((public/'protocol.json').read_bytes()).hexdigest(),
                scores_sha256=lock['scores_sha256'],blind_key_sha256=commitment,counts=counts,
                adherence_gain_fraction=sum(deltas)/129,paired_bootstrap_95pct=ci,
                adherence_noninferiority_passed=adherence_pass,text_gate_passed=text_pass,
                performance_gate_passed=speed_pass,research_replacement_gate_passed=adherence_pass and text_pass and speed_pass is True,
                historical_velocity_gate_passed=False,production_promoted=False,pairs=pairs,
                limitations=['One model-blind AI reviewer; one saved seed per prompt.',
                             'No independent external image-quality benchmark; no production promotion.'])
    (public/'review_summary.json').write_text(json.dumps(report,indent=2)+'\n')
    shutil.copyfile(key_path,public/'revealed_labels.json')
    print(json.dumps({k:v for k,v in report.items() if k not in ('pairs',)},indent=2))

if not (ROOT / 'data/phase7/blind_quality_v1/review_summary.json').exists():
    main()
else:
    print("Saved output present; not rebuilt.")

### What the results mean

| Panel | Teacher pass | A pass | A better / worse | Gate | Result |
|---|---|---|---|---|---|
| Adherence (129) | 89 | 78 | 6 / 17 | 95% lower bound of the gain >= -5 points | **fail**: -8.5 points, CI -15.5 to -1.6 |
| Text (30) | 28 | 18 | 0 / 10 | A within one of the teacher | **fail** |

**Candidate A is rejected as a base for further pruning.** It fails both image gates as well as the velocity gate.
The earlier impression that A matched the teacher came from a small unblinded screen.

## 3. Candidate A, measured speed

Thirty balanced, randomized teacher/A pairs on one H100, 1024x1024, 40 steps, BF16, warm models.

In [ ]:
# Candidate A, measured speed: reuses the saved run; launches GPU work only when its result is missing and ALLOW_GPU is True.

RUN = 'data/phase7/latency_benchmark/29f50a4db5f63a6c73135438800ab2d5413e32878bc3fa61284aebbb5c165f4c'

LAUNCH_ARGS = []

SOURCES['latency_benchmark_worker.py'] = r'''def benchmark_candidate(request, key):
    """Matched 40-step images for the Phase 7 candidates. family 'blocks45': teacher, C (bridge only), A and B (healed);
    family 'blocks25': D (28-block healed). Same prompt, CPU noise seed, size, scheduler and 40 steps for every model."""
    import gc
    import hashlib
    import inspect
    import json
    import os
    import time
    from pathlib import Path

    import torch
    from torch.nn.attention import sdpa_kernel, SDPBackend
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache, QwenImage21KVLayerCache

    directory = Path('/phase3/phase7/latency_benchmark') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete image attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def table_hash(value):
        return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

    variants = ['teacher', 'candidate_A']
    report = {'request_key': key, 'status': 'running', 'rows': [], 'artifacts': [],
              'calls': {v: 0 for v in variants}, 'optimizer_updates': 0}
    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('Expected H100.')
        load_started = time.perf_counter()
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
                                                   dtype=torch.bfloat16, local_files_only=True).to('cuda')
        torch.cuda.synchronize()
        pipeline_load_seconds = time.perf_counter()-load_started
        pipe.set_progress_bar_config(disable=True)
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Transformer source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--')) /
                       'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        # Derived position/time tables: use the audited values on every host (as in steps 7-11).
        derived = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
        derived['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
        expected = request['expected_derived_tables']
        mismatched = sorted(n for n, v in derived.items() if table_hash(v) != expected[n])
        if mismatched:
            canonical = torch.load(Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt'),
                                   weights_only=True, map_location='cpu')
            if any(table_hash(canonical[n]) != expected[n] for n in expected):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [canonical[f'pos_embed.freqs.{a}'].to(device=v.device, dtype=v.dtype) for a, v in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(canonical['time_text_embed.time_proj.freqs'])
            if any(table_hash(v) != expected[n] for n, v in
                   list({f'pos_embed.freqs.{a}': v for a, v in enumerate(model.pos_embed.freqs)}.items()) +
                   [('time_text_embed.time_proj.freqs', model.time_text_embed.time_proj.freqs)]):
                raise RuntimeError('Derived tables still differ after loading the audited copy.')
            report['canonical_tables_applied'] = mismatched

        import random
        import statistics
        from types import SimpleNamespace
        started_loading = time.perf_counter()
        teacher_state = {n:v.detach().cpu().clone() for n,v in model.state_dict().items()}
        original_blocks = list(model.transformer_blocks)
        namespace = {}
        for field in ('core_source','wrapper_source','components_source'):
            exec(compile(request[field],field+'.py','exec'),namespace)
        checkpoint = request['checkpoints']['A_healed_u60']
        path = Path(checkpoint['remote_path'])
        if path.stat().st_size != checkpoint['bytes'] or digest(path) != checkpoint['sha256']:
            raise RuntimeError('Candidate A checkpoint differs')
        state = torch.load(path,weights_only=True,map_location='cpu')
        if state['request_key']!=checkpoint['request_key'] or state['step']!=60:
            raise RuntimeError('Candidate A provenance differs')
        masters = state['master_state']
        del state
        student = request['students']['blocks45']
        region = student['region_bridge']
        region_path = Path('/phase3/phase7/region_bridge_fit')/region['fit_key']/region['checkpoint']
        if digest(region_path)!=region['sha256']:
            raise RuntimeError('Bridge checkpoint differs')
        region_value = torch.load(region_path,weights_only=True,map_location='cpu')
        report['cpu_snapshot_and_checkpoint_seconds']=time.perf_counter()-started_loading
        report['protocol']=request['benchmark_protocol']
        report['packages']={p:__import__('importlib.metadata',fromlist=['version']).version(p)
                            for p in ('torch','diffusers','transformers','accelerate','pillow')}
        report['torch_cuda']=torch.version.cuda
        report['cold_pipeline_load_seconds']=pipeline_load_seconds

        def restore(variant):
            model.transformer_blocks=torch.nn.ModuleList(original_blocks)
            model.to('cuda')
            model.load_state_dict(teacher_state,strict=True)
            if variant=='candidate_A':
                bridge=namespace['MasterBridge'](4096,512)
                bridge.load_state_dict(region_value['state_dict'])
                removed,_=namespace['insert_bridge'](model,4,bridge,QwenImage21KVCache)
                bridge.float()
                bridge.load_state_dict(region_value['state_dict'])
                del removed
                passthrough=namespace['ResidualBottleneckBridge'](4096,256)
                with torch.no_grad():passthrough.up.weight.zero_()
                removed,_=namespace['insert_bridge'](model,5,passthrough,QwenImage21KVCache)
                del removed
                for slot,block in enumerate(model.transformer_blocks):
                    if slot not in (4,5):
                        model.transformer_blocks[slot]=namespace['FunctionalCheckpointBlock'](block,QwenImage21KVLayerCache)
                frozen={'transformer_blocks.5.bridge.down.weight','transformer_blocks.5.bridge.up.weight'}
                parameters={n:p for n,p in model.named_parameters() if n not in frozen}
                if set(parameters)!=set(masters):raise RuntimeError('Candidate parameter names differ')
                with torch.no_grad():
                    for name,p in parameters.items():p.copy_(masters[name].to(device=p.device,dtype=p.dtype))
                original_blocks[4].cpu()
                original_blocks[5].cpu()
            model.requires_grad_(False).eval()
            model.gradient_checkpointing=False
            gc.collect()
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

        events=[]
        def before(module,args):
            start=torch.cuda.Event(enable_timing=True)
            end=torch.cuda.Event(enable_timing=True)
            start.record()
            events.append((start,end))
        def after(module,args,output):events[-1][1].record()
        pre=model.register_forward_pre_hook(before)
        post=model.register_forward_hook(after)

        def generate(row):
            events.clear()
            torch.cuda.synchronize()
            started=time.perf_counter()
            with torch.inference_mode(), sdpa_kernel([SDPBackend.MATH]):
                image=pipe(prompt=row['prompt'],width=row['width'],height=row['height'],
                           num_inference_steps=40,true_cfg_scale=teacher['true_cfg_scale'],
                           generator=torch.Generator('cpu').manual_seed(row['seed']),
                           use_kv_cache=True,output_type='pil').images[0]
            torch.cuda.synchronize()
            seconds=time.perf_counter()-started
            denoiser=sum(a.elapsed_time(b) for a,b in events)/1000
            return image,seconds,denoiser

        # Fifteen AB and fifteen BA pairs, shuffled before observing timings.
        rng=random.Random(request['benchmark_protocol']['order_seed'])
        orders=[('teacher','candidate_A')]*15+[('candidate_A','teacher')]*15
        rng.shuffle(orders)
        report['pair_orders']=orders
        report['rows']=[]
        for index,(row,order) in enumerate(zip(request['rows'],orders)):
            for position,variant in enumerate(order):
                restore_started=time.perf_counter()
                restore(variant)
                restore_seconds=time.perf_counter()-restore_started
                for _ in range(request['benchmark_protocol']['warmups_per_restore']):
                    warm,_,_=generate(row)
                    del warm
                gc.collect()
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats()
                image,seconds,denoiser=generate(row)
                record=dict(pair=index,position=position,variant=variant,source_id=row['source_id'],
                            seed=row['seed'],width=row['width'],height=row['height'],seconds=seconds,
                            denoiser_seconds=denoiser,restore_seconds_excluded=restore_seconds,
                            peak_allocated_gib=torch.cuda.max_memory_allocated()/2**30,
                            peak_reserved_gib=torch.cuda.max_memory_reserved()/2**30,
                            resident_allocated_gib=torch.cuda.memory_allocated()/2**30)
                image_path=directory/f'pair_{index:02d}_{variant}.png'
                image.save(image_path)
                record['image']=dict(name=image_path.name,sha256=digest(image_path),bytes=image_path.stat().st_size)
                del image
                report['rows'].append(record)
                report['calls'][variant]+=1
                (directory/'progress.json').write_text(json.dumps(report,indent=2))
                results_volume.commit()
                print(f'Pair {index+1}/30 {variant}: {seconds:.3f}s, {record["peak_allocated_gib"]:.3f} GiB',flush=True)
        pre.remove()
        post.remove()
        ratios=[]
        for index in range(30):
            pair={r['variant']:r for r in report['rows'] if r['pair']==index}
            ratios.append(pair['teacher']['seconds']/pair['candidate_A']['seconds'])
        bootstrap_rng=random.Random(20261017)
        boot=sorted(statistics.median(bootstrap_rng.choices(ratios,k=len(ratios))) for _ in range(20000))
        report['paired_speedup']=dict(median=statistics.median(ratios),bootstrap_95pct=[boot[499],boot[19499]],
                                      replicates=20000,ratios=ratios)
        report['summary']={}
        for variant in variants:
            rows=[r for r in report['rows'] if r['variant']==variant]
            values=sorted(r['seconds'] for r in rows)
            median=statistics.median(values)
            report['summary'][variant]=dict(pairs=len(rows),median_seconds=median,p90_seconds=values[26]+.1*(values[27]-values[26]),
                throughput_images_per_second=1/median,median_denoiser_seconds=statistics.median(r['denoiser_seconds'] for r in rows),
                peak_allocated_gib=max(r['peak_allocated_gib'] for r in rows),peak_reserved_gib=max(r['peak_reserved_gib'] for r in rows))
        report['status']='passed'
        report['scope']='Matched 40-step teacher vs provisional candidate A, one H100, 30 balanced randomized pairs; no quality promotion.'
    except Exception as exc:
        import traceback
        report['status']='failed'
        report['error']=repr(exc)
        report['traceback']=traceback.format_exc()
    (directory/'summary.json').write_text(json.dumps(report,indent=2))
    results_volume.commit()
    return report
'''


"""Launch the matched randomized latency/VRAM benchmark; reuse completed remote results."""
import argparse
import hashlib
import json
import os
import sys
from pathlib import Path


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--old-account', action='store_true')
    args = parser.parse_args(LAUNCH_ARGS)
    if args.old_account:
        parser.error('Old account is download-only; GPU execution must use the new account.')
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    root = ROOT
    recovered = root / 'data/recovered/modal'
    images_key = 'dda8c184e657dfb604c35fb261d83371c108c39fac033a0f57db66f7686c30ad'
    request = json.loads((recovered / 'phase7/candidate_images' / images_key / 'request.json').read_bytes())['request']
    source = SOURCES['latency_benchmark_worker.py']
    request.update(version='phase7-latency-benchmark-v1', worker_source=source, execution_account='new',
                   benchmark_protocol=dict(pairs=30, order_seed=20261017, warmups_per_restore=1,
                     precision='BF16 with original FP32 bridge masters cast per forward',
                     attention='SDPBackend.MATH', steps=40, batch_size=1, kv_cache=True,
                     timing='CUDA-synchronized pipeline wall time, excluding restoration, warmup and image serialization; CUDA events for denoiser',
                     memory='one active variant; unused teacher blocks moved to CPU; reset peaks after warmup; no offload of active pipeline'))
    if len(request['rows']) != 30:
        raise RuntimeError('Benchmark requires 30 saved prompt/seed pairs')
    from experiment_runtime import pin_reference_runtime
    pin_reference_runtime(request,recovered)
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase7/latency_benchmark' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2)+'\n',encoding='utf-8')
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase7/latency_benchmark/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        try:
            b''.join(results_volume.read_file(remote + '/started.json'))
        except FileNotFoundError:
            pass
        else:
            raise RuntimeError('Run already started; inspect its progress. No duplicate GPU scheduled.')
        app = modal.App('nihonga-phase7-latency-benchmark')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(source, 'latency_benchmark_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8,
                              memory=98304, max_containers=1, retries=0,
                              volumes={'/phase3':results_volume,
                                       '/root/.cache/huggingface':weights.with_mount_options(read_only=True)})(namespace['benchmark_candidate'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'),indent=2)+'\n')
            print('Request:',request_key,'Call:',call.object_id,flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2)+'\n',encoding='utf-8')
    print(json.dumps({k:report.get(k) for k in ('status','error','paired_speedup','summary')}),flush=True)
    if report.get('status') != 'passed':
        raise SystemExit(1)

summary = reuse_or_launch(RUN, main)

print(json.dumps(summary['summary'], indent=1)); print({k: v for k, v in summary['paired_speedup'].items() if k != 'ratios'})

### What the results mean

Median 21.91 s (teacher) vs 20.58 s (A): paired speedup **1.064x** (95% CI 1.063-1.065); peak memory 36.93 vs 36.14 GiB. Removing 2 of 32 blocks buys about 6% speed.

## 4. Candidate A, block importance scan

Each trial bypasses one surviving block of A, compares with the teacher on the 180 comparisons, then restores it. No weight is updated and bypasses never accumulate. All 90 intact-teacher controls must reproduce exactly and A must reproduce its saved predictions before any trial.

In [ ]:
# Candidate A, block importance scan: reuses the saved run; launches GPU work only when its result is missing and ALLOW_GPU is True.

RUN = 'data/phase7/conditional_importance/7454da6a5c1dc33af23dd46833e698f9b759d14985ca96340bc3d40cd47fb4f5'

LAUNCH_ARGS = []

SOURCES['conditional_scan_worker.py'] = r'''def scan_candidate(request, key):
    """Independently bypass surviving blocks inside frozen candidate A.

    Reuses the saved teacher replay contract before assembling the 30-block
    student. No training or architecture promotion is performed.
    """
    import hashlib
    import importlib.metadata
    import inspect
    import json
    import os
    import socket
    import uuid
    from pathlib import Path
    import torch
    from torch.nn.attention import sdpa_kernel, SDPBackend
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache

    directory = Path('/phase3/phase7/conditional_importance') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete healing attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()
    report = {'request_key': key, 'status': 'failed', 'mode': request['mode'], 'teacher_calls': 0,
              'student_calls': 0, 'optimizer_updates': 0, 'controls': [], 'artifacts': [],
              'process': {'pid': os.getpid(), 'host': socket.gethostname(), 'nonce': uuid.uuid4().hex,
                          'modal_task_id': os.environ.get('MODAL_TASK_ID')}}

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def gpu(value):
        if isinstance(value, torch.Tensor):
            return value.to('cuda')
        if isinstance(value, dict):
            return {k: gpu(v) for k, v in value.items()}
        if isinstance(value, (list, tuple)):
            return type(value)(gpu(v) for v in value)
        return value

    def compare(x, y):
        a, b = x.float(), y.float()
        if not torch.isfinite(a).all() or not torch.isfinite(b).all():
            raise RuntimeError('Nonfinite reference tensor.')
        d = a - b
        return {'relative_l2_error': float(d.norm()/b.norm().clamp_min(1e-12)),
                'max_abs_error': float(d.abs().max()), 'exact_equal': bool(torch.equal(x, y))}

    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('GPU differs.')
        report['packages'] = {p: importlib.metadata.version(p) for p in ('torch', 'diffusers', 'transformers', 'accelerate', 'pillow')}
        report['torch_cuda'] = torch.version.cuda
        report['cudnn_version'] = torch.backends.cudnn.version()
        report['settings'] = request['arithmetic_contract']
        if request['mode'] != 'verify':
            raise RuntimeError('Healing requires the verified teacher reference.')
        reference_dir = Path('/phase3/phase6/deterministic_reference') / request['reference_key']
        reference_report_path = reference_dir / 'summary.json'
        if hashlib.sha256(json.dumps(json.loads(reference_report_path.read_text()), sort_keys=True).encode()).hexdigest() != request['reference_report_semantic_sha256']:
            raise RuntimeError('Generation report differs.')
        reference_report = json.loads(reference_report_path.read_text())
        if reference_report['status'] != 'reference_generated':
            raise RuntimeError('Reference generation did not complete.')
        if reference_report['settings'] != request['arithmetic_contract']:
            raise RuntimeError('Arithmetic contract differs from generation.')
        report['reference_process'] = reference_report['process']
        if reference_report['process']['nonce'] == report['process']['nonce']:
            raise RuntimeError('Independent process requirement failed.')
        if reference_report['packages'] != report['packages'] or reference_report['torch_cuda'] != report['torch_cuda'] or reference_report['cudnn_version'] != report['cudnn_version']:
            raise RuntimeError('Numerical runtime differs across generation and verification.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
            dtype=torch.bfloat16, local_files_only=True).to('cuda')
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Model source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--'))
                       / 'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        state = {}
        for kind, values in (('parameter', model.named_parameters()), ('buffer', model.named_buffers())):
            for name, value in values:
                cpu = value.detach().cpu().contiguous()
                state[name] = {'kind': kind, 'dtype': str(value.dtype), 'shape': list(value.shape),
                    'stride': list(value.stride()), 'sha256': hashlib.sha256(memoryview(cpu.view(torch.uint8).numpy())).hexdigest()}
        for axis, value in enumerate(model.pos_embed.freqs):
            name = f'pos_embed.freqs.{axis}'
            cpu = value.detach().cpu().contiguous()
            state[name] = {'kind': 'unregistered_rope_table', 'dtype': str(value.dtype), 'shape': list(value.shape),
                'stride': list(value.stride()), 'sha256': hashlib.sha256(memoryview(cpu.view(torch.uint8).numpy())).hexdigest()}
        expected = request['expected_state']
        derived_names = sorted(n for n in expected if n.startswith('pos_embed.freqs.') or n == 'time_text_embed.time_proj.freqs')

        def derived_tensors():
            values = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
            values['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
            return values

        def table_hash(value):
            return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

        canonical_path = Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt')
        import time
        mismatched = [n for n in derived_names if state.get(n) != expected.get(n)]
        if not mismatched:
            if not canonical_path.exists():
                canonical_path.parent.mkdir(parents=True, exist_ok=True)
                temporary = canonical_path.with_suffix('.' + report['process']['nonce'] + '.tmp')
                torch.save({n: v.detach().cpu().contiguous() for n, v in derived_tensors().items()}, temporary)
                temporary.replace(canonical_path)
                results_volume.commit()
                report['canonical_tables_saved'] = True
        else:
            deadline = time.time() + 900
            while not canonical_path.exists():
                if time.time() > deadline:
                    raise RuntimeError('Derived tables differ on this host and no canonical copy became available.')
                time.sleep(20)
                results_volume.reload()
            saved = torch.load(canonical_path, weights_only=True, map_location='cpu')
            if sorted(saved) != derived_names or any(table_hash(saved[n]) != expected[n]['sha256'] for n in derived_names):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [saved[f'pos_embed.freqs.{axis}'].to(device=value.device, dtype=value.dtype)
                        for axis, value in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(saved['time_text_embed.time_proj.freqs'])
            for n, value in derived_tensors().items():
                state[n] = {**state[n], 'sha256': table_hash(value)}
            report['canonical_tables_applied'] = mismatched
        if state != request['expected_state']:
            expected = request['expected_state']
            different_names = sorted(name for name in expected.keys() & state.keys()
                                     if state[name] != expected[name])
            report['state_mismatch'] = {
                'actual_count': len(state), 'expected_count': len(expected),
                'missing_names': sorted(set(expected) - set(state))[:12],
                'extra_names': sorted(set(state) - set(expected))[:12],
                'different': [{'name': name, 'actual': state[name], 'expected': expected[name]}
                              for name in different_names][:12]}
            allowed_derived_tables = {'pos_embed.freqs.1', 'pos_embed.freqs.2', 'time_text_embed.time_proj.freqs'}
            if set(state) != set(expected) or not set(different_names).issubset(allowed_derived_tables):
                raise RuntimeError('Loaded teacher learned state differs from audited reference.')
            report['derived_table_variation_requires_output_replay'] = different_names
        report['state_sha256'] = hashlib.sha256(json.dumps(state, sort_keys=True).encode()).hexdigest()
        torch.cuda.synchronize()
        argument_order = ('hidden_states', 'timestep', 'encoder_hidden_states', 'encoder_hidden_states_mask',
                          'img_shapes', 'img_mask', 'attention_kwargs', 'kv_cache_mode', 'return_dict')
        # Every intact-teacher control must pass before any block is touched.
        for record in request['records']:
            path = Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt'
            if digest(path) != record['tensor_sha256']:
                raise RuntimeError('Legacy input bundle differs.')
            bundle = torch.load(path, weights_only=True, map_location='cpu')
            outputs, controls = [], []
            cache = QwenImage21KVCache(32)
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for stage in bundle['stages']:
                    full = {**bundle['shared_transformer_kwargs'], **stage['kwargs']}
                    kwargs = gpu({field: full[field] for field in argument_order})
                    kwargs['kv_cache'] = cache
                    pred = model(**kwargs)[0][:, -stage['target_image_tokens']:].detach().cpu().contiguous()
                    report['teacher_calls'] += 1
                    legacy = compare(pred, stage['teacher_velocity'])
                    entry = {'source_id': record['source_id'], 'split': bundle['row']['split'],
                             'stage': stage['stage'], 'legacy_difference': legacy}
                    outputs.append({'stage': stage['stage'], 'step_index': stage['step_index'],
                                    'scheduler_sigma': stage['scheduler_sigma'], 'velocity': pred})
                    controls.append(entry)
            filename = record['capture_id'] + '.pt'
            artifact = next(a for a in reference_report['artifacts'] if a['name'] == filename)
            if digest(reference_dir / filename) != artifact['sha256']:
                raise RuntimeError('Reference output checksum differs.')
            reference_bundle = torch.load(reference_dir / filename, weights_only=True, map_location='cpu')
            if reference_bundle['source_id'] != record['source_id'] or reference_bundle['input_sha256'] != record['tensor_sha256']:
                raise RuntimeError('Reference input provenance differs.')
            reference_outputs = {v['stage']: v['velocity'] for v in reference_bundle['outputs']}
            for entry, measured in zip(controls, outputs):
                entry['reference_difference'] = compare(measured['velocity'], reference_outputs[entry['stage']])
                entry['passed'] = entry['reference_difference']['relative_l2_error'] <= 0.001
            output_path = directory / filename
            temporary = output_path.with_suffix('.tmp')
            torch.save({'request_key': key, 'source_id': record['source_id'], 'input_sha256': record['tensor_sha256'],
                        'outputs': outputs, 'controls': controls}, temporary)
            temporary.replace(output_path)
            report['controls'].extend(controls)
            report['artifacts'].append({'name': filename, 'sha256': digest(output_path), 'bytes': output_path.stat().st_size})
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            if not all(c['passed'] for c in controls):
                raise RuntimeError('Independent-process reference replay exceeds tolerance.')
        if report['teacher_calls'] != 90:
            raise RuntimeError('Reference attempt did not cover all 90 saved stage inputs.')
        report['legacy_max_relative_l2'] = max(c['legacy_difference']['relative_l2_error'] for c in report['controls'])
        report['max_reference_relative_l2'] = max(c['reference_difference']['relative_l2_error'] for c in report['controls'])
        report['exact_equal_controls'] = sum(c['reference_difference']['exact_equal'] for c in report['controls'])
        if not all(c['passed'] for c in report['controls']):
            raise RuntimeError('Teacher controls did not pass.')
        report['status'] = 'teacher_verified'
        (directory / 'teacher_control_report.json').write_text(json.dumps(report, indent=2))
        results_volume.commit()
        print('All 90 intact-teacher controls passed; assembling frozen 30-block candidate A.', flush=True)

        import gc
        import time
        import statistics
        from types import SimpleNamespace
        from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVLayerCache

        # Inputs are independently identified; 10 gate and 50 wider development prompts.
        panels = {'gate10': [], 'wider50': []}
        for panel, records in (('gate10', [r for r in request['records'] if r['capture_id'].startswith('p5-validation-')]),
                               ('wider50', request['expanded_validation']['records'])):
            for record in records:
                path = Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt'
                if digest(path) != record['tensor_sha256']:
                    raise RuntimeError('Fixed input hash differs')
                item = torch.load(path, weights_only=True, map_location='cpu')
                if item['row']['source_id'] != record['source_id'] or item['row']['split'] != 'validation':
                    raise RuntimeError('Validation input identity differs')
                panels[panel].append(item)
        if [len(panels[p]) for p in ('gate10', 'wider50')] != [10, 50]:
            raise RuntimeError('Saved validation panel sizes differ')

        def forward(item, stage, cache):
            full = {**item['shared_transformer_kwargs'], **stage['kwargs']}
            kwargs = gpu({field: full[field] for field in argument_order})
            kwargs['kv_cache'] = cache
            return model(**kwargs)[0][:, -stage['target_image_tokens']:].detach().cpu().contiguous()

        def predictions():
            rows = []
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for panel, items in panels.items():
                    for item in items:
                        cache = QwenImage21KVCache(32)
                        for stage in item['stages']:
                            pred = forward(item, stage, cache)
                            if not torch.isfinite(pred).all() or pred.float().norm() == 0:
                                raise RuntimeError('Invalid reference velocity')
                            rows.append(dict(panel=panel, source_id=item['row']['source_id'],
                                             dimension=item['row']['dimension'],
                                             subdimension=item['row'].get('subdimension'),
                                             stage=stage['stage'], velocity=pred))
            return rows

        def save(name, value):
            path = directory / name
            temporary = path.with_suffix('.tmp')
            torch.save(value, temporary)
            temporary.replace(path)
            artifact = dict(name=name, sha256=digest(path), bytes=path.stat().st_size)
            report['artifacts'].append(artifact)
            return artifact

        teacher_predictions = predictions()
        save('teacher_predictions.pt', teacher_predictions)
        report['teacher_calls'] += len(teacher_predictions)
        # Reuse the exact assembly and FP32-master loading contract from the saved image worker.
        namespace = {}
        for field in ('core_source', 'wrapper_source', 'components_source'):
            exec(compile(request[field], field + '.py', 'exec'), namespace)
        student = request['image_request']['students']['blocks45']
        region = student['region_bridge']
        region_path = Path('/phase3/phase7/region_bridge_fit') / region['fit_key'] / region['checkpoint']
        if digest(region_path) != region['sha256']:
            raise RuntimeError('Region bridge changed')
        value = torch.load(region_path, weights_only=True, map_location='cpu')
        bridge = namespace['MasterBridge'](4096, region['rank'])
        bridge.load_state_dict(value['state_dict'])
        removed, _ = namespace['insert_bridge'](model, 4, bridge, QwenImage21KVCache)
        bridge.float()
        bridge.load_state_dict(value['state_dict'])
        del removed
        passthrough = namespace['ResidualBottleneckBridge'](4096, 256)
        with torch.no_grad():
            passthrough.up.weight.zero_()
        removed, _ = namespace['insert_bridge'](model, 5, passthrough, QwenImage21KVCache)
        del removed, value
        for slot, block in enumerate(model.transformer_blocks):
            if slot not in (4, 5):
                model.transformer_blocks[slot] = namespace['FunctionalCheckpointBlock'](block, QwenImage21KVLayerCache)
        checkpoint = request['image_request']['checkpoints']['A_healed_u60']
        path = Path(checkpoint['remote_path'])
        if path.stat().st_size != checkpoint['bytes'] or digest(path) != checkpoint['sha256']:
            raise RuntimeError('Candidate A checkpoint checksum differs')
        state = torch.load(path, weights_only=True, map_location='cpu')
        frozen = {f'transformer_blocks.5.bridge.{name}.weight' for name in ('down', 'up')}
        parameters = {n: p for n, p in model.named_parameters() if n not in frozen}
        if (state['request_key'] != checkpoint['request_key'] or state['step'] != 60 or
                set(parameters) != set(state['master_state'])):
            raise RuntimeError('Candidate A checkpoint structure differs')
        with torch.no_grad():
            for name, parameter in parameters.items():
                parameter.copy_(state['master_state'][name].to(device=parameter.device, dtype=parameter.dtype))
        del state, parameters
        model.requires_grad_(False).eval()
        model.gradient_checkpointing = False
        gc.collect()
        torch.cuda.empty_cache()
        baseline = predictions()
        report['student_calls'] += len(baseline)
        save('candidate_A_predictions.pt', baseline)
        historical_path = path.parent / 'validation_step_060.pt'
        if digest(historical_path) != request['historical_validation_sha256']:
            raise RuntimeError('Historical validation hash differs')
        historical = torch.load(historical_path, weights_only=True, map_location='cpu')
        reference = {(panel, r['source_id'], r['stage']): r['velocity']
                     for panel, field in (('gate10', 'gate_outputs'), ('wider50', 'monitor_outputs'))
                     for r in historical[field]}
        reproduction = [compare(r['velocity'], reference[(r['panel'], r['source_id'], r['stage'])])
                        for r in baseline]
        report['candidate_A_replay_max_relative_l2'] = max(r['relative_l2_error'] for r in reproduction)
        if report['candidate_A_replay_max_relative_l2'] > 0.001:
            raise RuntimeError('Candidate A failed historical output replay')
        del historical, reference

        class IdentityBlock(torch.nn.Module):
            def __init__(self, original):
                super().__init__()
                self.attn = SimpleNamespace(processor=original.attn.processor)

            def forward(self, hidden_states, *args, **kwargs):
                return hidden_states

        def metrics(rows):
            output = []
            for measured, base, teacher_row in zip(rows, baseline, teacher_predictions):
                identity = ('panel', 'source_id', 'stage')
                if any(measured[k] != base[k] or base[k] != teacher_row[k] for k in identity):
                    raise RuntimeError('Prediction alignment differs')
                teacher_error = compare(measured['velocity'], teacher_row['velocity'])['relative_l2_error']
                baseline_error = compare(base['velocity'], teacher_row['velocity'])['relative_l2_error']
                output.append({k: measured[k] for k in measured if k != 'velocity'} |
                              dict(baseline_displacement=compare(measured['velocity'], base['velocity'])['relative_l2_error'],
                                   teacher_relative_l2=teacher_error, baseline_teacher_relative_l2=baseline_error,
                                   teacher_error_change_percentage_points=100 * (teacher_error - baseline_error)))
            return output

        def aggregate(rows):
            fields = ('baseline_displacement', 'teacher_relative_l2', 'baseline_teacher_relative_l2',
                      'teacher_error_change_percentage_points')
            result = dict(comparisons=len(rows))
            for field in fields:
                values = sorted(r[field] for r in rows)
                def percentile(fraction):
                    position = fraction * (len(values) - 1)
                    low = int(position)
                    return values[low] + (values[min(low + 1, len(values) - 1)] - values[low]) * (position - low)
                result[field] = dict(mean=statistics.mean(values), maximum=max(values),
                                     p10=percentile(.1), p50=percentile(.5), p90=percentile(.9))
            result['worst_regressions'] = sorted(rows, key=lambda r:r['teacher_error_change_percentage_points'], reverse=True)[:10]
            return result

        report['baseline'] = aggregate(metrics(baseline))
        report['blocks'] = []
        weights = {n: (id(p), p._version) for n, p in model.named_parameters()}
        original_blocks = list(model.transformer_blocks)
        for slot in request['eligible_original_block_ids']:
            if list(model.transformer_blocks) != original_blocks:
                raise RuntimeError('Baseline A architecture not restored')
            if any((id(p), p._version) != weights[n] for n, p in model.named_parameters()):
                raise RuntimeError('Frozen candidate A weights changed')
            started = time.perf_counter()
            model.transformer_blocks[slot] = IdentityBlock(original_blocks[slot])
            try:
                rows = predictions()
            finally:
                model.transformer_blocks[slot] = original_blocks[slot]
            report['student_calls'] += len(rows)
            records = metrics(rows)
            artifact = save(f'block_{slot:02d}_predictions.pt', rows)
            result = dict(original_block_id=slot, overall=aggregate(records),
                          panels={panel:aggregate([r for r in records if r['panel']==panel]) for panel in panels},
                          stages={stage:aggregate([r for r in records if r['stage']==stage]) for stage in ('early','middle','late')},
                          prompt_groups={group:aggregate([r for r in records if r['dimension']==group])
                                         for group in sorted({r['dimension'] for r in records})},
                          records=records, artifact=artifact, seconds=time.perf_counter()-started)
            report['blocks'].append(result)
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            print(f'Completed independent block {slot}: {result["overall"]["baseline_displacement"]["mean"]:.6f}', flush=True)
        report['ranking'] = [r['original_block_id'] for r in sorted(report['blocks'],
                              key=lambda r:r['overall']['baseline_displacement']['mean'])]
        report['status'] = 'passed'
        report['optimizer_updates'] = 0
        report['scope'] = 'Conditional independent bypass screen only; no block accepted, no quality promotion.'
    except Exception as exc:
        import traceback
        report['status'] = 'failed'
        report['error'] = repr(exc)
        report['traceback'] = traceback.format_exc()
    (directory / 'summary.json').write_text(json.dumps(report, indent=2))
    results_volume.commit()
    return report
'''


"""Launch the hash-pinned conditional scan; reuse completed remote results."""
import argparse
import hashlib
import json
import os
import sys
from pathlib import Path


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--old-account', action='store_true')
    args = parser.parse_args(LAUNCH_ARGS)
    if args.old_account:
        parser.error('Old account is download-only; GPU execution must use the new account.')
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    root = ROOT
    recovered = root / 'data/recovered/modal'
    healing_key = '95fcd2cfae35cb7f124b8af2865c30f928cbd681a0d015df5e56e8ae5ba4fd78'
    images_key = 'dda8c184e657dfb604c35fb261d83371c108c39fac033a0f57db66f7686c30ad'
    healing = recovered / 'phase7/blocks45_healing' / healing_key
    request = json.loads((healing / 'request.json').read_bytes())['request']
    summary = json.loads((healing / 'summary.json').read_bytes())
    request['image_request'] = json.loads((recovered / 'phase7/candidate_images' / images_key / 'request.json').read_bytes())['request']
    request['historical_validation_sha256'] = next(a['sha256'] for a in summary['artifacts'] if a['name']=='validation_step_060.pt')
    source = SOURCES['conditional_scan_worker.py']
    request.update(version='phase7-conditional-importance-v1', worker_source=source, execution_account='new',
                   eligible_original_block_ids=[i for i in range(32) if i not in (4, 5)],
                   selection='Screening only: independent frozen bypasses; no pruning or quality promotion',
                   scope='Fixed gate10 and wider50 development prompts, three saved stages; compare A and teacher')
    from experiment_runtime import pin_reference_runtime
    pin_reference_runtime(request,recovered)
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase7/conditional_importance' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2)+'\n',encoding='utf-8')
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase7/conditional_importance/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        try:
            b''.join(results_volume.read_file(remote + '/started.json'))
        except FileNotFoundError:
            pass
        else:
            raise RuntimeError('Run already started; inspect its progress. No duplicate GPU scheduled.')
        app = modal.App('nihonga-phase7-conditional-importance')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(source, 'conditional_scan_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8,
                              memory=65536, max_containers=1, retries=0,
                              volumes={'/phase3':results_volume,
                                       '/root/.cache/huggingface':weights.with_mount_options(read_only=True)})(namespace['scan_candidate'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'),indent=2)+'\n')
            print('Request:',request_key,'Call:',call.object_id,flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2)+'\n',encoding='utf-8')
    print(json.dumps({k:report.get(k) for k in ('status','error','ranking','candidate_A_replay_max_relative_l2')}),flush=True)
    if report.get('status') != 'passed':
        raise SystemExit(1)

summary = reuse_or_launch(RUN, main)

rows = sorted((b['overall']['teacher_relative_l2']['mean'], b['original_block_id']) for b in summary['blocks'])
print('A alone:', pct(summary['baseline']['teacher_relative_l2']['mean']))
print([(block, pct(mean)) for mean, block in rows])

### What the results mean

A alone measures 7.49%. No single bypass stays within the gate: the gentlest, block 12, moves A to 9.19%. The middle blocks (6-17) cost least; blocks 0 and 31 cannot be skipped. Because A is rejected (step 2), this ranking describes a model we will not build on, so the scan is repeated inside B.

## 5. Block importance inside the accepted student (B)

The same scan inside B (31 blocks, pretrained block-5 bridge). B is rebuilt exactly as Phase 6 did, and its 10 check prompts must reproduce the saved Phase 6 velocities before any trial. Block 4 is eligible again.

In [ ]:
# Block importance inside the accepted student (B): reuses the saved run; launches GPU work only when its result is missing and ALLOW_GPU is True.

RUN = 'data/phase7/conditional_importance_b/b0a27f0cbe0f8e7f9938044fe088576a648c7b22e63a2ddbd771ee6b8e777aae'

LAUNCH_ARGS = []

SOURCES['conditional_scan_worker_b.py'] = r'''def scan_candidate(request, key):
    """Independently bypass surviving blocks inside the frozen block-5 student (candidate B).

    Reuses the saved teacher replay contract before assembling the selected
    Phase 6 student: 31 blocks plus the pretrained block-5 bridge. No training
    or architecture promotion is performed.
    """
    import hashlib
    import importlib.metadata
    import inspect
    import json
    import os
    import socket
    import uuid
    from pathlib import Path
    import torch
    from torch.nn.attention import sdpa_kernel, SDPBackend
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache

    directory = Path('/phase3/phase7/conditional_importance') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete healing attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()
    report = {'request_key': key, 'status': 'failed', 'mode': request['mode'], 'teacher_calls': 0,
              'student_calls': 0, 'optimizer_updates': 0, 'controls': [], 'artifacts': [],
              'process': {'pid': os.getpid(), 'host': socket.gethostname(), 'nonce': uuid.uuid4().hex,
                          'modal_task_id': os.environ.get('MODAL_TASK_ID')}}

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def gpu(value):
        if isinstance(value, torch.Tensor):
            return value.to('cuda')
        if isinstance(value, dict):
            return {k: gpu(v) for k, v in value.items()}
        if isinstance(value, (list, tuple)):
            return type(value)(gpu(v) for v in value)
        return value

    def compare(x, y):
        a, b = x.float(), y.float()
        if not torch.isfinite(a).all() or not torch.isfinite(b).all():
            raise RuntimeError('Nonfinite reference tensor.')
        d = a - b
        return {'relative_l2_error': float(d.norm()/b.norm().clamp_min(1e-12)),
                'max_abs_error': float(d.abs().max()), 'exact_equal': bool(torch.equal(x, y))}

    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('GPU differs.')
        report['packages'] = {p: importlib.metadata.version(p) for p in ('torch', 'diffusers', 'transformers', 'accelerate', 'pillow')}
        report['torch_cuda'] = torch.version.cuda
        report['cudnn_version'] = torch.backends.cudnn.version()
        report['settings'] = request['arithmetic_contract']
        if request['mode'] != 'verify':
            raise RuntimeError('Healing requires the verified teacher reference.')
        reference_dir = Path('/phase3/phase6/deterministic_reference') / request['reference_key']
        reference_report_path = reference_dir / 'summary.json'
        if hashlib.sha256(json.dumps(json.loads(reference_report_path.read_text()), sort_keys=True).encode()).hexdigest() != request['reference_report_semantic_sha256']:
            raise RuntimeError('Generation report differs.')
        reference_report = json.loads(reference_report_path.read_text())
        if reference_report['status'] != 'reference_generated':
            raise RuntimeError('Reference generation did not complete.')
        if reference_report['settings'] != request['arithmetic_contract']:
            raise RuntimeError('Arithmetic contract differs from generation.')
        report['reference_process'] = reference_report['process']
        if reference_report['process']['nonce'] == report['process']['nonce']:
            raise RuntimeError('Independent process requirement failed.')
        if reference_report['packages'] != report['packages'] or reference_report['torch_cuda'] != report['torch_cuda'] or reference_report['cudnn_version'] != report['cudnn_version']:
            raise RuntimeError('Numerical runtime differs across generation and verification.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
            dtype=torch.bfloat16, local_files_only=True).to('cuda')
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Model source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--'))
                       / 'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        state = {}
        for kind, values in (('parameter', model.named_parameters()), ('buffer', model.named_buffers())):
            for name, value in values:
                cpu = value.detach().cpu().contiguous()
                state[name] = {'kind': kind, 'dtype': str(value.dtype), 'shape': list(value.shape),
                    'stride': list(value.stride()), 'sha256': hashlib.sha256(memoryview(cpu.view(torch.uint8).numpy())).hexdigest()}
        for axis, value in enumerate(model.pos_embed.freqs):
            name = f'pos_embed.freqs.{axis}'
            cpu = value.detach().cpu().contiguous()
            state[name] = {'kind': 'unregistered_rope_table', 'dtype': str(value.dtype), 'shape': list(value.shape),
                'stride': list(value.stride()), 'sha256': hashlib.sha256(memoryview(cpu.view(torch.uint8).numpy())).hexdigest()}
        expected = request['expected_state']
        derived_names = sorted(n for n in expected if n.startswith('pos_embed.freqs.') or n == 'time_text_embed.time_proj.freqs')

        def derived_tensors():
            values = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
            values['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
            return values

        def table_hash(value):
            return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

        canonical_path = Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt')
        import time
        mismatched = [n for n in derived_names if state.get(n) != expected.get(n)]
        if not mismatched:
            if not canonical_path.exists():
                canonical_path.parent.mkdir(parents=True, exist_ok=True)
                temporary = canonical_path.with_suffix('.' + report['process']['nonce'] + '.tmp')
                torch.save({n: v.detach().cpu().contiguous() for n, v in derived_tensors().items()}, temporary)
                temporary.replace(canonical_path)
                results_volume.commit()
                report['canonical_tables_saved'] = True
        else:
            deadline = time.time() + 900
            while not canonical_path.exists():
                if time.time() > deadline:
                    raise RuntimeError('Derived tables differ on this host and no canonical copy became available.')
                time.sleep(20)
                results_volume.reload()
            saved = torch.load(canonical_path, weights_only=True, map_location='cpu')
            if sorted(saved) != derived_names or any(table_hash(saved[n]) != expected[n]['sha256'] for n in derived_names):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [saved[f'pos_embed.freqs.{axis}'].to(device=value.device, dtype=value.dtype)
                        for axis, value in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(saved['time_text_embed.time_proj.freqs'])
            for n, value in derived_tensors().items():
                state[n] = {**state[n], 'sha256': table_hash(value)}
            report['canonical_tables_applied'] = mismatched
        if state != request['expected_state']:
            expected = request['expected_state']
            different_names = sorted(name for name in expected.keys() & state.keys()
                                     if state[name] != expected[name])
            report['state_mismatch'] = {
                'actual_count': len(state), 'expected_count': len(expected),
                'missing_names': sorted(set(expected) - set(state))[:12],
                'extra_names': sorted(set(state) - set(expected))[:12],
                'different': [{'name': name, 'actual': state[name], 'expected': expected[name]}
                              for name in different_names][:12]}
            allowed_derived_tables = {'pos_embed.freqs.1', 'pos_embed.freqs.2', 'time_text_embed.time_proj.freqs'}
            if set(state) != set(expected) or not set(different_names).issubset(allowed_derived_tables):
                raise RuntimeError('Loaded teacher learned state differs from audited reference.')
            report['derived_table_variation_requires_output_replay'] = different_names
        report['state_sha256'] = hashlib.sha256(json.dumps(state, sort_keys=True).encode()).hexdigest()
        torch.cuda.synchronize()
        argument_order = ('hidden_states', 'timestep', 'encoder_hidden_states', 'encoder_hidden_states_mask',
                          'img_shapes', 'img_mask', 'attention_kwargs', 'kv_cache_mode', 'return_dict')
        # Every intact-teacher control must pass before any block is touched.
        for record in request['records']:
            path = Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt'
            if digest(path) != record['tensor_sha256']:
                raise RuntimeError('Legacy input bundle differs.')
            bundle = torch.load(path, weights_only=True, map_location='cpu')
            outputs, controls = [], []
            cache = QwenImage21KVCache(32)
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for stage in bundle['stages']:
                    full = {**bundle['shared_transformer_kwargs'], **stage['kwargs']}
                    kwargs = gpu({field: full[field] for field in argument_order})
                    kwargs['kv_cache'] = cache
                    pred = model(**kwargs)[0][:, -stage['target_image_tokens']:].detach().cpu().contiguous()
                    report['teacher_calls'] += 1
                    legacy = compare(pred, stage['teacher_velocity'])
                    entry = {'source_id': record['source_id'], 'split': bundle['row']['split'],
                             'stage': stage['stage'], 'legacy_difference': legacy}
                    outputs.append({'stage': stage['stage'], 'step_index': stage['step_index'],
                                    'scheduler_sigma': stage['scheduler_sigma'], 'velocity': pred})
                    controls.append(entry)
            filename = record['capture_id'] + '.pt'
            artifact = next(a for a in reference_report['artifacts'] if a['name'] == filename)
            if digest(reference_dir / filename) != artifact['sha256']:
                raise RuntimeError('Reference output checksum differs.')
            reference_bundle = torch.load(reference_dir / filename, weights_only=True, map_location='cpu')
            if reference_bundle['source_id'] != record['source_id'] or reference_bundle['input_sha256'] != record['tensor_sha256']:
                raise RuntimeError('Reference input provenance differs.')
            reference_outputs = {v['stage']: v['velocity'] for v in reference_bundle['outputs']}
            for entry, measured in zip(controls, outputs):
                entry['reference_difference'] = compare(measured['velocity'], reference_outputs[entry['stage']])
                entry['passed'] = entry['reference_difference']['relative_l2_error'] <= 0.001
            output_path = directory / filename
            temporary = output_path.with_suffix('.tmp')
            torch.save({'request_key': key, 'source_id': record['source_id'], 'input_sha256': record['tensor_sha256'],
                        'outputs': outputs, 'controls': controls}, temporary)
            temporary.replace(output_path)
            report['controls'].extend(controls)
            report['artifacts'].append({'name': filename, 'sha256': digest(output_path), 'bytes': output_path.stat().st_size})
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            if not all(c['passed'] for c in controls):
                raise RuntimeError('Independent-process reference replay exceeds tolerance.')
        if report['teacher_calls'] != 90:
            raise RuntimeError('Reference attempt did not cover all 90 saved stage inputs.')
        report['legacy_max_relative_l2'] = max(c['legacy_difference']['relative_l2_error'] for c in report['controls'])
        report['max_reference_relative_l2'] = max(c['reference_difference']['relative_l2_error'] for c in report['controls'])
        report['exact_equal_controls'] = sum(c['reference_difference']['exact_equal'] for c in report['controls'])
        if not all(c['passed'] for c in report['controls']):
            raise RuntimeError('Teacher controls did not pass.')
        report['status'] = 'teacher_verified'
        (directory / 'teacher_control_report.json').write_text(json.dumps(report, indent=2))
        results_volume.commit()
        print('All 90 intact-teacher controls passed; assembling frozen 31-block candidate B.', flush=True)

        import gc
        import time
        import statistics
        from types import SimpleNamespace
        
        # Inputs are independently identified; 10 gate and 50 wider development prompts.
        panels = {'gate10': [], 'wider50': []}
        for panel, records in (('gate10', [r for r in request['records'] if r['capture_id'].startswith('p5-validation-')]),
                               ('wider50', request['expanded_validation']['records'])):
            for record in records:
                path = Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt'
                if digest(path) != record['tensor_sha256']:
                    raise RuntimeError('Fixed input hash differs')
                item = torch.load(path, weights_only=True, map_location='cpu')
                if item['row']['source_id'] != record['source_id'] or item['row']['split'] != 'validation':
                    raise RuntimeError('Validation input identity differs')
                panels[panel].append(item)
        if [len(panels[p]) for p in ('gate10', 'wider50')] != [10, 50]:
            raise RuntimeError('Saved validation panel sizes differ')

        def forward(item, stage, cache):
            full = {**item['shared_transformer_kwargs'], **stage['kwargs']}
            kwargs = gpu({field: full[field] for field in argument_order})
            kwargs['kv_cache'] = cache
            return model(**kwargs)[0][:, -stage['target_image_tokens']:].detach().cpu().contiguous()

        def predictions():
            rows = []
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for panel, items in panels.items():
                    for item in items:
                        cache = QwenImage21KVCache(32)
                        for stage in item['stages']:
                            pred = forward(item, stage, cache)
                            if not torch.isfinite(pred).all() or pred.float().norm() == 0:
                                raise RuntimeError('Invalid reference velocity')
                            rows.append(dict(panel=panel, source_id=item['row']['source_id'],
                                             dimension=item['row']['dimension'],
                                             subdimension=item['row'].get('subdimension'),
                                             stage=stage['stage'], velocity=pred))
            return rows

        def save(name, value):
            path = directory / name
            temporary = path.with_suffix('.tmp')
            torch.save(value, temporary)
            temporary.replace(path)
            artifact = dict(name=name, sha256=digest(path), bytes=path.stat().st_size)
            report['artifacts'].append(artifact)
            return artifact

        teacher_predictions = predictions()
        save('teacher_predictions.pt', teacher_predictions)
        report['teacher_calls'] += len(teacher_predictions)
        # Rebuild the Phase 6 selected student exactly as its verified replay did:
        # 31 teacher blocks plus the frozen pretrained rank-256 bridge at slot 5.
        namespace = {}
        exec(compile(request['core_source'], 'bridge_definition.py', 'exec'), namespace)
        exec(compile(request['wrapper_source'], 'bridge_block_definition.py', 'exec'), namespace)
        candidate = request['selected_candidate']
        path = Path('/phase3/phase5/fitting') / request['fit_key'] / Path(candidate['checkpoint_path']).name
        if digest(path) != candidate['checkpoint_sha256']:
            raise RuntimeError('Selected bridge checkpoint differs.')
        checkpoint = torch.load(path, weights_only=True, map_location='cpu')
        if (checkpoint['request_key'] != request['fit_key'] or checkpoint['layer'] != 5
                or checkpoint['step'] != candidate['selected_update']):
            raise RuntimeError('Bridge provenance differs.')
        bridge = namespace['ResidualBottleneckBridge'](4096, 256)
        bridge.load_state_dict(checkpoint['state_dict'])
        bridge.requires_grad_(False)
        namespace['insert_bridge'](model, 5, bridge, QwenImage21KVCache)
        del checkpoint
        model.requires_grad_(False).eval()
        gc.collect()
        torch.cuda.empty_cache()
        baseline = predictions()
        report['student_calls'] += len(baseline)
        save('candidate_predictions.pt', baseline)
        # The 10 gate prompts must reproduce the saved Phase 6 trained-bridge velocities.
        replay = request['selected_replay']
        replay_dir = Path('/phase3/phase6/verified_student_replay') / replay['request_key']
        reference = {}
        for record in request['records']:
            if not record['capture_id'].startswith('p5-validation-'):
                continue
            name = 'students-' + record['capture_id'] + '.pt'
            if digest(replay_dir / name) != replay['artifacts'][name]:
                raise RuntimeError('Phase 6 student replay checksum differs.')
            saved = torch.load(replay_dir / name, weights_only=True, map_location='cpu')
            for output in saved['outputs']:
                if output['candidate_id'] == candidate['candidate_id'] and output['variant'] == 'trained':
                    reference[(output['source_id'], output['stage'])] = output['velocity']
        gate_rows = [r for r in baseline if r['panel'] == 'gate10']
        if len(reference) != 30 or len(gate_rows) != 30:
            raise RuntimeError('Phase 6 replay coverage differs.')
        reproduction = [compare(r['velocity'], reference[(r['source_id'], r['stage'])]) for r in gate_rows]
        report['candidate_replay_max_relative_l2'] = max(r['relative_l2_error'] for r in reproduction)
        if report['candidate_replay_max_relative_l2'] > 0.001:
            raise RuntimeError('Selected student failed Phase 6 output replay')
        del reference

        class IdentityBlock(torch.nn.Module):
            def __init__(self, original):
                super().__init__()
                self.attn = SimpleNamespace(processor=original.attn.processor)

            def forward(self, hidden_states, *args, **kwargs):
                return hidden_states

        def metrics(rows):
            output = []
            for measured, base, teacher_row in zip(rows, baseline, teacher_predictions):
                identity = ('panel', 'source_id', 'stage')
                if any(measured[k] != base[k] or base[k] != teacher_row[k] for k in identity):
                    raise RuntimeError('Prediction alignment differs')
                teacher_error = compare(measured['velocity'], teacher_row['velocity'])['relative_l2_error']
                baseline_error = compare(base['velocity'], teacher_row['velocity'])['relative_l2_error']
                output.append({k: measured[k] for k in measured if k != 'velocity'} |
                              dict(baseline_displacement=compare(measured['velocity'], base['velocity'])['relative_l2_error'],
                                   teacher_relative_l2=teacher_error, baseline_teacher_relative_l2=baseline_error,
                                   teacher_error_change_percentage_points=100 * (teacher_error - baseline_error)))
            return output

        def aggregate(rows):
            fields = ('baseline_displacement', 'teacher_relative_l2', 'baseline_teacher_relative_l2',
                      'teacher_error_change_percentage_points')
            result = dict(comparisons=len(rows))
            for field in fields:
                values = sorted(r[field] for r in rows)
                def percentile(fraction):
                    position = fraction * (len(values) - 1)
                    low = int(position)
                    return values[low] + (values[min(low + 1, len(values) - 1)] - values[low]) * (position - low)
                result[field] = dict(mean=statistics.mean(values), maximum=max(values),
                                     p10=percentile(.1), p50=percentile(.5), p90=percentile(.9))
            result['worst_regressions'] = sorted(rows, key=lambda r:r['teacher_error_change_percentage_points'], reverse=True)[:10]
            return result

        report['baseline'] = aggregate(metrics(baseline))
        report['blocks'] = []
        weights = {n: (id(p), p._version) for n, p in model.named_parameters()}
        original_blocks = list(model.transformer_blocks)
        for slot in request['eligible_original_block_ids']:
            if list(model.transformer_blocks) != original_blocks:
                raise RuntimeError('Baseline B architecture not restored')
            if any((id(p), p._version) != weights[n] for n, p in model.named_parameters()):
                raise RuntimeError('Frozen candidate B weights changed')
            started = time.perf_counter()
            model.transformer_blocks[slot] = IdentityBlock(original_blocks[slot])
            try:
                rows = predictions()
            finally:
                model.transformer_blocks[slot] = original_blocks[slot]
            report['student_calls'] += len(rows)
            records = metrics(rows)
            artifact = save(f'block_{slot:02d}_predictions.pt', rows)
            result = dict(original_block_id=slot, overall=aggregate(records),
                          panels={panel:aggregate([r for r in records if r['panel']==panel]) for panel in panels},
                          stages={stage:aggregate([r for r in records if r['stage']==stage]) for stage in ('early','middle','late')},
                          prompt_groups={group:aggregate([r for r in records if r['dimension']==group])
                                         for group in sorted({r['dimension'] for r in records})},
                          records=records, artifact=artifact, seconds=time.perf_counter()-started)
            report['blocks'].append(result)
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            print(f'Completed independent block {slot}: {result["overall"]["baseline_displacement"]["mean"]:.6f}', flush=True)
        report['ranking'] = [r['original_block_id'] for r in sorted(report['blocks'],
                              key=lambda r:r['overall']['baseline_displacement']['mean'])]
        report['status'] = 'passed'
        report['optimizer_updates'] = 0
        report['scope'] = 'Conditional independent bypass screen only; no block accepted, no quality promotion.'
    except Exception as exc:
        import traceback
        report['status'] = 'failed'
        report['error'] = repr(exc)
        report['traceback'] = traceback.format_exc()
    (directory / 'summary.json').write_text(json.dumps(report, indent=2))
    results_volume.commit()
    return report
'''


"""Launch the conditional scan inside candidate B (selected block-5 student); reuse completed results."""
import argparse
import hashlib
import json
import sys
from pathlib import Path


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--old-account', action='store_true')
    parser.add_argument('--dry-run', action='store_true', help='Write the request locally without scheduling GPU work.')
    args = parser.parse_args(LAUNCH_ARGS)
    if args.old_account:
        parser.error('Old account is download-only; GPU execution must use the new account.')
    root = ROOT
    recovered = root / 'data/recovered/modal'
    replay_key = 'ccabf81790b354ca69bcb5d0213c7559e15ba3458bf5d5c17036ecb79b73d12c'
    validation_key = '173d4a2d3ab854f15ae4c710e44b9e7afe122e429742f46d8fb7c72cce1880da'
    scan_a_key = '7454da6a5c1dc33af23dd46833e698f9b759d14985ca96340bc3d40cd47fb4f5'
    replay_dir = recovered / 'phase6/verified_student_replay' / replay_key
    replay = json.loads((replay_dir / 'request.json').read_bytes())['request']
    replay_summary = json.loads((replay_dir / 'summary.json').read_bytes())
    selected = json.loads((recovered / 'phase6/expanded_validation' / validation_key / 'request.json').read_bytes())['request']
    # Scan inputs (fixed panels, teacher contract, runtime) are shared with the candidate A scan.
    scan_a = json.loads((root / 'data/phase7/conditional_importance' / scan_a_key / 'request.json').read_bytes())
    if hashlib.sha256(json.dumps(scan_a['request'], sort_keys=True).encode()).hexdigest() != scan_a_key:
        raise RuntimeError('Candidate A scan request differs from its key.')
    scan_a = scan_a['request']
    candidate = selected['selected_candidate']
    if candidate['candidate_id'] != 'single_bridge_block_05' or selected['fit_key'] != replay['fit_key']:
        raise RuntimeError('Selected Phase 6 candidate differs.')
    if replay['core_source'] != scan_a['core_source'] or replay['wrapper_source'] != scan_a['wrapper_source']:
        raise RuntimeError('Bridge sources differ between Phase 6 replay and the scan contract.')
    shared = ('mode', 'teacher', 'handoff_manifest_sha256', 'replay_contract_sha256', 'expected_state',
              'arithmetic_contract', 'runtime_environment', 'reference_key', 'reference_report_semantic_sha256',
              'records', 'transformer_source_sha256', 'expanded_validation', 'core_source', 'wrapper_source',
              'canonical_tables_key', 'cache_policy', 'gate_rule', 'gate_references',
              'execution_packages', 'execution_reference_runtime')
    request = {k: scan_a[k] for k in shared}
    source = SOURCES['conditional_scan_worker_b.py']
    request.update(
        version='phase7-conditional-importance-b-v1', worker_source=source, execution_account='new',
        selected_candidate=candidate, fit_key=selected['fit_key'],
        selected_replay=dict(request_key=replay_key, artifacts={a['name']: a['sha256'] for a in replay_summary['artifacts']
                                                                  if a['name'].startswith('students-')}),
        eligible_original_block_ids=[i for i in range(32) if i != 5],
        selection='Screening only: independent frozen bypasses; no pruning or quality promotion',
        scope='Fixed gate10 and wider50 development prompts, three saved stages; compare B and teacher')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase7/conditional_importance_b' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2)+'\n', encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase7/conditional_importance/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        try:
            b''.join(results_volume.read_file(remote + '/started.json'))
        except FileNotFoundError:
            pass
        else:
            raise RuntimeError('Run already started; inspect its progress. No duplicate GPU scheduled.')
        app = modal.App('nihonga-phase7-conditional-importance-b')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(source, 'conditional_scan_worker_b.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8,
                              memory=65536, max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['scan_candidate'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2)+'\n')
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2)+'\n', encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'ranking', 'candidate_replay_max_relative_l2')}), flush=True)
    if report.get('status') != 'passed':
        raise SystemExit(1)

summary = reuse_or_launch(RUN, main)

rows = sorted((b['overall']['teacher_relative_l2']['mean'], b['original_block_id']) for b in summary['blocks'])
print('replay', summary['candidate_replay_max_relative_l2'], '| B alone:', pct(summary['baseline']['teacher_relative_l2']['mean']))
print([(block, pct(mean)) for mean, block in rows])

### What the results mean

B reproduces its Phase 6 outputs exactly (difference 0.0) and measures 5.40%. The gentlest bypass is **block 13** (7.63%), then 12, 14, 17 and 11: the same five as inside A. No bypass stays within the 6.52% gate, so any further removal needs a learned replacement. Early denoising suffers most for every candidate.

## 6. First learned bridge for block 13

Block 13 replaced by a rank-256 residual bridge (the Phase 5 design), fitted to block 13's own inputs and outputs recorded inside B on 60 TRAIN prompts plus 600 early TRAIN examples; the 10 check prompts only select the checkpoint. Gates were committed first (`data/phase7/block13_bridge/protocol.json`).

In [ ]:
# First learned bridge for block 13: reuses the saved run; launches GPU work only when its result is missing and ALLOW_GPU is True.

RUN = 'data/phase7/block13_bridge/6d44342e97eaa402dd0ccad4e36e29fc030adfa128a30b833f5c25c20b5893bc'

LAUNCH_ARGS = ['--slot', '13']

SOURCES['bridge_candidate_worker.py'] = r'''def fit_bridge_candidate(request, key):
    """Fit a learned bridge for one block inside the frozen block-5 student (candidate B).

    Reuses the verified teacher contract and exact B replay, captures the target
    block's input/output inside B, fits only the new bridge on TRAIN states, then
    evaluates the whole candidate against the teacher on the scan inputs.
    """
    import hashlib
    import importlib.metadata
    import inspect
    import json
    import os
    import socket
    import uuid
    from pathlib import Path
    import torch
    from torch.nn.attention import sdpa_kernel, SDPBackend
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache

    directory = Path('/phase3/phase7/bridge_candidate') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete healing attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()
    report = {'request_key': key, 'status': 'failed', 'mode': request['mode'], 'teacher_calls': 0,
              'student_calls': 0, 'optimizer_updates': 0, 'controls': [], 'artifacts': [],
              'process': {'pid': os.getpid(), 'host': socket.gethostname(), 'nonce': uuid.uuid4().hex,
                          'modal_task_id': os.environ.get('MODAL_TASK_ID')}}

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def gpu(value):
        if isinstance(value, torch.Tensor):
            return value.to('cuda')
        if isinstance(value, dict):
            return {k: gpu(v) for k, v in value.items()}
        if isinstance(value, (list, tuple)):
            return type(value)(gpu(v) for v in value)
        return value

    def compare(x, y):
        a, b = x.float(), y.float()
        if not torch.isfinite(a).all() or not torch.isfinite(b).all():
            raise RuntimeError('Nonfinite reference tensor.')
        d = a - b
        return {'relative_l2_error': float(d.norm()/b.norm().clamp_min(1e-12)),
                'max_abs_error': float(d.abs().max()), 'exact_equal': bool(torch.equal(x, y))}

    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('GPU differs.')
        report['packages'] = {p: importlib.metadata.version(p) for p in ('torch', 'diffusers', 'transformers', 'accelerate', 'pillow')}
        report['torch_cuda'] = torch.version.cuda
        report['cudnn_version'] = torch.backends.cudnn.version()
        report['settings'] = request['arithmetic_contract']
        if request['mode'] != 'verify':
            raise RuntimeError('Healing requires the verified teacher reference.')
        reference_dir = Path('/phase3/phase6/deterministic_reference') / request['reference_key']
        reference_report_path = reference_dir / 'summary.json'
        if hashlib.sha256(json.dumps(json.loads(reference_report_path.read_text()), sort_keys=True).encode()).hexdigest() != request['reference_report_semantic_sha256']:
            raise RuntimeError('Generation report differs.')
        reference_report = json.loads(reference_report_path.read_text())
        if reference_report['status'] != 'reference_generated':
            raise RuntimeError('Reference generation did not complete.')
        if reference_report['settings'] != request['arithmetic_contract']:
            raise RuntimeError('Arithmetic contract differs from generation.')
        report['reference_process'] = reference_report['process']
        if reference_report['process']['nonce'] == report['process']['nonce']:
            raise RuntimeError('Independent process requirement failed.')
        if reference_report['packages'] != report['packages'] or reference_report['torch_cuda'] != report['torch_cuda'] or reference_report['cudnn_version'] != report['cudnn_version']:
            raise RuntimeError('Numerical runtime differs across generation and verification.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
            dtype=torch.bfloat16, local_files_only=True).to('cuda')
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Model source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--'))
                       / 'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        state = {}
        for kind, values in (('parameter', model.named_parameters()), ('buffer', model.named_buffers())):
            for name, value in values:
                cpu = value.detach().cpu().contiguous()
                state[name] = {'kind': kind, 'dtype': str(value.dtype), 'shape': list(value.shape),
                    'stride': list(value.stride()), 'sha256': hashlib.sha256(memoryview(cpu.view(torch.uint8).numpy())).hexdigest()}
        for axis, value in enumerate(model.pos_embed.freqs):
            name = f'pos_embed.freqs.{axis}'
            cpu = value.detach().cpu().contiguous()
            state[name] = {'kind': 'unregistered_rope_table', 'dtype': str(value.dtype), 'shape': list(value.shape),
                'stride': list(value.stride()), 'sha256': hashlib.sha256(memoryview(cpu.view(torch.uint8).numpy())).hexdigest()}
        expected = request['expected_state']
        derived_names = sorted(n for n in expected if n.startswith('pos_embed.freqs.') or n == 'time_text_embed.time_proj.freqs')

        def derived_tensors():
            values = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
            values['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
            return values

        def table_hash(value):
            return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

        canonical_path = Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt')
        import time
        mismatched = [n for n in derived_names if state.get(n) != expected.get(n)]
        if not mismatched:
            if not canonical_path.exists():
                canonical_path.parent.mkdir(parents=True, exist_ok=True)
                temporary = canonical_path.with_suffix('.' + report['process']['nonce'] + '.tmp')
                torch.save({n: v.detach().cpu().contiguous() for n, v in derived_tensors().items()}, temporary)
                temporary.replace(canonical_path)
                results_volume.commit()
                report['canonical_tables_saved'] = True
        else:
            deadline = time.time() + 900
            while not canonical_path.exists():
                if time.time() > deadline:
                    raise RuntimeError('Derived tables differ on this host and no canonical copy became available.')
                time.sleep(20)
                results_volume.reload()
            saved = torch.load(canonical_path, weights_only=True, map_location='cpu')
            if sorted(saved) != derived_names or any(table_hash(saved[n]) != expected[n]['sha256'] for n in derived_names):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [saved[f'pos_embed.freqs.{axis}'].to(device=value.device, dtype=value.dtype)
                        for axis, value in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(saved['time_text_embed.time_proj.freqs'])
            for n, value in derived_tensors().items():
                state[n] = {**state[n], 'sha256': table_hash(value)}
            report['canonical_tables_applied'] = mismatched
        if state != request['expected_state']:
            expected = request['expected_state']
            different_names = sorted(name for name in expected.keys() & state.keys()
                                     if state[name] != expected[name])
            report['state_mismatch'] = {
                'actual_count': len(state), 'expected_count': len(expected),
                'missing_names': sorted(set(expected) - set(state))[:12],
                'extra_names': sorted(set(state) - set(expected))[:12],
                'different': [{'name': name, 'actual': state[name], 'expected': expected[name]}
                              for name in different_names][:12]}
            allowed_derived_tables = {'pos_embed.freqs.1', 'pos_embed.freqs.2', 'time_text_embed.time_proj.freqs'}
            if set(state) != set(expected) or not set(different_names).issubset(allowed_derived_tables):
                raise RuntimeError('Loaded teacher learned state differs from audited reference.')
            report['derived_table_variation_requires_output_replay'] = different_names
        report['state_sha256'] = hashlib.sha256(json.dumps(state, sort_keys=True).encode()).hexdigest()
        torch.cuda.synchronize()
        argument_order = ('hidden_states', 'timestep', 'encoder_hidden_states', 'encoder_hidden_states_mask',
                          'img_shapes', 'img_mask', 'attention_kwargs', 'kv_cache_mode', 'return_dict')
        # Every intact-teacher control must pass before any block is touched.
        for record in request['records']:
            path = Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt'
            if digest(path) != record['tensor_sha256']:
                raise RuntimeError('Legacy input bundle differs.')
            bundle = torch.load(path, weights_only=True, map_location='cpu')
            outputs, controls = [], []
            cache = QwenImage21KVCache(32)
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for stage in bundle['stages']:
                    full = {**bundle['shared_transformer_kwargs'], **stage['kwargs']}
                    kwargs = gpu({field: full[field] for field in argument_order})
                    kwargs['kv_cache'] = cache
                    pred = model(**kwargs)[0][:, -stage['target_image_tokens']:].detach().cpu().contiguous()
                    report['teacher_calls'] += 1
                    legacy = compare(pred, stage['teacher_velocity'])
                    entry = {'source_id': record['source_id'], 'split': bundle['row']['split'],
                             'stage': stage['stage'], 'legacy_difference': legacy}
                    outputs.append({'stage': stage['stage'], 'step_index': stage['step_index'],
                                    'scheduler_sigma': stage['scheduler_sigma'], 'velocity': pred})
                    controls.append(entry)
            filename = record['capture_id'] + '.pt'
            artifact = next(a for a in reference_report['artifacts'] if a['name'] == filename)
            if digest(reference_dir / filename) != artifact['sha256']:
                raise RuntimeError('Reference output checksum differs.')
            reference_bundle = torch.load(reference_dir / filename, weights_only=True, map_location='cpu')
            if reference_bundle['source_id'] != record['source_id'] or reference_bundle['input_sha256'] != record['tensor_sha256']:
                raise RuntimeError('Reference input provenance differs.')
            reference_outputs = {v['stage']: v['velocity'] for v in reference_bundle['outputs']}
            for entry, measured in zip(controls, outputs):
                entry['reference_difference'] = compare(measured['velocity'], reference_outputs[entry['stage']])
                entry['passed'] = entry['reference_difference']['relative_l2_error'] <= 0.001
            output_path = directory / filename
            temporary = output_path.with_suffix('.tmp')
            torch.save({'request_key': key, 'source_id': record['source_id'], 'input_sha256': record['tensor_sha256'],
                        'outputs': outputs, 'controls': controls}, temporary)
            temporary.replace(output_path)
            report['controls'].extend(controls)
            report['artifacts'].append({'name': filename, 'sha256': digest(output_path), 'bytes': output_path.stat().st_size})
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            if not all(c['passed'] for c in controls):
                raise RuntimeError('Independent-process reference replay exceeds tolerance.')
        if report['teacher_calls'] != 90:
            raise RuntimeError('Reference attempt did not cover all 90 saved stage inputs.')
        report['legacy_max_relative_l2'] = max(c['legacy_difference']['relative_l2_error'] for c in report['controls'])
        report['max_reference_relative_l2'] = max(c['reference_difference']['relative_l2_error'] for c in report['controls'])
        report['exact_equal_controls'] = sum(c['reference_difference']['exact_equal'] for c in report['controls'])
        if not all(c['passed'] for c in report['controls']):
            raise RuntimeError('Teacher controls did not pass.')
        report['status'] = 'teacher_verified'
        (directory / 'teacher_control_report.json').write_text(json.dumps(report, indent=2))
        results_volume.commit()
        print('All 90 intact-teacher controls passed; assembling frozen 31-block candidate B.', flush=True)

        import gc
        import time
        import statistics
        from types import SimpleNamespace
        
        # Inputs are independently identified; 10 gate and 50 wider development prompts.
        panels = {'gate10': [], 'wider50': []}
        for panel, records in (('gate10', [r for r in request['records'] if r['capture_id'].startswith('p5-validation-')]),
                               ('wider50', request['expanded_validation']['records'])):
            for record in records:
                path = Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt'
                if digest(path) != record['tensor_sha256']:
                    raise RuntimeError('Fixed input hash differs')
                item = torch.load(path, weights_only=True, map_location='cpu')
                if item['row']['source_id'] != record['source_id'] or item['row']['split'] != 'validation':
                    raise RuntimeError('Validation input identity differs')
                panels[panel].append(item)
        if [len(panels[p]) for p in ('gate10', 'wider50')] != [10, 50]:
            raise RuntimeError('Saved validation panel sizes differ')

        def forward(item, stage, cache):
            full = {**item['shared_transformer_kwargs'], **stage['kwargs']}
            kwargs = gpu({field: full[field] for field in argument_order})
            kwargs['kv_cache'] = cache
            return model(**kwargs)[0][:, -stage['target_image_tokens']:].detach().cpu().contiguous()

        def predictions():
            rows = []
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for panel, items in panels.items():
                    for item in items:
                        cache = QwenImage21KVCache(32)
                        for stage in item['stages']:
                            pred = forward(item, stage, cache)
                            if not torch.isfinite(pred).all() or pred.float().norm() == 0:
                                raise RuntimeError('Invalid reference velocity')
                            rows.append(dict(panel=panel, source_id=item['row']['source_id'],
                                             dimension=item['row']['dimension'],
                                             subdimension=item['row'].get('subdimension'),
                                             stage=stage['stage'], velocity=pred))
            return rows

        def save(name, value):
            path = directory / name
            temporary = path.with_suffix('.tmp')
            torch.save(value, temporary)
            temporary.replace(path)
            artifact = dict(name=name, sha256=digest(path), bytes=path.stat().st_size)
            report['artifacts'].append(artifact)
            return artifact

        teacher_predictions = predictions()
        save('teacher_predictions.pt', teacher_predictions)
        report['teacher_calls'] += len(teacher_predictions)
        # Rebuild the Phase 6 selected student exactly as its verified replay did:
        # 31 teacher blocks plus the frozen pretrained rank-256 bridge at slot 5.
        namespace = {}
        exec(compile(request['core_source'], 'bridge_definition.py', 'exec'), namespace)
        exec(compile(request['wrapper_source'], 'bridge_block_definition.py', 'exec'), namespace)
        candidate = request['selected_candidate']
        path = Path('/phase3/phase5/fitting') / request['fit_key'] / Path(candidate['checkpoint_path']).name
        if digest(path) != candidate['checkpoint_sha256']:
            raise RuntimeError('Selected bridge checkpoint differs.')
        checkpoint = torch.load(path, weights_only=True, map_location='cpu')
        if (checkpoint['request_key'] != request['fit_key'] or checkpoint['layer'] != 5
                or checkpoint['step'] != candidate['selected_update']):
            raise RuntimeError('Bridge provenance differs.')
        bridge = namespace['ResidualBottleneckBridge'](4096, 256)
        bridge.load_state_dict(checkpoint['state_dict'])
        bridge.requires_grad_(False)
        namespace['insert_bridge'](model, 5, bridge, QwenImage21KVCache)
        del checkpoint
        model.requires_grad_(False).eval()
        gc.collect()
        torch.cuda.empty_cache()
        baseline = predictions()
        report['student_calls'] += len(baseline)
        save('candidate_predictions.pt', baseline)
        # The 10 gate prompts must reproduce the saved Phase 6 trained-bridge velocities.
        replay = request['selected_replay']
        replay_dir = Path('/phase3/phase6/verified_student_replay') / replay['request_key']
        reference = {}
        for record in request['records']:
            if not record['capture_id'].startswith('p5-validation-'):
                continue
            name = 'students-' + record['capture_id'] + '.pt'
            if digest(replay_dir / name) != replay['artifacts'][name]:
                raise RuntimeError('Phase 6 student replay checksum differs.')
            saved = torch.load(replay_dir / name, weights_only=True, map_location='cpu')
            for output in saved['outputs']:
                if output['candidate_id'] == candidate['candidate_id'] and output['variant'] == 'trained':
                    reference[(output['source_id'], output['stage'])] = output['velocity']
        gate_rows = [r for r in baseline if r['panel'] == 'gate10']
        if len(reference) != 30 or len(gate_rows) != 30:
            raise RuntimeError('Phase 6 replay coverage differs.')
        reproduction = [compare(r['velocity'], reference[(r['source_id'], r['stage'])]) for r in gate_rows]
        report['candidate_replay_max_relative_l2'] = max(r['relative_l2_error'] for r in reproduction)
        if report['candidate_replay_max_relative_l2'] > 0.001:
            raise RuntimeError('Selected student failed Phase 6 output replay')
        del reference

        class IdentityBlock(torch.nn.Module):
            def __init__(self, original):
                super().__init__()
                self.attn = SimpleNamespace(processor=original.attn.processor)

            def forward(self, hidden_states, *args, **kwargs):
                return hidden_states

        def metrics(rows):
            output = []
            for measured, base, teacher_row in zip(rows, baseline, teacher_predictions):
                identity = ('panel', 'source_id', 'stage')
                if any(measured[k] != base[k] or base[k] != teacher_row[k] for k in identity):
                    raise RuntimeError('Prediction alignment differs')
                teacher_error = compare(measured['velocity'], teacher_row['velocity'])['relative_l2_error']
                baseline_error = compare(base['velocity'], teacher_row['velocity'])['relative_l2_error']
                output.append({k: measured[k] for k in measured if k != 'velocity'} |
                              dict(baseline_displacement=compare(measured['velocity'], base['velocity'])['relative_l2_error'],
                                   teacher_relative_l2=teacher_error, baseline_teacher_relative_l2=baseline_error,
                                   teacher_error_change_percentage_points=100 * (teacher_error - baseline_error)))
            return output

        def aggregate(rows):
            fields = ('baseline_displacement', 'teacher_relative_l2', 'baseline_teacher_relative_l2',
                      'teacher_error_change_percentage_points')
            result = dict(comparisons=len(rows))
            for field in fields:
                values = sorted(r[field] for r in rows)
                def percentile(fraction):
                    position = fraction * (len(values) - 1)
                    low = int(position)
                    return values[low] + (values[min(low + 1, len(values) - 1)] - values[low]) * (position - low)
                result[field] = dict(mean=statistics.mean(values), maximum=max(values),
                                     p10=percentile(.1), p50=percentile(.5), p90=percentile(.9))
            result['worst_regressions'] = sorted(rows, key=lambda r:r['teacher_error_change_percentage_points'], reverse=True)[:10]
            return result

        report['baseline'] = aggregate(metrics(baseline))

        # ---- Capture the target block's input and output inside frozen B. ----
        slot = request['slot']
        fit = request['fit']
        generator = torch.Generator().manual_seed(fit['capture_seed'])
        hook_state = {'on': False}

        def hook(module, args, kwargs, output):
            if hook_state['on']:
                hidden = kwargs['hidden_states'] if 'hidden_states' in kwargs else args[0]
                if not isinstance(output, torch.Tensor) or output.shape != hidden.shape:
                    raise RuntimeError('Unexpected block output layout.')
                hook_state['io'] = (hidden.detach(), output.detach())

        handle = model.transformer_blocks[slot].register_forward_hook(hook, with_kwargs=True)
        groups = ('prompt_tokens', 'early_image', 'middle_image', 'late_image')
        captured = {split: {g: ([], []) for g in groups} for split in ('train', 'validation')}

        def collect(item, split, tokens):
            cache = QwenImage21KVCache(32)
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for stage in item['stages']:
                    hook_state['on'] = True
                    forward(item, stage, cache)
                    hook_state['on'] = False
                    hidden, output = hook_state.pop('io')
                    total, image = hidden.shape[1], stage['target_image_tokens']
                    spans = [(stage['stage'] + '_image', total - image, total)]
                    if total > image:
                        spans.append(('prompt_tokens', 0, total - image))
                    for group, low, high in spans:
                        count = min(tokens, high - low)
                        index = (low + torch.randperm(high - low, generator=generator)[:count]).to(hidden.device)
                        captured[split][group][0].append(hidden[0].index_select(0, index).to(torch.bfloat16).cpu())
                        captured[split][group][1].append(output[0].index_select(0, index).to(torch.bfloat16).cpu())
                    report['student_calls'] += 1

        def load_verified(path, sha256, size=None):
            if (size is not None and path.stat().st_size != size) or digest(path) != sha256:
                raise RuntimeError('Input tensor differs: ' + str(path))
            return torch.load(path, weights_only=True, map_location='cpu')

        train_ids = set()
        counts = {'train_full': 0, 'train_early_pool': 0, 'validation': 0}
        for record in request['records']:
            item = load_verified(Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt', record['tensor_sha256'])
            split = item['row']['split']
            if item['row']['source_id'] != record['source_id'] or split not in ('train', 'validation'):
                raise RuntimeError('Capture record identity differs.')
            if split == 'train':
                train_ids.add(record['source_id'])
                counts['train_full'] += 1
            else:
                counts['validation'] += 1
            collect(item, split, fit['tokens_per_group_full'])
            del item
        fresh = request['fresh_aligned_train']
        for artifact in fresh['artifacts']:
            item = load_verified(Path(fresh['remote_directory']) / artifact['name'], artifact['sha256'], artifact['bytes'])
            if item['source_id'] != artifact['source_id'] or item['row']['split'] != 'train' or len(item['stages']) != 3:
                raise RuntimeError('Fresh aligned training provenance differs.')
            train_ids.add(artifact['source_id'])
            collect(item, 'train', fit['tokens_per_group_full'])
            counts['train_full'] += 1
            del item
        for example in request['early_pool']['examples']:
            item = load_verified(Path(example['remote_path']), example['sha256'], example['bytes'])
            if (item['capture_id'] != example['capture_id'] or item['row']['split'] != 'train'
                    or [s['stage'] for s in item['stages']] != ['early']):
                raise RuntimeError('Early pool example provenance differs.')
            collect(item, 'train', fit['tokens_per_group_early_pool'])
            counts['train_early_pool'] += 1
            del item
        handle.remove()
        validation_ids = {r['source_id'] for r in request['records'] if r['capture_id'].startswith('p5-validation-')}
        validation_ids |= {r['source_id'] for r in request['expanded_validation']['records']}
        if train_ids & validation_ids:
            raise RuntimeError('A validation prompt appears in the training set.')
        if counts != {'train_full': 60, 'train_early_pool': len(request['early_pool']['examples']), 'validation': 10}:
            raise RuntimeError('Capture counts differ: ' + json.dumps(counts))
        data = {split: {g: (torch.cat(v[0]).to('cuda'), torch.cat(v[1]).to('cuda')) for g, v in by_group.items() if v[0]}
                for split, by_group in captured.items()}
        del captured
        report['capture'] = {'counts': counts,
                             'tokens': {split: {g: int(v[0].shape[0]) for g, v in d.items()} for split, d in data.items()}}
        if set(data['train']) != set(groups) or set(data['validation']) != set(groups):
            raise RuntimeError('A token group is missing from the capture.')
        (directory / 'progress.json').write_text(json.dumps(report, indent=2))
        results_volume.commit()
        print('Captured block', slot, json.dumps(report['capture']), flush=True)

        # ---- Fit only the new bridge with the Phase 5 objective. ----
        torch.manual_seed(fit['init_seed'])
        new_bridge = namespace['ResidualBottleneckBridge'](4096, fit['rank']).to('cuda').float()
        optimizer = torch.optim.AdamW(new_bridge.parameters(), lr=fit['learning_rate'], weight_decay=fit['weight_decay'])
        sampler = torch.Generator(device='cuda').manual_seed(fit['sampling_seed'])

        def objective(prediction, target):
            prediction, target = prediction.float(), target.float()
            direction = (torch.nn.functional.normalize(prediction, dim=-1) -
                         torch.nn.functional.normalize(target, dim=-1)).square().sum(-1)
            relative = (prediction - target).square().sum(-1) / target.square().sum(-1).clamp_min(1e-12)
            return direction + fit['magnitude_weight'] * relative

        def run(inputs):
            with torch.autocast('cuda', dtype=torch.bfloat16):
                return new_bridge(inputs.float()).float()

        def validation_score():
            with torch.no_grad():
                values = {}
                for g, (inputs, targets) in data['validation'].items():
                    total = sum(objective(run(inputs[i:i + 4096]), targets[i:i + 4096]).sum()
                                for i in range(0, inputs.shape[0], 4096))
                    values[g] = float(total / inputs.shape[0])
                return sum(values.values()) / len(values), values

        curve = []
        best = None
        for step in range(fit['updates'] + 1):
            if step % fit['checkpoint_every'] == 0:
                score, per_group = validation_score()
                curve.append(dict(step=step, validation=score, groups=per_group))
                if best is None or score < best[0]:
                    best = (score, step, {k: v.detach().cpu().clone() for k, v in new_bridge.state_dict().items()})
                print(f'Bridge step {step}: validation {score:.6f}', flush=True)
            if step == fit['updates']:
                break
            loss = 0
            for g in groups:
                inputs, targets = data['train'][g]
                index = torch.randint(inputs.shape[0], (fit['tokens_per_group_update'],), device='cuda', generator=sampler)
                loss = loss + objective(run(inputs[index]), targets[index]).mean() / len(groups)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(new_bridge.parameters(), fit['gradient_clip'])
            optimizer.step()
            report['optimizer_updates'] += 1
        report['fit'] = dict(curve=curve, selected_step=best[1], selected_validation=best[0],
                             identity_validation=curve[0]['validation'])
        checkpoint_path = directory / f'bridge_block_{slot:02d}_selected.pt'
        temporary = checkpoint_path.with_suffix('.tmp')
        torch.save({'request_key': key, 'layer': slot, 'step': best[1], 'rank': fit['rank'], 'state_dict': best[2]}, temporary)
        temporary.replace(checkpoint_path)
        report['artifacts'].append(dict(name=checkpoint_path.name, sha256=digest(checkpoint_path),
                                        bytes=checkpoint_path.stat().st_size))
        del data, new_bridge, optimizer
        gc.collect()
        torch.cuda.empty_cache()
        (directory / 'progress.json').write_text(json.dumps(report, indent=2))
        results_volume.commit()

        # ---- Evaluate the whole candidate against the teacher on the scan inputs. ----
        selected = namespace['ResidualBottleneckBridge'](4096, fit['rank'])
        selected.load_state_dict(best[2])
        selected.requires_grad_(False)
        namespace['insert_bridge'](model, slot, selected, QwenImage21KVCache)
        model.requires_grad_(False).eval()
        rows = predictions()
        report['student_calls'] += len(rows)
        save('candidate_predictions_with_bridge.pt', rows)
        records = metrics(rows)
        report['candidate'] = dict(
            overall=aggregate(records),
            panels={p: aggregate([r for r in records if r['panel'] == p]) for p in panels},
            stages={s: aggregate([r for r in records if r['stage'] == s]) for s in ('early', 'middle', 'late')},
            prompt_groups={g: aggregate([r for r in records if r['dimension'] == g])
                           for g in sorted({r['dimension'] for r in records})},
            records=records)
        report['status'] = 'passed'
        report['scope'] = 'Bridge fit and fixed-input velocity evaluation only; gates are applied locally from the preregistered protocol.'
    except Exception as exc:
        import traceback
        report['status'] = 'failed'
        report['error'] = repr(exc)
        report['traceback'] = traceback.format_exc()
    (directory / 'summary.json').write_text(json.dumps(report, indent=2))
    results_volume.commit()
    return report
'''


"""Fit a learned bridge for one block inside candidate B and evaluate the whole student; reuse completed results."""
import argparse
import hashlib
import json
import sys
from pathlib import Path


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--slot', type=int, default=13)
    parser.add_argument('--old-account', action='store_true')
    parser.add_argument('--dry-run', action='store_true', help='Write the request locally without scheduling GPU work.')
    args = parser.parse_args(LAUNCH_ARGS)
    if args.old_account:
        parser.error('Old account is download-only; GPU execution must use the new account.')
    root = ROOT
    protocol_path = root / f'data/phase7/block{args.slot}_bridge/protocol.json'
    protocol = json.loads(protocol_path.read_bytes())
    scan_b = json.loads(next((root / 'data/phase7/conditional_importance_b').glob('*/request.json')).read_bytes())['request']
    scan_a = json.loads((root / 'data/phase7/conditional_importance/7454da6a5c1dc33af23dd46833e698f9b759d14985ca96340bc3d40cd47fb4f5/request.json').read_bytes())['request']
    if args.slot == 5 or not 0 < args.slot < 31:
        parser.error('Slot must be a surviving interior block.')
    request = {k: v for k, v in scan_b.items() if k not in ('worker_source', 'eligible_original_block_ids', 'selection', 'scope', 'version')}
    source = SOURCES['bridge_candidate_worker.py']
    request.update(
        version='phase7-bridge-candidate-v1', worker_source=source, slot=args.slot,
        protocol_sha256=hashlib.sha256(protocol_path.read_bytes()).hexdigest(),
        fresh_aligned_train=scan_a['fresh_aligned_train'], early_pool=scan_a['early_pool'],
        fit=dict(rank=256, init_seed=20261017, capture_seed=20261018, sampling_seed=20261019, learning_rate=1e-3,
                 weight_decay=0.01, gradient_clip=1.0, magnitude_weight=0.1, updates=1000, checkpoint_every=100,
                 tokens_per_group_full=256, tokens_per_group_early_pool=64, tokens_per_group_update=256),
        selection='Minimum validation hidden-state objective at saved checkpoints including identity start',
        scope=f'Bridge for block {args.slot} inside frozen B; TRAIN-only fitting; whole-student velocity on scan inputs; no healing, images or latency')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = protocol_path.parent / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2)+'\n', encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase7/bridge_candidate/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        try:
            b''.join(results_volume.read_file(remote + '/started.json'))
        except FileNotFoundError:
            pass
        else:
            raise RuntimeError('Run already started; inspect its progress. No duplicate GPU scheduled.')
        app = modal.App('nihonga-phase7-bridge-candidate')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(source, 'bridge_candidate_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8,
                              memory=98304, max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['fit_bridge_candidate'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2)+'\n')
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2)+'\n', encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'candidate_replay_max_relative_l2')}), flush=True)
    if report.get('fit'):
        print('Selected step', report['fit']['selected_step'], 'validation', report['fit']['selected_validation'], flush=True)
    if report.get('status') != 'passed':
        raise SystemExit(1)

summary = reuse_or_launch(RUN, main)

velocity_table([velocity_row('C: first bridge', summary['candidate'])])
print('hidden objective', summary['fit']['identity_validation'], '->', summary['fit']['selected_validation'])

### What the results mean

The fit works on its own terms (hidden objective -38%), but the whole student barely moves: 7.37% on the check prompts against 7.56% for a plain skip and 5.52% for B; only 10% of the skip penalty is recovered. The velocity gates fail.

## 6b. Qualitative images: teacher, B and the first bridge

Because the teacher is not a perfect reference, the 30 earlier Phase 7 prompts are rendered with matched seeds and judged against each prompt. The regenerated teacher images match the earlier renders byte for byte.

In [ ]:
# Qualitative images: teacher, B and the first bridge: reuses the saved run; launches GPU work only when its result is missing and ALLOW_GPU is True.

RUN = 'data/phase7/bridge_images/d4530312b0f69c14bb9bbc8ad15d2514f3e8cb009caa860eed3b1702ff22ad7b'

LAUNCH_ARGS = []

SOURCES['bridge_images_worker.py'] = r'''def generate_bridge_images(request, key):
    """Matched 40-step images: teacher, B (block-5 pretrained bridge) and C (B plus the learned block-13 bridge).
    Same prompt, CPU noise seed, size, scheduler and 40 steps for every model. Qualitative screen only."""
    import gc
    import hashlib
    import inspect
    import json
    import os
    import time
    from pathlib import Path

    import torch
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache, QwenImage21KVLayerCache

    directory = Path('/phase3/phase7/bridge_images') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete image attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def table_hash(value):
        return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

    variants = ['teacher', 'B', 'C']
    report = {'request_key': key, 'status': 'running', 'rows': [], 'artifacts': [],
              'calls': {v: 0 for v in variants}, 'optimizer_updates': 0}
    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('Expected H100.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
                                                   dtype=torch.bfloat16, local_files_only=True).to('cuda')
        pipe.set_progress_bar_config(disable=True)
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Transformer source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--')) /
                       'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        # Derived position/time tables: use the audited values on every host (as in steps 7-11).
        derived = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
        derived['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
        expected = request['expected_derived_tables']
        mismatched = sorted(n for n, v in derived.items() if table_hash(v) != expected[n])
        if mismatched:
            canonical = torch.load(Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt'),
                                   weights_only=True, map_location='cpu')
            if any(table_hash(canonical[n]) != expected[n] for n in expected):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [canonical[f'pos_embed.freqs.{a}'].to(device=v.device, dtype=v.dtype) for a, v in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(canonical['time_text_embed.time_proj.freqs'])
            if any(table_hash(v) != expected[n] for n, v in
                   list({f'pos_embed.freqs.{a}': v for a, v in enumerate(model.pos_embed.freqs)}.items()) +
                   [('time_text_embed.time_proj.freqs', model.time_text_embed.time_proj.freqs)]):
                raise RuntimeError('Derived tables still differ after loading the audited copy.')
            report['canonical_tables_applied'] = mismatched

        def generate(row, variant):
            path = directory / (row['source_id'] + '-' + variant + '.png')
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'],
                             num_inference_steps=40, true_cfg_scale=teacher['true_cfg_scale'],
                             generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=True, output_type='pil').images[0]
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            temporary = path.with_suffix('.tmp')
            image.save(temporary, format='PNG')
            temporary.replace(path)
            results_volume.commit()
            report['calls'][variant] += 1
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            return {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size, 'seconds': elapsed}

        namespace = {}
        exec(compile(request['core_source'], 'bridge_definition.py', 'exec'), namespace)
        exec(compile(request['wrapper_source'], 'bridge_block_definition.py', 'exec'), namespace)

        def add_bridge(slot, spec):
            path = Path(spec['remote_path'])
            if digest(path) != spec['sha256']:
                raise RuntimeError('Bridge checkpoint differs: ' + spec['remote_path'])
            checkpoint = torch.load(path, weights_only=True, map_location='cpu')
            if checkpoint['layer'] != slot or checkpoint['step'] != spec['step'] or checkpoint['request_key'] != spec['request_key']:
                raise RuntimeError('Bridge provenance differs.')
            bridge = namespace['ResidualBottleneckBridge'](4096, spec['rank'])
            bridge.load_state_dict(checkpoint['state_dict'])
            bridge.requires_grad_(False)
            namespace['insert_bridge'](model, slot, bridge, QwenImage21KVCache)
            model.eval()

        outputs = {v: {} for v in variants}
        for row in request['rows']:
            outputs['teacher'][row['source_id']] = generate(row, 'teacher')
        add_bridge(5, request['bridges']['5'])
        for row in request['rows']:
            outputs['B'][row['source_id']] = generate(row, 'B')
        add_bridge(13, request['bridges']['13'])
        report['surviving_blocks'] = sum(not hasattr(b, 'bridge') for b in model.transformer_blocks)
        if report['surviving_blocks'] != 30:
            raise RuntimeError('Candidate C assembly differs.')
        for row in request['rows']:
            outputs['C'][row['source_id']] = generate(row, 'C')
        for row in request['rows']:
            report['rows'].append({**row, **{v: outputs[v][row['source_id']] for v in variants}})
            report['artifacts'].extend(outputs[v][row['source_id']] for v in variants)
        report['status'] = 'passed'
    except Exception as exc:
        report['status'] = 'failed'
        report['error'] = type(exc).__name__ + ': ' + str(exc)
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=True) + '\n')
    results_volume.commit()
    return report
'''


"""Render teacher, B and C (B plus learned block-13 bridge) on the 30 earlier Phase 7 image prompts; qualitative only."""
import argparse
import hashlib
import json
import sys
from pathlib import Path


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--dry-run', action='store_true')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    recovered = root / 'data/recovered/modal'
    images_key = 'dda8c184e657dfb604c35fb261d83371c108c39fac033a0f57db66f7686c30ad'
    base = json.loads((recovered / 'phase7/candidate_images' / images_key / 'request.json').read_bytes())['request']
    scan_b = json.loads(next((root / 'data/phase7/conditional_importance_b').glob('*/request.json')).read_bytes())['request']
    fit = json.loads(next((root / 'data/phase7/block13_bridge').glob('*/summary.json')).read_bytes())
    if fit.get('status') != 'passed':
        raise RuntimeError('Block-13 bridge run did not complete.')
    selected = scan_b['selected_candidate']
    bridge13 = next(a for a in fit['artifacts'] if a['name'] == 'bridge_block_13_selected.pt')
    request = {k: base[k] for k in ('teacher', 'transformer_source_sha256', 'runtime_environment', 'expected_derived_tables',
                                    'canonical_tables_key', 'core_source', 'wrapper_source', 'rows')}
    request.update(
        version='phase7-bridge-images-v1', worker_source=SOURCES['bridge_images_worker.py'],
        execution_account='new',
        bridges={'5': dict(remote_path=f"/phase3/phase5/fitting/{scan_b['fit_key']}/{Path(selected['checkpoint_path']).name}",
                           sha256=selected['checkpoint_sha256'], step=selected['selected_update'], rank=256,
                           request_key=scan_b['fit_key']),
                 '13': dict(remote_path=f"/phase3/phase7/bridge_candidate/{fit['request_key']}/{bridge13['name']}",
                            sha256=bridge13['sha256'], step=fit['fit']['selected_step'], rank=256, request_key=fit['request_key'])},
        scope='Qualitative matched-seed screen of teacher, B and C on the 30 earlier Phase 7 image prompts; cannot override the failed velocity gates')
    from experiment_runtime import pin_reference_runtime
    pin_reference_runtime(request, recovered)
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase7/bridge_images' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2)+'\n', encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase7/bridge_images/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        try:
            b''.join(results_volume.read_file(remote + '/started.json'))
        except FileNotFoundError:
            pass
        else:
            raise RuntimeError('Run already started; inspect its progress. No duplicate GPU scheduled.')
        app = modal.App('nihonga-phase7-bridge-images')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'bridge_images_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8,
                              memory=98304, max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['generate_bridge_images'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2)+'\n')
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2)+'\n', encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'calls')}), flush=True)
    if report.get('status') != 'passed':
        raise SystemExit(1)

summary = reuse_or_launch(RUN, main)

show_images(RUN + '/sheets', ['P14.jpg', 'P28.jpg'])

### What the results mean

Against B, the first bridge was worse on 15 prompts, similar on 13 and better on 2: hair smearing into the jaw (P14), dissolving circuit traces, misspelled signs ("Happy Birtliday Emma", "The Rusty Achor"), a missing alarm clock. B stays close to the teacher. Sheets: `sheets/P01.jpg`-`P30.jpg`.

### Build the teacher | B | C sheets

Downloads and hash-checks the images, then lays out one sheet per prompt.

In [ ]:
# Build the teacher | B | C sheets: local work only (no GPU).
LAUNCH_ARGS = []

"""Download the teacher/B/C images and build labeled comparison sheets (no GPU work)."""
import hashlib
import json
import textwrap
from pathlib import Path


def main():
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='download')
    import modal
    from PIL import Image, ImageDraw, ImageFont
    root = ROOT
    run = next((root / 'data/phase7/bridge_images').glob('*/summary.json'))
    report = json.loads(run.read_bytes())
    if report.get('status') != 'passed':
        raise RuntimeError('Image run did not pass.')
    key = report['request_key']
    store = root / '.archive/bridge_images' / key
    store.mkdir(parents=True, exist_ok=True)
    volume = modal.Volume.from_name('nihonga-phase3-artifacts')
    for artifact in report['artifacts']:
        path = store / artifact['name']
        if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != artifact['sha256']:
            path.write_bytes(b''.join(volume.read_file(f'/phase7/bridge_images/{key}/{artifact["name"]}')))
        if hashlib.sha256(path.read_bytes()).hexdigest() != artifact['sha256']:
            raise RuntimeError('Image hash mismatch: ' + artifact['name'])
    # Determinism check against the earlier Phase 7 teacher renders of the same prompts and seeds.
    earlier = json.loads(next((root / 'data/recovered/modal/phase7/candidate_images').glob('*/summary_blocks45.json')).read_bytes())
    earlier_hashes = {a['name']: a['sha256'] for a in earlier.get('artifacts', [])}
    matches = []
    for row in report['rows']:
        old = next((h for n, h in earlier_hashes.items() if n.startswith(row['source_id']) and 'teacher' in n), None)
        matches.append(None if old is None else old == row['teacher']['sha256'])
    # A CJK-capable font so the Chinese prompts render.
    face = next((f for f in ('C:/Windows/Fonts/msyh.ttc', 'C:/Windows/Fonts/simsun.ttc') if Path(f).exists()), 'C:/Windows/Fonts/arial.ttf')
    font = ImageFont.truetype(face, 20)
    big = ImageFont.truetype(face, 26)
    sheets = run.parent / 'sheets'
    sheets.mkdir(exist_ok=True)
    tile = 640
    for index, row in enumerate(report['rows']):
        canvas = Image.new('RGB', (3 * tile, tile + 150), 'white')
        draw = ImageDraw.Draw(canvas)
        draw.text((10, 6), f"P{index + 1:02d} | {row['source_id']} | {row.get('subdimension', '')}", font=big, fill='black')
        for line, text in enumerate(textwrap.wrap(row['prompt'], 170 if row['prompt'].isascii() else 85)[:3]):
            draw.text((10, 42 + 24 * line), text, font=font, fill='black')
        for column, variant in enumerate(('teacher', 'B', 'C')):
            label = {'teacher': 'Teacher (32 blocks)', 'B': 'B: block 5 bridged (accepted)', 'C': 'C: blocks 5 + 13 bridged'}[variant]
            draw.text((column * tile + 10, 118), label, font=big, fill='black')
            with Image.open(store / row[variant]['name']) as image:
                canvas.paste(image.convert('RGB').resize((tile, tile), Image.Resampling.LANCZOS), (column * tile, 150))
        canvas.save(sheets / f'P{index + 1:02d}.jpg', quality=90)
    print(json.dumps(dict(sheets=len(report['rows']), teacher_matches_earlier=sum(m is True for m in matches),
                          teacher_differs=sum(m is False for m in matches), teacher_unmatched=sum(m is None for m in matches))))

if not (ROOT / 'data/phase7/bridge_images/d4530312b0f69c14bb9bbc8ad15d2514f3e8cb009caa860eed3b1702ff22ad7b/sheets/P30.jpg').exists():
    main()
else:
    print("Saved output present; not rebuilt.")

## 7. Greedy selection of further blocks (stopped)

A greedy identity-skip search for three more blocks was launched and stopped at the user's request after a few
minutes: the work stays with blocks 5 and 13 and better bridges. No result exists; the launcher is not reproduced.

## 8. Bridge architectures for block 13: capacity and timestep conditioning

Every real block receives the model's shared timestep modulation [attention scale, attention gate, MLP scale, MLP gate]; the first bridge ignored it, and its errors concentrate at early, noisy steps. Three variants are fitted on identical data: a 4x wider plain bridge, and a conditioned bridge h + tanh(gate) * MLP(LN(h) * (1 + scale)), with scale and gate learned mixes of the model's own modulation, at rank 256 and 1024.

In [ ]:
# Bridge architectures for block 13: capacity and timestep conditioning: reuses the saved run; launches GPU work only when its result is missing and ALLOW_GPU is True.

RUN = 'data/phase7/block13_bridge_v2/ccd87daf9af35e7c3d69a01f15b46d94d4e509778dcf65a353d516cc1ebc23c0'

LAUNCH_ARGS = ['--slot', '13']

SOURCES['bridge_variants_worker.py'] = r'''def fit_bridge_variants(request, key):
    """Fit a learned bridge for one block inside the frozen block-5 student (candidate B).

    Reuses the verified teacher contract and exact B replay, captures the target
    block's input/output inside B, fits only the new bridge on TRAIN states, then
    evaluates the whole candidate against the teacher on the scan inputs.
    """
    import hashlib
    import importlib.metadata
    import inspect
    import json
    import os
    import socket
    import uuid
    from pathlib import Path
    import torch
    from torch.nn.attention import sdpa_kernel, SDPBackend
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache

    directory = Path('/phase3/phase7/bridge_variants') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete healing attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()
    report = {'request_key': key, 'status': 'failed', 'mode': request['mode'], 'teacher_calls': 0,
              'student_calls': 0, 'optimizer_updates': 0, 'controls': [], 'artifacts': [],
              'process': {'pid': os.getpid(), 'host': socket.gethostname(), 'nonce': uuid.uuid4().hex,
                          'modal_task_id': os.environ.get('MODAL_TASK_ID')}}

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def gpu(value):
        if isinstance(value, torch.Tensor):
            return value.to('cuda')
        if isinstance(value, dict):
            return {k: gpu(v) for k, v in value.items()}
        if isinstance(value, (list, tuple)):
            return type(value)(gpu(v) for v in value)
        return value

    def compare(x, y):
        a, b = x.float(), y.float()
        if not torch.isfinite(a).all() or not torch.isfinite(b).all():
            raise RuntimeError('Nonfinite reference tensor.')
        d = a - b
        return {'relative_l2_error': float(d.norm()/b.norm().clamp_min(1e-12)),
                'max_abs_error': float(d.abs().max()), 'exact_equal': bool(torch.equal(x, y))}

    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('GPU differs.')
        report['packages'] = {p: importlib.metadata.version(p) for p in ('torch', 'diffusers', 'transformers', 'accelerate', 'pillow')}
        report['torch_cuda'] = torch.version.cuda
        report['cudnn_version'] = torch.backends.cudnn.version()
        report['settings'] = request['arithmetic_contract']
        if request['mode'] != 'verify':
            raise RuntimeError('Healing requires the verified teacher reference.')
        reference_dir = Path('/phase3/phase6/deterministic_reference') / request['reference_key']
        reference_report_path = reference_dir / 'summary.json'
        if hashlib.sha256(json.dumps(json.loads(reference_report_path.read_text()), sort_keys=True).encode()).hexdigest() != request['reference_report_semantic_sha256']:
            raise RuntimeError('Generation report differs.')
        reference_report = json.loads(reference_report_path.read_text())
        if reference_report['status'] != 'reference_generated':
            raise RuntimeError('Reference generation did not complete.')
        if reference_report['settings'] != request['arithmetic_contract']:
            raise RuntimeError('Arithmetic contract differs from generation.')
        report['reference_process'] = reference_report['process']
        if reference_report['process']['nonce'] == report['process']['nonce']:
            raise RuntimeError('Independent process requirement failed.')
        if reference_report['packages'] != report['packages'] or reference_report['torch_cuda'] != report['torch_cuda'] or reference_report['cudnn_version'] != report['cudnn_version']:
            raise RuntimeError('Numerical runtime differs across generation and verification.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
            dtype=torch.bfloat16, local_files_only=True).to('cuda')
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Model source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--'))
                       / 'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        state = {}
        for kind, values in (('parameter', model.named_parameters()), ('buffer', model.named_buffers())):
            for name, value in values:
                cpu = value.detach().cpu().contiguous()
                state[name] = {'kind': kind, 'dtype': str(value.dtype), 'shape': list(value.shape),
                    'stride': list(value.stride()), 'sha256': hashlib.sha256(memoryview(cpu.view(torch.uint8).numpy())).hexdigest()}
        for axis, value in enumerate(model.pos_embed.freqs):
            name = f'pos_embed.freqs.{axis}'
            cpu = value.detach().cpu().contiguous()
            state[name] = {'kind': 'unregistered_rope_table', 'dtype': str(value.dtype), 'shape': list(value.shape),
                'stride': list(value.stride()), 'sha256': hashlib.sha256(memoryview(cpu.view(torch.uint8).numpy())).hexdigest()}
        expected = request['expected_state']
        derived_names = sorted(n for n in expected if n.startswith('pos_embed.freqs.') or n == 'time_text_embed.time_proj.freqs')

        def derived_tensors():
            values = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
            values['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
            return values

        def table_hash(value):
            return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

        canonical_path = Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt')
        import time
        mismatched = [n for n in derived_names if state.get(n) != expected.get(n)]
        if not mismatched:
            if not canonical_path.exists():
                canonical_path.parent.mkdir(parents=True, exist_ok=True)
                temporary = canonical_path.with_suffix('.' + report['process']['nonce'] + '.tmp')
                torch.save({n: v.detach().cpu().contiguous() for n, v in derived_tensors().items()}, temporary)
                temporary.replace(canonical_path)
                results_volume.commit()
                report['canonical_tables_saved'] = True
        else:
            deadline = time.time() + 900
            while not canonical_path.exists():
                if time.time() > deadline:
                    raise RuntimeError('Derived tables differ on this host and no canonical copy became available.')
                time.sleep(20)
                results_volume.reload()
            saved = torch.load(canonical_path, weights_only=True, map_location='cpu')
            if sorted(saved) != derived_names or any(table_hash(saved[n]) != expected[n]['sha256'] for n in derived_names):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [saved[f'pos_embed.freqs.{axis}'].to(device=value.device, dtype=value.dtype)
                        for axis, value in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(saved['time_text_embed.time_proj.freqs'])
            for n, value in derived_tensors().items():
                state[n] = {**state[n], 'sha256': table_hash(value)}
            report['canonical_tables_applied'] = mismatched
        if state != request['expected_state']:
            expected = request['expected_state']
            different_names = sorted(name for name in expected.keys() & state.keys()
                                     if state[name] != expected[name])
            report['state_mismatch'] = {
                'actual_count': len(state), 'expected_count': len(expected),
                'missing_names': sorted(set(expected) - set(state))[:12],
                'extra_names': sorted(set(state) - set(expected))[:12],
                'different': [{'name': name, 'actual': state[name], 'expected': expected[name]}
                              for name in different_names][:12]}
            allowed_derived_tables = {'pos_embed.freqs.1', 'pos_embed.freqs.2', 'time_text_embed.time_proj.freqs'}
            if set(state) != set(expected) or not set(different_names).issubset(allowed_derived_tables):
                raise RuntimeError('Loaded teacher learned state differs from audited reference.')
            report['derived_table_variation_requires_output_replay'] = different_names
        report['state_sha256'] = hashlib.sha256(json.dumps(state, sort_keys=True).encode()).hexdigest()
        torch.cuda.synchronize()
        argument_order = ('hidden_states', 'timestep', 'encoder_hidden_states', 'encoder_hidden_states_mask',
                          'img_shapes', 'img_mask', 'attention_kwargs', 'kv_cache_mode', 'return_dict')
        # Every intact-teacher control must pass before any block is touched.
        for record in request['records']:
            path = Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt'
            if digest(path) != record['tensor_sha256']:
                raise RuntimeError('Legacy input bundle differs.')
            bundle = torch.load(path, weights_only=True, map_location='cpu')
            outputs, controls = [], []
            cache = QwenImage21KVCache(32)
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for stage in bundle['stages']:
                    full = {**bundle['shared_transformer_kwargs'], **stage['kwargs']}
                    kwargs = gpu({field: full[field] for field in argument_order})
                    kwargs['kv_cache'] = cache
                    pred = model(**kwargs)[0][:, -stage['target_image_tokens']:].detach().cpu().contiguous()
                    report['teacher_calls'] += 1
                    legacy = compare(pred, stage['teacher_velocity'])
                    entry = {'source_id': record['source_id'], 'split': bundle['row']['split'],
                             'stage': stage['stage'], 'legacy_difference': legacy}
                    outputs.append({'stage': stage['stage'], 'step_index': stage['step_index'],
                                    'scheduler_sigma': stage['scheduler_sigma'], 'velocity': pred})
                    controls.append(entry)
            filename = record['capture_id'] + '.pt'
            artifact = next(a for a in reference_report['artifacts'] if a['name'] == filename)
            if digest(reference_dir / filename) != artifact['sha256']:
                raise RuntimeError('Reference output checksum differs.')
            reference_bundle = torch.load(reference_dir / filename, weights_only=True, map_location='cpu')
            if reference_bundle['source_id'] != record['source_id'] or reference_bundle['input_sha256'] != record['tensor_sha256']:
                raise RuntimeError('Reference input provenance differs.')
            reference_outputs = {v['stage']: v['velocity'] for v in reference_bundle['outputs']}
            for entry, measured in zip(controls, outputs):
                entry['reference_difference'] = compare(measured['velocity'], reference_outputs[entry['stage']])
                entry['passed'] = entry['reference_difference']['relative_l2_error'] <= 0.001
            output_path = directory / filename
            temporary = output_path.with_suffix('.tmp')
            torch.save({'request_key': key, 'source_id': record['source_id'], 'input_sha256': record['tensor_sha256'],
                        'outputs': outputs, 'controls': controls}, temporary)
            temporary.replace(output_path)
            report['controls'].extend(controls)
            report['artifacts'].append({'name': filename, 'sha256': digest(output_path), 'bytes': output_path.stat().st_size})
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            if not all(c['passed'] for c in controls):
                raise RuntimeError('Independent-process reference replay exceeds tolerance.')
        if report['teacher_calls'] != 90:
            raise RuntimeError('Reference attempt did not cover all 90 saved stage inputs.')
        report['legacy_max_relative_l2'] = max(c['legacy_difference']['relative_l2_error'] for c in report['controls'])
        report['max_reference_relative_l2'] = max(c['reference_difference']['relative_l2_error'] for c in report['controls'])
        report['exact_equal_controls'] = sum(c['reference_difference']['exact_equal'] for c in report['controls'])
        if not all(c['passed'] for c in report['controls']):
            raise RuntimeError('Teacher controls did not pass.')
        report['status'] = 'teacher_verified'
        (directory / 'teacher_control_report.json').write_text(json.dumps(report, indent=2))
        results_volume.commit()
        print('All 90 intact-teacher controls passed; assembling frozen 31-block candidate B.', flush=True)

        import gc
        import time
        import statistics
        from types import SimpleNamespace
        
        # Inputs are independently identified; 10 gate and 50 wider development prompts.
        panels = {'gate10': [], 'wider50': []}
        for panel, records in (('gate10', [r for r in request['records'] if r['capture_id'].startswith('p5-validation-')]),
                               ('wider50', request['expanded_validation']['records'])):
            for record in records:
                path = Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt'
                if digest(path) != record['tensor_sha256']:
                    raise RuntimeError('Fixed input hash differs')
                item = torch.load(path, weights_only=True, map_location='cpu')
                if item['row']['source_id'] != record['source_id'] or item['row']['split'] != 'validation':
                    raise RuntimeError('Validation input identity differs')
                panels[panel].append(item)
        if [len(panels[p]) for p in ('gate10', 'wider50')] != [10, 50]:
            raise RuntimeError('Saved validation panel sizes differ')

        def forward(item, stage, cache):
            full = {**item['shared_transformer_kwargs'], **stage['kwargs']}
            kwargs = gpu({field: full[field] for field in argument_order})
            kwargs['kv_cache'] = cache
            return model(**kwargs)[0][:, -stage['target_image_tokens']:].detach().cpu().contiguous()

        def predictions():
            rows = []
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for panel, items in panels.items():
                    for item in items:
                        cache = QwenImage21KVCache(32)
                        for stage in item['stages']:
                            pred = forward(item, stage, cache)
                            if not torch.isfinite(pred).all() or pred.float().norm() == 0:
                                raise RuntimeError('Invalid reference velocity')
                            rows.append(dict(panel=panel, source_id=item['row']['source_id'],
                                             dimension=item['row']['dimension'],
                                             subdimension=item['row'].get('subdimension'),
                                             stage=stage['stage'], velocity=pred))
            return rows

        def save(name, value):
            path = directory / name
            temporary = path.with_suffix('.tmp')
            torch.save(value, temporary)
            temporary.replace(path)
            artifact = dict(name=name, sha256=digest(path), bytes=path.stat().st_size)
            report['artifacts'].append(artifact)
            return artifact

        teacher_predictions = predictions()
        save('teacher_predictions.pt', teacher_predictions)
        report['teacher_calls'] += len(teacher_predictions)
        # Rebuild the Phase 6 selected student exactly as its verified replay did:
        # 31 teacher blocks plus the frozen pretrained rank-256 bridge at slot 5.
        namespace = {}
        exec(compile(request['core_source'], 'bridge_definition.py', 'exec'), namespace)
        exec(compile(request['wrapper_source'], 'bridge_block_definition.py', 'exec'), namespace)
        candidate = request['selected_candidate']
        path = Path('/phase3/phase5/fitting') / request['fit_key'] / Path(candidate['checkpoint_path']).name
        if digest(path) != candidate['checkpoint_sha256']:
            raise RuntimeError('Selected bridge checkpoint differs.')
        checkpoint = torch.load(path, weights_only=True, map_location='cpu')
        if (checkpoint['request_key'] != request['fit_key'] or checkpoint['layer'] != 5
                or checkpoint['step'] != candidate['selected_update']):
            raise RuntimeError('Bridge provenance differs.')
        bridge = namespace['ResidualBottleneckBridge'](4096, 256)
        bridge.load_state_dict(checkpoint['state_dict'])
        bridge.requires_grad_(False)
        namespace['insert_bridge'](model, 5, bridge, QwenImage21KVCache)
        del checkpoint
        model.requires_grad_(False).eval()
        gc.collect()
        torch.cuda.empty_cache()
        baseline = predictions()
        report['student_calls'] += len(baseline)
        save('candidate_predictions.pt', baseline)
        # The 10 gate prompts must reproduce the saved Phase 6 trained-bridge velocities.
        replay = request['selected_replay']
        replay_dir = Path('/phase3/phase6/verified_student_replay') / replay['request_key']
        reference = {}
        for record in request['records']:
            if not record['capture_id'].startswith('p5-validation-'):
                continue
            name = 'students-' + record['capture_id'] + '.pt'
            if digest(replay_dir / name) != replay['artifacts'][name]:
                raise RuntimeError('Phase 6 student replay checksum differs.')
            saved = torch.load(replay_dir / name, weights_only=True, map_location='cpu')
            for output in saved['outputs']:
                if output['candidate_id'] == candidate['candidate_id'] and output['variant'] == 'trained':
                    reference[(output['source_id'], output['stage'])] = output['velocity']
        gate_rows = [r for r in baseline if r['panel'] == 'gate10']
        if len(reference) != 30 or len(gate_rows) != 30:
            raise RuntimeError('Phase 6 replay coverage differs.')
        reproduction = [compare(r['velocity'], reference[(r['source_id'], r['stage'])]) for r in gate_rows]
        report['candidate_replay_max_relative_l2'] = max(r['relative_l2_error'] for r in reproduction)
        if report['candidate_replay_max_relative_l2'] > 0.001:
            raise RuntimeError('Selected student failed Phase 6 output replay')
        del reference

        class IdentityBlock(torch.nn.Module):
            def __init__(self, original):
                super().__init__()
                self.attn = SimpleNamespace(processor=original.attn.processor)

            def forward(self, hidden_states, *args, **kwargs):
                return hidden_states

        def metrics(rows):
            output = []
            for measured, base, teacher_row in zip(rows, baseline, teacher_predictions):
                identity = ('panel', 'source_id', 'stage')
                if any(measured[k] != base[k] or base[k] != teacher_row[k] for k in identity):
                    raise RuntimeError('Prediction alignment differs')
                teacher_error = compare(measured['velocity'], teacher_row['velocity'])['relative_l2_error']
                baseline_error = compare(base['velocity'], teacher_row['velocity'])['relative_l2_error']
                output.append({k: measured[k] for k in measured if k != 'velocity'} |
                              dict(baseline_displacement=compare(measured['velocity'], base['velocity'])['relative_l2_error'],
                                   teacher_relative_l2=teacher_error, baseline_teacher_relative_l2=baseline_error,
                                   teacher_error_change_percentage_points=100 * (teacher_error - baseline_error)))
            return output

        def aggregate(rows):
            fields = ('baseline_displacement', 'teacher_relative_l2', 'baseline_teacher_relative_l2',
                      'teacher_error_change_percentage_points')
            result = dict(comparisons=len(rows))
            for field in fields:
                values = sorted(r[field] for r in rows)
                def percentile(fraction):
                    position = fraction * (len(values) - 1)
                    low = int(position)
                    return values[low] + (values[min(low + 1, len(values) - 1)] - values[low]) * (position - low)
                result[field] = dict(mean=statistics.mean(values), maximum=max(values),
                                     p10=percentile(.1), p50=percentile(.5), p90=percentile(.9))
            result['worst_regressions'] = sorted(rows, key=lambda r:r['teacher_error_change_percentage_points'], reverse=True)[:10]
            return result

        report['baseline'] = aggregate(metrics(baseline))

        # ---- Capture the target block's input, output and modulation rows inside frozen B. ----
        slot = request['slot']
        fit = request['fit']
        conditioned_namespace = {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned_namespace)
        ConditionedBridge = conditioned_namespace['ConditionedBridge']
        generator = torch.Generator().manual_seed(fit['capture_seed'])
        hook_state = {'on': False}

        def hook(module, args, kwargs, output):
            if hook_state['on']:
                hidden = kwargs['hidden_states'] if 'hidden_states' in kwargs else args[0]
                if not isinstance(output, torch.Tensor) or output.shape != hidden.shape or hidden.shape[0] != 1:
                    raise RuntimeError('Unexpected block input/output layout.')
                hook_state['io'] = (hidden.detach(), output.detach(), kwargs['modulation'].detach(),
                                    kwargs.get('target_token_mask'))

        handle = model.transformer_blocks[slot].register_forward_hook(hook, with_kwargs=True)
        groups = ('prompt_tokens', 'early_image', 'middle_image', 'late_image')
        captured = {split: {g: ([], [], []) for g in groups} for split in ('train', 'validation')}
        table = []

        def collect(item, split, tokens):
            cache = QwenImage21KVCache(32)
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for stage in item['stages']:
                    hook_state['on'] = True
                    forward(item, stage, cache)
                    hook_state['on'] = False
                    hidden, output, modulation, mask = hook_state.pop('io')
                    if modulation.dim() != 2 or modulation.shape[-1] != 4 * hidden.shape[-1]:
                        raise RuntimeError('Unexpected modulation layout: ' + str(tuple(modulation.shape)))
                    real = len(table)
                    table.append(modulation[0].to(torch.bfloat16).cpu())
                    zero = real
                    if mask is not None:
                        zero = len(table)
                        table.append(modulation[-1].to(torch.bfloat16).cpu())
                    total, image = hidden.shape[1], stage['target_image_tokens']
                    spans = [(stage['stage'] + '_image', total - image, total)]
                    if total > image:
                        spans.append(('prompt_tokens', 0, total - image))
                    for group, low, high in spans:
                        count = min(tokens, high - low)
                        index = low + torch.randperm(high - low, generator=generator)[:count]
                        device_index = index.to(hidden.device)
                        rows = torch.full((count,), real, dtype=torch.long)
                        if mask is not None:
                            rows = torch.where(mask.cpu()[index], rows, torch.full_like(rows, zero))
                        entry = captured[split][group]
                        entry[0].append(hidden[0].index_select(0, device_index).to(torch.bfloat16).cpu())
                        entry[1].append(output[0].index_select(0, device_index).to(torch.bfloat16).cpu())
                        entry[2].append(rows)
                    report['student_calls'] += 1

        def load_verified(path, sha256, size=None):
            if (size is not None and path.stat().st_size != size) or digest(path) != sha256:
                raise RuntimeError('Input tensor differs: ' + str(path))
            return torch.load(path, weights_only=True, map_location='cpu')

        train_ids = set()
        counts = {'train_full': 0, 'train_early_pool': 0, 'validation': 0}
        for record in request['records']:
            item = load_verified(Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt', record['tensor_sha256'])
            split = item['row']['split']
            if item['row']['source_id'] != record['source_id'] or split not in ('train', 'validation'):
                raise RuntimeError('Capture record identity differs.')
            if split == 'train':
                train_ids.add(record['source_id'])
                counts['train_full'] += 1
            else:
                counts['validation'] += 1
            collect(item, split, fit['tokens_per_group_full'])
            del item
        fresh = request['fresh_aligned_train']
        for artifact in fresh['artifacts']:
            item = load_verified(Path(fresh['remote_directory']) / artifact['name'], artifact['sha256'], artifact['bytes'])
            if item['source_id'] != artifact['source_id'] or item['row']['split'] != 'train' or len(item['stages']) != 3:
                raise RuntimeError('Fresh aligned training provenance differs.')
            train_ids.add(artifact['source_id'])
            collect(item, 'train', fit['tokens_per_group_full'])
            counts['train_full'] += 1
            del item
        for example in request['early_pool']['examples']:
            item = load_verified(Path(example['remote_path']), example['sha256'], example['bytes'])
            if (item['capture_id'] != example['capture_id'] or item['row']['split'] != 'train'
                    or [s['stage'] for s in item['stages']] != ['early']):
                raise RuntimeError('Early pool example provenance differs.')
            collect(item, 'train', fit['tokens_per_group_early_pool'])
            counts['train_early_pool'] += 1
            del item
        handle.remove()
        validation_ids = {r['source_id'] for r in request['records'] if r['capture_id'].startswith('p5-validation-')}
        validation_ids |= {r['source_id'] for r in request['expanded_validation']['records']}
        if train_ids & validation_ids:
            raise RuntimeError('A validation prompt appears in the training set.')
        if counts != {'train_full': 60, 'train_early_pool': len(request['early_pool']['examples']), 'validation': 10}:
            raise RuntimeError('Capture counts differ: ' + json.dumps(counts))
        table = torch.stack(table).to('cuda')
        data = {split: {g: tuple(torch.cat(part).to('cuda') for part in v) for g, v in by_group.items() if v[0]}
                for split, by_group in captured.items()}
        del captured
        if set(data['train']) != set(groups) or set(data['validation']) != set(groups):
            raise RuntimeError('A token group is missing from the capture.')
        report['capture'] = {'counts': counts, 'modulation_rows': int(table.shape[0]),
                             'tokens': {split: {g: int(v[0].shape[0]) for g, v in d.items()} for split, d in data.items()}}
        (directory / 'progress.json').write_text(json.dumps(report, indent=2))
        results_volume.commit()
        print('Captured block', slot, json.dumps(report['capture']), flush=True)

        def objective(prediction, target):
            prediction, target = prediction.float(), target.float()
            direction = (torch.nn.functional.normalize(prediction, dim=-1) -
                         torch.nn.functional.normalize(target, dim=-1)).square().sum(-1)
            relative = (prediction - target).square().sum(-1) / target.square().sum(-1).clamp_min(1e-12)
            return direction + fit['magnitude_weight'] * relative

        original_block = model.transformer_blocks[slot]
        report['variants'] = {}
        for variant in request['variants']:
            # ---- Fit one bridge variant on TRAIN states only. ----
            torch.manual_seed(fit['init_seed'])
            bridge = ConditionedBridge(4096, variant['rank'], variant['conditioned']).to('cuda').float()
            optimizer = torch.optim.AdamW(bridge.parameters(), lr=fit['learning_rate'], weight_decay=fit['weight_decay'])
            sampler = torch.Generator(device='cuda').manual_seed(fit['sampling_seed'])

            def run(inputs, rows):
                with torch.autocast('cuda', dtype=torch.bfloat16):
                    return bridge(inputs.float(), table[rows].float()).float()

            def validation_score():
                with torch.no_grad():
                    values = {}
                    for g, (inputs, targets, rows) in data['validation'].items():
                        total = sum(objective(run(inputs[i:i + 2048], rows[i:i + 2048]), targets[i:i + 2048]).sum()
                                    for i in range(0, inputs.shape[0], 2048))
                        values[g] = float(total / inputs.shape[0])
                    return sum(values.values()) / len(values), values

            curve, best = [], None
            for step in range(fit['updates'] + 1):
                if step % fit['checkpoint_every'] == 0:
                    score, per_group = validation_score()
                    curve.append(dict(step=step, validation=score, groups=per_group))
                    if best is None or score < best[0]:
                        best = (score, step, {k: v.detach().cpu().clone() for k, v in bridge.state_dict().items()})
                    print(f'{variant["name"]} step {step}: validation {score:.6f}', flush=True)
                if step == fit['updates']:
                    break
                loss = 0
                for g in groups:
                    inputs, targets, rows = data['train'][g]
                    index = torch.randint(inputs.shape[0], (fit['tokens_per_group_update'],), device='cuda', generator=sampler)
                    loss = loss + objective(run(inputs[index], rows[index]), targets[index]).mean() / len(groups)
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(bridge.parameters(), fit['gradient_clip'])
                optimizer.step()
                report['optimizer_updates'] += 1
            checkpoint_path = directory / f'bridge_block_{slot:02d}_{variant["name"]}.pt'
            temporary = checkpoint_path.with_suffix('.tmp')
            torch.save({'request_key': key, 'layer': slot, 'variant': variant, 'step': best[1], 'state_dict': best[2]}, temporary)
            temporary.replace(checkpoint_path)
            artifact = dict(name=checkpoint_path.name, sha256=digest(checkpoint_path), bytes=checkpoint_path.stat().st_size)
            report['artifacts'].append(artifact)
            del bridge, optimizer
            gc.collect()
            torch.cuda.empty_cache()

            # ---- Evaluate the whole student with this bridge at the slot. ----
            selected = ConditionedBridge(4096, variant['rank'], variant['conditioned'])
            selected.load_state_dict(best[2])
            selected.requires_grad_(False)
            conditioned_namespace['insert_conditioned_bridge'](model, slot, selected)
            model.requires_grad_(False).eval()
            try:
                rows = predictions()
            finally:
                model.transformer_blocks[slot] = original_block
            report['student_calls'] += len(rows)
            records = metrics(rows)
            report['variants'][variant['name']] = dict(
                variant=variant, checkpoint=artifact, parameters=sum(v.numel() for v in best[2].values()),
                fit=dict(curve=curve, selected_step=best[1], selected_validation=best[0], identity_validation=curve[0]['validation']),
                overall=aggregate(records),
                panels={p: aggregate([r for r in records if r['panel'] == p]) for p in panels},
                stages={s: aggregate([r for r in records if r['stage'] == s]) for s in ('early', 'middle', 'late')},
                records=records)
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            print(f'{variant["name"]}: teacher-relative {report["variants"][variant["name"]]["overall"]["teacher_relative_l2"]["mean"]:.6f}', flush=True)
        report['status'] = 'passed'
        report['scope'] = 'Bridge variant fits and fixed-input velocity evaluation only; images and gates are separate.'
    except Exception as exc:
        import traceback
        report['status'] = 'failed'
        report['error'] = repr(exc)
        report['traceback'] = traceback.format_exc()
    (directory / 'summary.json').write_text(json.dumps(report, indent=2))
    results_volume.commit()
    return report
'''


SOURCES['conditioned_bridge.py'] = r'''import torch
from torch import nn
from types import SimpleNamespace


class ConditionedBridge(nn.Module):
    """Residual bottleneck bridge that can read the model's shared timestep modulation.

    The transformer computes one modulation tensor per timestep, [attn scale, attn gate, MLP scale, MLP gate],
    shared by every block. A conditioned bridge mirrors a block's MLP branch:
    h + tanh(gate) * up(gelu(down(LN(h) * (1 + scale)))), with scale and gate learned per-channel mixes of the
    shared values. Unconditioned, it is the original Phase 4 bridge: h + up(gelu(down(h))).
    """

    def __init__(self, hidden_dim=4096, rank=256, conditioned=True, eps=1e-6):
        super().__init__()
        self.conditioned = conditioned
        self.down = nn.Linear(hidden_dim, rank, bias=False)
        self.up = nn.Linear(rank, hidden_dim, bias=False)
        nn.init.zeros_(self.up.weight)
        if conditioned:
            self.norm = nn.LayerNorm(hidden_dim, elementwise_affine=False, eps=eps)
            # Start from the block's own MLP modulation (weight 1 on the MLP terms, 0 on the attention terms).
            mix = torch.zeros(2, hidden_dim)
            mix[1] = 1
            self.scale_mix = nn.Parameter(mix.clone())
            self.gate_mix = nn.Parameter(mix.clone())
            self.scale_bias = nn.Parameter(torch.zeros(hidden_dim))
            self.gate_bias = nn.Parameter(torch.zeros(hidden_dim))

    def forward(self, hidden_states, modulation_rows=None):
        if not self.conditioned:
            return hidden_states + self.up(nn.functional.gelu(self.down(hidden_states)))
        scale1, gate1, scale2, gate2 = modulation_rows.chunk(4, dim=-1)
        scale = self.scale_mix[0] * scale1 + self.scale_mix[1] * scale2 + self.scale_bias
        gate = self.gate_mix[0] * gate1 + self.gate_mix[1] * gate2 + self.gate_bias
        update = self.up(nn.functional.gelu(self.down(self.norm(hidden_states) * (1 + scale))))
        return hidden_states + torch.tanh(gate) * update


def modulation_rows(modulation, target_token_mask):
    """Per-token modulation rows, selected exactly as the transformer block does."""
    if target_token_mask is None:
        return modulation.unsqueeze(1)
    real, zero = modulation[:-1].unsqueeze(1), modulation[-1:].unsqueeze(0)
    return torch.where(target_token_mask.view(1, -1, 1), real, zero)


class ConditionedBridgeBlock(nn.Module):
    def __init__(self, bridge, processor):
        super().__init__()
        self.bridge = bridge
        # Parent forward inspects this processor to choose mask construction; no attention is performed here.
        self.attn = SimpleNamespace(processor=processor)

    def forward(self, hidden_states, modulation, rotary_emb=None, attention_mask=None, target_token_mask=None,
                layer_cache=None, kv_cache_mode=None, cache_write_slice=None, segments=None, key_valid=None):
        if kv_cache_mode not in (None, 'extract', 'cached'):
            raise ValueError('Unknown KV cache mode.')
        rows = modulation_rows(modulation, target_token_mask) if self.bridge.conditioned else None
        return self.bridge(hidden_states, rows)


def insert_conditioned_bridge(transformer, layer, bridge):
    """Replace one slot; callers must use fresh KV caches afterwards."""
    old = transformer.transformer_blocks[layer]
    reference = next(transformer.parameters())
    bridge = bridge.to(device=reference.device, dtype=reference.dtype)
    wrapper = ConditionedBridgeBlock(bridge, old.attn.processor)
    wrapper.train(transformer.training)
    transformer.transformer_blocks[layer] = wrapper
    return old
'''


"""Fit several bridge architectures for one block inside candidate B and evaluate each whole student; reuse completed results."""
import argparse
import hashlib
import json
import sys
from pathlib import Path


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--slot', type=int, default=13)
    parser.add_argument('--old-account', action='store_true')
    parser.add_argument('--dry-run', action='store_true', help='Write the request locally without scheduling GPU work.')
    args = parser.parse_args(LAUNCH_ARGS)
    if args.old_account:
        parser.error('Old account is download-only; GPU execution must use the new account.')
    root = ROOT
    protocol_path = root / f'data/phase7/block{args.slot}_bridge_v2/protocol.json'
    protocol = json.loads(protocol_path.read_bytes())
    scan_b = json.loads(next((root / 'data/phase7/conditional_importance_b').glob('*/request.json')).read_bytes())['request']
    scan_a = json.loads((root / 'data/phase7/conditional_importance/7454da6a5c1dc33af23dd46833e698f9b759d14985ca96340bc3d40cd47fb4f5/request.json').read_bytes())['request']
    if args.slot == 5 or not 0 < args.slot < 31:
        parser.error('Slot must be a surviving interior block.')
    request = {k: v for k, v in scan_b.items() if k not in ('worker_source', 'eligible_original_block_ids', 'selection', 'scope', 'version')}
    source = SOURCES['bridge_variants_worker.py']
    request.update(
        version='phase7-bridge-variants-v1', worker_source=source, slot=args.slot,
        conditioned_bridge_source=SOURCES['conditioned_bridge.py'],
        variants=[dict(name='plain_r1024', rank=1024, conditioned=False), dict(name='cond_r256', rank=256, conditioned=True),
                  dict(name='cond_r1024', rank=1024, conditioned=True)],
        protocol_sha256=hashlib.sha256(protocol_path.read_bytes()).hexdigest(),
        fresh_aligned_train=scan_a['fresh_aligned_train'], early_pool=scan_a['early_pool'],
        fit=dict(init_seed=20261017, capture_seed=20261018, sampling_seed=20261019, learning_rate=1e-3,
                 weight_decay=0.01, gradient_clip=1.0, magnitude_weight=0.1, updates=1000, checkpoint_every=100,
                 tokens_per_group_full=256, tokens_per_group_early_pool=64, tokens_per_group_update=256),
        selection='Minimum validation hidden-state objective at saved checkpoints including identity start',
        scope=f'Bridge variants for block {args.slot} inside frozen B; TRAIN-only fitting; whole-student velocity on scan inputs; no healing, images or latency')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = protocol_path.parent / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2)+'\n', encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase7/bridge_variants/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        try:
            b''.join(results_volume.read_file(remote + '/started.json'))
        except FileNotFoundError:
            pass
        else:
            raise RuntimeError('Run already started; inspect its progress. No duplicate GPU scheduled.')
        app = modal.App('nihonga-phase7-bridge-variants')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(source, 'bridge_variants_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8,
                              memory=98304, max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['fit_bridge_variants'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2)+'\n')
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2)+'\n', encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'candidate_replay_max_relative_l2')}), flush=True)
    for name, v in (report.get('variants') or {}).items():
        print(name, 'step', v['fit']['selected_step'], 'teacher-relative', v['overall']['teacher_relative_l2']['mean'], flush=True)
    if report.get('status') != 'passed':
        raise SystemExit(1)

summary = reuse_or_launch(RUN, main)

show_variants(summary)

### What the results mean

Capacity alone does nothing (plain r1024: 7.35%); conditioning helps (cond r256: 7.02%; **cond r1024: 6.77%** overall, 6.68% wider). Curves were still falling at 1000 updates.

## 8b. Images of the bridge variants

The three variants on the same 30 prompts and seeds (teacher and B reused).

In [ ]:
# Images of the bridge variants: reuses the saved run; launches GPU work only when its result is missing and ALLOW_GPU is True.

RUN = 'data/phase7/bridge_variant_images/011adc43b3545ae0f974fe1141959be08e20214fd3046f35e45ec6982b40442f'

LAUNCH_ARGS = []

SOURCES['bridge_variant_images_worker.py'] = r'''def generate_variant_images(request, key):
    """Matched 40-step images for block-13 bridge variants inside B (teacher and B images are reused from the
    earlier run). Same prompt, CPU noise seed, size, scheduler and 40 steps. Qualitative screen only."""
    import gc
    import hashlib
    import inspect
    import json
    import os
    import time
    from pathlib import Path

    import torch
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache, QwenImage21KVLayerCache

    directory = Path('/phase3/phase7/bridge_variant_images') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete image attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def table_hash(value):
        return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

    variants = [v['name'] for v in request['variants']]
    report = {'request_key': key, 'status': 'running', 'rows': [], 'artifacts': [],
              'calls': {v: 0 for v in variants}, 'optimizer_updates': 0}
    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('Expected H100.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
                                                   dtype=torch.bfloat16, local_files_only=True).to('cuda')
        pipe.set_progress_bar_config(disable=True)
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Transformer source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--')) /
                       'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        # Derived position/time tables: use the audited values on every host (as in steps 7-11).
        derived = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
        derived['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
        expected = request['expected_derived_tables']
        mismatched = sorted(n for n, v in derived.items() if table_hash(v) != expected[n])
        if mismatched:
            canonical = torch.load(Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt'),
                                   weights_only=True, map_location='cpu')
            if any(table_hash(canonical[n]) != expected[n] for n in expected):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [canonical[f'pos_embed.freqs.{a}'].to(device=v.device, dtype=v.dtype) for a, v in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(canonical['time_text_embed.time_proj.freqs'])
            if any(table_hash(v) != expected[n] for n, v in
                   list({f'pos_embed.freqs.{a}': v for a, v in enumerate(model.pos_embed.freqs)}.items()) +
                   [('time_text_embed.time_proj.freqs', model.time_text_embed.time_proj.freqs)]):
                raise RuntimeError('Derived tables still differ after loading the audited copy.')
            report['canonical_tables_applied'] = mismatched

        def generate(row, variant):
            path = directory / (row['source_id'] + '-' + variant + '.png')
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'],
                             num_inference_steps=40, true_cfg_scale=teacher['true_cfg_scale'],
                             generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=True, output_type='pil').images[0]
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            temporary = path.with_suffix('.tmp')
            image.save(temporary, format='PNG')
            temporary.replace(path)
            results_volume.commit()
            report['calls'][variant] += 1
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            return {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size, 'seconds': elapsed}

        namespace = {}
        exec(compile(request['core_source'], 'bridge_definition.py', 'exec'), namespace)
        exec(compile(request['wrapper_source'], 'bridge_block_definition.py', 'exec'), namespace)
        conditioned = {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned)

        def load_checkpoint(spec):
            path = Path(spec['remote_path'])
            if digest(path) != spec['sha256']:
                raise RuntimeError('Bridge checkpoint differs: ' + spec['remote_path'])
            return torch.load(path, weights_only=True, map_location='cpu')

        block5 = request['block5_bridge']
        checkpoint = load_checkpoint(block5)
        if checkpoint['layer'] != 5 or checkpoint['step'] != block5['step'] or checkpoint['request_key'] != block5['request_key']:
            raise RuntimeError('Block-5 bridge provenance differs.')
        bridge = namespace['ResidualBottleneckBridge'](4096, 256)
        bridge.load_state_dict(checkpoint['state_dict'])
        bridge.requires_grad_(False)
        namespace['insert_bridge'](model, 5, bridge, QwenImage21KVCache)
        original13 = model.transformer_blocks[13]
        outputs = {v: {} for v in variants}
        for variant in request['variants']:
            checkpoint = load_checkpoint(variant)
            if checkpoint['layer'] != 13 or checkpoint['variant']['name'] != variant['name']:
                raise RuntimeError('Variant provenance differs.')
            spec = checkpoint['variant']
            bridge13 = conditioned['ConditionedBridge'](4096, spec['rank'], spec['conditioned'])
            bridge13.load_state_dict(checkpoint['state_dict'])
            bridge13.requires_grad_(False)
            conditioned['insert_conditioned_bridge'](model, 13, bridge13)
            model.eval()
            try:
                for row in request['rows']:
                    outputs[variant['name']][row['source_id']] = generate(row, variant['name'])
            finally:
                model.transformer_blocks[13] = original13
        for row in request['rows']:
            report['rows'].append({**row, **{v: outputs[v][row['source_id']] for v in variants}})
            report['artifacts'].extend(outputs[v][row['source_id']] for v in variants)
        report['status'] = 'passed'
    except Exception as exc:
        report['status'] = 'failed'
        report['error'] = type(exc).__name__ + ': ' + str(exc)
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=True) + '\n')
    results_volume.commit()
    return report
'''


SOURCES['conditioned_bridge.py'] = r'''import torch
from torch import nn
from types import SimpleNamespace


class ConditionedBridge(nn.Module):
    """Residual bottleneck bridge that can read the model's shared timestep modulation.

    The transformer computes one modulation tensor per timestep, [attn scale, attn gate, MLP scale, MLP gate],
    shared by every block. A conditioned bridge mirrors a block's MLP branch:
    h + tanh(gate) * up(gelu(down(LN(h) * (1 + scale)))), with scale and gate learned per-channel mixes of the
    shared values. Unconditioned, it is the original Phase 4 bridge: h + up(gelu(down(h))).
    """

    def __init__(self, hidden_dim=4096, rank=256, conditioned=True, eps=1e-6):
        super().__init__()
        self.conditioned = conditioned
        self.down = nn.Linear(hidden_dim, rank, bias=False)
        self.up = nn.Linear(rank, hidden_dim, bias=False)
        nn.init.zeros_(self.up.weight)
        if conditioned:
            self.norm = nn.LayerNorm(hidden_dim, elementwise_affine=False, eps=eps)
            # Start from the block's own MLP modulation (weight 1 on the MLP terms, 0 on the attention terms).
            mix = torch.zeros(2, hidden_dim)
            mix[1] = 1
            self.scale_mix = nn.Parameter(mix.clone())
            self.gate_mix = nn.Parameter(mix.clone())
            self.scale_bias = nn.Parameter(torch.zeros(hidden_dim))
            self.gate_bias = nn.Parameter(torch.zeros(hidden_dim))

    def forward(self, hidden_states, modulation_rows=None):
        if not self.conditioned:
            return hidden_states + self.up(nn.functional.gelu(self.down(hidden_states)))
        scale1, gate1, scale2, gate2 = modulation_rows.chunk(4, dim=-1)
        scale = self.scale_mix[0] * scale1 + self.scale_mix[1] * scale2 + self.scale_bias
        gate = self.gate_mix[0] * gate1 + self.gate_mix[1] * gate2 + self.gate_bias
        update = self.up(nn.functional.gelu(self.down(self.norm(hidden_states) * (1 + scale))))
        return hidden_states + torch.tanh(gate) * update


def modulation_rows(modulation, target_token_mask):
    """Per-token modulation rows, selected exactly as the transformer block does."""
    if target_token_mask is None:
        return modulation.unsqueeze(1)
    real, zero = modulation[:-1].unsqueeze(1), modulation[-1:].unsqueeze(0)
    return torch.where(target_token_mask.view(1, -1, 1), real, zero)


class ConditionedBridgeBlock(nn.Module):
    def __init__(self, bridge, processor):
        super().__init__()
        self.bridge = bridge
        # Parent forward inspects this processor to choose mask construction; no attention is performed here.
        self.attn = SimpleNamespace(processor=processor)

    def forward(self, hidden_states, modulation, rotary_emb=None, attention_mask=None, target_token_mask=None,
                layer_cache=None, kv_cache_mode=None, cache_write_slice=None, segments=None, key_valid=None):
        if kv_cache_mode not in (None, 'extract', 'cached'):
            raise ValueError('Unknown KV cache mode.')
        rows = modulation_rows(modulation, target_token_mask) if self.bridge.conditioned else None
        return self.bridge(hidden_states, rows)


def insert_conditioned_bridge(transformer, layer, bridge):
    """Replace one slot; callers must use fresh KV caches afterwards."""
    old = transformer.transformer_blocks[layer]
    reference = next(transformer.parameters())
    bridge = bridge.to(device=reference.device, dtype=reference.dtype)
    wrapper = ConditionedBridgeBlock(bridge, old.attn.processor)
    wrapper.train(transformer.training)
    transformer.transformer_blocks[layer] = wrapper
    return old
'''


"""Render block-13 bridge variants inside B on the 30 earlier Phase 7 image prompts (teacher and B reused); qualitative only."""
import argparse
import hashlib
import json
import sys
from pathlib import Path


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--dry-run', action='store_true')
    parser.add_argument('--variants-dir', default='block13_bridge_v2')
    parser.add_argument('--order', nargs='*', help='Variant names in rendering order (default: as fitted).')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    recovered = root / 'data/recovered/modal'
    images_key = 'dda8c184e657dfb604c35fb261d83371c108c39fac033a0f57db66f7686c30ad'
    base = json.loads((recovered / 'phase7/candidate_images' / images_key / 'request.json').read_bytes())['request']
    scan_b = json.loads(next((root / 'data/phase7/conditional_importance_b').glob('*/request.json')).read_bytes())['request']
    fit = next(r for r in (json.loads(p.read_bytes()) for p in (root / 'data/phase7' / args.variants_dir).glob('*/summary.json')) if r.get('status') == 'passed')
    if fit.get('status') != 'passed':
        raise RuntimeError('Bridge variants run did not complete.')
    selected = scan_b['selected_candidate']
    request = {k: base[k] for k in ('teacher', 'transformer_source_sha256', 'runtime_environment', 'expected_derived_tables',
                                    'canonical_tables_key', 'core_source', 'wrapper_source', 'rows')}
    request.update(
        version='phase7-bridge-variant-images-v1',
        worker_source=SOURCES['bridge_variant_images_worker.py'],
        conditioned_bridge_source=SOURCES['conditioned_bridge.py'],
        execution_account='new',
        block5_bridge=dict(remote_path=f"/phase3/phase5/fitting/{scan_b['fit_key']}/{Path(selected['checkpoint_path']).name}",
                           sha256=selected['checkpoint_sha256'], step=selected['selected_update'], request_key=scan_b['fit_key']),
        variants=[dict(name=name, remote_path=f"/phase3/phase7/{'bridge_variants' if args.variants_dir == 'block13_bridge_v2' else 'bridge_variants_v3'}/{fit['request_key']}/{v['checkpoint']['name']}",
                       sha256=v['checkpoint']['sha256']) for name, v in sorted(fit['variants'].items(), key=lambda kv: (args.order or list(fit['variants'])).index(kv[0]))],
        scope='Qualitative matched-seed screen of block-13 bridge variants inside B; teacher and B images reused from the earlier run')
    from experiment_runtime import pin_reference_runtime
    pin_reference_runtime(request, recovered)
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase7/bridge_variant_images' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2)+'\n', encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase7/bridge_variant_images/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        try:
            b''.join(results_volume.read_file(remote + '/started.json'))
        except FileNotFoundError:
            pass
        else:
            raise RuntimeError('Run already started; inspect its progress. No duplicate GPU scheduled.')
        app = modal.App('nihonga-phase7-bridge-variant-images')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'bridge_variant_images_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8,
                              memory=98304, max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['generate_variant_images'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2)+'\n')
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2)+'\n', encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'calls')}), flush=True)
    if report.get('status') != 'passed':
        raise SystemExit(1)

summary = reuse_or_launch(RUN, main)

show_images(RUN + '/sheets', ['P14.jpg', 'P28.jpg'])

### What the results mean

cond r1024 was worse than B on 8 prompts, similar on 20, better on 2 (first bridge: 15 worse). Conditioning fixes the hair (P14), spells "Happy Birthday Emma" on this seed, keeps the alarm clock and stops duplicating a pot. It does not recover fine detail (lace, circuit board, peacock eye), the bent spoon or melting clocks, and "The Rusty Anchor" stays misspelled.

## 9. Longer training and local token mixing

cond r1024 trained 3000 updates, and the same bridge plus a zero-initialised depthwise 3x3 convolution over the image-token grid (each token sees its neighbours; the grid comes from each image's own `img_shapes`).

In [ ]:
# Longer training and local token mixing: reuses the saved run; launches GPU work only when its result is missing and ALLOW_GPU is True.

RUN = 'data/phase7/block13_bridge_v3/e75e274953b560d2d9143f8f6c6e83b73309a4ba88d35e233f39608b2b50239e'

LAUNCH_ARGS = ['--slot', '13']

SOURCES['bridge_variants_v3_worker.py'] = r'''def fit_bridge_variants(request, key):
    """Fit a learned bridge for one block inside the frozen block-5 student (candidate B).

    Reuses the verified teacher contract and exact B replay, captures the target
    block's input/output inside B, fits only the new bridge on TRAIN states, then
    evaluates the whole candidate against the teacher on the scan inputs.
    """
    import hashlib
    import importlib.metadata
    import inspect
    import json
    import os
    import socket
    import uuid
    from pathlib import Path
    import torch
    from torch.nn.attention import sdpa_kernel, SDPBackend
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache

    directory = Path('/phase3/phase7/bridge_variants_v3') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete healing attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()
    report = {'request_key': key, 'status': 'failed', 'mode': request['mode'], 'teacher_calls': 0,
              'student_calls': 0, 'optimizer_updates': 0, 'controls': [], 'artifacts': [],
              'process': {'pid': os.getpid(), 'host': socket.gethostname(), 'nonce': uuid.uuid4().hex,
                          'modal_task_id': os.environ.get('MODAL_TASK_ID')}}

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def gpu(value):
        if isinstance(value, torch.Tensor):
            return value.to('cuda')
        if isinstance(value, dict):
            return {k: gpu(v) for k, v in value.items()}
        if isinstance(value, (list, tuple)):
            return type(value)(gpu(v) for v in value)
        return value

    def compare(x, y):
        a, b = x.float(), y.float()
        if not torch.isfinite(a).all() or not torch.isfinite(b).all():
            raise RuntimeError('Nonfinite reference tensor.')
        d = a - b
        return {'relative_l2_error': float(d.norm()/b.norm().clamp_min(1e-12)),
                'max_abs_error': float(d.abs().max()), 'exact_equal': bool(torch.equal(x, y))}

    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('GPU differs.')
        report['packages'] = {p: importlib.metadata.version(p) for p in ('torch', 'diffusers', 'transformers', 'accelerate', 'pillow')}
        report['torch_cuda'] = torch.version.cuda
        report['cudnn_version'] = torch.backends.cudnn.version()
        report['settings'] = request['arithmetic_contract']
        if request['mode'] != 'verify':
            raise RuntimeError('Healing requires the verified teacher reference.')
        reference_dir = Path('/phase3/phase6/deterministic_reference') / request['reference_key']
        reference_report_path = reference_dir / 'summary.json'
        if hashlib.sha256(json.dumps(json.loads(reference_report_path.read_text()), sort_keys=True).encode()).hexdigest() != request['reference_report_semantic_sha256']:
            raise RuntimeError('Generation report differs.')
        reference_report = json.loads(reference_report_path.read_text())
        if reference_report['status'] != 'reference_generated':
            raise RuntimeError('Reference generation did not complete.')
        if reference_report['settings'] != request['arithmetic_contract']:
            raise RuntimeError('Arithmetic contract differs from generation.')
        report['reference_process'] = reference_report['process']
        if reference_report['process']['nonce'] == report['process']['nonce']:
            raise RuntimeError('Independent process requirement failed.')
        if reference_report['packages'] != report['packages'] or reference_report['torch_cuda'] != report['torch_cuda'] or reference_report['cudnn_version'] != report['cudnn_version']:
            raise RuntimeError('Numerical runtime differs across generation and verification.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
            dtype=torch.bfloat16, local_files_only=True).to('cuda')
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Model source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--'))
                       / 'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        state = {}
        for kind, values in (('parameter', model.named_parameters()), ('buffer', model.named_buffers())):
            for name, value in values:
                cpu = value.detach().cpu().contiguous()
                state[name] = {'kind': kind, 'dtype': str(value.dtype), 'shape': list(value.shape),
                    'stride': list(value.stride()), 'sha256': hashlib.sha256(memoryview(cpu.view(torch.uint8).numpy())).hexdigest()}
        for axis, value in enumerate(model.pos_embed.freqs):
            name = f'pos_embed.freqs.{axis}'
            cpu = value.detach().cpu().contiguous()
            state[name] = {'kind': 'unregistered_rope_table', 'dtype': str(value.dtype), 'shape': list(value.shape),
                'stride': list(value.stride()), 'sha256': hashlib.sha256(memoryview(cpu.view(torch.uint8).numpy())).hexdigest()}
        expected = request['expected_state']
        derived_names = sorted(n for n in expected if n.startswith('pos_embed.freqs.') or n == 'time_text_embed.time_proj.freqs')

        def derived_tensors():
            values = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
            values['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
            return values

        def table_hash(value):
            return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

        canonical_path = Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt')
        import time
        mismatched = [n for n in derived_names if state.get(n) != expected.get(n)]
        if not mismatched:
            if not canonical_path.exists():
                canonical_path.parent.mkdir(parents=True, exist_ok=True)
                temporary = canonical_path.with_suffix('.' + report['process']['nonce'] + '.tmp')
                torch.save({n: v.detach().cpu().contiguous() for n, v in derived_tensors().items()}, temporary)
                temporary.replace(canonical_path)
                results_volume.commit()
                report['canonical_tables_saved'] = True
        else:
            deadline = time.time() + 900
            while not canonical_path.exists():
                if time.time() > deadline:
                    raise RuntimeError('Derived tables differ on this host and no canonical copy became available.')
                time.sleep(20)
                results_volume.reload()
            saved = torch.load(canonical_path, weights_only=True, map_location='cpu')
            if sorted(saved) != derived_names or any(table_hash(saved[n]) != expected[n]['sha256'] for n in derived_names):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [saved[f'pos_embed.freqs.{axis}'].to(device=value.device, dtype=value.dtype)
                        for axis, value in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(saved['time_text_embed.time_proj.freqs'])
            for n, value in derived_tensors().items():
                state[n] = {**state[n], 'sha256': table_hash(value)}
            report['canonical_tables_applied'] = mismatched
        if state != request['expected_state']:
            expected = request['expected_state']
            different_names = sorted(name for name in expected.keys() & state.keys()
                                     if state[name] != expected[name])
            report['state_mismatch'] = {
                'actual_count': len(state), 'expected_count': len(expected),
                'missing_names': sorted(set(expected) - set(state))[:12],
                'extra_names': sorted(set(state) - set(expected))[:12],
                'different': [{'name': name, 'actual': state[name], 'expected': expected[name]}
                              for name in different_names][:12]}
            allowed_derived_tables = {'pos_embed.freqs.1', 'pos_embed.freqs.2', 'time_text_embed.time_proj.freqs'}
            if set(state) != set(expected) or not set(different_names).issubset(allowed_derived_tables):
                raise RuntimeError('Loaded teacher learned state differs from audited reference.')
            report['derived_table_variation_requires_output_replay'] = different_names
        report['state_sha256'] = hashlib.sha256(json.dumps(state, sort_keys=True).encode()).hexdigest()
        torch.cuda.synchronize()
        argument_order = ('hidden_states', 'timestep', 'encoder_hidden_states', 'encoder_hidden_states_mask',
                          'img_shapes', 'img_mask', 'attention_kwargs', 'kv_cache_mode', 'return_dict')
        # Every intact-teacher control must pass before any block is touched.
        for record in request['records']:
            path = Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt'
            if digest(path) != record['tensor_sha256']:
                raise RuntimeError('Legacy input bundle differs.')
            bundle = torch.load(path, weights_only=True, map_location='cpu')
            outputs, controls = [], []
            cache = QwenImage21KVCache(32)
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for stage in bundle['stages']:
                    full = {**bundle['shared_transformer_kwargs'], **stage['kwargs']}
                    kwargs = gpu({field: full[field] for field in argument_order})
                    kwargs['kv_cache'] = cache
                    pred = model(**kwargs)[0][:, -stage['target_image_tokens']:].detach().cpu().contiguous()
                    report['teacher_calls'] += 1
                    legacy = compare(pred, stage['teacher_velocity'])
                    entry = {'source_id': record['source_id'], 'split': bundle['row']['split'],
                             'stage': stage['stage'], 'legacy_difference': legacy}
                    outputs.append({'stage': stage['stage'], 'step_index': stage['step_index'],
                                    'scheduler_sigma': stage['scheduler_sigma'], 'velocity': pred})
                    controls.append(entry)
            filename = record['capture_id'] + '.pt'
            artifact = next(a for a in reference_report['artifacts'] if a['name'] == filename)
            if digest(reference_dir / filename) != artifact['sha256']:
                raise RuntimeError('Reference output checksum differs.')
            reference_bundle = torch.load(reference_dir / filename, weights_only=True, map_location='cpu')
            if reference_bundle['source_id'] != record['source_id'] or reference_bundle['input_sha256'] != record['tensor_sha256']:
                raise RuntimeError('Reference input provenance differs.')
            reference_outputs = {v['stage']: v['velocity'] for v in reference_bundle['outputs']}
            for entry, measured in zip(controls, outputs):
                entry['reference_difference'] = compare(measured['velocity'], reference_outputs[entry['stage']])
                entry['passed'] = entry['reference_difference']['relative_l2_error'] <= 0.001
            output_path = directory / filename
            temporary = output_path.with_suffix('.tmp')
            torch.save({'request_key': key, 'source_id': record['source_id'], 'input_sha256': record['tensor_sha256'],
                        'outputs': outputs, 'controls': controls}, temporary)
            temporary.replace(output_path)
            report['controls'].extend(controls)
            report['artifacts'].append({'name': filename, 'sha256': digest(output_path), 'bytes': output_path.stat().st_size})
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            if not all(c['passed'] for c in controls):
                raise RuntimeError('Independent-process reference replay exceeds tolerance.')
        if report['teacher_calls'] != 90:
            raise RuntimeError('Reference attempt did not cover all 90 saved stage inputs.')
        report['legacy_max_relative_l2'] = max(c['legacy_difference']['relative_l2_error'] for c in report['controls'])
        report['max_reference_relative_l2'] = max(c['reference_difference']['relative_l2_error'] for c in report['controls'])
        report['exact_equal_controls'] = sum(c['reference_difference']['exact_equal'] for c in report['controls'])
        if not all(c['passed'] for c in report['controls']):
            raise RuntimeError('Teacher controls did not pass.')
        report['status'] = 'teacher_verified'
        (directory / 'teacher_control_report.json').write_text(json.dumps(report, indent=2))
        results_volume.commit()
        print('All 90 intact-teacher controls passed; assembling frozen 31-block candidate B.', flush=True)

        import gc
        import time
        import statistics
        from types import SimpleNamespace
        
        # Inputs are independently identified; 10 gate and 50 wider development prompts.
        panels = {'gate10': [], 'wider50': []}
        for panel, records in (('gate10', [r for r in request['records'] if r['capture_id'].startswith('p5-validation-')]),
                               ('wider50', request['expanded_validation']['records'])):
            for record in records:
                path = Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt'
                if digest(path) != record['tensor_sha256']:
                    raise RuntimeError('Fixed input hash differs')
                item = torch.load(path, weights_only=True, map_location='cpu')
                if item['row']['source_id'] != record['source_id'] or item['row']['split'] != 'validation':
                    raise RuntimeError('Validation input identity differs')
                panels[panel].append(item)
        if [len(panels[p]) for p in ('gate10', 'wider50')] != [10, 50]:
            raise RuntimeError('Saved validation panel sizes differ')

        def forward(item, stage, cache):
            full = {**item['shared_transformer_kwargs'], **stage['kwargs']}
            kwargs = gpu({field: full[field] for field in argument_order})
            kwargs['kv_cache'] = cache
            return model(**kwargs)[0][:, -stage['target_image_tokens']:].detach().cpu().contiguous()

        def predictions():
            rows = []
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for panel, items in panels.items():
                    for item in items:
                        cache = QwenImage21KVCache(32)
                        for stage in item['stages']:
                            pred = forward(item, stage, cache)
                            if not torch.isfinite(pred).all() or pred.float().norm() == 0:
                                raise RuntimeError('Invalid reference velocity')
                            rows.append(dict(panel=panel, source_id=item['row']['source_id'],
                                             dimension=item['row']['dimension'],
                                             subdimension=item['row'].get('subdimension'),
                                             stage=stage['stage'], velocity=pred))
            return rows

        def save(name, value):
            path = directory / name
            temporary = path.with_suffix('.tmp')
            torch.save(value, temporary)
            temporary.replace(path)
            artifact = dict(name=name, sha256=digest(path), bytes=path.stat().st_size)
            report['artifacts'].append(artifact)
            return artifact

        teacher_predictions = predictions()
        save('teacher_predictions.pt', teacher_predictions)
        report['teacher_calls'] += len(teacher_predictions)
        # Rebuild the Phase 6 selected student exactly as its verified replay did:
        # 31 teacher blocks plus the frozen pretrained rank-256 bridge at slot 5.
        namespace = {}
        exec(compile(request['core_source'], 'bridge_definition.py', 'exec'), namespace)
        exec(compile(request['wrapper_source'], 'bridge_block_definition.py', 'exec'), namespace)
        candidate = request['selected_candidate']
        path = Path('/phase3/phase5/fitting') / request['fit_key'] / Path(candidate['checkpoint_path']).name
        if digest(path) != candidate['checkpoint_sha256']:
            raise RuntimeError('Selected bridge checkpoint differs.')
        checkpoint = torch.load(path, weights_only=True, map_location='cpu')
        if (checkpoint['request_key'] != request['fit_key'] or checkpoint['layer'] != 5
                or checkpoint['step'] != candidate['selected_update']):
            raise RuntimeError('Bridge provenance differs.')
        bridge = namespace['ResidualBottleneckBridge'](4096, 256)
        bridge.load_state_dict(checkpoint['state_dict'])
        bridge.requires_grad_(False)
        namespace['insert_bridge'](model, 5, bridge, QwenImage21KVCache)
        del checkpoint
        model.requires_grad_(False).eval()
        gc.collect()
        torch.cuda.empty_cache()
        baseline = predictions()
        report['student_calls'] += len(baseline)
        save('candidate_predictions.pt', baseline)
        # The 10 gate prompts must reproduce the saved Phase 6 trained-bridge velocities.
        replay = request['selected_replay']
        replay_dir = Path('/phase3/phase6/verified_student_replay') / replay['request_key']
        reference = {}
        for record in request['records']:
            if not record['capture_id'].startswith('p5-validation-'):
                continue
            name = 'students-' + record['capture_id'] + '.pt'
            if digest(replay_dir / name) != replay['artifacts'][name]:
                raise RuntimeError('Phase 6 student replay checksum differs.')
            saved = torch.load(replay_dir / name, weights_only=True, map_location='cpu')
            for output in saved['outputs']:
                if output['candidate_id'] == candidate['candidate_id'] and output['variant'] == 'trained':
                    reference[(output['source_id'], output['stage'])] = output['velocity']
        gate_rows = [r for r in baseline if r['panel'] == 'gate10']
        if len(reference) != 30 or len(gate_rows) != 30:
            raise RuntimeError('Phase 6 replay coverage differs.')
        reproduction = [compare(r['velocity'], reference[(r['source_id'], r['stage'])]) for r in gate_rows]
        report['candidate_replay_max_relative_l2'] = max(r['relative_l2_error'] for r in reproduction)
        if report['candidate_replay_max_relative_l2'] > 0.001:
            raise RuntimeError('Selected student failed Phase 6 output replay')
        del reference

        class IdentityBlock(torch.nn.Module):
            def __init__(self, original):
                super().__init__()
                self.attn = SimpleNamespace(processor=original.attn.processor)

            def forward(self, hidden_states, *args, **kwargs):
                return hidden_states

        def metrics(rows):
            output = []
            for measured, base, teacher_row in zip(rows, baseline, teacher_predictions):
                identity = ('panel', 'source_id', 'stage')
                if any(measured[k] != base[k] or base[k] != teacher_row[k] for k in identity):
                    raise RuntimeError('Prediction alignment differs')
                teacher_error = compare(measured['velocity'], teacher_row['velocity'])['relative_l2_error']
                baseline_error = compare(base['velocity'], teacher_row['velocity'])['relative_l2_error']
                output.append({k: measured[k] for k in measured if k != 'velocity'} |
                              dict(baseline_displacement=compare(measured['velocity'], base['velocity'])['relative_l2_error'],
                                   teacher_relative_l2=teacher_error, baseline_teacher_relative_l2=baseline_error,
                                   teacher_error_change_percentage_points=100 * (teacher_error - baseline_error)))
            return output

        def aggregate(rows):
            fields = ('baseline_displacement', 'teacher_relative_l2', 'baseline_teacher_relative_l2',
                      'teacher_error_change_percentage_points')
            result = dict(comparisons=len(rows))
            for field in fields:
                values = sorted(r[field] for r in rows)
                def percentile(fraction):
                    position = fraction * (len(values) - 1)
                    low = int(position)
                    return values[low] + (values[min(low + 1, len(values) - 1)] - values[low]) * (position - low)
                result[field] = dict(mean=statistics.mean(values), maximum=max(values),
                                     p10=percentile(.1), p50=percentile(.5), p90=percentile(.9))
            result['worst_regressions'] = sorted(rows, key=lambda r:r['teacher_error_change_percentage_points'], reverse=True)[:10]
            return result

        report['baseline'] = aggregate(metrics(baseline))

        # ---- Capture the target block's input, output and modulation rows inside frozen B. ----
        slot = request['slot']
        fit = request['fit']
        conditioned_namespace = {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned_namespace)
        ConditionedBridge = conditioned_namespace['ConditionedBridge']
        generator = torch.Generator().manual_seed(fit['capture_seed'])
        hook_state = {'on': False}

        def hook(module, args, kwargs, output):
            if hook_state['on']:
                hidden = kwargs['hidden_states'] if 'hidden_states' in kwargs else args[0]
                if not isinstance(output, torch.Tensor) or output.shape != hidden.shape or hidden.shape[0] != 1:
                    raise RuntimeError('Unexpected block input/output layout.')
                hook_state['io'] = (hidden.detach(), output.detach(), kwargs['modulation'].detach(),
                                    kwargs.get('target_token_mask'))

        handle = model.transformer_blocks[slot].register_forward_hook(hook, with_kwargs=True)
        groups = ('prompt_tokens', 'early_image', 'middle_image', 'late_image')
        captured = {split: {g: ([], [], [], []) for g in groups} for split in ('train', 'validation')}
        offsets_y = torch.tensor([-1, -1, -1, 0, 0, 0, 1, 1, 1])
        offsets_x = torch.tensor([-1, 0, 1, -1, 0, 1, -1, 0, 1])
        table = []

        def collect(item, split, tokens):
            cache = QwenImage21KVCache(32)
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for stage in item['stages']:
                    hook_state['on'] = True
                    forward(item, stage, cache)
                    hook_state['on'] = False
                    hidden, output, modulation, mask = hook_state.pop('io')
                    if modulation.dim() != 2 or modulation.shape[-1] != 4 * hidden.shape[-1]:
                        raise RuntimeError('Unexpected modulation layout: ' + str(tuple(modulation.shape)))
                    real = len(table)
                    table.append(modulation[0].to(torch.bfloat16).cpu())
                    zero = real
                    if mask is not None:
                        zero = len(table)
                        table.append(modulation[-1].to(torch.bfloat16).cpu())
                    total, image = hidden.shape[1], stage['target_image_tokens']
                    spans = [(stage['stage'] + '_image', total - image, total)]
                    if total > image:
                        spans.append(('prompt_tokens', 0, total - image))
                    for group, low, high in spans:
                        count = min(tokens, high - low)
                        index = low + torch.randperm(high - low, generator=generator)[:count]
                        device_index = index.to(hidden.device)
                        rows = torch.full((count,), real, dtype=torch.long)
                        if mask is not None:
                            rows = torch.where(mask.cpu()[index], rows, torch.full_like(rows, zero))
                        entry = captured[split][group]
                        entry[0].append(hidden[0].index_select(0, device_index).to(torch.bfloat16).cpu())
                        entry[1].append(output[0].index_select(0, device_index).to(torch.bfloat16).cpu())
                        entry[2].append(rows)
                        if group != 'prompt_tokens':
                            frames, rows_grid, cols_grid = {**item['shared_transformer_kwargs'], **stage['kwargs']}['img_shapes'][0][-1]
                            rows_grid, cols_grid = int(rows_grid), int(cols_grid)
                            if frames != 1 or rows_grid * cols_grid != high - low:
                                raise RuntimeError('Target image grid does not match the image-token count.')
                            position = index - low
                            y = position.div(cols_grid, rounding_mode='floor')[:, None] + offsets_y
                            x = (position % cols_grid)[:, None] + offsets_x
                            valid = (y >= 0) & (y < rows_grid) & (x >= 0) & (x < cols_grid)
                            neighbors = low + y.clamp(0, rows_grid - 1) * cols_grid + x.clamp(0, cols_grid - 1)
                            gathered = hidden[0][neighbors.to(hidden.device)] * valid[..., None].to(hidden.device, hidden.dtype)
                            entry[3].append(gathered.to(torch.bfloat16).cpu())
                    report['student_calls'] += 1

        def load_verified(path, sha256, size=None):
            if (size is not None and path.stat().st_size != size) or digest(path) != sha256:
                raise RuntimeError('Input tensor differs: ' + str(path))
            return torch.load(path, weights_only=True, map_location='cpu')

        train_ids = set()
        counts = {'train_full': 0, 'train_early_pool': 0, 'validation': 0}
        for record in request['records']:
            item = load_verified(Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt', record['tensor_sha256'])
            split = item['row']['split']
            if item['row']['source_id'] != record['source_id'] or split not in ('train', 'validation'):
                raise RuntimeError('Capture record identity differs.')
            if split == 'train':
                train_ids.add(record['source_id'])
                counts['train_full'] += 1
            else:
                counts['validation'] += 1
            collect(item, split, fit['tokens_per_group_full'])
            del item
        fresh = request['fresh_aligned_train']
        for artifact in fresh['artifacts']:
            item = load_verified(Path(fresh['remote_directory']) / artifact['name'], artifact['sha256'], artifact['bytes'])
            if item['source_id'] != artifact['source_id'] or item['row']['split'] != 'train' or len(item['stages']) != 3:
                raise RuntimeError('Fresh aligned training provenance differs.')
            train_ids.add(artifact['source_id'])
            collect(item, 'train', fit['tokens_per_group_full'])
            counts['train_full'] += 1
            del item
        for example in request['early_pool']['examples']:
            item = load_verified(Path(example['remote_path']), example['sha256'], example['bytes'])
            if (item['capture_id'] != example['capture_id'] or item['row']['split'] != 'train'
                    or [s['stage'] for s in item['stages']] != ['early']):
                raise RuntimeError('Early pool example provenance differs.')
            collect(item, 'train', fit['tokens_per_group_early_pool'])
            counts['train_early_pool'] += 1
            del item
        handle.remove()
        validation_ids = {r['source_id'] for r in request['records'] if r['capture_id'].startswith('p5-validation-')}
        validation_ids |= {r['source_id'] for r in request['expanded_validation']['records']}
        if train_ids & validation_ids:
            raise RuntimeError('A validation prompt appears in the training set.')
        if counts != {'train_full': 60, 'train_early_pool': len(request['early_pool']['examples']), 'validation': 10}:
            raise RuntimeError('Capture counts differ: ' + json.dumps(counts))
        table = torch.stack(table).to('cuda')
        data = {split: {g: tuple(torch.cat(part).to('cuda') if part else None for part in v) for g, v in by_group.items() if v[0]}
                for split, by_group in captured.items()}
        del captured
        if set(data['train']) != set(groups) or set(data['validation']) != set(groups):
            raise RuntimeError('A token group is missing from the capture.')
        report['capture'] = {'counts': counts, 'modulation_rows': int(table.shape[0]),
                             'tokens': {split: {g: int(v[0].shape[0]) for g, v in d.items()} for split, d in data.items()}}
        (directory / 'progress.json').write_text(json.dumps(report, indent=2))
        results_volume.commit()
        print('Captured block', slot, json.dumps(report['capture']), flush=True)

        def objective(prediction, target):
            prediction, target = prediction.float(), target.float()
            direction = (torch.nn.functional.normalize(prediction, dim=-1) -
                         torch.nn.functional.normalize(target, dim=-1)).square().sum(-1)
            relative = (prediction - target).square().sum(-1) / target.square().sum(-1).clamp_min(1e-12)
            return direction + fit['magnitude_weight'] * relative

        original_block = model.transformer_blocks[slot]
        report['variants'] = {}
        for variant in request['variants']:
            # ---- Fit one bridge variant on TRAIN states only. ----
            torch.manual_seed(fit['init_seed'])
            local = variant.get('local', False)
            updates = variant.get('updates', fit['updates'])

            def build():
                if local:
                    return conditioned_namespace['LocalConditionedBridge'](4096, variant['rank'])
                return ConditionedBridge(4096, variant['rank'], variant['conditioned'])
            bridge = build().to('cuda').float()
            optimizer = torch.optim.AdamW(bridge.parameters(), lr=fit['learning_rate'], weight_decay=fit['weight_decay'])
            sampler = torch.Generator(device='cuda').manual_seed(fit['sampling_seed'])

            def run(inputs, rows, neighborhood=None):
                with torch.autocast('cuda', dtype=torch.bfloat16):
                    if local and neighborhood is not None:
                        return bridge.forward_neighborhood(neighborhood.float(), table[rows].float()).float()
                    return bridge(inputs.float(), table[rows].float()).float()

            def validation_score():
                with torch.no_grad():
                    values = {}
                    for g, (inputs, targets, rows, hoods) in data['validation'].items():
                        total = sum(objective(run(inputs[i:i + 1024], rows[i:i + 1024],
                                                  None if hoods is None else hoods[i:i + 1024]), targets[i:i + 1024]).sum()
                                    for i in range(0, inputs.shape[0], 1024))
                        values[g] = float(total / inputs.shape[0])
                    return sum(values.values()) / len(values), values

            curve, best = [], None
            for step in range(updates + 1):
                if step % fit['checkpoint_every'] == 0:
                    score, per_group = validation_score()
                    curve.append(dict(step=step, validation=score, groups=per_group))
                    if best is None or score < best[0]:
                        best = (score, step, {k: v.detach().cpu().clone() for k, v in bridge.state_dict().items()})
                    print(f'{variant["name"]} step {step}: validation {score:.6f}', flush=True)
                if step == updates:
                    break
                loss = 0
                for g in groups:
                    inputs, targets, rows, hoods = data['train'][g]
                    index = torch.randint(inputs.shape[0], (fit['tokens_per_group_update'],), device='cuda', generator=sampler)
                    loss = loss + objective(run(inputs[index], rows[index], None if hoods is None else hoods[index]),
                                            targets[index]).mean() / len(groups)
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(bridge.parameters(), fit['gradient_clip'])
                optimizer.step()
                report['optimizer_updates'] += 1
            checkpoint_path = directory / f'bridge_block_{slot:02d}_{variant["name"]}.pt'
            temporary = checkpoint_path.with_suffix('.tmp')
            torch.save({'request_key': key, 'layer': slot, 'variant': variant, 'step': best[1], 'state_dict': best[2]}, temporary)
            temporary.replace(checkpoint_path)
            artifact = dict(name=checkpoint_path.name, sha256=digest(checkpoint_path), bytes=checkpoint_path.stat().st_size)
            report['artifacts'].append(artifact)
            del bridge, optimizer
            gc.collect()
            torch.cuda.empty_cache()

            # ---- Evaluate the whole student with this bridge at the slot. ----
            selected = build()
            selected.load_state_dict(best[2])
            selected.requires_grad_(False)
            conditioned_namespace['insert_conditioned_bridge'](model, slot, selected)
            model.requires_grad_(False).eval()
            try:
                rows = predictions()
            finally:
                model.transformer_blocks[slot] = original_block
            report['student_calls'] += len(rows)
            records = metrics(rows)
            report['variants'][variant['name']] = dict(
                variant=variant, checkpoint=artifact, parameters=sum(v.numel() for v in best[2].values()),
                fit=dict(curve=curve, selected_step=best[1], selected_validation=best[0], identity_validation=curve[0]['validation']),
                overall=aggregate(records),
                panels={p: aggregate([r for r in records if r['panel'] == p]) for p in panels},
                stages={s: aggregate([r for r in records if r['stage'] == s]) for s in ('early', 'middle', 'late')},
                records=records)
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            print(f'{variant["name"]}: teacher-relative {report["variants"][variant["name"]]["overall"]["teacher_relative_l2"]["mean"]:.6f}', flush=True)
        report['status'] = 'passed'
        report['scope'] = 'Bridge variant fits and fixed-input velocity evaluation only; images and gates are separate.'
    except Exception as exc:
        import traceback
        report['status'] = 'failed'
        report['error'] = repr(exc)
        report['traceback'] = traceback.format_exc()
    (directory / 'summary.json').write_text(json.dumps(report, indent=2))
    results_volume.commit()
    return report
'''


SOURCES['conditioned_bridge.py'] = r'''import torch
from torch import nn
from types import SimpleNamespace


class ConditionedBridge(nn.Module):
    """Residual bottleneck bridge that can read the model's shared timestep modulation.

    The transformer computes one modulation tensor per timestep, [attn scale, attn gate, MLP scale, MLP gate],
    shared by every block. A conditioned bridge mirrors a block's MLP branch:
    h + tanh(gate) * up(gelu(down(LN(h) * (1 + scale)))), with scale and gate learned per-channel mixes of the
    shared values. Unconditioned, it is the original Phase 4 bridge: h + up(gelu(down(h))).
    """

    def __init__(self, hidden_dim=4096, rank=256, conditioned=True, eps=1e-6):
        super().__init__()
        self.conditioned = conditioned
        self.down = nn.Linear(hidden_dim, rank, bias=False)
        self.up = nn.Linear(rank, hidden_dim, bias=False)
        nn.init.zeros_(self.up.weight)
        if conditioned:
            self.norm = nn.LayerNorm(hidden_dim, elementwise_affine=False, eps=eps)
            # Start from the block's own MLP modulation (weight 1 on the MLP terms, 0 on the attention terms).
            mix = torch.zeros(2, hidden_dim)
            mix[1] = 1
            self.scale_mix = nn.Parameter(mix.clone())
            self.gate_mix = nn.Parameter(mix.clone())
            self.scale_bias = nn.Parameter(torch.zeros(hidden_dim))
            self.gate_bias = nn.Parameter(torch.zeros(hidden_dim))

    def scale_gate(self, modulation_rows):
        scale1, gate1, scale2, gate2 = modulation_rows.chunk(4, dim=-1)
        scale = self.scale_mix[0] * scale1 + self.scale_mix[1] * scale2 + self.scale_bias
        gate = self.gate_mix[0] * gate1 + self.gate_mix[1] * gate2 + self.gate_bias
        return scale, gate

    def forward(self, hidden_states, modulation_rows=None):
        if not self.conditioned:
            return hidden_states + self.up(nn.functional.gelu(self.down(hidden_states)))
        scale, gate = self.scale_gate(modulation_rows)
        update = self.up(nn.functional.gelu(self.down(self.norm(hidden_states) * (1 + scale))))
        return hidden_states + torch.tanh(gate) * update


class LocalConditionedBridge(ConditionedBridge):
    """Conditioned bridge plus a depthwise 3x3 convolution over the image-token grid inside the bottleneck.

    The convolution starts at zero (identity start) and is applied to target-image tokens only; prompt tokens have
    no grid and pass through the per-token path.
    """
    local = True

    def __init__(self, hidden_dim=4096, rank=1024, eps=1e-6):
        super().__init__(hidden_dim, rank, True, eps)
        self.mix = nn.Parameter(torch.zeros(rank, 9))

    def forward(self, hidden_states, modulation_rows=None, image_mask=None, grid=None):
        scale, gate = self.scale_gate(modulation_rows)
        z = self.down(self.norm(hidden_states) * (1 + scale))
        if grid is not None:
            height, width = grid
            image = z[:, image_mask]
            batch, tokens, rank = image.shape
            planes = image.transpose(1, 2).reshape(batch, rank, height, width)
            mixed = nn.functional.conv2d(planes, self.mix.view(rank, 1, 3, 3).to(planes.dtype), padding=1, groups=rank)
            z = z.clone()
            z[:, image_mask] = image + mixed.reshape(batch, rank, tokens).transpose(1, 2)
        return hidden_states + torch.tanh(gate) * self.up(nn.functional.gelu(z))

    def forward_neighborhood(self, neighborhood, modulation_rows):
        """Training path: neighborhood is (tokens, 9, hidden) in 3x3 raster order, zeros outside the grid."""
        scale, gate = self.scale_gate(modulation_rows)
        z = self.down(self.norm(neighborhood) * (1 + scale.unsqueeze(1)))
        z = z[:, 4] + (z * self.mix.t().unsqueeze(0)).sum(1)
        return neighborhood[:, 4] + torch.tanh(gate) * self.up(nn.functional.gelu(z))


def modulation_rows(modulation, target_token_mask):
    """Per-token modulation rows, selected exactly as the transformer block does."""
    if target_token_mask is None:
        return modulation.unsqueeze(1)
    real, zero = modulation[:-1].unsqueeze(1), modulation[-1:].unsqueeze(0)
    return torch.where(target_token_mask.view(1, -1, 1), real, zero)


class ConditionedBridgeBlock(nn.Module):
    def __init__(self, bridge, processor):
        super().__init__()
        self.bridge = bridge
        # Parent forward inspects this processor to choose mask construction; no attention is performed here.
        self.attn = SimpleNamespace(processor=processor)

    def forward(self, hidden_states, modulation, rotary_emb=None, attention_mask=None, target_token_mask=None,
                layer_cache=None, kv_cache_mode=None, cache_write_slice=None, segments=None, key_valid=None):
        if kv_cache_mode not in (None, 'extract', 'cached'):
            raise ValueError('Unknown KV cache mode.')
        rows = modulation_rows(modulation, target_token_mask) if self.bridge.conditioned else None
        if getattr(self.bridge, 'local', False):
            mask = target_token_mask if target_token_mask is not None else torch.ones(
                hidden_states.shape[1], dtype=torch.bool, device=hidden_states.device)
            height, width = self.grid
            if height * width != int(mask.sum()):
                raise RuntimeError('Target image grid does not match the image-token count.')
            return self.bridge(hidden_states, rows, mask, (height, width))
        return self.bridge(hidden_states, rows)


def insert_conditioned_bridge(transformer, layer, bridge):
    """Replace one slot; callers must use fresh KV caches afterwards."""
    old = transformer.transformer_blocks[layer]
    reference = next(transformer.parameters())
    bridge = bridge.to(device=reference.device, dtype=reference.dtype)
    wrapper = ConditionedBridgeBlock(bridge, old.attn.processor)
    wrapper.train(transformer.training)
    transformer.transformer_blocks[layer] = wrapper
    if getattr(bridge, 'local', False) and not getattr(transformer, '_bridge_grid_hook', False):
        # The block never sees img_shapes; record the target image's (height, width) in tokens on every call.
        def record_grid(module, args, kwargs):
            frames, height, width = kwargs['img_shapes'][0][-1]
            for block in module.transformer_blocks:
                if isinstance(block, ConditionedBridgeBlock):
                    block.grid = (int(height), int(width))
        transformer.register_forward_pre_hook(record_grid, with_kwargs=True)
        transformer._bridge_grid_hook = True
    return old
'''


"""Fit several bridge architectures for one block inside candidate B and evaluate each whole student; reuse completed results."""
import argparse
import hashlib
import json
import sys
from pathlib import Path


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--slot', type=int, default=13)
    parser.add_argument('--old-account', action='store_true')
    parser.add_argument('--dry-run', action='store_true', help='Write the request locally without scheduling GPU work.')
    args = parser.parse_args(LAUNCH_ARGS)
    if args.old_account:
        parser.error('Old account is download-only; GPU execution must use the new account.')
    root = ROOT
    protocol_path = root / f'data/phase7/block{args.slot}_bridge_v3/protocol.json'
    protocol = json.loads(protocol_path.read_bytes())
    scan_b = json.loads(next((root / 'data/phase7/conditional_importance_b').glob('*/request.json')).read_bytes())['request']
    scan_a = json.loads((root / 'data/phase7/conditional_importance/7454da6a5c1dc33af23dd46833e698f9b759d14985ca96340bc3d40cd47fb4f5/request.json').read_bytes())['request']
    if args.slot == 5 or not 0 < args.slot < 31:
        parser.error('Slot must be a surviving interior block.')
    request = {k: v for k, v in scan_b.items() if k not in ('worker_source', 'eligible_original_block_ids', 'selection', 'scope', 'version')}
    source = SOURCES['bridge_variants_v3_worker.py']
    request.update(
        version='phase7-bridge-variants-v3', worker_source=source, slot=args.slot,
        conditioned_bridge_source=SOURCES['conditioned_bridge.py'],
        variants=[dict(name='cond_r1024_long', rank=1024, conditioned=True, updates=3000),
                  dict(name='cond_local_r1024', rank=1024, conditioned=True, local=True, updates=3000)],
        protocol_sha256=hashlib.sha256(protocol_path.read_bytes()).hexdigest(),
        fresh_aligned_train=scan_a['fresh_aligned_train'], early_pool=scan_a['early_pool'],
        fit=dict(init_seed=20261017, capture_seed=20261018, sampling_seed=20261019, learning_rate=1e-3,
                 weight_decay=0.01, gradient_clip=1.0, magnitude_weight=0.1, updates=1000, checkpoint_every=100,
                 tokens_per_group_full=256, tokens_per_group_early_pool=64, tokens_per_group_update=256),
        selection='Minimum validation hidden-state objective at saved checkpoints including identity start',
        scope=f'Bridge variants for block {args.slot} inside frozen B; TRAIN-only fitting; whole-student velocity on scan inputs; no healing, images or latency')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = protocol_path.parent / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2)+'\n', encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase7/bridge_variants_v3/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        try:
            b''.join(results_volume.read_file(remote + '/started.json'))
        except FileNotFoundError:
            pass
        else:
            raise RuntimeError('Run already started; inspect its progress. No duplicate GPU scheduled.')
        app = modal.App('nihonga-phase7-bridge-variants-v3')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(source, 'bridge_variants_v3_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8,
                              memory=98304, max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['fit_bridge_variants'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2)+'\n')
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2)+'\n', encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'candidate_replay_max_relative_l2')}), flush=True)
    for name, v in (report.get('variants') or {}).items():
        print(name, 'step', v['fit']['selected_step'], 'teacher-relative', v['overall']['teacher_relative_l2']['mean'], flush=True)
    if report.get('status') != 'passed':
        raise SystemExit(1)

summary = reuse_or_launch(RUN, main)

show_variants(summary)

### What the results mean

Both move the error by only about 0.05 points (6.72% and 6.74%): the per-token fitting objective has reached its ceiling. (A first launch stopped on purpose when a check found non-square image grids; the corrected run is shown.)

### Variant comparison sheets

Builds teacher | B | first C | variant sheets from any rendered variant run.

In [ ]:
# Variant comparison sheets: local work only (no GPU).
LAUNCH_ARGS = ['--columns', 'cond_r1024', 'cond_r1024_long', 'cond_local_r1024', '--out', 'sheets_best']

"""Build teacher | B | first C | variant comparison sheets from whatever images exist so far (no GPU work)."""
import hashlib
import json
import textwrap
from pathlib import Path


def main():
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='download')
    import modal
    from PIL import Image, ImageDraw, ImageFont
    root = ROOT
    first = json.loads(next((root / 'data/phase7/bridge_images').glob('*/summary.json')).read_bytes())
    first_store = root / '.archive/bridge_images' / first['request_key']
    import argparse
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--columns', nargs='*', default=None, help='Variant columns after teacher, B and first C.')
    parser.add_argument('--out', default=None, help='Sheet folder name under data/phase7/bridge_variant_images.')
    args = parser.parse_args(LAUNCH_ARGS)
    volume = modal.Volume.from_name('nihonga-phase3-artifacts')
    sources, order = {}, []
    for request_path in sorted((root / 'data/phase7/bridge_variant_images').glob('*/request.json')):
        run_dir = request_path.parent
        request = json.loads(request_path.read_bytes())
        key = request['request_key']
        remote = f'/phase7/bridge_variant_images/{key}'
        try:
            summary = json.loads(b''.join(volume.read_file(remote + '/summary.json')))
            (run_dir / 'summary.json').write_text(json.dumps(summary, indent=2) + '\n', encoding='utf-8')
        except FileNotFoundError:
            summary = None
        expected = {a['name']: a['sha256'] for a in summary['artifacts']} if summary else {}
        for v in request['request']['variants']:
            sources[v['name']] = (key, remote, expected)
            order.append(v['name'])
    variants = args.columns or order
    sheets = root / 'data/phase7/bridge_variant_images' / (args.out or 'sheets_' + '_'.join(variants))
    face = next((f for f in ('C:/Windows/Fonts/msyh.ttc', 'C:/Windows/Fonts/simsun.ttc') if Path(f).exists()), 'C:/Windows/Fonts/arial.ttf')
    font, big = ImageFont.truetype(face, 20), ImageFont.truetype(face, 24)
    labels = {'teacher': 'Teacher', 'B': 'B (block 13 kept)', 'C': 'First C (plain r256)',
              'plain_r1024': 'plain r1024', 'cond_r256': 'cond r256', 'cond_r1024': 'cond r1024',
              'cond_r1024_long': 'cond r1024, 3k steps', 'cond_local_r1024': 'cond + local 3x3'}
    sheets.mkdir(exist_ok=True)
    tile, columns = 480, ['teacher', 'B', 'C'] + variants
    ready = 0
    for index, row in enumerate(first['rows']):
        images = {v: first_store / row[v]['name'] for v in ('teacher', 'B', 'C')}
        for variant in variants:
            key, remote, expected = sources[variant]
            store = root / '.archive/bridge_variant_images' / key
            store.mkdir(parents=True, exist_ok=True)
            name = f"{row['source_id']}-{variant}.png"
            path = store / name
            if not path.exists():
                try:
                    data = b''.join(volume.read_file(f'{remote}/{name}'))
                except FileNotFoundError:
                    continue
                path.write_bytes(data)
            if name in expected and hashlib.sha256(path.read_bytes()).hexdigest() != expected[name]:
                raise RuntimeError('Image hash mismatch: ' + name)
            images[variant] = path
        canvas = Image.new('RGB', (len(columns) * tile, tile + 140), 'white')
        draw = ImageDraw.Draw(canvas)
        draw.text((10, 6), f"P{index + 1:02d} | {row.get('subdimension', '')}", font=big, fill='black')
        for line, text in enumerate(textwrap.wrap(row['prompt'], 250 if row['prompt'].isascii() else 120)[:2]):
            draw.text((10, 40 + 24 * line), text, font=font, fill='black')
        for column, variant in enumerate(columns):
            draw.text((column * tile + 8, 105), labels[variant], font=big, fill='black')
            if variant in images:
                with Image.open(images[variant]) as image:
                    canvas.paste(image.convert('RGB').resize((tile, tile), Image.Resampling.LANCZOS), (column * tile, 140))
            else:
                draw.text((column * tile + 150, 140 + tile // 2), 'rendering...', font=big, fill='gray')
        canvas.save(sheets / f'P{index + 1:02d}.jpg', quality=88)
        ready += all(v in images for v in variants)
    print(json.dumps(dict(sheets=len(first['rows']), complete=ready, folder=str(sheets))))

if not (ROOT / 'data/phase7/bridge_variant_images/sheets_best/P30.jpg').exists():
    main()
else:
    print("Saved output present; not rebuilt.")

## 9b. Images of the longer-trained and local-mixing bridges

Both variants on the same 30 prompts and seeds.

In [ ]:
# Images of the longer-trained and local-mixing bridges: reuses the saved run; launches GPU work only when its result is missing and ALLOW_GPU is True.

RUN = 'data/phase7/bridge_variant_images/aa4f5189b1d06f81ec32eb4e01b400f790f0b1bbe545e923ae2f38ff40f2c286'

LAUNCH_ARGS = ['--variants-dir', 'block13_bridge_v3', '--order', 'cond_local_r1024', 'cond_r1024_long']

SOURCES['bridge_variant_images_worker.py'] = r'''def generate_variant_images(request, key):
    """Matched 40-step images for block-13 bridge variants inside B (teacher and B images are reused from the
    earlier run). Same prompt, CPU noise seed, size, scheduler and 40 steps. Qualitative screen only."""
    import gc
    import hashlib
    import inspect
    import json
    import os
    import time
    from pathlib import Path

    import torch
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache, QwenImage21KVLayerCache

    directory = Path('/phase3/phase7/bridge_variant_images') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete image attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def table_hash(value):
        return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

    variants = [v['name'] for v in request['variants']]
    report = {'request_key': key, 'status': 'running', 'rows': [], 'artifacts': [],
              'calls': {v: 0 for v in variants}, 'optimizer_updates': 0}
    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('Expected H100.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
                                                   dtype=torch.bfloat16, local_files_only=True).to('cuda')
        pipe.set_progress_bar_config(disable=True)
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Transformer source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--')) /
                       'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        # Derived position/time tables: use the audited values on every host (as in steps 7-11).
        derived = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
        derived['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
        expected = request['expected_derived_tables']
        mismatched = sorted(n for n, v in derived.items() if table_hash(v) != expected[n])
        if mismatched:
            canonical = torch.load(Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt'),
                                   weights_only=True, map_location='cpu')
            if any(table_hash(canonical[n]) != expected[n] for n in expected):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [canonical[f'pos_embed.freqs.{a}'].to(device=v.device, dtype=v.dtype) for a, v in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(canonical['time_text_embed.time_proj.freqs'])
            if any(table_hash(v) != expected[n] for n, v in
                   list({f'pos_embed.freqs.{a}': v for a, v in enumerate(model.pos_embed.freqs)}.items()) +
                   [('time_text_embed.time_proj.freqs', model.time_text_embed.time_proj.freqs)]):
                raise RuntimeError('Derived tables still differ after loading the audited copy.')
            report['canonical_tables_applied'] = mismatched

        def generate(row, variant):
            path = directory / (row['source_id'] + '-' + variant + '.png')
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'],
                             num_inference_steps=40, true_cfg_scale=teacher['true_cfg_scale'],
                             generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=True, output_type='pil').images[0]
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            temporary = path.with_suffix('.tmp')
            image.save(temporary, format='PNG')
            temporary.replace(path)
            results_volume.commit()
            report['calls'][variant] += 1
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            return {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size, 'seconds': elapsed}

        namespace = {}
        exec(compile(request['core_source'], 'bridge_definition.py', 'exec'), namespace)
        exec(compile(request['wrapper_source'], 'bridge_block_definition.py', 'exec'), namespace)
        conditioned = {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned)

        def load_checkpoint(spec):
            path = Path(spec['remote_path'])
            if digest(path) != spec['sha256']:
                raise RuntimeError('Bridge checkpoint differs: ' + spec['remote_path'])
            return torch.load(path, weights_only=True, map_location='cpu')

        block5 = request['block5_bridge']
        checkpoint = load_checkpoint(block5)
        if checkpoint['layer'] != 5 or checkpoint['step'] != block5['step'] or checkpoint['request_key'] != block5['request_key']:
            raise RuntimeError('Block-5 bridge provenance differs.')
        bridge = namespace['ResidualBottleneckBridge'](4096, 256)
        bridge.load_state_dict(checkpoint['state_dict'])
        bridge.requires_grad_(False)
        namespace['insert_bridge'](model, 5, bridge, QwenImage21KVCache)
        original13 = model.transformer_blocks[13]
        outputs = {v: {} for v in variants}
        for variant in request['variants']:
            checkpoint = load_checkpoint(variant)
            if checkpoint['layer'] != 13 or checkpoint['variant']['name'] != variant['name']:
                raise RuntimeError('Variant provenance differs.')
            spec = checkpoint['variant']
            if spec.get('local'):
                bridge13 = conditioned['LocalConditionedBridge'](4096, spec['rank'])
            else:
                bridge13 = conditioned['ConditionedBridge'](4096, spec['rank'], spec['conditioned'])
            bridge13.load_state_dict(checkpoint['state_dict'])
            bridge13.requires_grad_(False)
            conditioned['insert_conditioned_bridge'](model, 13, bridge13)
            model.eval()
            try:
                for row in request['rows']:
                    outputs[variant['name']][row['source_id']] = generate(row, variant['name'])
            finally:
                model.transformer_blocks[13] = original13
        for row in request['rows']:
            report['rows'].append({**row, **{v: outputs[v][row['source_id']] for v in variants}})
            report['artifacts'].extend(outputs[v][row['source_id']] for v in variants)
        report['status'] = 'passed'
    except Exception as exc:
        report['status'] = 'failed'
        report['error'] = type(exc).__name__ + ': ' + str(exc)
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=True) + '\n')
    results_volume.commit()
    return report
'''


SOURCES['conditioned_bridge.py'] = r'''import torch
from torch import nn
from types import SimpleNamespace


class ConditionedBridge(nn.Module):
    """Residual bottleneck bridge that can read the model's shared timestep modulation.

    The transformer computes one modulation tensor per timestep, [attn scale, attn gate, MLP scale, MLP gate],
    shared by every block. A conditioned bridge mirrors a block's MLP branch:
    h + tanh(gate) * up(gelu(down(LN(h) * (1 + scale)))), with scale and gate learned per-channel mixes of the
    shared values. Unconditioned, it is the original Phase 4 bridge: h + up(gelu(down(h))).
    """

    def __init__(self, hidden_dim=4096, rank=256, conditioned=True, eps=1e-6):
        super().__init__()
        self.conditioned = conditioned
        self.down = nn.Linear(hidden_dim, rank, bias=False)
        self.up = nn.Linear(rank, hidden_dim, bias=False)
        nn.init.zeros_(self.up.weight)
        if conditioned:
            self.norm = nn.LayerNorm(hidden_dim, elementwise_affine=False, eps=eps)
            # Start from the block's own MLP modulation (weight 1 on the MLP terms, 0 on the attention terms).
            mix = torch.zeros(2, hidden_dim)
            mix[1] = 1
            self.scale_mix = nn.Parameter(mix.clone())
            self.gate_mix = nn.Parameter(mix.clone())
            self.scale_bias = nn.Parameter(torch.zeros(hidden_dim))
            self.gate_bias = nn.Parameter(torch.zeros(hidden_dim))

    def scale_gate(self, modulation_rows):
        scale1, gate1, scale2, gate2 = modulation_rows.chunk(4, dim=-1)
        scale = self.scale_mix[0] * scale1 + self.scale_mix[1] * scale2 + self.scale_bias
        gate = self.gate_mix[0] * gate1 + self.gate_mix[1] * gate2 + self.gate_bias
        return scale, gate

    def forward(self, hidden_states, modulation_rows=None):
        if not self.conditioned:
            return hidden_states + self.up(nn.functional.gelu(self.down(hidden_states)))
        scale, gate = self.scale_gate(modulation_rows)
        update = self.up(nn.functional.gelu(self.down(self.norm(hidden_states) * (1 + scale))))
        return hidden_states + torch.tanh(gate) * update


class LocalConditionedBridge(ConditionedBridge):
    """Conditioned bridge plus a depthwise 3x3 convolution over the image-token grid inside the bottleneck.

    The convolution starts at zero (identity start) and is applied to target-image tokens only; prompt tokens have
    no grid and pass through the per-token path.
    """
    local = True

    def __init__(self, hidden_dim=4096, rank=1024, eps=1e-6):
        super().__init__(hidden_dim, rank, True, eps)
        self.mix = nn.Parameter(torch.zeros(rank, 9))

    def forward(self, hidden_states, modulation_rows=None, image_mask=None, grid=None):
        scale, gate = self.scale_gate(modulation_rows)
        z = self.down(self.norm(hidden_states) * (1 + scale))
        if grid is not None:
            height, width = grid
            image = z[:, image_mask]
            batch, tokens, rank = image.shape
            planes = image.transpose(1, 2).reshape(batch, rank, height, width)
            mixed = nn.functional.conv2d(planes, self.mix.view(rank, 1, 3, 3).to(planes.dtype), padding=1, groups=rank)
            z = z.clone()
            z[:, image_mask] = image + mixed.reshape(batch, rank, tokens).transpose(1, 2)
        return hidden_states + torch.tanh(gate) * self.up(nn.functional.gelu(z))

    def forward_neighborhood(self, neighborhood, modulation_rows):
        """Training path: neighborhood is (tokens, 9, hidden) in 3x3 raster order, zeros outside the grid."""
        scale, gate = self.scale_gate(modulation_rows)
        z = self.down(self.norm(neighborhood) * (1 + scale.unsqueeze(1)))
        z = z[:, 4] + (z * self.mix.t().unsqueeze(0)).sum(1)
        return neighborhood[:, 4] + torch.tanh(gate) * self.up(nn.functional.gelu(z))


def modulation_rows(modulation, target_token_mask):
    """Per-token modulation rows, selected exactly as the transformer block does."""
    if target_token_mask is None:
        return modulation.unsqueeze(1)
    real, zero = modulation[:-1].unsqueeze(1), modulation[-1:].unsqueeze(0)
    return torch.where(target_token_mask.view(1, -1, 1), real, zero)


class ConditionedBridgeBlock(nn.Module):
    def __init__(self, bridge, processor):
        super().__init__()
        self.bridge = bridge
        # Parent forward inspects this processor to choose mask construction; no attention is performed here.
        self.attn = SimpleNamespace(processor=processor)

    def forward(self, hidden_states, modulation, rotary_emb=None, attention_mask=None, target_token_mask=None,
                layer_cache=None, kv_cache_mode=None, cache_write_slice=None, segments=None, key_valid=None):
        if kv_cache_mode not in (None, 'extract', 'cached'):
            raise ValueError('Unknown KV cache mode.')
        rows = modulation_rows(modulation, target_token_mask) if self.bridge.conditioned else None
        if getattr(self.bridge, 'local', False):
            mask = target_token_mask if target_token_mask is not None else torch.ones(
                hidden_states.shape[1], dtype=torch.bool, device=hidden_states.device)
            height, width = self.grid
            if height * width != int(mask.sum()):
                raise RuntimeError('Target image grid does not match the image-token count.')
            return self.bridge(hidden_states, rows, mask, (height, width))
        return self.bridge(hidden_states, rows)


def insert_conditioned_bridge(transformer, layer, bridge):
    """Replace one slot; callers must use fresh KV caches afterwards."""
    old = transformer.transformer_blocks[layer]
    reference = next(transformer.parameters())
    bridge = bridge.to(device=reference.device, dtype=reference.dtype)
    wrapper = ConditionedBridgeBlock(bridge, old.attn.processor)
    wrapper.train(transformer.training)
    transformer.transformer_blocks[layer] = wrapper
    if getattr(bridge, 'local', False) and not getattr(transformer, '_bridge_grid_hook', False):
        # The block never sees img_shapes; record the target image's (height, width) in tokens on every call.
        def record_grid(module, args, kwargs):
            frames, height, width = kwargs['img_shapes'][0][-1]
            for block in module.transformer_blocks:
                if isinstance(block, ConditionedBridgeBlock):
                    block.grid = (int(height), int(width))
        transformer.register_forward_pre_hook(record_grid, with_kwargs=True)
        transformer._bridge_grid_hook = True
    return old
'''


"""Render block-13 bridge variants inside B on the 30 earlier Phase 7 image prompts (teacher and B reused); qualitative only."""
import argparse
import hashlib
import json
import sys
from pathlib import Path


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--dry-run', action='store_true')
    parser.add_argument('--variants-dir', default='block13_bridge_v2')
    parser.add_argument('--order', nargs='*', help='Variant names in rendering order (default: as fitted).')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    recovered = root / 'data/recovered/modal'
    images_key = 'dda8c184e657dfb604c35fb261d83371c108c39fac033a0f57db66f7686c30ad'
    base = json.loads((recovered / 'phase7/candidate_images' / images_key / 'request.json').read_bytes())['request']
    scan_b = json.loads(next((root / 'data/phase7/conditional_importance_b').glob('*/request.json')).read_bytes())['request']
    fit = next(r for r in (json.loads(p.read_bytes()) for p in (root / 'data/phase7' / args.variants_dir).glob('*/summary.json')) if r.get('status') == 'passed')
    if fit.get('status') != 'passed':
        raise RuntimeError('Bridge variants run did not complete.')
    selected = scan_b['selected_candidate']
    request = {k: base[k] for k in ('teacher', 'transformer_source_sha256', 'runtime_environment', 'expected_derived_tables',
                                    'canonical_tables_key', 'core_source', 'wrapper_source', 'rows')}
    request.update(
        version='phase7-bridge-variant-images-v1',
        worker_source=SOURCES['bridge_variant_images_worker.py'],
        conditioned_bridge_source=SOURCES['conditioned_bridge.py'],
        execution_account='new',
        block5_bridge=dict(remote_path=f"/phase3/phase5/fitting/{scan_b['fit_key']}/{Path(selected['checkpoint_path']).name}",
                           sha256=selected['checkpoint_sha256'], step=selected['selected_update'], request_key=scan_b['fit_key']),
        variants=[dict(name=name, remote_path=f"/phase3/phase7/{'bridge_variants' if args.variants_dir == 'block13_bridge_v2' else 'bridge_variants_v3'}/{fit['request_key']}/{v['checkpoint']['name']}",
                       sha256=v['checkpoint']['sha256']) for name, v in sorted(fit['variants'].items(), key=lambda kv: (args.order or list(fit['variants'])).index(kv[0]))],
        scope='Qualitative matched-seed screen of block-13 bridge variants inside B; teacher and B images reused from the earlier run')
    from experiment_runtime import pin_reference_runtime
    pin_reference_runtime(request, recovered)
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase7/bridge_variant_images' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2)+'\n', encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase7/bridge_variant_images/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        try:
            b''.join(results_volume.read_file(remote + '/started.json'))
        except FileNotFoundError:
            pass
        else:
            raise RuntimeError('Run already started; inspect its progress. No duplicate GPU scheduled.')
        app = modal.App('nihonga-phase7-bridge-variant-images')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'bridge_variant_images_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8,
                              memory=98304, max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['generate_variant_images'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2)+'\n')
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2)+'\n', encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'calls')}), flush=True)
    if report.get('status') != 'passed':
        raise SystemExit(1)

summary = reuse_or_launch(RUN, main)

show_images('data/phase7/bridge_variant_images/sheets_best', ['P28.jpg'])

### What the results mean

Images confirm the velocity plateau: neither variant changes the picture against cond r1024.

## 10. Is the misspelling systematic? Four seeds per text prompt

The 5 text prompts rendered with three more seeds for teacher, B and the best per-token bridge (cond r1024, 3000 updates).

In [ ]:
# Is the misspelling systematic? Four seeds per text prompt: reuses the saved run; launches GPU work only when its result is missing and ALLOW_GPU is True.

RUN = 'data/phase7/multiseed_text/af4a34ba1a0c739121706991b89e79ec92bff3debaf1b1ddbf5b6188c182f514'

LAUNCH_ARGS = []

SOURCES['multiseed_text_worker.py'] = r'''def generate_multiseed_text(request, key):
    """Extra seeds for the text prompts: teacher, B, and C with the best block-13 bridge.
    Same prompt, CPU noise seed, size, scheduler and 40 steps for every model. Qualitative screen only."""
    import gc
    import hashlib
    import inspect
    import json
    import os
    import time
    from pathlib import Path

    import torch
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache, QwenImage21KVLayerCache

    directory = Path('/phase3/phase7/multiseed_text') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete image attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def table_hash(value):
        return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

    variants = ['teacher', 'B', request['c_variant']['name']]
    report = {'request_key': key, 'status': 'running', 'rows': [], 'artifacts': [],
              'calls': {v: 0 for v in variants}, 'optimizer_updates': 0}
    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('Expected H100.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
                                                   dtype=torch.bfloat16, local_files_only=True).to('cuda')
        pipe.set_progress_bar_config(disable=True)
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Transformer source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--')) /
                       'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        # Derived position/time tables: use the audited values on every host (as in steps 7-11).
        derived = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
        derived['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
        expected = request['expected_derived_tables']
        mismatched = sorted(n for n, v in derived.items() if table_hash(v) != expected[n])
        if mismatched:
            canonical = torch.load(Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt'),
                                   weights_only=True, map_location='cpu')
            if any(table_hash(canonical[n]) != expected[n] for n in expected):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [canonical[f'pos_embed.freqs.{a}'].to(device=v.device, dtype=v.dtype) for a, v in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(canonical['time_text_embed.time_proj.freqs'])
            if any(table_hash(v) != expected[n] for n, v in
                   list({f'pos_embed.freqs.{a}': v for a, v in enumerate(model.pos_embed.freqs)}.items()) +
                   [('time_text_embed.time_proj.freqs', model.time_text_embed.time_proj.freqs)]):
                raise RuntimeError('Derived tables still differ after loading the audited copy.')
            report['canonical_tables_applied'] = mismatched

        def generate(row, variant):
            path = directory / (row['source_id'] + '-' + variant + '.png')
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'],
                             num_inference_steps=40, true_cfg_scale=teacher['true_cfg_scale'],
                             generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=True, output_type='pil').images[0]
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            temporary = path.with_suffix('.tmp')
            image.save(temporary, format='PNG')
            temporary.replace(path)
            results_volume.commit()
            report['calls'][variant] += 1
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            return {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size, 'seconds': elapsed}

        namespace = {}
        exec(compile(request['core_source'], 'bridge_definition.py', 'exec'), namespace)
        exec(compile(request['wrapper_source'], 'bridge_block_definition.py', 'exec'), namespace)
        conditioned = {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned)

        def load_checkpoint(spec):
            path = Path(spec['remote_path'])
            if digest(path) != spec['sha256']:
                raise RuntimeError('Bridge checkpoint differs: ' + spec['remote_path'])
            return torch.load(path, weights_only=True, map_location='cpu')

        outputs = {v: {} for v in variants}
        for row in request['rows']:
            outputs['teacher'][row['source_id']] = generate(row, 'teacher')
        block5 = request['block5_bridge']
        checkpoint = load_checkpoint(block5)
        if checkpoint['layer'] != 5 or checkpoint['step'] != block5['step']:
            raise RuntimeError('Block-5 bridge provenance differs.')
        bridge = namespace['ResidualBottleneckBridge'](4096, 256)
        bridge.load_state_dict(checkpoint['state_dict'])
        bridge.requires_grad_(False)
        namespace['insert_bridge'](model, 5, bridge, QwenImage21KVCache)
        model.eval()
        for row in request['rows']:
            outputs['B'][row['source_id']] = generate(row, 'B')
        spec = request['c_variant']
        checkpoint = load_checkpoint(spec)
        if checkpoint['layer'] != 13 or checkpoint['variant']['name'] != spec['name']:
            raise RuntimeError('Block-13 bridge provenance differs.')
        fitted = checkpoint['variant']
        if fitted.get('local'):
            bridge13 = conditioned['LocalConditionedBridge'](4096, fitted['rank'])
        else:
            bridge13 = conditioned['ConditionedBridge'](4096, fitted['rank'], fitted['conditioned'])
        bridge13.load_state_dict(checkpoint['state_dict'])
        bridge13.requires_grad_(False)
        conditioned['insert_conditioned_bridge'](model, 13, bridge13)
        model.eval()
        for row in request['rows']:
            outputs[spec['name']][row['source_id']] = generate(row, spec['name'])
        for row in request['rows']:
            report['rows'].append({**row, **{v: outputs[v][row['source_id']] for v in variants}})
            report['artifacts'].extend(outputs[v][row['source_id']] for v in variants)
        report['status'] = 'passed'
    except Exception as exc:
        report['status'] = 'failed'
        report['error'] = type(exc).__name__ + ': ' + str(exc)
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=True) + '\n')
    results_volume.commit()
    return report
'''


SOURCES['conditioned_bridge.py'] = r'''import torch
from torch import nn
from types import SimpleNamespace


class ConditionedBridge(nn.Module):
    """Residual bottleneck bridge that can read the model's shared timestep modulation.

    The transformer computes one modulation tensor per timestep, [attn scale, attn gate, MLP scale, MLP gate],
    shared by every block. A conditioned bridge mirrors a block's MLP branch:
    h + tanh(gate) * up(gelu(down(LN(h) * (1 + scale)))), with scale and gate learned per-channel mixes of the
    shared values. Unconditioned, it is the original Phase 4 bridge: h + up(gelu(down(h))).
    """

    def __init__(self, hidden_dim=4096, rank=256, conditioned=True, eps=1e-6):
        super().__init__()
        self.conditioned = conditioned
        self.down = nn.Linear(hidden_dim, rank, bias=False)
        self.up = nn.Linear(rank, hidden_dim, bias=False)
        nn.init.zeros_(self.up.weight)
        if conditioned:
            self.norm = nn.LayerNorm(hidden_dim, elementwise_affine=False, eps=eps)
            # Start from the block's own MLP modulation (weight 1 on the MLP terms, 0 on the attention terms).
            mix = torch.zeros(2, hidden_dim)
            mix[1] = 1
            self.scale_mix = nn.Parameter(mix.clone())
            self.gate_mix = nn.Parameter(mix.clone())
            self.scale_bias = nn.Parameter(torch.zeros(hidden_dim))
            self.gate_bias = nn.Parameter(torch.zeros(hidden_dim))

    def scale_gate(self, modulation_rows):
        scale1, gate1, scale2, gate2 = modulation_rows.chunk(4, dim=-1)
        scale = self.scale_mix[0] * scale1 + self.scale_mix[1] * scale2 + self.scale_bias
        gate = self.gate_mix[0] * gate1 + self.gate_mix[1] * gate2 + self.gate_bias
        return scale, gate

    def forward(self, hidden_states, modulation_rows=None):
        if not self.conditioned:
            return hidden_states + self.up(nn.functional.gelu(self.down(hidden_states)))
        scale, gate = self.scale_gate(modulation_rows)
        update = self.up(nn.functional.gelu(self.down(self.norm(hidden_states) * (1 + scale))))
        return hidden_states + torch.tanh(gate) * update


class LocalConditionedBridge(ConditionedBridge):
    """Conditioned bridge plus a depthwise 3x3 convolution over the image-token grid inside the bottleneck.

    The convolution starts at zero (identity start) and is applied to target-image tokens only; prompt tokens have
    no grid and pass through the per-token path.
    """
    local = True

    def __init__(self, hidden_dim=4096, rank=1024, eps=1e-6):
        super().__init__(hidden_dim, rank, True, eps)
        self.mix = nn.Parameter(torch.zeros(rank, 9))

    def forward(self, hidden_states, modulation_rows=None, image_mask=None, grid=None):
        scale, gate = self.scale_gate(modulation_rows)
        z = self.down(self.norm(hidden_states) * (1 + scale))
        if grid is not None:
            height, width = grid
            image = z[:, image_mask]
            batch, tokens, rank = image.shape
            planes = image.transpose(1, 2).reshape(batch, rank, height, width)
            mixed = nn.functional.conv2d(planes, self.mix.view(rank, 1, 3, 3).to(planes.dtype), padding=1, groups=rank)
            z = z.clone()
            z[:, image_mask] = image + mixed.reshape(batch, rank, tokens).transpose(1, 2)
        return hidden_states + torch.tanh(gate) * self.up(nn.functional.gelu(z))

    def forward_neighborhood(self, neighborhood, modulation_rows):
        """Training path: neighborhood is (tokens, 9, hidden) in 3x3 raster order, zeros outside the grid."""
        scale, gate = self.scale_gate(modulation_rows)
        z = self.down(self.norm(neighborhood) * (1 + scale.unsqueeze(1)))
        z = z[:, 4] + (z * self.mix.t().unsqueeze(0)).sum(1)
        return neighborhood[:, 4] + torch.tanh(gate) * self.up(nn.functional.gelu(z))


def modulation_rows(modulation, target_token_mask):
    """Per-token modulation rows, selected exactly as the transformer block does."""
    if target_token_mask is None:
        return modulation.unsqueeze(1)
    real, zero = modulation[:-1].unsqueeze(1), modulation[-1:].unsqueeze(0)
    return torch.where(target_token_mask.view(1, -1, 1), real, zero)


class ConditionedBridgeBlock(nn.Module):
    def __init__(self, bridge, processor):
        super().__init__()
        self.bridge = bridge
        # Parent forward inspects this processor to choose mask construction; no attention is performed here.
        self.attn = SimpleNamespace(processor=processor)

    def forward(self, hidden_states, modulation, rotary_emb=None, attention_mask=None, target_token_mask=None,
                layer_cache=None, kv_cache_mode=None, cache_write_slice=None, segments=None, key_valid=None):
        if kv_cache_mode not in (None, 'extract', 'cached'):
            raise ValueError('Unknown KV cache mode.')
        rows = modulation_rows(modulation, target_token_mask) if self.bridge.conditioned else None
        if getattr(self.bridge, 'local', False):
            mask = target_token_mask if target_token_mask is not None else torch.ones(
                hidden_states.shape[1], dtype=torch.bool, device=hidden_states.device)
            height, width = self.grid
            if height * width != int(mask.sum()):
                raise RuntimeError('Target image grid does not match the image-token count.')
            return self.bridge(hidden_states, rows, mask, (height, width))
        return self.bridge(hidden_states, rows)


def insert_conditioned_bridge(transformer, layer, bridge):
    """Replace one slot; callers must use fresh KV caches afterwards."""
    old = transformer.transformer_blocks[layer]
    reference = next(transformer.parameters())
    bridge = bridge.to(device=reference.device, dtype=reference.dtype)
    wrapper = ConditionedBridgeBlock(bridge, old.attn.processor)
    wrapper.train(transformer.training)
    transformer.transformer_blocks[layer] = wrapper
    if getattr(bridge, 'local', False) and not getattr(transformer, '_bridge_grid_hook', False):
        # The block never sees img_shapes; record the target image's (height, width) in tokens on every call.
        def record_grid(module, args, kwargs):
            frames, height, width = kwargs['img_shapes'][0][-1]
            for block in module.transformer_blocks:
                if isinstance(block, ConditionedBridgeBlock):
                    block.grid = (int(height), int(width))
        transformer.register_forward_pre_hook(record_grid, with_kwargs=True)
        transformer._bridge_grid_hook = True
    return old
'''


"""Render the text prompts with extra seeds for teacher, B and C (best block-13 bridge); qualitative only."""
import argparse
import hashlib
import json
import sys
from pathlib import Path


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--dry-run', action='store_true')
    parser.add_argument('--variant', default='cond_r1024_long')
    parser.add_argument('--source', default='block13_bridge_v3', help='Run folder holding the bridge checkpoint.')
    parser.add_argument('--with-base-prompts', action='store_true', help='Also render the 30 base prompts at seed 0; candidate only.')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    recovered = root / 'data/recovered/modal'
    images_key = 'dda8c184e657dfb604c35fb261d83371c108c39fac033a0f57db66f7686c30ad'
    base = json.loads((recovered / 'phase7/candidate_images' / images_key / 'request.json').read_bytes())['request']
    scan_b = json.loads(next((root / 'data/phase7/conditional_importance_b').glob('*/request.json')).read_bytes())['request']
    fit = next(r for r in (json.loads(p.read_bytes()) for p in (root / 'data/phase7' / args.source).glob('*/summary.json')) if r.get('status') == 'passed')
    best = fit['variants'][args.variant]
    remote_dir = {'block13_bridge_v3': 'bridge_variants_v3', 'block13_bridge_e2e': 'bridge_end_to_end', 'block13_bridge_e2e_text': 'bridge_end_to_end_v2', 'block13_bridge_e2e_text8': 'bridge_end_to_end_v3'}[args.source]
    selected = scan_b['selected_candidate']
    text_rows = [r for r in base['rows'] if r['subdimension'] in ('text_rendering', 'posters')]
    rows = [{**r, 'source_id': f"{r['source_id']}-s{k}", 'seed': (r['seed'] + 1000003 * k) % 2**32}
            for r in text_rows for k in (1, 2, 3)]
    if args.with_base_prompts:
        rows += base['rows']
    request = {k: base[k] for k in ('teacher', 'transformer_source_sha256', 'runtime_environment', 'expected_derived_tables',
                                    'canonical_tables_key', 'core_source', 'wrapper_source')}
    request.update(
        version='phase7-multiseed-text-v1', rows=rows,
        worker_source=SOURCES['multiseed_text_worker.py'],
        conditioned_bridge_source=SOURCES['conditioned_bridge.py'],
        execution_account='new',
        block5_bridge=dict(remote_path=f"/phase3/phase5/fitting/{scan_b['fit_key']}/{Path(selected['checkpoint_path']).name}",
                           sha256=selected['checkpoint_sha256'], step=selected['selected_update']),
        c_variant=dict(name=args.variant, remote_path=f"/phase3/phase7/{remote_dir}/{fit['request_key']}/{best['checkpoint']['name']}",
                       sha256=best['checkpoint']['sha256']),
        candidate_only=args.with_base_prompts,
        scope='Extra seeds (3 per prompt) for the 5 text prompts of the Phase 7 image set' + ('; plus the 30 base prompts at seed 0, candidate only' if args.with_base_prompts else '; seed 0 images are reused'))
    from experiment_runtime import pin_reference_runtime
    pin_reference_runtime(request, recovered)
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase7/multiseed_text' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2)+'\n', encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase7/multiseed_text/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        try:
            b''.join(results_volume.read_file(remote + '/started.json'))
        except FileNotFoundError:
            pass
        else:
            raise RuntimeError('Run already started; inspect its progress. No duplicate GPU scheduled.')
        app = modal.App('nihonga-phase7-multiseed-text')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'multiseed_text_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8,
                              memory=98304, max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['generate_multiseed_text'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2)+'\n')
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2)+'\n', encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'calls')}), flush=True)
    if report.get('status') != 'passed':
        raise SystemExit(1)

summary = reuse_or_launch(RUN, main)

show_images(RUN + '/sheets', ['p1-2fa54a3c4eb8.jpg'])

### What the results mean

Systematic: "The Rusty Anchor" 0/4 (B 3/4), "Happy Birthday Emma" 1/4 (B 3/4), "Moonlight Motel" 1/4 (B 3/4). Short or common phrases and Chinese survive. Without block 13 the model loses the spelling of longer, less common Latin words.

## 11. Attention bridge trained on the final output

Two changes, tested separately. **End-to-end training**: only the bridge learns, from the final velocity error against live teacher targets, with gradients through the frozen blocks 14-31 (functional checkpointing). **Attention branch**: 4 heads x 128, image tokens attend to the prompt and to each other with the model's RoPE and key masks, its own prefix K/V in the freed cache slot, zero-initialised output. `e2e_cond` changes only the objective; `e2e_attn` adds attention. Both start from cond r1024 (3000 updates).

In [ ]:
# Attention bridge trained on the final output: reuses the saved run; launches GPU work only when its result is missing and ALLOW_GPU is True.

RUN = 'data/phase7/block13_bridge_e2e/0ff2fb58663d578f0109f8d65bcc11514d269deceb5c25f25da4c8530aa8da47'

LAUNCH_ARGS = ['--slot', '13']

SOURCES['bridge_end_to_end_worker.py'] = r'''def train_bridge_end_to_end(request, key):
    """Train a bridge for one block inside frozen B against the final velocity (bridge weights only).

    Verifies the teacher contract and exact B replay, computes live teacher targets for TRAIN inputs, then trains
    each bridge variant end to end through the frozen downstream blocks and evaluates the whole student.
    """
    import hashlib
    import importlib.metadata
    import inspect
    import json
    import os
    import socket
    import uuid
    from pathlib import Path
    import torch
    from torch.nn.attention import sdpa_kernel, SDPBackend
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache

    directory = Path('/phase3/phase7/bridge_end_to_end') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete healing attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()
    report = {'request_key': key, 'status': 'failed', 'mode': request['mode'], 'teacher_calls': 0,
              'student_calls': 0, 'optimizer_updates': 0, 'controls': [], 'artifacts': [],
              'process': {'pid': os.getpid(), 'host': socket.gethostname(), 'nonce': uuid.uuid4().hex,
                          'modal_task_id': os.environ.get('MODAL_TASK_ID')}}

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def gpu(value):
        if isinstance(value, torch.Tensor):
            return value.to('cuda')
        if isinstance(value, dict):
            return {k: gpu(v) for k, v in value.items()}
        if isinstance(value, (list, tuple)):
            return type(value)(gpu(v) for v in value)
        return value

    def compare(x, y):
        a, b = x.float(), y.float()
        if not torch.isfinite(a).all() or not torch.isfinite(b).all():
            raise RuntimeError('Nonfinite reference tensor.')
        d = a - b
        return {'relative_l2_error': float(d.norm()/b.norm().clamp_min(1e-12)),
                'max_abs_error': float(d.abs().max()), 'exact_equal': bool(torch.equal(x, y))}

    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('GPU differs.')
        report['packages'] = {p: importlib.metadata.version(p) for p in ('torch', 'diffusers', 'transformers', 'accelerate', 'pillow')}
        report['torch_cuda'] = torch.version.cuda
        report['cudnn_version'] = torch.backends.cudnn.version()
        report['settings'] = request['arithmetic_contract']
        if request['mode'] != 'verify':
            raise RuntimeError('Healing requires the verified teacher reference.')
        reference_dir = Path('/phase3/phase6/deterministic_reference') / request['reference_key']
        reference_report_path = reference_dir / 'summary.json'
        if hashlib.sha256(json.dumps(json.loads(reference_report_path.read_text()), sort_keys=True).encode()).hexdigest() != request['reference_report_semantic_sha256']:
            raise RuntimeError('Generation report differs.')
        reference_report = json.loads(reference_report_path.read_text())
        if reference_report['status'] != 'reference_generated':
            raise RuntimeError('Reference generation did not complete.')
        if reference_report['settings'] != request['arithmetic_contract']:
            raise RuntimeError('Arithmetic contract differs from generation.')
        report['reference_process'] = reference_report['process']
        if reference_report['process']['nonce'] == report['process']['nonce']:
            raise RuntimeError('Independent process requirement failed.')
        if reference_report['packages'] != report['packages'] or reference_report['torch_cuda'] != report['torch_cuda'] or reference_report['cudnn_version'] != report['cudnn_version']:
            raise RuntimeError('Numerical runtime differs across generation and verification.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
            dtype=torch.bfloat16, local_files_only=True).to('cuda')
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Model source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--'))
                       / 'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        state = {}
        for kind, values in (('parameter', model.named_parameters()), ('buffer', model.named_buffers())):
            for name, value in values:
                cpu = value.detach().cpu().contiguous()
                state[name] = {'kind': kind, 'dtype': str(value.dtype), 'shape': list(value.shape),
                    'stride': list(value.stride()), 'sha256': hashlib.sha256(memoryview(cpu.view(torch.uint8).numpy())).hexdigest()}
        for axis, value in enumerate(model.pos_embed.freqs):
            name = f'pos_embed.freqs.{axis}'
            cpu = value.detach().cpu().contiguous()
            state[name] = {'kind': 'unregistered_rope_table', 'dtype': str(value.dtype), 'shape': list(value.shape),
                'stride': list(value.stride()), 'sha256': hashlib.sha256(memoryview(cpu.view(torch.uint8).numpy())).hexdigest()}
        expected = request['expected_state']
        derived_names = sorted(n for n in expected if n.startswith('pos_embed.freqs.') or n == 'time_text_embed.time_proj.freqs')

        def derived_tensors():
            values = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
            values['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
            return values

        def table_hash(value):
            return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

        canonical_path = Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt')
        import time
        mismatched = [n for n in derived_names if state.get(n) != expected.get(n)]
        if not mismatched:
            if not canonical_path.exists():
                canonical_path.parent.mkdir(parents=True, exist_ok=True)
                temporary = canonical_path.with_suffix('.' + report['process']['nonce'] + '.tmp')
                torch.save({n: v.detach().cpu().contiguous() for n, v in derived_tensors().items()}, temporary)
                temporary.replace(canonical_path)
                results_volume.commit()
                report['canonical_tables_saved'] = True
        else:
            deadline = time.time() + 900
            while not canonical_path.exists():
                if time.time() > deadline:
                    raise RuntimeError('Derived tables differ on this host and no canonical copy became available.')
                time.sleep(20)
                results_volume.reload()
            saved = torch.load(canonical_path, weights_only=True, map_location='cpu')
            if sorted(saved) != derived_names or any(table_hash(saved[n]) != expected[n]['sha256'] for n in derived_names):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [saved[f'pos_embed.freqs.{axis}'].to(device=value.device, dtype=value.dtype)
                        for axis, value in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(saved['time_text_embed.time_proj.freqs'])
            for n, value in derived_tensors().items():
                state[n] = {**state[n], 'sha256': table_hash(value)}
            report['canonical_tables_applied'] = mismatched
        if state != request['expected_state']:
            expected = request['expected_state']
            different_names = sorted(name for name in expected.keys() & state.keys()
                                     if state[name] != expected[name])
            report['state_mismatch'] = {
                'actual_count': len(state), 'expected_count': len(expected),
                'missing_names': sorted(set(expected) - set(state))[:12],
                'extra_names': sorted(set(state) - set(expected))[:12],
                'different': [{'name': name, 'actual': state[name], 'expected': expected[name]}
                              for name in different_names][:12]}
            allowed_derived_tables = {'pos_embed.freqs.1', 'pos_embed.freqs.2', 'time_text_embed.time_proj.freqs'}
            if set(state) != set(expected) or not set(different_names).issubset(allowed_derived_tables):
                raise RuntimeError('Loaded teacher learned state differs from audited reference.')
            report['derived_table_variation_requires_output_replay'] = different_names
        report['state_sha256'] = hashlib.sha256(json.dumps(state, sort_keys=True).encode()).hexdigest()
        torch.cuda.synchronize()
        argument_order = ('hidden_states', 'timestep', 'encoder_hidden_states', 'encoder_hidden_states_mask',
                          'img_shapes', 'img_mask', 'attention_kwargs', 'kv_cache_mode', 'return_dict')
        # Every intact-teacher control must pass before any block is touched.
        for record in request['records']:
            path = Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt'
            if digest(path) != record['tensor_sha256']:
                raise RuntimeError('Legacy input bundle differs.')
            bundle = torch.load(path, weights_only=True, map_location='cpu')
            outputs, controls = [], []
            cache = QwenImage21KVCache(32)
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for stage in bundle['stages']:
                    full = {**bundle['shared_transformer_kwargs'], **stage['kwargs']}
                    kwargs = gpu({field: full[field] for field in argument_order})
                    kwargs['kv_cache'] = cache
                    pred = model(**kwargs)[0][:, -stage['target_image_tokens']:].detach().cpu().contiguous()
                    report['teacher_calls'] += 1
                    legacy = compare(pred, stage['teacher_velocity'])
                    entry = {'source_id': record['source_id'], 'split': bundle['row']['split'],
                             'stage': stage['stage'], 'legacy_difference': legacy}
                    outputs.append({'stage': stage['stage'], 'step_index': stage['step_index'],
                                    'scheduler_sigma': stage['scheduler_sigma'], 'velocity': pred})
                    controls.append(entry)
            filename = record['capture_id'] + '.pt'
            artifact = next(a for a in reference_report['artifacts'] if a['name'] == filename)
            if digest(reference_dir / filename) != artifact['sha256']:
                raise RuntimeError('Reference output checksum differs.')
            reference_bundle = torch.load(reference_dir / filename, weights_only=True, map_location='cpu')
            if reference_bundle['source_id'] != record['source_id'] or reference_bundle['input_sha256'] != record['tensor_sha256']:
                raise RuntimeError('Reference input provenance differs.')
            reference_outputs = {v['stage']: v['velocity'] for v in reference_bundle['outputs']}
            for entry, measured in zip(controls, outputs):
                entry['reference_difference'] = compare(measured['velocity'], reference_outputs[entry['stage']])
                entry['passed'] = entry['reference_difference']['relative_l2_error'] <= 0.001
            output_path = directory / filename
            temporary = output_path.with_suffix('.tmp')
            torch.save({'request_key': key, 'source_id': record['source_id'], 'input_sha256': record['tensor_sha256'],
                        'outputs': outputs, 'controls': controls}, temporary)
            temporary.replace(output_path)
            report['controls'].extend(controls)
            report['artifacts'].append({'name': filename, 'sha256': digest(output_path), 'bytes': output_path.stat().st_size})
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            if not all(c['passed'] for c in controls):
                raise RuntimeError('Independent-process reference replay exceeds tolerance.')
        if report['teacher_calls'] != 90:
            raise RuntimeError('Reference attempt did not cover all 90 saved stage inputs.')
        report['legacy_max_relative_l2'] = max(c['legacy_difference']['relative_l2_error'] for c in report['controls'])
        report['max_reference_relative_l2'] = max(c['reference_difference']['relative_l2_error'] for c in report['controls'])
        report['exact_equal_controls'] = sum(c['reference_difference']['exact_equal'] for c in report['controls'])
        if not all(c['passed'] for c in report['controls']):
            raise RuntimeError('Teacher controls did not pass.')
        report['status'] = 'teacher_verified'
        (directory / 'teacher_control_report.json').write_text(json.dumps(report, indent=2))
        results_volume.commit()
        print('All 90 intact-teacher controls passed; assembling frozen 31-block candidate B.', flush=True)

        import gc
        import time
        import statistics
        from types import SimpleNamespace
        
        # Inputs are independently identified; 10 gate and 50 wider development prompts.
        panels = {'gate10': [], 'wider50': []}
        for panel, records in (('gate10', [r for r in request['records'] if r['capture_id'].startswith('p5-validation-')]),
                               ('wider50', request['expanded_validation']['records'])):
            for record in records:
                path = Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt'
                if digest(path) != record['tensor_sha256']:
                    raise RuntimeError('Fixed input hash differs')
                item = torch.load(path, weights_only=True, map_location='cpu')
                if item['row']['source_id'] != record['source_id'] or item['row']['split'] != 'validation':
                    raise RuntimeError('Validation input identity differs')
                panels[panel].append(item)
        if [len(panels[p]) for p in ('gate10', 'wider50')] != [10, 50]:
            raise RuntimeError('Saved validation panel sizes differ')

        def forward(item, stage, cache):
            full = {**item['shared_transformer_kwargs'], **stage['kwargs']}
            kwargs = gpu({field: full[field] for field in argument_order})
            kwargs['kv_cache'] = cache
            return model(**kwargs)[0][:, -stage['target_image_tokens']:].detach().cpu().contiguous()

        def predictions():
            rows = []
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for panel, items in panels.items():
                    for item in items:
                        cache = QwenImage21KVCache(32)
                        for stage in item['stages']:
                            pred = forward(item, stage, cache)
                            if not torch.isfinite(pred).all() or pred.float().norm() == 0:
                                raise RuntimeError('Invalid reference velocity')
                            rows.append(dict(panel=panel, source_id=item['row']['source_id'],
                                             dimension=item['row']['dimension'],
                                             subdimension=item['row'].get('subdimension'),
                                             stage=stage['stage'], velocity=pred))
            return rows

        def save(name, value):
            path = directory / name
            temporary = path.with_suffix('.tmp')
            torch.save(value, temporary)
            temporary.replace(path)
            artifact = dict(name=name, sha256=digest(path), bytes=path.stat().st_size)
            report['artifacts'].append(artifact)
            return artifact

        teacher_predictions = predictions()
        save('teacher_predictions.pt', teacher_predictions)
        report['teacher_calls'] += len(teacher_predictions)
        # ---- Live teacher targets for TRAIN inputs (stored legacy targets differ by up to ~2.6%). ----
        def load_verified(path, sha256, size=None):
            if (size is not None and path.stat().st_size != size) or digest(path) != sha256:
                raise RuntimeError('Input tensor differs: ' + str(path))
            return torch.load(path, weights_only=True, map_location='cpu')

        train_full, early_pool = [], []
        for record in request['records']:
            if not record['capture_id'].startswith('p5-train-'):
                continue
            item = load_verified(Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt', record['tensor_sha256'])
            if item['row']['split'] != 'train' or item['row']['source_id'] != record['source_id']:
                raise RuntimeError('Training record identity differs.')
            train_full.append(item)
        fresh = request['fresh_aligned_train']
        for artifact in fresh['artifacts']:
            item = load_verified(Path(fresh['remote_directory']) / artifact['name'], artifact['sha256'], artifact['bytes'])
            if item['source_id'] != artifact['source_id'] or item['row']['split'] != 'train' or len(item['stages']) != 3:
                raise RuntimeError('Fresh aligned training provenance differs.')
            train_full.append(item)
        for example in request['early_pool']['examples']:
            item = load_verified(Path(example['remote_path']), example['sha256'], example['bytes'])
            if (item['capture_id'] != example['capture_id'] or item['row']['split'] != 'train'
                    or [s['stage'] for s in item['stages']] != ['early']):
                raise RuntimeError('Early pool example provenance differs.')
            early_pool.append(item)
        validation_ids = {r['source_id'] for r in request['records'] if r['capture_id'].startswith('p5-validation-')}
        validation_ids |= {r['source_id'] for r in request['expanded_validation']['records']}
        train_ids = {i['row']['source_id'] for i in train_full} | {i['row']['source_id'] for i in early_pool}
        if len(train_full) != 60 or train_ids & validation_ids:
            raise RuntimeError('Training set size or split differs.')
        for item in train_full + early_pool:
            cache = QwenImage21KVCache(32)
            item['live_targets'] = []
            for stage in item['stages']:
                with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                    item['live_targets'].append(forward(item, stage, cache).to(torch.bfloat16))
                report['teacher_calls'] += 1
        report['train_counts'] = {'full_prompts': len(train_full), 'early_pool': len(early_pool)}
        print('Live teacher targets for', report['train_counts'], flush=True)

        # Rebuild the Phase 6 selected student exactly as its verified replay did:
        # 31 teacher blocks plus the frozen pretrained rank-256 bridge at slot 5.
        namespace = {}
        exec(compile(request['core_source'], 'bridge_definition.py', 'exec'), namespace)
        exec(compile(request['wrapper_source'], 'bridge_block_definition.py', 'exec'), namespace)
        candidate = request['selected_candidate']
        path = Path('/phase3/phase5/fitting') / request['fit_key'] / Path(candidate['checkpoint_path']).name
        if digest(path) != candidate['checkpoint_sha256']:
            raise RuntimeError('Selected bridge checkpoint differs.')
        checkpoint = torch.load(path, weights_only=True, map_location='cpu')
        if (checkpoint['request_key'] != request['fit_key'] or checkpoint['layer'] != 5
                or checkpoint['step'] != candidate['selected_update']):
            raise RuntimeError('Bridge provenance differs.')
        bridge = namespace['ResidualBottleneckBridge'](4096, 256)
        bridge.load_state_dict(checkpoint['state_dict'])
        bridge.requires_grad_(False)
        namespace['insert_bridge'](model, 5, bridge, QwenImage21KVCache)
        del checkpoint
        model.requires_grad_(False).eval()
        gc.collect()
        torch.cuda.empty_cache()
        baseline = predictions()
        report['student_calls'] += len(baseline)
        save('candidate_predictions.pt', baseline)
        # The 10 gate prompts must reproduce the saved Phase 6 trained-bridge velocities.
        replay = request['selected_replay']
        replay_dir = Path('/phase3/phase6/verified_student_replay') / replay['request_key']
        reference = {}
        for record in request['records']:
            if not record['capture_id'].startswith('p5-validation-'):
                continue
            name = 'students-' + record['capture_id'] + '.pt'
            if digest(replay_dir / name) != replay['artifacts'][name]:
                raise RuntimeError('Phase 6 student replay checksum differs.')
            saved = torch.load(replay_dir / name, weights_only=True, map_location='cpu')
            for output in saved['outputs']:
                if output['candidate_id'] == candidate['candidate_id'] and output['variant'] == 'trained':
                    reference[(output['source_id'], output['stage'])] = output['velocity']
        gate_rows = [r for r in baseline if r['panel'] == 'gate10']
        if len(reference) != 30 or len(gate_rows) != 30:
            raise RuntimeError('Phase 6 replay coverage differs.')
        reproduction = [compare(r['velocity'], reference[(r['source_id'], r['stage'])]) for r in gate_rows]
        report['candidate_replay_max_relative_l2'] = max(r['relative_l2_error'] for r in reproduction)
        if report['candidate_replay_max_relative_l2'] > 0.001:
            raise RuntimeError('Selected student failed Phase 6 output replay')
        del reference

        class IdentityBlock(torch.nn.Module):
            def __init__(self, original):
                super().__init__()
                self.attn = SimpleNamespace(processor=original.attn.processor)

            def forward(self, hidden_states, *args, **kwargs):
                return hidden_states

        def metrics(rows):
            output = []
            for measured, base, teacher_row in zip(rows, baseline, teacher_predictions):
                identity = ('panel', 'source_id', 'stage')
                if any(measured[k] != base[k] or base[k] != teacher_row[k] for k in identity):
                    raise RuntimeError('Prediction alignment differs')
                teacher_error = compare(measured['velocity'], teacher_row['velocity'])['relative_l2_error']
                baseline_error = compare(base['velocity'], teacher_row['velocity'])['relative_l2_error']
                output.append({k: measured[k] for k in measured if k != 'velocity'} |
                              dict(baseline_displacement=compare(measured['velocity'], base['velocity'])['relative_l2_error'],
                                   teacher_relative_l2=teacher_error, baseline_teacher_relative_l2=baseline_error,
                                   teacher_error_change_percentage_points=100 * (teacher_error - baseline_error)))
            return output

        def aggregate(rows):
            fields = ('baseline_displacement', 'teacher_relative_l2', 'baseline_teacher_relative_l2',
                      'teacher_error_change_percentage_points')
            result = dict(comparisons=len(rows))
            for field in fields:
                values = sorted(r[field] for r in rows)
                def percentile(fraction):
                    position = fraction * (len(values) - 1)
                    low = int(position)
                    return values[low] + (values[min(low + 1, len(values) - 1)] - values[low]) * (position - low)
                result[field] = dict(mean=statistics.mean(values), maximum=max(values),
                                     p10=percentile(.1), p50=percentile(.5), p90=percentile(.9))
            result['worst_regressions'] = sorted(rows, key=lambda r:r['teacher_error_change_percentage_points'], reverse=True)[:10]
            return result

        report['baseline'] = aggregate(metrics(baseline))

        # ---- End-to-end bridge training: only the bridge at the slot learns, from the final velocity error. ----
        import random
        from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVLayerCache
        slot = request['slot']
        fit = request['fit']
        conditioned = {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned)
        components = {}
        exec(compile(request['components_source'], 'healing_components.py', 'exec'), components)
        warm_spec = request['warm_start']
        warm_path = Path(warm_spec['remote_path'])
        if digest(warm_path) != warm_spec['sha256']:
            raise RuntimeError('Warm-start checkpoint differs.')
        warm = torch.load(warm_path, weights_only=True, map_location='cpu')
        if warm['layer'] != slot or warm['variant']['name'] != warm_spec['name']:
            raise RuntimeError('Warm-start provenance differs.')
        original_blocks = list(model.transformer_blocks)
        for index in range(slot + 1, len(original_blocks)):
            model.transformer_blocks[index] = components['FunctionalCheckpointBlock'](original_blocks[index], QwenImage21KVLayerCache)
        model.requires_grad_(False)
        model.gradient_checkpointing = False
        gate_teacher = [r for r in teacher_predictions if r['panel'] == 'gate10']

        def gate_error():
            errors = []
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                position = 0
                for item in panels['gate10']:
                    cache = QwenImage21KVCache(32)
                    for stage in item['stages']:
                        reference = gate_teacher[position]
                        if reference['source_id'] != item['row']['source_id'] or reference['stage'] != stage['stage']:
                            raise RuntimeError('Gate alignment differs.')
                        errors.append(compare(forward(item, stage, cache), reference['velocity'])['relative_l2_error'])
                        position += 1
            return sum(errors) / len(errors)

        def train_loss(item):
            cache = QwenImage21KVCache(32)
            losses = []
            with model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for stage, target in zip(item['stages'], item['live_targets']):
                    full = {**item['shared_transformer_kwargs'], **stage['kwargs']}
                    kwargs = gpu({field: full[field] for field in argument_order})
                    kwargs['kv_cache'] = cache
                    prediction = model(**kwargs)[0][:, -stage['target_image_tokens']:].float()
                    target = target.to('cuda').float()
                    losses.append((prediction - target).square().mean() / target.square().mean().clamp_min(1e-12))
                    report['student_calls'] += 1
            loss = torch.stack(losses).mean()
            if not torch.isfinite(loss):
                raise RuntimeError('Nonfinite training loss.')
            return loss

        report['variants'] = {}
        for variant in request['variants']:
            if variant['attention']:
                bridge = conditioned['AttentionConditionedBridge'](4096, 1024, variant['heads'], 128)
            else:
                bridge = conditioned['ConditionedBridge'](4096, 1024, True)
            missing, unexpected = bridge.load_state_dict(warm['state_dict'], strict=False)
            if unexpected or (missing and not variant['attention']):
                raise RuntimeError(f'Warm start mismatch: missing {missing}, unexpected {unexpected}')
            torch.manual_seed(fit['init_seed'])
            if variant['attention']:
                for projection in (bridge.to_q, bridge.to_k, bridge.to_v):
                    torch.nn.init.normal_(projection.weight, std=projection.in_features ** -0.5)
            bridge = bridge.float()
            conditioned['insert_conditioned_bridge'](model, slot, bridge, keep_dtype=True)
            bridge = model.transformer_blocks[slot].bridge
            bridge.requires_grad_(True)
            optimizer = torch.optim.AdamW(bridge.parameters(), lr=fit['learning_rate'], weight_decay=0.0)
            rng = random.Random(fit['sampling_seed'])
            curve, best = [], None
            started = time.perf_counter()
            for step in range(fit['updates'] + 1):
                if step % fit['eval_every'] == 0:
                    bridge.eval()
                    score = gate_error()
                    curve.append(dict(step=step, gate10=score, seconds=time.perf_counter() - started))
                    if best is None or score < best[0]:
                        best = (score, step, {k: v.detach().cpu().clone() for k, v in bridge.state_dict().items()})
                    print(f'{variant["name"]} step {step}: gate10 {score:.6f} ({time.perf_counter() - started:.0f}s)', flush=True)
                    (directory / 'progress.json').write_text(json.dumps(report | {'current_curve': curve}, indent=2))
                    results_volume.commit()
                if step == fit['updates']:
                    break
                for group in optimizer.param_groups:
                    group['lr'] = fit['learning_rate'] * min(1.0, (step + 1) / fit['warmup'])
                optimizer.zero_grad(set_to_none=True)
                # Backward recomputes checkpointed blocks: keep the same attention kernel and cache context as forward.
                with model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                    (train_loss(rng.choice(train_full)) * (1 - fit['early_weight'])).backward()
                    (train_loss(rng.choice(early_pool)) * fit['early_weight']).backward()
                torch.nn.utils.clip_grad_norm_(bridge.parameters(), fit['gradient_clip'])
                optimizer.step()
                report['optimizer_updates'] += 1
            bridge.load_state_dict(best[2])
            bridge.requires_grad_(False)
            bridge.eval()
            checkpoint_path = directory / f'bridge_block_{slot:02d}_{variant["name"]}.pt'
            temporary = checkpoint_path.with_suffix('.tmp')
            torch.save({'request_key': key, 'layer': slot, 'variant': variant, 'step': best[1], 'state_dict': best[2]}, temporary)
            temporary.replace(checkpoint_path)
            artifact = dict(name=checkpoint_path.name, sha256=digest(checkpoint_path), bytes=checkpoint_path.stat().st_size)
            report['artifacts'].append(artifact)
            rows = predictions()
            report['student_calls'] += len(rows)
            records = metrics(rows)
            report['variants'][variant['name']] = dict(
                variant=variant, checkpoint=artifact, parameters=sum(v.numel() for v in best[2].values()),
                fit=dict(curve=curve, selected_step=best[1], selected_gate10=best[0]),
                overall=aggregate(records),
                panels={p: aggregate([r for r in records if r['panel'] == p]) for p in panels},
                stages={s: aggregate([r for r in records if r['stage'] == s]) for s in ('early', 'middle', 'late')},
                records=records)
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            print(f'{variant["name"]}: teacher-relative {report["variants"][variant["name"]]["overall"]["teacher_relative_l2"]["mean"]:.6f}', flush=True)
            del optimizer
            gc.collect()
            torch.cuda.empty_cache()
        report['status'] = 'passed'
        report['scope'] = 'End-to-end bridge training (bridge weights only) and fixed-input velocity evaluation; images are separate.'
    except Exception as exc:
        import traceback
        report['status'] = 'failed'
        report['error'] = repr(exc)
        report['traceback'] = traceback.format_exc()
    (directory / 'summary.json').write_text(json.dumps(report, indent=2))
    results_volume.commit()
    return report
'''


SOURCES['conditioned_bridge.py'] = r'''import torch
from torch import nn
from types import SimpleNamespace


class ConditionedBridge(nn.Module):
    """Residual bottleneck bridge that can read the model's shared timestep modulation.

    The transformer computes one modulation tensor per timestep, [attn scale, attn gate, MLP scale, MLP gate],
    shared by every block. A conditioned bridge mirrors a block's MLP branch:
    h + tanh(gate) * up(gelu(down(LN(h) * (1 + scale)))), with scale and gate learned per-channel mixes of the
    shared values. Unconditioned, it is the original Phase 4 bridge: h + up(gelu(down(h))).
    """

    def __init__(self, hidden_dim=4096, rank=256, conditioned=True, eps=1e-6):
        super().__init__()
        self.conditioned = conditioned
        self.down = nn.Linear(hidden_dim, rank, bias=False)
        self.up = nn.Linear(rank, hidden_dim, bias=False)
        nn.init.zeros_(self.up.weight)
        if conditioned:
            self.norm = nn.LayerNorm(hidden_dim, elementwise_affine=False, eps=eps)
            # Start from the block's own MLP modulation (weight 1 on the MLP terms, 0 on the attention terms).
            mix = torch.zeros(2, hidden_dim)
            mix[1] = 1
            self.scale_mix = nn.Parameter(mix.clone())
            self.gate_mix = nn.Parameter(mix.clone())
            self.scale_bias = nn.Parameter(torch.zeros(hidden_dim))
            self.gate_bias = nn.Parameter(torch.zeros(hidden_dim))

    def scale_gate(self, modulation_rows):
        scale1, gate1, scale2, gate2 = modulation_rows.chunk(4, dim=-1)
        scale = self.scale_mix[0] * scale1 + self.scale_mix[1] * scale2 + self.scale_bias
        gate = self.gate_mix[0] * gate1 + self.gate_mix[1] * gate2 + self.gate_bias
        return scale, gate

    def forward(self, hidden_states, modulation_rows=None):
        if not self.conditioned:
            return hidden_states + self.up(nn.functional.gelu(self.down(hidden_states)))
        scale, gate = self.scale_gate(modulation_rows)
        update = self.up(nn.functional.gelu(self.down(self.norm(hidden_states) * (1 + scale))))
        return hidden_states + torch.tanh(gate) * update


class LocalConditionedBridge(ConditionedBridge):
    """Conditioned bridge plus a depthwise 3x3 convolution over the image-token grid inside the bottleneck.

    The convolution starts at zero (identity start) and is applied to target-image tokens only; prompt tokens have
    no grid and pass through the per-token path.
    """
    local = True

    def __init__(self, hidden_dim=4096, rank=1024, eps=1e-6):
        super().__init__(hidden_dim, rank, True, eps)
        self.mix = nn.Parameter(torch.zeros(rank, 9))

    def forward(self, hidden_states, modulation_rows=None, image_mask=None, grid=None):
        scale, gate = self.scale_gate(modulation_rows)
        z = self.down(self.norm(hidden_states) * (1 + scale))
        if grid is not None:
            height, width = grid
            image = z[:, image_mask]
            batch, tokens, rank = image.shape
            planes = image.transpose(1, 2).reshape(batch, rank, height, width)
            mixed = nn.functional.conv2d(planes, self.mix.view(rank, 1, 3, 3).to(planes.dtype), padding=1, groups=rank)
            z = z.clone()
            z[:, image_mask] = image + mixed.reshape(batch, rank, tokens).transpose(1, 2)
        return hidden_states + torch.tanh(gate) * self.up(nn.functional.gelu(z))

    def forward_neighborhood(self, neighborhood, modulation_rows):
        """Training path: neighborhood is (tokens, 9, hidden) in 3x3 raster order, zeros outside the grid."""
        scale, gate = self.scale_gate(modulation_rows)
        z = self.down(self.norm(neighborhood) * (1 + scale.unsqueeze(1)))
        z = z[:, 4] + (z * self.mix.t().unsqueeze(0)).sum(1)
        return neighborhood[:, 4] + torch.tanh(gate) * self.up(nn.functional.gelu(z))


class AttentionConditionedBridge(ConditionedBridge):
    """Conditioned bridge with a small attention branch in front of its MLP, mirroring a block.

    Target-image tokens attend to the prompt prefix and all image tokens with the model's RoPE and key masks; prefix
    tokens only take the MLP path. The branch keeps its own prefix K/V in the layer cache of the slot it replaces.
    The output projection starts at zero, so a fresh branch leaves the conditioned MLP bridge unchanged.
    """
    attention = True

    def __init__(self, hidden_dim=4096, rank=1024, heads=4, head_dim=128, eps=1e-6):
        super().__init__(hidden_dim, rank, True, eps)
        inner = heads * head_dim
        self.heads, self.head_dim, self.eps = heads, head_dim, eps
        self.to_q = nn.Linear(hidden_dim, inner, bias=False)
        self.to_k = nn.Linear(hidden_dim, inner, bias=False)
        self.to_v = nn.Linear(hidden_dim, inner, bias=False)
        self.to_out = nn.Linear(inner, hidden_dim, bias=False)
        nn.init.zeros_(self.to_out.weight)
        self.norm_q = nn.Parameter(torch.ones(head_dim))
        self.norm_k = nn.Parameter(torch.ones(head_dim))
        # Attention modulation starts from the block's own attention terms (weight 1 on scale1/gate1).
        mix = torch.zeros(2, hidden_dim)
        mix[0] = 1
        self.attn_scale_mix = nn.Parameter(mix.clone())
        self.attn_gate_mix = nn.Parameter(mix.clone())
        self.attn_scale_bias = nn.Parameter(torch.zeros(hidden_dim))
        self.attn_gate_bias = nn.Parameter(torch.zeros(hidden_dim))

    def _rms(self, x, weight):
        return x * torch.rsqrt(x.float().square().mean(-1, keepdim=True) + self.eps).to(x.dtype) * weight

    @staticmethod
    def _rope(x, freqs):
        rotated = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2)) * freqs.unsqueeze(1)
        return torch.view_as_real(rotated).flatten(3).type_as(x)

    def attend(self, hidden_states, modulation_rows, rotary_emb, target_count, layer_cache, kv_cache_mode,
               cache_write_slice, key_mask):
        scale1, gate1, scale2, gate2 = modulation_rows.chunk(4, dim=-1)
        scale = self.attn_scale_mix[0] * scale1 + self.attn_scale_mix[1] * scale2 + self.attn_scale_bias
        gate = self.attn_gate_mix[0] * gate1 + self.attn_gate_mix[1] * gate2 + self.attn_gate_bias
        x = self.norm(hidden_states) * (1 + scale)
        q, k, v = (proj(x).unflatten(-1, (self.heads, self.head_dim)) for proj in (self.to_q, self.to_k, self.to_v))
        q, k = self._rms(q, self.norm_q), self._rms(k, self.norm_k)
        if rotary_emb is not None:
            q, k = self._rope(q, rotary_emb), self._rope(k, rotary_emb)
        if layer_cache is not None:
            if kv_cache_mode == 'extract' and cache_write_slice is not None:
                layer_cache.store(k[:, cache_write_slice].clone(), v[:, cache_write_slice].clone())
            elif kv_cache_mode == 'cached':
                cached_k, cached_v = layer_cache.get()
                k, v = torch.cat([cached_k, k], dim=1), torch.cat([cached_v, v], dim=1)
        prefix = hidden_states.shape[1] - target_count
        out = nn.functional.scaled_dot_product_attention(
            q[:, prefix:].transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2), attn_mask=key_mask)
        update = torch.tanh(gate[:, prefix:]) * self.to_out(out.transpose(1, 2).flatten(2))
        return torch.cat([hidden_states[:, :prefix], hidden_states[:, prefix:] + update], dim=1)


def modulation_rows(modulation, target_token_mask):
    """Per-token modulation rows, selected exactly as the transformer block does."""
    if target_token_mask is None:
        return modulation.unsqueeze(1)
    real, zero = modulation[:-1].unsqueeze(1), modulation[-1:].unsqueeze(0)
    return torch.where(target_token_mask.view(1, -1, 1), real, zero)


class ConditionedBridgeBlock(nn.Module):
    def __init__(self, bridge, processor):
        super().__init__()
        self.bridge = bridge
        # Parent forward inspects this processor to choose mask construction; no attention is performed here.
        self.attn = SimpleNamespace(processor=processor)

    def forward(self, hidden_states, modulation, rotary_emb=None, attention_mask=None, target_token_mask=None,
                layer_cache=None, kv_cache_mode=None, cache_write_slice=None, segments=None, key_valid=None):
        if kv_cache_mode not in (None, 'extract', 'cached'):
            raise ValueError('Unknown KV cache mode.')
        rows = modulation_rows(modulation, target_token_mask) if self.bridge.conditioned else None
        dtype = hidden_states.dtype
        bridge_dtype = next(self.bridge.parameters()).dtype
        if getattr(self.bridge, 'attention', False):
            # Small bridges may keep FP32 weights for training; compute in their dtype and return the model dtype.
            target_count = hidden_states.shape[1] if target_token_mask is None else int(target_token_mask.sum())
            key_mask = key_valid[:, None, None, :] if key_valid is not None else attention_mask
            h = hidden_states.to(bridge_dtype)
            r = rows.to(bridge_dtype)
            h = self.bridge.attend(h, r, rotary_emb, target_count, layer_cache, kv_cache_mode, cache_write_slice, key_mask)
            return self.bridge(h, r).to(dtype)
        if bridge_dtype != dtype:
            return self.bridge(hidden_states.to(bridge_dtype), None if rows is None else rows.to(bridge_dtype)).to(dtype)
        if getattr(self.bridge, 'local', False):
            mask = target_token_mask if target_token_mask is not None else torch.ones(
                hidden_states.shape[1], dtype=torch.bool, device=hidden_states.device)
            height, width = self.grid
            if height * width != int(mask.sum()):
                raise RuntimeError('Target image grid does not match the image-token count.')
            return self.bridge(hidden_states, rows, mask, (height, width))
        return self.bridge(hidden_states, rows)


def insert_conditioned_bridge(transformer, layer, bridge, keep_dtype=False):
    """Replace one slot; callers must use fresh KV caches afterwards. keep_dtype keeps FP32 bridge weights."""
    old = transformer.transformer_blocks[layer]
    reference = next(transformer.parameters())
    bridge = bridge.to(device=reference.device) if keep_dtype else bridge.to(device=reference.device, dtype=reference.dtype)
    wrapper = ConditionedBridgeBlock(bridge, old.attn.processor)
    wrapper.train(transformer.training)
    transformer.transformer_blocks[layer] = wrapper
    if getattr(bridge, 'local', False) and not getattr(transformer, '_bridge_grid_hook', False):
        # The block never sees img_shapes; record the target image's (height, width) in tokens on every call.
        def record_grid(module, args, kwargs):
            frames, height, width = kwargs['img_shapes'][0][-1]
            for block in module.transformer_blocks:
                if isinstance(block, ConditionedBridgeBlock):
                    block.grid = (int(height), int(width))
        transformer.register_forward_pre_hook(record_grid, with_kwargs=True)
        transformer._bridge_grid_hook = True
    return old
'''


"""Train block bridges end to end (final velocity loss, bridge weights only) inside candidate B; reuse completed results."""
import argparse
import hashlib
import json
import sys
from pathlib import Path


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--slot', type=int, default=13)
    parser.add_argument('--old-account', action='store_true')
    parser.add_argument('--dry-run', action='store_true', help='Write the request locally without scheduling GPU work.')
    args = parser.parse_args(LAUNCH_ARGS)
    if args.old_account:
        parser.error('Old account is download-only; GPU execution must use the new account.')
    root = ROOT
    protocol_path = root / f'data/phase7/block{args.slot}_bridge_e2e/protocol.json'
    protocol = json.loads(protocol_path.read_bytes())
    scan_b = json.loads(next((root / 'data/phase7/conditional_importance_b').glob('*/request.json')).read_bytes())['request']
    scan_a = json.loads((root / 'data/phase7/conditional_importance/7454da6a5c1dc33af23dd46833e698f9b759d14985ca96340bc3d40cd47fb4f5/request.json').read_bytes())['request']
    if args.slot == 5 or not 0 < args.slot < 31:
        parser.error('Slot must be a surviving interior block.')
    request = {k: v for k, v in scan_b.items() if k not in ('worker_source', 'eligible_original_block_ids', 'selection', 'scope', 'version')}
    warm = next(r for r in (json.loads(p.read_bytes()) for p in (root / 'data/phase7/block13_bridge_v3').glob('*/summary.json')) if r.get('status') == 'passed')
    source = SOURCES['bridge_end_to_end_worker.py']
    request.update(
        version='phase7-bridge-end-to-end-v1', worker_source=source, slot=args.slot,
        protocol_sha256=hashlib.sha256(protocol_path.read_bytes()).hexdigest(),
        conditioned_bridge_source=SOURCES['conditioned_bridge.py'],
        components_source=scan_a['components_source'],
        fresh_aligned_train=scan_a['fresh_aligned_train'], early_pool=scan_a['early_pool'],
        warm_start=dict(name='cond_r1024_long', remote_path=f"/phase3/phase7/bridge_variants_v3/{warm['request_key']}/{warm['variants']['cond_r1024_long']['checkpoint']['name']}",
                        sha256=warm['variants']['cond_r1024_long']['checkpoint']['sha256']),
        variants=[dict(name='e2e_attn', attention=True, heads=4), dict(name='e2e_cond', attention=False)],
        fit=dict(init_seed=20261017, sampling_seed=20261019, learning_rate=2e-4, warmup=20, gradient_clip=1.0,
                 updates=300, eval_every=50, early_weight=0.3),
        selection='Lowest gate10 velocity error at evaluations every 50 updates, including the warm start',
        scope=f'End-to-end bridges for block {args.slot} inside frozen B; TRAIN-only; whole-student velocity on scan inputs; no images or latency')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = protocol_path.parent / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2)+'\n', encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase7/bridge_end_to_end/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        try:
            b''.join(results_volume.read_file(remote + '/started.json'))
        except FileNotFoundError:
            pass
        else:
            raise RuntimeError('Run already started; inspect its progress. No duplicate GPU scheduled.')
        app = modal.App('nihonga-phase7-bridge-end-to-end')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(source, 'bridge_end_to_end_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8,
                              memory=98304, max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['train_bridge_end_to_end'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2)+'\n')
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2)+'\n', encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'candidate_replay_max_relative_l2')}), flush=True)
    for name, v in (report.get('variants') or {}).items():
        print(name, 'step', v['fit']['selected_step'], 'teacher-relative', v['overall']['teacher_relative_l2']['mean'], flush=True)
    if report.get('status') != 'passed':
        raise SystemExit(1)

summary = reuse_or_launch(RUN, main)

show_variants(summary)

### What the results mean

The objective alone gives nothing (`e2e_cond` never beats its start). Attention gives the first clear step: **6.42%** overall, 6.29% wider. (The first launch stopped at the first backward pass because checkpoint recomputation ran outside the forward attention-kernel context; the fixed run is shown.)

## 11b. Text check of the attention bridge

Four seeds per text prompt plus the 30 standard prompts, candidate only (teacher and B reused).

In [ ]:
# Text check of the attention bridge: reuses the saved run; launches GPU work only when its result is missing and ALLOW_GPU is True.

RUN = 'data/phase7/multiseed_text/8611910a2e5f829c8a5f663372457de6ab2481528914e1f3ea7a49023b9a055b'

LAUNCH_ARGS = ['--source', 'block13_bridge_e2e', '--variant', 'e2e_attn', '--with-base-prompts']

SOURCES['multiseed_text_worker.py'] = r'''def generate_multiseed_text(request, key):
    """Extra seeds for the text prompts: teacher, B, and C with the best block-13 bridge.
    Same prompt, CPU noise seed, size, scheduler and 40 steps for every model. Qualitative screen only."""
    import gc
    import hashlib
    import inspect
    import json
    import os
    import time
    from pathlib import Path

    import torch
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache, QwenImage21KVLayerCache

    directory = Path('/phase3/phase7/multiseed_text') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete image attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def table_hash(value):
        return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

    variants = ['teacher', 'B', request['c_variant']['name']]
    report = {'request_key': key, 'status': 'running', 'rows': [], 'artifacts': [],
              'calls': {v: 0 for v in variants}, 'optimizer_updates': 0}
    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('Expected H100.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
                                                   dtype=torch.bfloat16, local_files_only=True).to('cuda')
        pipe.set_progress_bar_config(disable=True)
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Transformer source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--')) /
                       'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        # Derived position/time tables: use the audited values on every host (as in steps 7-11).
        derived = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
        derived['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
        expected = request['expected_derived_tables']
        mismatched = sorted(n for n, v in derived.items() if table_hash(v) != expected[n])
        if mismatched:
            canonical = torch.load(Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt'),
                                   weights_only=True, map_location='cpu')
            if any(table_hash(canonical[n]) != expected[n] for n in expected):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [canonical[f'pos_embed.freqs.{a}'].to(device=v.device, dtype=v.dtype) for a, v in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(canonical['time_text_embed.time_proj.freqs'])
            if any(table_hash(v) != expected[n] for n, v in
                   list({f'pos_embed.freqs.{a}': v for a, v in enumerate(model.pos_embed.freqs)}.items()) +
                   [('time_text_embed.time_proj.freqs', model.time_text_embed.time_proj.freqs)]):
                raise RuntimeError('Derived tables still differ after loading the audited copy.')
            report['canonical_tables_applied'] = mismatched

        def generate(row, variant):
            path = directory / (row['source_id'] + '-' + variant + '.png')
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'],
                             num_inference_steps=40, true_cfg_scale=teacher['true_cfg_scale'],
                             generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=True, output_type='pil').images[0]
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            temporary = path.with_suffix('.tmp')
            image.save(temporary, format='PNG')
            temporary.replace(path)
            results_volume.commit()
            report['calls'][variant] += 1
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            return {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size, 'seconds': elapsed}

        namespace = {}
        exec(compile(request['core_source'], 'bridge_definition.py', 'exec'), namespace)
        exec(compile(request['wrapper_source'], 'bridge_block_definition.py', 'exec'), namespace)
        conditioned = {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned)

        def load_checkpoint(spec):
            path = Path(spec['remote_path'])
            if digest(path) != spec['sha256']:
                raise RuntimeError('Bridge checkpoint differs: ' + spec['remote_path'])
            return torch.load(path, weights_only=True, map_location='cpu')

        outputs = {v: {} for v in variants}
        references = not request.get('candidate_only', False)
        if references:
            for row in request['rows']:
                outputs['teacher'][row['source_id']] = generate(row, 'teacher')
        block5 = request['block5_bridge']
        checkpoint = load_checkpoint(block5)
        if checkpoint['layer'] != 5 or checkpoint['step'] != block5['step']:
            raise RuntimeError('Block-5 bridge provenance differs.')
        bridge = namespace['ResidualBottleneckBridge'](4096, 256)
        bridge.load_state_dict(checkpoint['state_dict'])
        bridge.requires_grad_(False)
        namespace['insert_bridge'](model, 5, bridge, QwenImage21KVCache)
        model.eval()
        if references:
            for row in request['rows']:
                outputs['B'][row['source_id']] = generate(row, 'B')
        spec = request['c_variant']
        checkpoint = load_checkpoint(spec)
        if checkpoint['layer'] != 13 or checkpoint['variant']['name'] != spec['name']:
            raise RuntimeError('Block-13 bridge provenance differs.')
        fitted = checkpoint['variant']
        keep_dtype = False
        if fitted.get('attention'):
            bridge13 = conditioned['AttentionConditionedBridge'](4096, 1024, fitted['heads'], 128)
            keep_dtype = True
        elif 'attention' in fitted:
            bridge13 = conditioned['ConditionedBridge'](4096, 1024, True)
            keep_dtype = True
        elif fitted.get('local'):
            bridge13 = conditioned['LocalConditionedBridge'](4096, fitted['rank'])
        else:
            bridge13 = conditioned['ConditionedBridge'](4096, fitted['rank'], fitted['conditioned'])
        bridge13.load_state_dict(checkpoint['state_dict'])
        bridge13.requires_grad_(False)
        conditioned['insert_conditioned_bridge'](model, 13, bridge13.float() if keep_dtype else bridge13, keep_dtype=keep_dtype)
        model.eval()
        for row in request['rows']:
            outputs[spec['name']][row['source_id']] = generate(row, spec['name'])
        rendered = [v for v in variants if outputs[v]]
        for row in request['rows']:
            report['rows'].append({**row, **{v: outputs[v][row['source_id']] for v in rendered}})
            report['artifacts'].extend(outputs[v][row['source_id']] for v in rendered)
        report['status'] = 'passed'
    except Exception as exc:
        report['status'] = 'failed'
        report['error'] = type(exc).__name__ + ': ' + str(exc)
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=True) + '\n')
    results_volume.commit()
    return report
'''


SOURCES['conditioned_bridge.py'] = r'''import torch
from torch import nn
from types import SimpleNamespace


class ConditionedBridge(nn.Module):
    """Residual bottleneck bridge that can read the model's shared timestep modulation.

    The transformer computes one modulation tensor per timestep, [attn scale, attn gate, MLP scale, MLP gate],
    shared by every block. A conditioned bridge mirrors a block's MLP branch:
    h + tanh(gate) * up(gelu(down(LN(h) * (1 + scale)))), with scale and gate learned per-channel mixes of the
    shared values. Unconditioned, it is the original Phase 4 bridge: h + up(gelu(down(h))).
    """

    def __init__(self, hidden_dim=4096, rank=256, conditioned=True, eps=1e-6):
        super().__init__()
        self.conditioned = conditioned
        self.down = nn.Linear(hidden_dim, rank, bias=False)
        self.up = nn.Linear(rank, hidden_dim, bias=False)
        nn.init.zeros_(self.up.weight)
        if conditioned:
            self.norm = nn.LayerNorm(hidden_dim, elementwise_affine=False, eps=eps)
            # Start from the block's own MLP modulation (weight 1 on the MLP terms, 0 on the attention terms).
            mix = torch.zeros(2, hidden_dim)
            mix[1] = 1
            self.scale_mix = nn.Parameter(mix.clone())
            self.gate_mix = nn.Parameter(mix.clone())
            self.scale_bias = nn.Parameter(torch.zeros(hidden_dim))
            self.gate_bias = nn.Parameter(torch.zeros(hidden_dim))

    def scale_gate(self, modulation_rows):
        scale1, gate1, scale2, gate2 = modulation_rows.chunk(4, dim=-1)
        scale = self.scale_mix[0] * scale1 + self.scale_mix[1] * scale2 + self.scale_bias
        gate = self.gate_mix[0] * gate1 + self.gate_mix[1] * gate2 + self.gate_bias
        return scale, gate

    def forward(self, hidden_states, modulation_rows=None):
        if not self.conditioned:
            return hidden_states + self.up(nn.functional.gelu(self.down(hidden_states)))
        scale, gate = self.scale_gate(modulation_rows)
        update = self.up(nn.functional.gelu(self.down(self.norm(hidden_states) * (1 + scale))))
        return hidden_states + torch.tanh(gate) * update


class LocalConditionedBridge(ConditionedBridge):
    """Conditioned bridge plus a depthwise 3x3 convolution over the image-token grid inside the bottleneck.

    The convolution starts at zero (identity start) and is applied to target-image tokens only; prompt tokens have
    no grid and pass through the per-token path.
    """
    local = True

    def __init__(self, hidden_dim=4096, rank=1024, eps=1e-6):
        super().__init__(hidden_dim, rank, True, eps)
        self.mix = nn.Parameter(torch.zeros(rank, 9))

    def forward(self, hidden_states, modulation_rows=None, image_mask=None, grid=None):
        scale, gate = self.scale_gate(modulation_rows)
        z = self.down(self.norm(hidden_states) * (1 + scale))
        if grid is not None:
            height, width = grid
            image = z[:, image_mask]
            batch, tokens, rank = image.shape
            planes = image.transpose(1, 2).reshape(batch, rank, height, width)
            mixed = nn.functional.conv2d(planes, self.mix.view(rank, 1, 3, 3).to(planes.dtype), padding=1, groups=rank)
            z = z.clone()
            z[:, image_mask] = image + mixed.reshape(batch, rank, tokens).transpose(1, 2)
        return hidden_states + torch.tanh(gate) * self.up(nn.functional.gelu(z))

    def forward_neighborhood(self, neighborhood, modulation_rows):
        """Training path: neighborhood is (tokens, 9, hidden) in 3x3 raster order, zeros outside the grid."""
        scale, gate = self.scale_gate(modulation_rows)
        z = self.down(self.norm(neighborhood) * (1 + scale.unsqueeze(1)))
        z = z[:, 4] + (z * self.mix.t().unsqueeze(0)).sum(1)
        return neighborhood[:, 4] + torch.tanh(gate) * self.up(nn.functional.gelu(z))


class AttentionConditionedBridge(ConditionedBridge):
    """Conditioned bridge with a small attention branch in front of its MLP, mirroring a block.

    Target-image tokens attend to the prompt prefix and all image tokens with the model's RoPE and key masks; prefix
    tokens only take the MLP path. The branch keeps its own prefix K/V in the layer cache of the slot it replaces.
    The output projection starts at zero, so a fresh branch leaves the conditioned MLP bridge unchanged.
    """
    attention = True

    def __init__(self, hidden_dim=4096, rank=1024, heads=4, head_dim=128, eps=1e-6):
        super().__init__(hidden_dim, rank, True, eps)
        inner = heads * head_dim
        self.heads, self.head_dim, self.eps = heads, head_dim, eps
        self.to_q = nn.Linear(hidden_dim, inner, bias=False)
        self.to_k = nn.Linear(hidden_dim, inner, bias=False)
        self.to_v = nn.Linear(hidden_dim, inner, bias=False)
        self.to_out = nn.Linear(inner, hidden_dim, bias=False)
        nn.init.zeros_(self.to_out.weight)
        self.norm_q = nn.Parameter(torch.ones(head_dim))
        self.norm_k = nn.Parameter(torch.ones(head_dim))
        # Attention modulation starts from the block's own attention terms (weight 1 on scale1/gate1).
        mix = torch.zeros(2, hidden_dim)
        mix[0] = 1
        self.attn_scale_mix = nn.Parameter(mix.clone())
        self.attn_gate_mix = nn.Parameter(mix.clone())
        self.attn_scale_bias = nn.Parameter(torch.zeros(hidden_dim))
        self.attn_gate_bias = nn.Parameter(torch.zeros(hidden_dim))

    def _rms(self, x, weight):
        return x * torch.rsqrt(x.float().square().mean(-1, keepdim=True) + self.eps).to(x.dtype) * weight

    @staticmethod
    def _rope(x, freqs):
        rotated = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2)) * freqs.unsqueeze(1)
        return torch.view_as_real(rotated).flatten(3).type_as(x)

    def attend(self, hidden_states, modulation_rows, rotary_emb, target_count, layer_cache, kv_cache_mode,
               cache_write_slice, key_mask):
        scale1, gate1, scale2, gate2 = modulation_rows.chunk(4, dim=-1)
        scale = self.attn_scale_mix[0] * scale1 + self.attn_scale_mix[1] * scale2 + self.attn_scale_bias
        gate = self.attn_gate_mix[0] * gate1 + self.attn_gate_mix[1] * gate2 + self.attn_gate_bias
        x = self.norm(hidden_states) * (1 + scale)
        q, k, v = (proj(x).unflatten(-1, (self.heads, self.head_dim)) for proj in (self.to_q, self.to_k, self.to_v))
        q, k = self._rms(q, self.norm_q), self._rms(k, self.norm_k)
        if rotary_emb is not None:
            q, k = self._rope(q, rotary_emb), self._rope(k, rotary_emb)
        if layer_cache is not None:
            if kv_cache_mode == 'extract' and cache_write_slice is not None:
                layer_cache.store(k[:, cache_write_slice].clone(), v[:, cache_write_slice].clone())
            elif kv_cache_mode == 'cached':
                cached_k, cached_v = layer_cache.get()
                k, v = torch.cat([cached_k, k], dim=1), torch.cat([cached_v, v], dim=1)
        prefix = hidden_states.shape[1] - target_count
        out = nn.functional.scaled_dot_product_attention(
            q[:, prefix:].transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2), attn_mask=key_mask)
        update = torch.tanh(gate[:, prefix:]) * self.to_out(out.transpose(1, 2).flatten(2))
        return torch.cat([hidden_states[:, :prefix], hidden_states[:, prefix:] + update], dim=1)


def modulation_rows(modulation, target_token_mask):
    """Per-token modulation rows, selected exactly as the transformer block does."""
    if target_token_mask is None:
        return modulation.unsqueeze(1)
    real, zero = modulation[:-1].unsqueeze(1), modulation[-1:].unsqueeze(0)
    return torch.where(target_token_mask.view(1, -1, 1), real, zero)


class ConditionedBridgeBlock(nn.Module):
    def __init__(self, bridge, processor):
        super().__init__()
        self.bridge = bridge
        # Parent forward inspects this processor to choose mask construction; no attention is performed here.
        self.attn = SimpleNamespace(processor=processor)

    def forward(self, hidden_states, modulation, rotary_emb=None, attention_mask=None, target_token_mask=None,
                layer_cache=None, kv_cache_mode=None, cache_write_slice=None, segments=None, key_valid=None):
        if kv_cache_mode not in (None, 'extract', 'cached'):
            raise ValueError('Unknown KV cache mode.')
        rows = modulation_rows(modulation, target_token_mask) if self.bridge.conditioned else None
        dtype = hidden_states.dtype
        bridge_dtype = next(self.bridge.parameters()).dtype
        if getattr(self.bridge, 'attention', False):
            # Small bridges may keep FP32 weights for training; compute in their dtype and return the model dtype.
            target_count = hidden_states.shape[1] if target_token_mask is None else int(target_token_mask.sum())
            key_mask = key_valid[:, None, None, :] if key_valid is not None else attention_mask
            h = hidden_states.to(bridge_dtype)
            r = rows.to(bridge_dtype)
            h = self.bridge.attend(h, r, rotary_emb, target_count, layer_cache, kv_cache_mode, cache_write_slice, key_mask)
            return self.bridge(h, r).to(dtype)
        if bridge_dtype != dtype:
            return self.bridge(hidden_states.to(bridge_dtype), None if rows is None else rows.to(bridge_dtype)).to(dtype)
        if getattr(self.bridge, 'local', False):
            mask = target_token_mask if target_token_mask is not None else torch.ones(
                hidden_states.shape[1], dtype=torch.bool, device=hidden_states.device)
            height, width = self.grid
            if height * width != int(mask.sum()):
                raise RuntimeError('Target image grid does not match the image-token count.')
            return self.bridge(hidden_states, rows, mask, (height, width))
        return self.bridge(hidden_states, rows)


def insert_conditioned_bridge(transformer, layer, bridge, keep_dtype=False):
    """Replace one slot; callers must use fresh KV caches afterwards. keep_dtype keeps FP32 bridge weights."""
    old = transformer.transformer_blocks[layer]
    reference = next(transformer.parameters())
    bridge = bridge.to(device=reference.device) if keep_dtype else bridge.to(device=reference.device, dtype=reference.dtype)
    wrapper = ConditionedBridgeBlock(bridge, old.attn.processor)
    wrapper.train(transformer.training)
    transformer.transformer_blocks[layer] = wrapper
    if getattr(bridge, 'local', False) and not getattr(transformer, '_bridge_grid_hook', False):
        # The block never sees img_shapes; record the target image's (height, width) in tokens on every call.
        def record_grid(module, args, kwargs):
            frames, height, width = kwargs['img_shapes'][0][-1]
            for block in module.transformer_blocks:
                if isinstance(block, ConditionedBridgeBlock):
                    block.grid = (int(height), int(width))
        transformer.register_forward_pre_hook(record_grid, with_kwargs=True)
        transformer._bridge_grid_hook = True
    return old
'''


"""Render the text prompts with extra seeds for teacher, B and C (best block-13 bridge); qualitative only."""
import argparse
import hashlib
import json
import sys
from pathlib import Path


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--dry-run', action='store_true')
    parser.add_argument('--variant', default='cond_r1024_long')
    parser.add_argument('--source', default='block13_bridge_v3', help='Run folder holding the bridge checkpoint.')
    parser.add_argument('--with-base-prompts', action='store_true', help='Also render the 30 base prompts at seed 0; candidate only.')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    recovered = root / 'data/recovered/modal'
    images_key = 'dda8c184e657dfb604c35fb261d83371c108c39fac033a0f57db66f7686c30ad'
    base = json.loads((recovered / 'phase7/candidate_images' / images_key / 'request.json').read_bytes())['request']
    scan_b = json.loads(next((root / 'data/phase7/conditional_importance_b').glob('*/request.json')).read_bytes())['request']
    fit = next(r for r in (json.loads(p.read_bytes()) for p in (root / 'data/phase7' / args.source).glob('*/summary.json')) if r.get('status') == 'passed')
    best = fit['variants'][args.variant]
    remote_dir = {'block13_bridge_v3': 'bridge_variants_v3', 'block13_bridge_e2e': 'bridge_end_to_end', 'block13_bridge_e2e_text': 'bridge_end_to_end_v2', 'block13_bridge_e2e_text8': 'bridge_end_to_end_v3'}[args.source]
    selected = scan_b['selected_candidate']
    text_rows = [r for r in base['rows'] if r['subdimension'] in ('text_rendering', 'posters')]
    rows = [{**r, 'source_id': f"{r['source_id']}-s{k}", 'seed': (r['seed'] + 1000003 * k) % 2**32}
            for r in text_rows for k in (1, 2, 3)]
    if args.with_base_prompts:
        rows += base['rows']
    request = {k: base[k] for k in ('teacher', 'transformer_source_sha256', 'runtime_environment', 'expected_derived_tables',
                                    'canonical_tables_key', 'core_source', 'wrapper_source')}
    request.update(
        version='phase7-multiseed-text-v1', rows=rows,
        worker_source=SOURCES['multiseed_text_worker.py'],
        conditioned_bridge_source=SOURCES['conditioned_bridge.py'],
        execution_account='new',
        block5_bridge=dict(remote_path=f"/phase3/phase5/fitting/{scan_b['fit_key']}/{Path(selected['checkpoint_path']).name}",
                           sha256=selected['checkpoint_sha256'], step=selected['selected_update']),
        c_variant=dict(name=args.variant, remote_path=f"/phase3/phase7/{remote_dir}/{fit['request_key']}/{best['checkpoint']['name']}",
                       sha256=best['checkpoint']['sha256']),
        candidate_only=args.with_base_prompts,
        scope='Extra seeds (3 per prompt) for the 5 text prompts of the Phase 7 image set' + ('; plus the 30 base prompts at seed 0, candidate only' if args.with_base_prompts else '; seed 0 images are reused'))
    from experiment_runtime import pin_reference_runtime
    pin_reference_runtime(request, recovered)
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase7/multiseed_text' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2)+'\n', encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase7/multiseed_text/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        try:
            b''.join(results_volume.read_file(remote + '/started.json'))
        except FileNotFoundError:
            pass
        else:
            raise RuntimeError('Run already started; inspect its progress. No duplicate GPU scheduled.')
        app = modal.App('nihonga-phase7-multiseed-text')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'multiseed_text_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8,
                              memory=98304, max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['generate_multiseed_text'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2)+'\n')
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2)+'\n', encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'calls')}), flush=True)
    if report.get('status') != 'passed':
        raise SystemExit(1)

summary = reuse_or_launch(RUN, main)

show_images('../niwaki-page/nihonga/images', ['phase7-anchor-seed0.jpg', 'phase7-anchor-seeds.jpg'])

### What the results mean

On the original seed the attention bridge is the first to write "The Rusty Anchor" correctly; across four seeds it is right once (B three times). The cause is clear: only **2 of its 60** full training prompts contain text.

## 12. More text in training: 150 recorded text prompts

200 new TRAIN prompts (150 text, 50 general) are recorded with the intact teacher inside the job (steps 0 and 19 of the 40-step trajectory); a replay must reproduce the teacher exactly. The attention bridge then continues for 600 updates, 40% of them on text.

In [ ]:
# More text in training: 150 recorded text prompts: reuses the saved run; launches GPU work only when its result is missing and ALLOW_GPU is True.

RUN = 'data/phase7/block13_bridge_e2e_text/b5a259abba4f24b392a11b81e4d448a97365941066d09a57e107d45efe390d92'

LAUNCH_ARGS = ['--slot', '13']

SOURCES['bridge_end_to_end_v2_worker.py'] = r'''def train_bridge_end_to_end(request, key):
    """Train a bridge for one block inside frozen B against the final velocity (bridge weights only).

    Verifies the teacher contract and exact B replay, computes live teacher targets for TRAIN inputs, then trains
    each bridge variant end to end through the frozen downstream blocks and evaluates the whole student.
    """
    import hashlib
    import importlib.metadata
    import inspect
    import json
    import os
    import socket
    import uuid
    from pathlib import Path
    import torch
    from torch.nn.attention import sdpa_kernel, SDPBackend
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache

    directory = Path('/phase3/phase7/bridge_end_to_end_v2') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete healing attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()
    report = {'request_key': key, 'status': 'failed', 'mode': request['mode'], 'teacher_calls': 0,
              'student_calls': 0, 'optimizer_updates': 0, 'controls': [], 'artifacts': [],
              'process': {'pid': os.getpid(), 'host': socket.gethostname(), 'nonce': uuid.uuid4().hex,
                          'modal_task_id': os.environ.get('MODAL_TASK_ID')}}

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def gpu(value):
        if isinstance(value, torch.Tensor):
            return value.to('cuda')
        if isinstance(value, dict):
            return {k: gpu(v) for k, v in value.items()}
        if isinstance(value, (list, tuple)):
            return type(value)(gpu(v) for v in value)
        return value

    def compare(x, y):
        a, b = x.float(), y.float()
        if not torch.isfinite(a).all() or not torch.isfinite(b).all():
            raise RuntimeError('Nonfinite reference tensor.')
        d = a - b
        return {'relative_l2_error': float(d.norm()/b.norm().clamp_min(1e-12)),
                'max_abs_error': float(d.abs().max()), 'exact_equal': bool(torch.equal(x, y))}

    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('GPU differs.')
        report['packages'] = {p: importlib.metadata.version(p) for p in ('torch', 'diffusers', 'transformers', 'accelerate', 'pillow')}
        report['torch_cuda'] = torch.version.cuda
        report['cudnn_version'] = torch.backends.cudnn.version()
        report['settings'] = request['arithmetic_contract']
        if request['mode'] != 'verify':
            raise RuntimeError('Healing requires the verified teacher reference.')
        reference_dir = Path('/phase3/phase6/deterministic_reference') / request['reference_key']
        reference_report_path = reference_dir / 'summary.json'
        if hashlib.sha256(json.dumps(json.loads(reference_report_path.read_text()), sort_keys=True).encode()).hexdigest() != request['reference_report_semantic_sha256']:
            raise RuntimeError('Generation report differs.')
        reference_report = json.loads(reference_report_path.read_text())
        if reference_report['status'] != 'reference_generated':
            raise RuntimeError('Reference generation did not complete.')
        if reference_report['settings'] != request['arithmetic_contract']:
            raise RuntimeError('Arithmetic contract differs from generation.')
        report['reference_process'] = reference_report['process']
        if reference_report['process']['nonce'] == report['process']['nonce']:
            raise RuntimeError('Independent process requirement failed.')
        if reference_report['packages'] != report['packages'] or reference_report['torch_cuda'] != report['torch_cuda'] or reference_report['cudnn_version'] != report['cudnn_version']:
            raise RuntimeError('Numerical runtime differs across generation and verification.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
            dtype=torch.bfloat16, local_files_only=True).to('cuda')
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Model source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--'))
                       / 'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        state = {}
        for kind, values in (('parameter', model.named_parameters()), ('buffer', model.named_buffers())):
            for name, value in values:
                cpu = value.detach().cpu().contiguous()
                state[name] = {'kind': kind, 'dtype': str(value.dtype), 'shape': list(value.shape),
                    'stride': list(value.stride()), 'sha256': hashlib.sha256(memoryview(cpu.view(torch.uint8).numpy())).hexdigest()}
        for axis, value in enumerate(model.pos_embed.freqs):
            name = f'pos_embed.freqs.{axis}'
            cpu = value.detach().cpu().contiguous()
            state[name] = {'kind': 'unregistered_rope_table', 'dtype': str(value.dtype), 'shape': list(value.shape),
                'stride': list(value.stride()), 'sha256': hashlib.sha256(memoryview(cpu.view(torch.uint8).numpy())).hexdigest()}
        expected = request['expected_state']
        derived_names = sorted(n for n in expected if n.startswith('pos_embed.freqs.') or n == 'time_text_embed.time_proj.freqs')

        def derived_tensors():
            values = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
            values['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
            return values

        def table_hash(value):
            return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

        canonical_path = Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt')
        import time
        mismatched = [n for n in derived_names if state.get(n) != expected.get(n)]
        if not mismatched:
            if not canonical_path.exists():
                canonical_path.parent.mkdir(parents=True, exist_ok=True)
                temporary = canonical_path.with_suffix('.' + report['process']['nonce'] + '.tmp')
                torch.save({n: v.detach().cpu().contiguous() for n, v in derived_tensors().items()}, temporary)
                temporary.replace(canonical_path)
                results_volume.commit()
                report['canonical_tables_saved'] = True
        else:
            deadline = time.time() + 900
            while not canonical_path.exists():
                if time.time() > deadline:
                    raise RuntimeError('Derived tables differ on this host and no canonical copy became available.')
                time.sleep(20)
                results_volume.reload()
            saved = torch.load(canonical_path, weights_only=True, map_location='cpu')
            if sorted(saved) != derived_names or any(table_hash(saved[n]) != expected[n]['sha256'] for n in derived_names):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [saved[f'pos_embed.freqs.{axis}'].to(device=value.device, dtype=value.dtype)
                        for axis, value in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(saved['time_text_embed.time_proj.freqs'])
            for n, value in derived_tensors().items():
                state[n] = {**state[n], 'sha256': table_hash(value)}
            report['canonical_tables_applied'] = mismatched
        if state != request['expected_state']:
            expected = request['expected_state']
            different_names = sorted(name for name in expected.keys() & state.keys()
                                     if state[name] != expected[name])
            report['state_mismatch'] = {
                'actual_count': len(state), 'expected_count': len(expected),
                'missing_names': sorted(set(expected) - set(state))[:12],
                'extra_names': sorted(set(state) - set(expected))[:12],
                'different': [{'name': name, 'actual': state[name], 'expected': expected[name]}
                              for name in different_names][:12]}
            allowed_derived_tables = {'pos_embed.freqs.1', 'pos_embed.freqs.2', 'time_text_embed.time_proj.freqs'}
            if set(state) != set(expected) or not set(different_names).issubset(allowed_derived_tables):
                raise RuntimeError('Loaded teacher learned state differs from audited reference.')
            report['derived_table_variation_requires_output_replay'] = different_names
        report['state_sha256'] = hashlib.sha256(json.dumps(state, sort_keys=True).encode()).hexdigest()
        torch.cuda.synchronize()
        argument_order = ('hidden_states', 'timestep', 'encoder_hidden_states', 'encoder_hidden_states_mask',
                          'img_shapes', 'img_mask', 'attention_kwargs', 'kv_cache_mode', 'return_dict')
        # Every intact-teacher control must pass before any block is touched.
        for record in request['records']:
            path = Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt'
            if digest(path) != record['tensor_sha256']:
                raise RuntimeError('Legacy input bundle differs.')
            bundle = torch.load(path, weights_only=True, map_location='cpu')
            outputs, controls = [], []
            cache = QwenImage21KVCache(32)
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for stage in bundle['stages']:
                    full = {**bundle['shared_transformer_kwargs'], **stage['kwargs']}
                    kwargs = gpu({field: full[field] for field in argument_order})
                    kwargs['kv_cache'] = cache
                    pred = model(**kwargs)[0][:, -stage['target_image_tokens']:].detach().cpu().contiguous()
                    report['teacher_calls'] += 1
                    legacy = compare(pred, stage['teacher_velocity'])
                    entry = {'source_id': record['source_id'], 'split': bundle['row']['split'],
                             'stage': stage['stage'], 'legacy_difference': legacy}
                    outputs.append({'stage': stage['stage'], 'step_index': stage['step_index'],
                                    'scheduler_sigma': stage['scheduler_sigma'], 'velocity': pred})
                    controls.append(entry)
            filename = record['capture_id'] + '.pt'
            artifact = next(a for a in reference_report['artifacts'] if a['name'] == filename)
            if digest(reference_dir / filename) != artifact['sha256']:
                raise RuntimeError('Reference output checksum differs.')
            reference_bundle = torch.load(reference_dir / filename, weights_only=True, map_location='cpu')
            if reference_bundle['source_id'] != record['source_id'] or reference_bundle['input_sha256'] != record['tensor_sha256']:
                raise RuntimeError('Reference input provenance differs.')
            reference_outputs = {v['stage']: v['velocity'] for v in reference_bundle['outputs']}
            for entry, measured in zip(controls, outputs):
                entry['reference_difference'] = compare(measured['velocity'], reference_outputs[entry['stage']])
                entry['passed'] = entry['reference_difference']['relative_l2_error'] <= 0.001
            output_path = directory / filename
            temporary = output_path.with_suffix('.tmp')
            torch.save({'request_key': key, 'source_id': record['source_id'], 'input_sha256': record['tensor_sha256'],
                        'outputs': outputs, 'controls': controls}, temporary)
            temporary.replace(output_path)
            report['controls'].extend(controls)
            report['artifacts'].append({'name': filename, 'sha256': digest(output_path), 'bytes': output_path.stat().st_size})
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            if not all(c['passed'] for c in controls):
                raise RuntimeError('Independent-process reference replay exceeds tolerance.')
        if report['teacher_calls'] != 90:
            raise RuntimeError('Reference attempt did not cover all 90 saved stage inputs.')
        report['legacy_max_relative_l2'] = max(c['legacy_difference']['relative_l2_error'] for c in report['controls'])
        report['max_reference_relative_l2'] = max(c['reference_difference']['relative_l2_error'] for c in report['controls'])
        report['exact_equal_controls'] = sum(c['reference_difference']['exact_equal'] for c in report['controls'])
        if not all(c['passed'] for c in report['controls']):
            raise RuntimeError('Teacher controls did not pass.')
        report['status'] = 'teacher_verified'
        (directory / 'teacher_control_report.json').write_text(json.dumps(report, indent=2))
        results_volume.commit()
        print('All 90 intact-teacher controls passed; assembling frozen 31-block candidate B.', flush=True)

        import gc
        import time
        import statistics
        from types import SimpleNamespace
        
        # Inputs are independently identified; 10 gate and 50 wider development prompts.
        panels = {'gate10': [], 'wider50': []}
        for panel, records in (('gate10', [r for r in request['records'] if r['capture_id'].startswith('p5-validation-')]),
                               ('wider50', request['expanded_validation']['records'])):
            for record in records:
                path = Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt'
                if digest(path) != record['tensor_sha256']:
                    raise RuntimeError('Fixed input hash differs')
                item = torch.load(path, weights_only=True, map_location='cpu')
                if item['row']['source_id'] != record['source_id'] or item['row']['split'] != 'validation':
                    raise RuntimeError('Validation input identity differs')
                panels[panel].append(item)
        if [len(panels[p]) for p in ('gate10', 'wider50')] != [10, 50]:
            raise RuntimeError('Saved validation panel sizes differ')

        def forward(item, stage, cache):
            full = {**item['shared_transformer_kwargs'], **stage['kwargs']}
            kwargs = gpu({field: full[field] for field in argument_order})
            kwargs['kv_cache'] = cache
            return model(**kwargs)[0][:, -stage['target_image_tokens']:].detach().cpu().contiguous()

        def predictions():
            rows = []
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for panel, items in panels.items():
                    for item in items:
                        cache = QwenImage21KVCache(32)
                        for stage in item['stages']:
                            pred = forward(item, stage, cache)
                            if not torch.isfinite(pred).all() or pred.float().norm() == 0:
                                raise RuntimeError('Invalid reference velocity')
                            rows.append(dict(panel=panel, source_id=item['row']['source_id'],
                                             dimension=item['row']['dimension'],
                                             subdimension=item['row'].get('subdimension'),
                                             stage=stage['stage'], velocity=pred))
            return rows

        def save(name, value):
            path = directory / name
            temporary = path.with_suffix('.tmp')
            torch.save(value, temporary)
            temporary.replace(path)
            artifact = dict(name=name, sha256=digest(path), bytes=path.stat().st_size)
            report['artifacts'].append(artifact)
            return artifact

        teacher_predictions = predictions()
        save('teacher_predictions.pt', teacher_predictions)
        report['teacher_calls'] += len(teacher_predictions)
        # ---- Live teacher targets for TRAIN inputs (stored legacy targets differ by up to ~2.6%). ----
        def load_verified(path, sha256, size=None):
            if (size is not None and path.stat().st_size != size) or digest(path) != sha256:
                raise RuntimeError('Input tensor differs: ' + str(path))
            return torch.load(path, weights_only=True, map_location='cpu')

        train_full, early_pool = [], []
        for record in request['records']:
            if not record['capture_id'].startswith('p5-train-'):
                continue
            item = load_verified(Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt', record['tensor_sha256'])
            if item['row']['split'] != 'train' or item['row']['source_id'] != record['source_id']:
                raise RuntimeError('Training record identity differs.')
            train_full.append(item)
        fresh = request['fresh_aligned_train']
        for artifact in fresh['artifacts']:
            item = load_verified(Path(fresh['remote_directory']) / artifact['name'], artifact['sha256'], artifact['bytes'])
            if item['source_id'] != artifact['source_id'] or item['row']['split'] != 'train' or len(item['stages']) != 3:
                raise RuntimeError('Fresh aligned training provenance differs.')
            train_full.append(item)
        for example in request['early_pool']['examples']:
            item = load_verified(Path(example['remote_path']), example['sha256'], example['bytes'])
            if (item['capture_id'] != example['capture_id'] or item['row']['split'] != 'train'
                    or [s['stage'] for s in item['stages']] != ['early']):
                raise RuntimeError('Early pool example provenance differs.')
            early_pool.append(item)
        validation_ids = {r['source_id'] for r in request['records'] if r['capture_id'].startswith('p5-validation-')}
        validation_ids |= {r['source_id'] for r in request['expanded_validation']['records']}
        train_ids = {i['row']['source_id'] for i in train_full} | {i['row']['source_id'] for i in early_pool}
        if len(train_full) != 60 or train_ids & validation_ids:
            raise RuntimeError('Training set size or split differs.')
        for item in train_full + early_pool:
            cache = QwenImage21KVCache(32)
            item['live_targets'] = []
            for stage in item['stages']:
                with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                    item['live_targets'].append(forward(item, stage, cache).to(torch.bfloat16))
                report['teacher_calls'] += 1
        report['train_counts'] = {'full_prompts': len(train_full), 'early_pool': len(early_pool)}
        print('Live teacher targets for', report['train_counts'], flush=True)

        # ---- Record new TRAIN prompts with the intact teacher (steps 0 and 19 of the 40-step trajectory). ----
        import math

        class StopCapture(Exception):
            pass

        record_steps = {0: 'early', 19: 'middle'}
        capture_state = {'calls': 0, 'stages': [], 'kwargs': None}

        def cpu_tree(value):
            if isinstance(value, torch.Tensor):
                return value.detach().cpu().clone()
            if isinstance(value, dict):
                return {k: cpu_tree(v) for k, v in value.items()}
            if isinstance(value, (list, tuple)):
                return type(value)(cpu_tree(v) for v in value)
            return value

        def pre_hook(module, args, kwargs):
            if capture_state['calls'] in record_steps:
                capture_state['kwargs'] = {field: cpu_tree(kwargs.get(field)) for field in argument_order}

        def post_hook(module, args, kwargs, output):
            step = capture_state['calls']
            capture_state['calls'] += 1
            if step in record_steps:
                stage_kwargs = capture_state['kwargs']
                tokens = math.prod(int(v) for v in stage_kwargs['img_shapes'][0][-1])
                velocity = output[0][:, -tokens:].detach().to(torch.bfloat16).cpu()
                capture_state['stages'].append(dict(stage=record_steps[step], step_index=step, kwargs=stage_kwargs,
                                                    target_image_tokens=tokens, velocity=velocity))
                if step == max(record_steps):
                    raise StopCapture()

        handles = [model.register_forward_pre_hook(pre_hook, with_kwargs=True),
                   model.register_forward_hook(post_hook, with_kwargs=True)]
        new_text, new_general = [], []
        try:
            for row in request['new_prompts']:
                capture_state.update(calls=0, stages=[], kwargs=None)
                try:
                    with torch.inference_mode(), sdpa_kernel([SDPBackend.MATH]):
                        pipe(prompt=row['prompt'], width=row['width'], height=row['height'], num_inference_steps=40,
                             true_cfg_scale=teacher['true_cfg_scale'], generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=True, output_type='latent')
                except StopCapture:
                    pass
                if [s['stage'] for s in capture_state['stages']] != ['early', 'middle']:
                    raise RuntimeError('Capture did not record the expected steps for ' + row['source_id'])
                report['teacher_calls'] += capture_state['calls']
                item = {'row': {k: row[k] for k in ('source_id', 'dimension', 'subdimension')} | {'split': 'train'},
                        'shared_transformer_kwargs': {},
                        'stages': [{k: v for k, v in s.items() if k != 'velocity'} for s in capture_state['stages']],
                        'live_targets': [s['velocity'] for s in capture_state['stages']]}
                (new_text if row['subdimension'] == 'text_rendering' else new_general).append(item)
        finally:
            for handle in handles:
                handle.remove()
        # The replay path used for training must reproduce the recorded teacher outputs.
        probe = (new_text + new_general)[0]
        cache = QwenImage21KVCache(32)
        replay = []
        with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
            for stage, target in zip(probe['stages'], probe['live_targets']):
                replay.append(compare(forward(probe, stage, cache).to(torch.bfloat16), target)['relative_l2_error'])
        report['new_prompt_replay_relative_l2'] = replay
        if max(replay) > 0.001:
            raise RuntimeError('Recorded new-prompt inputs do not replay the teacher: ' + json.dumps(replay))
        new_ids = {i['row']['source_id'] for i in new_text + new_general}
        if new_ids & (train_ids | validation_ids):
            raise RuntimeError('New prompts overlap earlier training or validation prompts.')
        report['new_prompt_counts'] = {'text': len(new_text), 'general': len(new_general)}
        print('Recorded new prompts', report['new_prompt_counts'], 'replay', replay, flush=True)

        # Rebuild the Phase 6 selected student exactly as its verified replay did:
        # 31 teacher blocks plus the frozen pretrained rank-256 bridge at slot 5.
        namespace = {}
        exec(compile(request['core_source'], 'bridge_definition.py', 'exec'), namespace)
        exec(compile(request['wrapper_source'], 'bridge_block_definition.py', 'exec'), namespace)
        candidate = request['selected_candidate']
        path = Path('/phase3/phase5/fitting') / request['fit_key'] / Path(candidate['checkpoint_path']).name
        if digest(path) != candidate['checkpoint_sha256']:
            raise RuntimeError('Selected bridge checkpoint differs.')
        checkpoint = torch.load(path, weights_only=True, map_location='cpu')
        if (checkpoint['request_key'] != request['fit_key'] or checkpoint['layer'] != 5
                or checkpoint['step'] != candidate['selected_update']):
            raise RuntimeError('Bridge provenance differs.')
        bridge = namespace['ResidualBottleneckBridge'](4096, 256)
        bridge.load_state_dict(checkpoint['state_dict'])
        bridge.requires_grad_(False)
        namespace['insert_bridge'](model, 5, bridge, QwenImage21KVCache)
        del checkpoint
        model.requires_grad_(False).eval()
        gc.collect()
        torch.cuda.empty_cache()
        baseline = predictions()
        report['student_calls'] += len(baseline)
        save('candidate_predictions.pt', baseline)
        # The 10 gate prompts must reproduce the saved Phase 6 trained-bridge velocities.
        replay = request['selected_replay']
        replay_dir = Path('/phase3/phase6/verified_student_replay') / replay['request_key']
        reference = {}
        for record in request['records']:
            if not record['capture_id'].startswith('p5-validation-'):
                continue
            name = 'students-' + record['capture_id'] + '.pt'
            if digest(replay_dir / name) != replay['artifacts'][name]:
                raise RuntimeError('Phase 6 student replay checksum differs.')
            saved = torch.load(replay_dir / name, weights_only=True, map_location='cpu')
            for output in saved['outputs']:
                if output['candidate_id'] == candidate['candidate_id'] and output['variant'] == 'trained':
                    reference[(output['source_id'], output['stage'])] = output['velocity']
        gate_rows = [r for r in baseline if r['panel'] == 'gate10']
        if len(reference) != 30 or len(gate_rows) != 30:
            raise RuntimeError('Phase 6 replay coverage differs.')
        reproduction = [compare(r['velocity'], reference[(r['source_id'], r['stage'])]) for r in gate_rows]
        report['candidate_replay_max_relative_l2'] = max(r['relative_l2_error'] for r in reproduction)
        if report['candidate_replay_max_relative_l2'] > 0.001:
            raise RuntimeError('Selected student failed Phase 6 output replay')
        del reference

        class IdentityBlock(torch.nn.Module):
            def __init__(self, original):
                super().__init__()
                self.attn = SimpleNamespace(processor=original.attn.processor)

            def forward(self, hidden_states, *args, **kwargs):
                return hidden_states

        def metrics(rows):
            output = []
            for measured, base, teacher_row in zip(rows, baseline, teacher_predictions):
                identity = ('panel', 'source_id', 'stage')
                if any(measured[k] != base[k] or base[k] != teacher_row[k] for k in identity):
                    raise RuntimeError('Prediction alignment differs')
                teacher_error = compare(measured['velocity'], teacher_row['velocity'])['relative_l2_error']
                baseline_error = compare(base['velocity'], teacher_row['velocity'])['relative_l2_error']
                output.append({k: measured[k] for k in measured if k != 'velocity'} |
                              dict(baseline_displacement=compare(measured['velocity'], base['velocity'])['relative_l2_error'],
                                   teacher_relative_l2=teacher_error, baseline_teacher_relative_l2=baseline_error,
                                   teacher_error_change_percentage_points=100 * (teacher_error - baseline_error)))
            return output

        def aggregate(rows):
            fields = ('baseline_displacement', 'teacher_relative_l2', 'baseline_teacher_relative_l2',
                      'teacher_error_change_percentage_points')
            result = dict(comparisons=len(rows))
            for field in fields:
                values = sorted(r[field] for r in rows)
                def percentile(fraction):
                    position = fraction * (len(values) - 1)
                    low = int(position)
                    return values[low] + (values[min(low + 1, len(values) - 1)] - values[low]) * (position - low)
                result[field] = dict(mean=statistics.mean(values), maximum=max(values),
                                     p10=percentile(.1), p50=percentile(.5), p90=percentile(.9))
            result['worst_regressions'] = sorted(rows, key=lambda r:r['teacher_error_change_percentage_points'], reverse=True)[:10]
            return result

        report['baseline'] = aggregate(metrics(baseline))

        # ---- End-to-end bridge training: only the bridge at the slot learns, from the final velocity error. ----
        import random
        from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVLayerCache
        slot = request['slot']
        fit = request['fit']
        conditioned = {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned)
        components = {}
        exec(compile(request['components_source'], 'healing_components.py', 'exec'), components)
        warm_spec = request['warm_start']
        warm_path = Path(warm_spec['remote_path'])
        if digest(warm_path) != warm_spec['sha256']:
            raise RuntimeError('Warm-start checkpoint differs.')
        warm = torch.load(warm_path, weights_only=True, map_location='cpu')
        if warm['layer'] != slot or warm['variant']['name'] != warm_spec['name']:
            raise RuntimeError('Warm-start provenance differs.')
        original_blocks = list(model.transformer_blocks)
        for index in range(slot + 1, len(original_blocks)):
            model.transformer_blocks[index] = components['FunctionalCheckpointBlock'](original_blocks[index], QwenImage21KVLayerCache)
        model.requires_grad_(False)
        model.gradient_checkpointing = False
        gate_teacher = [r for r in teacher_predictions if r['panel'] == 'gate10']

        def gate_error():
            errors = []
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                position = 0
                for item in panels['gate10']:
                    cache = QwenImage21KVCache(32)
                    for stage in item['stages']:
                        reference = gate_teacher[position]
                        if reference['source_id'] != item['row']['source_id'] or reference['stage'] != stage['stage']:
                            raise RuntimeError('Gate alignment differs.')
                        errors.append(compare(forward(item, stage, cache), reference['velocity'])['relative_l2_error'])
                        position += 1
            return sum(errors) / len(errors)

        def train_loss(item):
            cache = QwenImage21KVCache(32)
            losses = []
            with model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for stage, target in zip(item['stages'], item['live_targets']):
                    full = {**item['shared_transformer_kwargs'], **stage['kwargs']}
                    kwargs = gpu({field: full[field] for field in argument_order})
                    kwargs['kv_cache'] = cache
                    prediction = model(**kwargs)[0][:, -stage['target_image_tokens']:].float()
                    target = target.to('cuda').float()
                    losses.append((prediction - target).square().mean() / target.square().mean().clamp_min(1e-12))
                    report['student_calls'] += 1
            loss = torch.stack(losses).mean()
            if not torch.isfinite(loss):
                raise RuntimeError('Nonfinite training loss.')
            return loss

        general_pool = train_full + new_general
        report['variants'] = {}
        for variant in request['variants']:
            if variant['attention']:
                bridge = conditioned['AttentionConditionedBridge'](4096, 1024, variant['heads'], 128)
            else:
                bridge = conditioned['ConditionedBridge'](4096, 1024, True)
            missing, unexpected = bridge.load_state_dict(warm['state_dict'], strict=False)
            if unexpected or (missing and not variant['attention']):
                raise RuntimeError(f'Warm start mismatch: missing {missing}, unexpected {unexpected}')
            torch.manual_seed(fit['init_seed'])
            if variant['attention'] and missing:
                for projection in (bridge.to_q, bridge.to_k, bridge.to_v):
                    torch.nn.init.normal_(projection.weight, std=projection.in_features ** -0.5)
            bridge = bridge.float()
            conditioned['insert_conditioned_bridge'](model, slot, bridge, keep_dtype=True)
            bridge = model.transformer_blocks[slot].bridge
            bridge.requires_grad_(True)
            optimizer = torch.optim.AdamW(bridge.parameters(), lr=fit['learning_rate'], weight_decay=0.0)
            rng = random.Random(fit['sampling_seed'])
            curve, best = [], None
            started = time.perf_counter()
            for step in range(fit['updates'] + 1):
                if step % fit['eval_every'] == 0:
                    bridge.eval()
                    score = gate_error()
                    curve.append(dict(step=step, gate10=score, seconds=time.perf_counter() - started))
                    if best is None or score < best[0]:
                        best = (score, step, {k: v.detach().cpu().clone() for k, v in bridge.state_dict().items()})
                    print(f'{variant["name"]} step {step}: gate10 {score:.6f} ({time.perf_counter() - started:.0f}s)', flush=True)
                    (directory / 'progress.json').write_text(json.dumps(report | {'current_curve': curve}, indent=2))
                    results_volume.commit()
                if step == fit['updates']:
                    break
                for group in optimizer.param_groups:
                    group['lr'] = fit['learning_rate'] * min(1.0, (step + 1) / fit['warmup'])
                optimizer.zero_grad(set_to_none=True)
                # Backward recomputes checkpointed blocks: keep the same attention kernel and cache context as forward.
                with model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                    pool = new_text if rng.random() < fit['text_fraction'] else general_pool
                    (train_loss(rng.choice(pool)) * (1 - fit['early_weight'])).backward()
                    (train_loss(rng.choice(early_pool)) * fit['early_weight']).backward()
                torch.nn.utils.clip_grad_norm_(bridge.parameters(), fit['gradient_clip'])
                optimizer.step()
                report['optimizer_updates'] += 1
            bridge.load_state_dict(best[2])
            bridge.requires_grad_(False)
            bridge.eval()
            checkpoint_path = directory / f'bridge_block_{slot:02d}_{variant["name"]}.pt'
            temporary = checkpoint_path.with_suffix('.tmp')
            torch.save({'request_key': key, 'layer': slot, 'variant': variant, 'step': best[1], 'state_dict': best[2]}, temporary)
            temporary.replace(checkpoint_path)
            artifact = dict(name=checkpoint_path.name, sha256=digest(checkpoint_path), bytes=checkpoint_path.stat().st_size)
            report['artifacts'].append(artifact)
            rows = predictions()
            report['student_calls'] += len(rows)
            records = metrics(rows)
            report['variants'][variant['name']] = dict(
                variant=variant, checkpoint=artifact, parameters=sum(v.numel() for v in best[2].values()),
                fit=dict(curve=curve, selected_step=best[1], selected_gate10=best[0]),
                overall=aggregate(records),
                panels={p: aggregate([r for r in records if r['panel'] == p]) for p in panels},
                stages={s: aggregate([r for r in records if r['stage'] == s]) for s in ('early', 'middle', 'late')},
                records=records)
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            print(f'{variant["name"]}: teacher-relative {report["variants"][variant["name"]]["overall"]["teacher_relative_l2"]["mean"]:.6f}', flush=True)
            del optimizer
            gc.collect()
            torch.cuda.empty_cache()
        report['status'] = 'passed'
        report['scope'] = 'End-to-end bridge training (bridge weights only) and fixed-input velocity evaluation; images are separate.'
    except Exception as exc:
        import traceback
        report['status'] = 'failed'
        report['error'] = repr(exc)
        report['traceback'] = traceback.format_exc()
    (directory / 'summary.json').write_text(json.dumps(report, indent=2))
    results_volume.commit()
    return report
'''


SOURCES['conditioned_bridge.py'] = r'''import torch
from torch import nn
from types import SimpleNamespace


class ConditionedBridge(nn.Module):
    """Residual bottleneck bridge that can read the model's shared timestep modulation.

    The transformer computes one modulation tensor per timestep, [attn scale, attn gate, MLP scale, MLP gate],
    shared by every block. A conditioned bridge mirrors a block's MLP branch:
    h + tanh(gate) * up(gelu(down(LN(h) * (1 + scale)))), with scale and gate learned per-channel mixes of the
    shared values. Unconditioned, it is the original Phase 4 bridge: h + up(gelu(down(h))).
    """

    def __init__(self, hidden_dim=4096, rank=256, conditioned=True, eps=1e-6):
        super().__init__()
        self.conditioned = conditioned
        self.down = nn.Linear(hidden_dim, rank, bias=False)
        self.up = nn.Linear(rank, hidden_dim, bias=False)
        nn.init.zeros_(self.up.weight)
        if conditioned:
            self.norm = nn.LayerNorm(hidden_dim, elementwise_affine=False, eps=eps)
            # Start from the block's own MLP modulation (weight 1 on the MLP terms, 0 on the attention terms).
            mix = torch.zeros(2, hidden_dim)
            mix[1] = 1
            self.scale_mix = nn.Parameter(mix.clone())
            self.gate_mix = nn.Parameter(mix.clone())
            self.scale_bias = nn.Parameter(torch.zeros(hidden_dim))
            self.gate_bias = nn.Parameter(torch.zeros(hidden_dim))

    def scale_gate(self, modulation_rows):
        scale1, gate1, scale2, gate2 = modulation_rows.chunk(4, dim=-1)
        scale = self.scale_mix[0] * scale1 + self.scale_mix[1] * scale2 + self.scale_bias
        gate = self.gate_mix[0] * gate1 + self.gate_mix[1] * gate2 + self.gate_bias
        return scale, gate

    def forward(self, hidden_states, modulation_rows=None):
        if not self.conditioned:
            return hidden_states + self.up(nn.functional.gelu(self.down(hidden_states)))
        scale, gate = self.scale_gate(modulation_rows)
        update = self.up(nn.functional.gelu(self.down(self.norm(hidden_states) * (1 + scale))))
        return hidden_states + torch.tanh(gate) * update


class LocalConditionedBridge(ConditionedBridge):
    """Conditioned bridge plus a depthwise 3x3 convolution over the image-token grid inside the bottleneck.

    The convolution starts at zero (identity start) and is applied to target-image tokens only; prompt tokens have
    no grid and pass through the per-token path.
    """
    local = True

    def __init__(self, hidden_dim=4096, rank=1024, eps=1e-6):
        super().__init__(hidden_dim, rank, True, eps)
        self.mix = nn.Parameter(torch.zeros(rank, 9))

    def forward(self, hidden_states, modulation_rows=None, image_mask=None, grid=None):
        scale, gate = self.scale_gate(modulation_rows)
        z = self.down(self.norm(hidden_states) * (1 + scale))
        if grid is not None:
            height, width = grid
            image = z[:, image_mask]
            batch, tokens, rank = image.shape
            planes = image.transpose(1, 2).reshape(batch, rank, height, width)
            mixed = nn.functional.conv2d(planes, self.mix.view(rank, 1, 3, 3).to(planes.dtype), padding=1, groups=rank)
            z = z.clone()
            z[:, image_mask] = image + mixed.reshape(batch, rank, tokens).transpose(1, 2)
        return hidden_states + torch.tanh(gate) * self.up(nn.functional.gelu(z))

    def forward_neighborhood(self, neighborhood, modulation_rows):
        """Training path: neighborhood is (tokens, 9, hidden) in 3x3 raster order, zeros outside the grid."""
        scale, gate = self.scale_gate(modulation_rows)
        z = self.down(self.norm(neighborhood) * (1 + scale.unsqueeze(1)))
        z = z[:, 4] + (z * self.mix.t().unsqueeze(0)).sum(1)
        return neighborhood[:, 4] + torch.tanh(gate) * self.up(nn.functional.gelu(z))


class AttentionConditionedBridge(ConditionedBridge):
    """Conditioned bridge with a small attention branch in front of its MLP, mirroring a block.

    Target-image tokens attend to the prompt prefix and all image tokens with the model's RoPE and key masks; prefix
    tokens only take the MLP path. The branch keeps its own prefix K/V in the layer cache of the slot it replaces.
    The output projection starts at zero, so a fresh branch leaves the conditioned MLP bridge unchanged.
    """
    attention = True

    def __init__(self, hidden_dim=4096, rank=1024, heads=4, head_dim=128, eps=1e-6):
        super().__init__(hidden_dim, rank, True, eps)
        inner = heads * head_dim
        self.heads, self.head_dim, self.eps = heads, head_dim, eps
        self.to_q = nn.Linear(hidden_dim, inner, bias=False)
        self.to_k = nn.Linear(hidden_dim, inner, bias=False)
        self.to_v = nn.Linear(hidden_dim, inner, bias=False)
        self.to_out = nn.Linear(inner, hidden_dim, bias=False)
        nn.init.zeros_(self.to_out.weight)
        self.norm_q = nn.Parameter(torch.ones(head_dim))
        self.norm_k = nn.Parameter(torch.ones(head_dim))
        # Attention modulation starts from the block's own attention terms (weight 1 on scale1/gate1).
        mix = torch.zeros(2, hidden_dim)
        mix[0] = 1
        self.attn_scale_mix = nn.Parameter(mix.clone())
        self.attn_gate_mix = nn.Parameter(mix.clone())
        self.attn_scale_bias = nn.Parameter(torch.zeros(hidden_dim))
        self.attn_gate_bias = nn.Parameter(torch.zeros(hidden_dim))

    def _rms(self, x, weight):
        return x * torch.rsqrt(x.float().square().mean(-1, keepdim=True) + self.eps).to(x.dtype) * weight

    @staticmethod
    def _rope(x, freqs):
        rotated = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2)) * freqs.unsqueeze(1)
        return torch.view_as_real(rotated).flatten(3).type_as(x)

    def attend(self, hidden_states, modulation_rows, rotary_emb, target_count, layer_cache, kv_cache_mode,
               cache_write_slice, key_mask):
        scale1, gate1, scale2, gate2 = modulation_rows.chunk(4, dim=-1)
        scale = self.attn_scale_mix[0] * scale1 + self.attn_scale_mix[1] * scale2 + self.attn_scale_bias
        gate = self.attn_gate_mix[0] * gate1 + self.attn_gate_mix[1] * gate2 + self.attn_gate_bias
        x = self.norm(hidden_states) * (1 + scale)
        q, k, v = (proj(x).unflatten(-1, (self.heads, self.head_dim)) for proj in (self.to_q, self.to_k, self.to_v))
        q, k = self._rms(q, self.norm_q), self._rms(k, self.norm_k)
        if rotary_emb is not None:
            q, k = self._rope(q, rotary_emb), self._rope(k, rotary_emb)
        if layer_cache is not None:
            if kv_cache_mode == 'extract' and cache_write_slice is not None:
                layer_cache.store(k[:, cache_write_slice].clone(), v[:, cache_write_slice].clone())
            elif kv_cache_mode == 'cached':
                cached_k, cached_v = layer_cache.get()
                k, v = torch.cat([cached_k, k], dim=1), torch.cat([cached_v, v], dim=1)
        prefix = hidden_states.shape[1] - target_count
        out = nn.functional.scaled_dot_product_attention(
            q[:, prefix:].transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2), attn_mask=key_mask)
        update = torch.tanh(gate[:, prefix:]) * self.to_out(out.transpose(1, 2).flatten(2))
        return torch.cat([hidden_states[:, :prefix], hidden_states[:, prefix:] + update], dim=1)


def modulation_rows(modulation, target_token_mask):
    """Per-token modulation rows, selected exactly as the transformer block does."""
    if target_token_mask is None:
        return modulation.unsqueeze(1)
    real, zero = modulation[:-1].unsqueeze(1), modulation[-1:].unsqueeze(0)
    return torch.where(target_token_mask.view(1, -1, 1), real, zero)


class ConditionedBridgeBlock(nn.Module):
    def __init__(self, bridge, processor):
        super().__init__()
        self.bridge = bridge
        # Parent forward inspects this processor to choose mask construction; no attention is performed here.
        self.attn = SimpleNamespace(processor=processor)

    def forward(self, hidden_states, modulation, rotary_emb=None, attention_mask=None, target_token_mask=None,
                layer_cache=None, kv_cache_mode=None, cache_write_slice=None, segments=None, key_valid=None):
        if kv_cache_mode not in (None, 'extract', 'cached'):
            raise ValueError('Unknown KV cache mode.')
        rows = modulation_rows(modulation, target_token_mask) if self.bridge.conditioned else None
        dtype = hidden_states.dtype
        bridge_dtype = next(self.bridge.parameters()).dtype
        if getattr(self.bridge, 'attention', False):
            # Small bridges may keep FP32 weights for training; compute in their dtype and return the model dtype.
            target_count = hidden_states.shape[1] if target_token_mask is None else int(target_token_mask.sum())
            key_mask = key_valid[:, None, None, :] if key_valid is not None else attention_mask
            h = hidden_states.to(bridge_dtype)
            r = rows.to(bridge_dtype)
            h = self.bridge.attend(h, r, rotary_emb, target_count, layer_cache, kv_cache_mode, cache_write_slice, key_mask)
            return self.bridge(h, r).to(dtype)
        if bridge_dtype != dtype:
            return self.bridge(hidden_states.to(bridge_dtype), None if rows is None else rows.to(bridge_dtype)).to(dtype)
        if getattr(self.bridge, 'local', False):
            mask = target_token_mask if target_token_mask is not None else torch.ones(
                hidden_states.shape[1], dtype=torch.bool, device=hidden_states.device)
            height, width = self.grid
            if height * width != int(mask.sum()):
                raise RuntimeError('Target image grid does not match the image-token count.')
            return self.bridge(hidden_states, rows, mask, (height, width))
        return self.bridge(hidden_states, rows)


def insert_conditioned_bridge(transformer, layer, bridge, keep_dtype=False):
    """Replace one slot; callers must use fresh KV caches afterwards. keep_dtype keeps FP32 bridge weights."""
    old = transformer.transformer_blocks[layer]
    reference = next(transformer.parameters())
    bridge = bridge.to(device=reference.device) if keep_dtype else bridge.to(device=reference.device, dtype=reference.dtype)
    wrapper = ConditionedBridgeBlock(bridge, old.attn.processor)
    wrapper.train(transformer.training)
    transformer.transformer_blocks[layer] = wrapper
    if getattr(bridge, 'local', False) and not getattr(transformer, '_bridge_grid_hook', False):
        # The block never sees img_shapes; record the target image's (height, width) in tokens on every call.
        def record_grid(module, args, kwargs):
            frames, height, width = kwargs['img_shapes'][0][-1]
            for block in module.transformer_blocks:
                if isinstance(block, ConditionedBridgeBlock):
                    block.grid = (int(height), int(width))
        transformer.register_forward_pre_hook(record_grid, with_kwargs=True)
        transformer._bridge_grid_hook = True
    return old
'''


"""Continue the end-to-end attention bridge on text-heavy TRAIN prompts recorded in the same job; reuse completed results."""
import argparse
import hashlib
import json
import sys
from pathlib import Path


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--slot', type=int, default=13)
    parser.add_argument('--old-account', action='store_true')
    parser.add_argument('--dry-run', action='store_true', help='Write the request locally without scheduling GPU work.')
    args = parser.parse_args(LAUNCH_ARGS)
    if args.old_account:
        parser.error('Old account is download-only; GPU execution must use the new account.')
    root = ROOT
    protocol_path = root / f'data/phase7/block{args.slot}_bridge_e2e_text/protocol.json'
    protocol = json.loads(protocol_path.read_bytes())
    scan_b = json.loads(next((root / 'data/phase7/conditional_importance_b').glob('*/request.json')).read_bytes())['request']
    scan_a = json.loads((root / 'data/phase7/conditional_importance/7454da6a5c1dc33af23dd46833e698f9b759d14985ca96340bc3d40cd47fb4f5/request.json').read_bytes())['request']
    if args.slot == 5 or not 0 < args.slot < 31:
        parser.error('Slot must be a surviving interior block.')
    request = {k: v for k, v in scan_b.items() if k not in ('worker_source', 'eligible_original_block_ids', 'selection', 'scope', 'version')}
    warm = next(r for r in (json.loads(p.read_bytes()) for p in (root / 'data/phase7/block13_bridge_e2e').glob('*/summary.json')) if r.get('status') == 'passed')
    import random
    used = {r['source_id'] for r in scan_a['records']} | {a['source_id'] for a in scan_a['fresh_aligned_train']['artifacts']}
    used |= {'p1-' + e['capture_id'].split('p1-')[1].rsplit('-s', 1)[0] for e in scan_a['early_pool']['examples']}
    used |= {r['source_id'] for r in scan_a['expanded_validation']['records']}
    resolutions = json.loads((root / 'data/phase2/protocol.json').read_bytes())['RESOLUTIONS']
    pilot = [json.loads(line) for line in (root / 'data/phase1/pilot_20k/train.jsonl').read_text(encoding='utf-8').splitlines()]
    pilot = [r for r in pilot if r['split'] == 'train' and r['id'] not in used and r['aspect_ratio'] in resolutions]
    picker = random.Random(20261020)
    text_rows = picker.sample(sorted((r for r in pilot if r['subdimension'] == 'text_rendering'), key=lambda r: r['id']), 150)
    general_rows = picker.sample(sorted((r for r in pilot if r['subdimension'] != 'text_rendering'), key=lambda r: r['id']), 50)
    new_prompts = [dict(source_id=r['id'], prompt=r['prompt'], seed=r['seed'], dimension=r['dimension'], subdimension=r['subdimension'],
                        width=resolutions[r['aspect_ratio']][0], height=resolutions[r['aspect_ratio']][1]) for r in text_rows + general_rows]
    source = SOURCES['bridge_end_to_end_v2_worker.py']
    request.update(
        version='phase7-bridge-end-to-end-text-v1', new_prompts=new_prompts, worker_source=source, slot=args.slot,
        protocol_sha256=hashlib.sha256(protocol_path.read_bytes()).hexdigest(),
        conditioned_bridge_source=SOURCES['conditioned_bridge.py'],
        components_source=scan_a['components_source'],
        fresh_aligned_train=scan_a['fresh_aligned_train'], early_pool=scan_a['early_pool'],
        warm_start=dict(name='e2e_attn', remote_path=f"/phase3/phase7/bridge_end_to_end/{warm['request_key']}/{warm['variants']['e2e_attn']['checkpoint']['name']}",
                        sha256=warm['variants']['e2e_attn']['checkpoint']['sha256']),
        variants=[dict(name='e2e_attn_text', attention=True, heads=4)],
        fit=dict(init_seed=20261017, sampling_seed=20261021, learning_rate=1e-4, warmup=20, gradient_clip=1.0,
                 updates=600, eval_every=100, early_weight=0.3, text_fraction=0.4),
        selection='Lowest gate10 velocity error at evaluations every 50 updates, including the warm start',
        scope=f'End-to-end bridges for block {args.slot} inside frozen B; TRAIN-only; whole-student velocity on scan inputs; no images or latency')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = protocol_path.parent / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2)+'\n', encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase7/bridge_end_to_end_v2/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        try:
            b''.join(results_volume.read_file(remote + '/started.json'))
        except FileNotFoundError:
            pass
        else:
            raise RuntimeError('Run already started; inspect its progress. No duplicate GPU scheduled.')
        app = modal.App('nihonga-phase7-bridge-end-to-end-text')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(source, 'bridge_end_to_end_v2_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8,
                              memory=98304, max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['train_bridge_end_to_end'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2)+'\n')
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2)+'\n', encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'candidate_replay_max_relative_l2')}), flush=True)
    for name, v in (report.get('variants') or {}).items():
        print(name, 'step', v['fit']['selected_step'], 'teacher-relative', v['overall']['teacher_relative_l2']['mean'], flush=True)
    if report.get('status') != 'passed':
        raise SystemExit(1)

summary = reuse_or_launch(RUN, main)

print('replay', summary['new_prompt_replay_relative_l2'], summary['new_prompt_counts'])
show_variants(summary)

### What the results mean

6.25% overall, 6.21% wider, worst comparison 18.5% (B 17.9%). Exact signs across four seeds: 5/12 (B 9/12).

## 12b. Text check after 150 text prompts

Same text grid and 30 prompts.

In [ ]:
# Text check after 150 text prompts: reuses the saved run; launches GPU work only when its result is missing and ALLOW_GPU is True.

RUN = 'data/phase7/multiseed_text/d89c3819cef9b80c4eda2a2738451fc02d60af553af6e5c8124ecde32b93361b'

LAUNCH_ARGS = ['--source', 'block13_bridge_e2e_text', '--variant', 'e2e_attn_text', '--with-base-prompts']

SOURCES['multiseed_text_worker.py'] = r'''def generate_multiseed_text(request, key):
    """Extra seeds for the text prompts: teacher, B, and C with the best block-13 bridge.
    Same prompt, CPU noise seed, size, scheduler and 40 steps for every model. Qualitative screen only."""
    import gc
    import hashlib
    import inspect
    import json
    import os
    import time
    from pathlib import Path

    import torch
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache, QwenImage21KVLayerCache

    directory = Path('/phase3/phase7/multiseed_text') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete image attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def table_hash(value):
        return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

    variants = ['teacher', 'B', request['c_variant']['name']]
    report = {'request_key': key, 'status': 'running', 'rows': [], 'artifacts': [],
              'calls': {v: 0 for v in variants}, 'optimizer_updates': 0}
    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('Expected H100.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
                                                   dtype=torch.bfloat16, local_files_only=True).to('cuda')
        pipe.set_progress_bar_config(disable=True)
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Transformer source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--')) /
                       'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        # Derived position/time tables: use the audited values on every host (as in steps 7-11).
        derived = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
        derived['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
        expected = request['expected_derived_tables']
        mismatched = sorted(n for n, v in derived.items() if table_hash(v) != expected[n])
        if mismatched:
            canonical = torch.load(Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt'),
                                   weights_only=True, map_location='cpu')
            if any(table_hash(canonical[n]) != expected[n] for n in expected):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [canonical[f'pos_embed.freqs.{a}'].to(device=v.device, dtype=v.dtype) for a, v in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(canonical['time_text_embed.time_proj.freqs'])
            if any(table_hash(v) != expected[n] for n, v in
                   list({f'pos_embed.freqs.{a}': v for a, v in enumerate(model.pos_embed.freqs)}.items()) +
                   [('time_text_embed.time_proj.freqs', model.time_text_embed.time_proj.freqs)]):
                raise RuntimeError('Derived tables still differ after loading the audited copy.')
            report['canonical_tables_applied'] = mismatched

        def generate(row, variant):
            path = directory / (row['source_id'] + '-' + variant + '.png')
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'],
                             num_inference_steps=40, true_cfg_scale=teacher['true_cfg_scale'],
                             generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=True, output_type='pil').images[0]
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            temporary = path.with_suffix('.tmp')
            image.save(temporary, format='PNG')
            temporary.replace(path)
            results_volume.commit()
            report['calls'][variant] += 1
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            return {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size, 'seconds': elapsed}

        namespace = {}
        exec(compile(request['core_source'], 'bridge_definition.py', 'exec'), namespace)
        exec(compile(request['wrapper_source'], 'bridge_block_definition.py', 'exec'), namespace)
        conditioned = {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned)

        def load_checkpoint(spec):
            path = Path(spec['remote_path'])
            if digest(path) != spec['sha256']:
                raise RuntimeError('Bridge checkpoint differs: ' + spec['remote_path'])
            return torch.load(path, weights_only=True, map_location='cpu')

        outputs = {v: {} for v in variants}
        references = not request.get('candidate_only', False)
        if references:
            for row in request['rows']:
                outputs['teacher'][row['source_id']] = generate(row, 'teacher')
        block5 = request['block5_bridge']
        checkpoint = load_checkpoint(block5)
        if checkpoint['layer'] != 5 or checkpoint['step'] != block5['step']:
            raise RuntimeError('Block-5 bridge provenance differs.')
        bridge = namespace['ResidualBottleneckBridge'](4096, 256)
        bridge.load_state_dict(checkpoint['state_dict'])
        bridge.requires_grad_(False)
        namespace['insert_bridge'](model, 5, bridge, QwenImage21KVCache)
        model.eval()
        if references:
            for row in request['rows']:
                outputs['B'][row['source_id']] = generate(row, 'B')
        spec = request['c_variant']
        checkpoint = load_checkpoint(spec)
        if checkpoint['layer'] != 13 or checkpoint['variant']['name'] != spec['name']:
            raise RuntimeError('Block-13 bridge provenance differs.')
        fitted = checkpoint['variant']
        keep_dtype = False
        if fitted.get('attention'):
            bridge13 = conditioned['AttentionConditionedBridge'](4096, 1024, fitted['heads'], 128)
            keep_dtype = True
        elif 'attention' in fitted:
            bridge13 = conditioned['ConditionedBridge'](4096, 1024, True)
            keep_dtype = True
        elif fitted.get('local'):
            bridge13 = conditioned['LocalConditionedBridge'](4096, fitted['rank'])
        else:
            bridge13 = conditioned['ConditionedBridge'](4096, fitted['rank'], fitted['conditioned'])
        bridge13.load_state_dict(checkpoint['state_dict'])
        bridge13.requires_grad_(False)
        conditioned['insert_conditioned_bridge'](model, 13, bridge13.float() if keep_dtype else bridge13, keep_dtype=keep_dtype)
        model.eval()
        for row in request['rows']:
            outputs[spec['name']][row['source_id']] = generate(row, spec['name'])
        rendered = [v for v in variants if outputs[v]]
        for row in request['rows']:
            report['rows'].append({**row, **{v: outputs[v][row['source_id']] for v in rendered}})
            report['artifacts'].extend(outputs[v][row['source_id']] for v in rendered)
        report['status'] = 'passed'
    except Exception as exc:
        report['status'] = 'failed'
        report['error'] = type(exc).__name__ + ': ' + str(exc)
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=True) + '\n')
    results_volume.commit()
    return report
'''


SOURCES['conditioned_bridge.py'] = r'''import torch
from torch import nn
from types import SimpleNamespace


class ConditionedBridge(nn.Module):
    """Residual bottleneck bridge that can read the model's shared timestep modulation.

    The transformer computes one modulation tensor per timestep, [attn scale, attn gate, MLP scale, MLP gate],
    shared by every block. A conditioned bridge mirrors a block's MLP branch:
    h + tanh(gate) * up(gelu(down(LN(h) * (1 + scale)))), with scale and gate learned per-channel mixes of the
    shared values. Unconditioned, it is the original Phase 4 bridge: h + up(gelu(down(h))).
    """

    def __init__(self, hidden_dim=4096, rank=256, conditioned=True, eps=1e-6):
        super().__init__()
        self.conditioned = conditioned
        self.down = nn.Linear(hidden_dim, rank, bias=False)
        self.up = nn.Linear(rank, hidden_dim, bias=False)
        nn.init.zeros_(self.up.weight)
        if conditioned:
            self.norm = nn.LayerNorm(hidden_dim, elementwise_affine=False, eps=eps)
            # Start from the block's own MLP modulation (weight 1 on the MLP terms, 0 on the attention terms).
            mix = torch.zeros(2, hidden_dim)
            mix[1] = 1
            self.scale_mix = nn.Parameter(mix.clone())
            self.gate_mix = nn.Parameter(mix.clone())
            self.scale_bias = nn.Parameter(torch.zeros(hidden_dim))
            self.gate_bias = nn.Parameter(torch.zeros(hidden_dim))

    def scale_gate(self, modulation_rows):
        scale1, gate1, scale2, gate2 = modulation_rows.chunk(4, dim=-1)
        scale = self.scale_mix[0] * scale1 + self.scale_mix[1] * scale2 + self.scale_bias
        gate = self.gate_mix[0] * gate1 + self.gate_mix[1] * gate2 + self.gate_bias
        return scale, gate

    def forward(self, hidden_states, modulation_rows=None):
        if not self.conditioned:
            return hidden_states + self.up(nn.functional.gelu(self.down(hidden_states)))
        scale, gate = self.scale_gate(modulation_rows)
        update = self.up(nn.functional.gelu(self.down(self.norm(hidden_states) * (1 + scale))))
        return hidden_states + torch.tanh(gate) * update


class LocalConditionedBridge(ConditionedBridge):
    """Conditioned bridge plus a depthwise 3x3 convolution over the image-token grid inside the bottleneck.

    The convolution starts at zero (identity start) and is applied to target-image tokens only; prompt tokens have
    no grid and pass through the per-token path.
    """
    local = True

    def __init__(self, hidden_dim=4096, rank=1024, eps=1e-6):
        super().__init__(hidden_dim, rank, True, eps)
        self.mix = nn.Parameter(torch.zeros(rank, 9))

    def forward(self, hidden_states, modulation_rows=None, image_mask=None, grid=None):
        scale, gate = self.scale_gate(modulation_rows)
        z = self.down(self.norm(hidden_states) * (1 + scale))
        if grid is not None:
            height, width = grid
            image = z[:, image_mask]
            batch, tokens, rank = image.shape
            planes = image.transpose(1, 2).reshape(batch, rank, height, width)
            mixed = nn.functional.conv2d(planes, self.mix.view(rank, 1, 3, 3).to(planes.dtype), padding=1, groups=rank)
            z = z.clone()
            z[:, image_mask] = image + mixed.reshape(batch, rank, tokens).transpose(1, 2)
        return hidden_states + torch.tanh(gate) * self.up(nn.functional.gelu(z))

    def forward_neighborhood(self, neighborhood, modulation_rows):
        """Training path: neighborhood is (tokens, 9, hidden) in 3x3 raster order, zeros outside the grid."""
        scale, gate = self.scale_gate(modulation_rows)
        z = self.down(self.norm(neighborhood) * (1 + scale.unsqueeze(1)))
        z = z[:, 4] + (z * self.mix.t().unsqueeze(0)).sum(1)
        return neighborhood[:, 4] + torch.tanh(gate) * self.up(nn.functional.gelu(z))


class AttentionConditionedBridge(ConditionedBridge):
    """Conditioned bridge with a small attention branch in front of its MLP, mirroring a block.

    Target-image tokens attend to the prompt prefix and all image tokens with the model's RoPE and key masks; prefix
    tokens only take the MLP path. The branch keeps its own prefix K/V in the layer cache of the slot it replaces.
    The output projection starts at zero, so a fresh branch leaves the conditioned MLP bridge unchanged.
    """
    attention = True

    def __init__(self, hidden_dim=4096, rank=1024, heads=4, head_dim=128, eps=1e-6):
        super().__init__(hidden_dim, rank, True, eps)
        inner = heads * head_dim
        self.heads, self.head_dim, self.eps = heads, head_dim, eps
        self.to_q = nn.Linear(hidden_dim, inner, bias=False)
        self.to_k = nn.Linear(hidden_dim, inner, bias=False)
        self.to_v = nn.Linear(hidden_dim, inner, bias=False)
        self.to_out = nn.Linear(inner, hidden_dim, bias=False)
        nn.init.zeros_(self.to_out.weight)
        self.norm_q = nn.Parameter(torch.ones(head_dim))
        self.norm_k = nn.Parameter(torch.ones(head_dim))
        # Attention modulation starts from the block's own attention terms (weight 1 on scale1/gate1).
        mix = torch.zeros(2, hidden_dim)
        mix[0] = 1
        self.attn_scale_mix = nn.Parameter(mix.clone())
        self.attn_gate_mix = nn.Parameter(mix.clone())
        self.attn_scale_bias = nn.Parameter(torch.zeros(hidden_dim))
        self.attn_gate_bias = nn.Parameter(torch.zeros(hidden_dim))

    def _rms(self, x, weight):
        return x * torch.rsqrt(x.float().square().mean(-1, keepdim=True) + self.eps).to(x.dtype) * weight

    @staticmethod
    def _rope(x, freqs):
        rotated = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2)) * freqs.unsqueeze(1)
        return torch.view_as_real(rotated).flatten(3).type_as(x)

    def attend(self, hidden_states, modulation_rows, rotary_emb, target_count, layer_cache, kv_cache_mode,
               cache_write_slice, key_mask):
        scale1, gate1, scale2, gate2 = modulation_rows.chunk(4, dim=-1)
        scale = self.attn_scale_mix[0] * scale1 + self.attn_scale_mix[1] * scale2 + self.attn_scale_bias
        gate = self.attn_gate_mix[0] * gate1 + self.attn_gate_mix[1] * gate2 + self.attn_gate_bias
        x = self.norm(hidden_states) * (1 + scale)
        q, k, v = (proj(x).unflatten(-1, (self.heads, self.head_dim)) for proj in (self.to_q, self.to_k, self.to_v))
        q, k = self._rms(q, self.norm_q), self._rms(k, self.norm_k)
        if rotary_emb is not None:
            q, k = self._rope(q, rotary_emb), self._rope(k, rotary_emb)
        if layer_cache is not None:
            if kv_cache_mode == 'extract' and cache_write_slice is not None:
                layer_cache.store(k[:, cache_write_slice].clone(), v[:, cache_write_slice].clone())
            elif kv_cache_mode == 'cached':
                cached_k, cached_v = layer_cache.get()
                k, v = torch.cat([cached_k, k], dim=1), torch.cat([cached_v, v], dim=1)
        prefix = hidden_states.shape[1] - target_count
        out = nn.functional.scaled_dot_product_attention(
            q[:, prefix:].transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2), attn_mask=key_mask)
        update = torch.tanh(gate[:, prefix:]) * self.to_out(out.transpose(1, 2).flatten(2))
        return torch.cat([hidden_states[:, :prefix], hidden_states[:, prefix:] + update], dim=1)


def modulation_rows(modulation, target_token_mask):
    """Per-token modulation rows, selected exactly as the transformer block does."""
    if target_token_mask is None:
        return modulation.unsqueeze(1)
    real, zero = modulation[:-1].unsqueeze(1), modulation[-1:].unsqueeze(0)
    return torch.where(target_token_mask.view(1, -1, 1), real, zero)


class ConditionedBridgeBlock(nn.Module):
    def __init__(self, bridge, processor):
        super().__init__()
        self.bridge = bridge
        # Parent forward inspects this processor to choose mask construction; no attention is performed here.
        self.attn = SimpleNamespace(processor=processor)

    def forward(self, hidden_states, modulation, rotary_emb=None, attention_mask=None, target_token_mask=None,
                layer_cache=None, kv_cache_mode=None, cache_write_slice=None, segments=None, key_valid=None):
        if kv_cache_mode not in (None, 'extract', 'cached'):
            raise ValueError('Unknown KV cache mode.')
        rows = modulation_rows(modulation, target_token_mask) if self.bridge.conditioned else None
        dtype = hidden_states.dtype
        bridge_dtype = next(self.bridge.parameters()).dtype
        if getattr(self.bridge, 'attention', False):
            # Small bridges may keep FP32 weights for training; compute in their dtype and return the model dtype.
            target_count = hidden_states.shape[1] if target_token_mask is None else int(target_token_mask.sum())
            key_mask = key_valid[:, None, None, :] if key_valid is not None else attention_mask
            h = hidden_states.to(bridge_dtype)
            r = rows.to(bridge_dtype)
            h = self.bridge.attend(h, r, rotary_emb, target_count, layer_cache, kv_cache_mode, cache_write_slice, key_mask)
            return self.bridge(h, r).to(dtype)
        if bridge_dtype != dtype:
            return self.bridge(hidden_states.to(bridge_dtype), None if rows is None else rows.to(bridge_dtype)).to(dtype)
        if getattr(self.bridge, 'local', False):
            mask = target_token_mask if target_token_mask is not None else torch.ones(
                hidden_states.shape[1], dtype=torch.bool, device=hidden_states.device)
            height, width = self.grid
            if height * width != int(mask.sum()):
                raise RuntimeError('Target image grid does not match the image-token count.')
            return self.bridge(hidden_states, rows, mask, (height, width))
        return self.bridge(hidden_states, rows)


def insert_conditioned_bridge(transformer, layer, bridge, keep_dtype=False):
    """Replace one slot; callers must use fresh KV caches afterwards. keep_dtype keeps FP32 bridge weights."""
    old = transformer.transformer_blocks[layer]
    reference = next(transformer.parameters())
    bridge = bridge.to(device=reference.device) if keep_dtype else bridge.to(device=reference.device, dtype=reference.dtype)
    wrapper = ConditionedBridgeBlock(bridge, old.attn.processor)
    wrapper.train(transformer.training)
    transformer.transformer_blocks[layer] = wrapper
    if getattr(bridge, 'local', False) and not getattr(transformer, '_bridge_grid_hook', False):
        # The block never sees img_shapes; record the target image's (height, width) in tokens on every call.
        def record_grid(module, args, kwargs):
            frames, height, width = kwargs['img_shapes'][0][-1]
            for block in module.transformer_blocks:
                if isinstance(block, ConditionedBridgeBlock):
                    block.grid = (int(height), int(width))
        transformer.register_forward_pre_hook(record_grid, with_kwargs=True)
        transformer._bridge_grid_hook = True
    return old
'''


"""Render the text prompts with extra seeds for teacher, B and C (best block-13 bridge); qualitative only."""
import argparse
import hashlib
import json
import sys
from pathlib import Path


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--dry-run', action='store_true')
    parser.add_argument('--variant', default='cond_r1024_long')
    parser.add_argument('--source', default='block13_bridge_v3', help='Run folder holding the bridge checkpoint.')
    parser.add_argument('--with-base-prompts', action='store_true', help='Also render the 30 base prompts at seed 0; candidate only.')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    recovered = root / 'data/recovered/modal'
    images_key = 'dda8c184e657dfb604c35fb261d83371c108c39fac033a0f57db66f7686c30ad'
    base = json.loads((recovered / 'phase7/candidate_images' / images_key / 'request.json').read_bytes())['request']
    scan_b = json.loads(next((root / 'data/phase7/conditional_importance_b').glob('*/request.json')).read_bytes())['request']
    fit = next(r for r in (json.loads(p.read_bytes()) for p in (root / 'data/phase7' / args.source).glob('*/summary.json')) if r.get('status') == 'passed')
    best = fit['variants'][args.variant]
    remote_dir = {'block13_bridge_v3': 'bridge_variants_v3', 'block13_bridge_e2e': 'bridge_end_to_end', 'block13_bridge_e2e_text': 'bridge_end_to_end_v2', 'block13_bridge_e2e_text8': 'bridge_end_to_end_v3'}[args.source]
    selected = scan_b['selected_candidate']
    text_rows = [r for r in base['rows'] if r['subdimension'] in ('text_rendering', 'posters')]
    rows = [{**r, 'source_id': f"{r['source_id']}-s{k}", 'seed': (r['seed'] + 1000003 * k) % 2**32}
            for r in text_rows for k in (1, 2, 3)]
    if args.with_base_prompts:
        rows += base['rows']
    request = {k: base[k] for k in ('teacher', 'transformer_source_sha256', 'runtime_environment', 'expected_derived_tables',
                                    'canonical_tables_key', 'core_source', 'wrapper_source')}
    request.update(
        version='phase7-multiseed-text-v1', rows=rows,
        worker_source=SOURCES['multiseed_text_worker.py'],
        conditioned_bridge_source=SOURCES['conditioned_bridge.py'],
        execution_account='new',
        block5_bridge=dict(remote_path=f"/phase3/phase5/fitting/{scan_b['fit_key']}/{Path(selected['checkpoint_path']).name}",
                           sha256=selected['checkpoint_sha256'], step=selected['selected_update']),
        c_variant=dict(name=args.variant, remote_path=f"/phase3/phase7/{remote_dir}/{fit['request_key']}/{best['checkpoint']['name']}",
                       sha256=best['checkpoint']['sha256']),
        candidate_only=args.with_base_prompts,
        scope='Extra seeds (3 per prompt) for the 5 text prompts of the Phase 7 image set' + ('; plus the 30 base prompts at seed 0, candidate only' if args.with_base_prompts else '; seed 0 images are reused'))
    from experiment_runtime import pin_reference_runtime
    pin_reference_runtime(request, recovered)
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase7/multiseed_text' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2)+'\n', encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase7/multiseed_text/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        try:
            b''.join(results_volume.read_file(remote + '/started.json'))
        except FileNotFoundError:
            pass
        else:
            raise RuntimeError('Run already started; inspect its progress. No duplicate GPU scheduled.')
        app = modal.App('nihonga-phase7-multiseed-text')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'multiseed_text_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8,
                              memory=98304, max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['generate_multiseed_text'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2)+'\n')
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2)+'\n', encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'calls')}), flush=True)
    if report.get('status') != 'passed':
        raise SystemExit(1)

summary = reuse_or_launch(RUN, main)

show_images(RUN + '/sheets', ['p1-c9344c076abf.jpg'])

### What the results mean

"Emma" correct on 2 of 4 seeds; "The Rusty Anchor" still 1 of 4 (seed 3 now "The Rust Anchor").

## 13. Decisive test: 8 attention heads and 450 text prompts

Is the remaining spelling gap capacity or data? The attention branch grows to 8 heads (trained heads copied, new heads start with zero output) and trains on 500 recorded prompts (450 text), now saved on the volume for reuse. Decision rule committed first (`data/phase7/block13_bridge_e2e_text8/protocol.json`): >= 8/12 exact signs means continue with bridges; <= 6/12 means a bridge alone does not carry spelling.

In [ ]:
# Decisive test: 8 attention heads and 450 text prompts: reuses the saved run; launches GPU work only when its result is missing and ALLOW_GPU is True.

RUN = 'data/phase7/block13_bridge_e2e_text8/ff9d259ae589bdf15b98076930c4d16340ef41f4a2c3db897ea3167e95397054'

LAUNCH_ARGS = ['--slot', '13']

SOURCES['bridge_end_to_end_v3_worker.py'] = r'''def train_bridge_end_to_end(request, key):
    """Train a bridge for one block inside frozen B against the final velocity (bridge weights only).

    Verifies the teacher contract and exact B replay, computes live teacher targets for TRAIN inputs, then trains
    each bridge variant end to end through the frozen downstream blocks and evaluates the whole student.
    """
    import hashlib
    import importlib.metadata
    import inspect
    import json
    import os
    import socket
    import uuid
    from pathlib import Path
    import torch
    from torch.nn.attention import sdpa_kernel, SDPBackend
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache

    directory = Path('/phase3/phase7/bridge_end_to_end_v3') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete healing attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()
    report = {'request_key': key, 'status': 'failed', 'mode': request['mode'], 'teacher_calls': 0,
              'student_calls': 0, 'optimizer_updates': 0, 'controls': [], 'artifacts': [],
              'process': {'pid': os.getpid(), 'host': socket.gethostname(), 'nonce': uuid.uuid4().hex,
                          'modal_task_id': os.environ.get('MODAL_TASK_ID')}}

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def gpu(value):
        if isinstance(value, torch.Tensor):
            return value.to('cuda')
        if isinstance(value, dict):
            return {k: gpu(v) for k, v in value.items()}
        if isinstance(value, (list, tuple)):
            return type(value)(gpu(v) for v in value)
        return value

    def compare(x, y):
        a, b = x.float(), y.float()
        if not torch.isfinite(a).all() or not torch.isfinite(b).all():
            raise RuntimeError('Nonfinite reference tensor.')
        d = a - b
        return {'relative_l2_error': float(d.norm()/b.norm().clamp_min(1e-12)),
                'max_abs_error': float(d.abs().max()), 'exact_equal': bool(torch.equal(x, y))}

    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('GPU differs.')
        report['packages'] = {p: importlib.metadata.version(p) for p in ('torch', 'diffusers', 'transformers', 'accelerate', 'pillow')}
        report['torch_cuda'] = torch.version.cuda
        report['cudnn_version'] = torch.backends.cudnn.version()
        report['settings'] = request['arithmetic_contract']
        if request['mode'] != 'verify':
            raise RuntimeError('Healing requires the verified teacher reference.')
        reference_dir = Path('/phase3/phase6/deterministic_reference') / request['reference_key']
        reference_report_path = reference_dir / 'summary.json'
        if hashlib.sha256(json.dumps(json.loads(reference_report_path.read_text()), sort_keys=True).encode()).hexdigest() != request['reference_report_semantic_sha256']:
            raise RuntimeError('Generation report differs.')
        reference_report = json.loads(reference_report_path.read_text())
        if reference_report['status'] != 'reference_generated':
            raise RuntimeError('Reference generation did not complete.')
        if reference_report['settings'] != request['arithmetic_contract']:
            raise RuntimeError('Arithmetic contract differs from generation.')
        report['reference_process'] = reference_report['process']
        if reference_report['process']['nonce'] == report['process']['nonce']:
            raise RuntimeError('Independent process requirement failed.')
        if reference_report['packages'] != report['packages'] or reference_report['torch_cuda'] != report['torch_cuda'] or reference_report['cudnn_version'] != report['cudnn_version']:
            raise RuntimeError('Numerical runtime differs across generation and verification.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
            dtype=torch.bfloat16, local_files_only=True).to('cuda')
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Model source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--'))
                       / 'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        state = {}
        for kind, values in (('parameter', model.named_parameters()), ('buffer', model.named_buffers())):
            for name, value in values:
                cpu = value.detach().cpu().contiguous()
                state[name] = {'kind': kind, 'dtype': str(value.dtype), 'shape': list(value.shape),
                    'stride': list(value.stride()), 'sha256': hashlib.sha256(memoryview(cpu.view(torch.uint8).numpy())).hexdigest()}
        for axis, value in enumerate(model.pos_embed.freqs):
            name = f'pos_embed.freqs.{axis}'
            cpu = value.detach().cpu().contiguous()
            state[name] = {'kind': 'unregistered_rope_table', 'dtype': str(value.dtype), 'shape': list(value.shape),
                'stride': list(value.stride()), 'sha256': hashlib.sha256(memoryview(cpu.view(torch.uint8).numpy())).hexdigest()}
        expected = request['expected_state']
        derived_names = sorted(n for n in expected if n.startswith('pos_embed.freqs.') or n == 'time_text_embed.time_proj.freqs')

        def derived_tensors():
            values = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
            values['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
            return values

        def table_hash(value):
            return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

        canonical_path = Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt')
        import time
        mismatched = [n for n in derived_names if state.get(n) != expected.get(n)]
        if not mismatched:
            if not canonical_path.exists():
                canonical_path.parent.mkdir(parents=True, exist_ok=True)
                temporary = canonical_path.with_suffix('.' + report['process']['nonce'] + '.tmp')
                torch.save({n: v.detach().cpu().contiguous() for n, v in derived_tensors().items()}, temporary)
                temporary.replace(canonical_path)
                results_volume.commit()
                report['canonical_tables_saved'] = True
        else:
            deadline = time.time() + 900
            while not canonical_path.exists():
                if time.time() > deadline:
                    raise RuntimeError('Derived tables differ on this host and no canonical copy became available.')
                time.sleep(20)
                results_volume.reload()
            saved = torch.load(canonical_path, weights_only=True, map_location='cpu')
            if sorted(saved) != derived_names or any(table_hash(saved[n]) != expected[n]['sha256'] for n in derived_names):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [saved[f'pos_embed.freqs.{axis}'].to(device=value.device, dtype=value.dtype)
                        for axis, value in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(saved['time_text_embed.time_proj.freqs'])
            for n, value in derived_tensors().items():
                state[n] = {**state[n], 'sha256': table_hash(value)}
            report['canonical_tables_applied'] = mismatched
        if state != request['expected_state']:
            expected = request['expected_state']
            different_names = sorted(name for name in expected.keys() & state.keys()
                                     if state[name] != expected[name])
            report['state_mismatch'] = {
                'actual_count': len(state), 'expected_count': len(expected),
                'missing_names': sorted(set(expected) - set(state))[:12],
                'extra_names': sorted(set(state) - set(expected))[:12],
                'different': [{'name': name, 'actual': state[name], 'expected': expected[name]}
                              for name in different_names][:12]}
            allowed_derived_tables = {'pos_embed.freqs.1', 'pos_embed.freqs.2', 'time_text_embed.time_proj.freqs'}
            if set(state) != set(expected) or not set(different_names).issubset(allowed_derived_tables):
                raise RuntimeError('Loaded teacher learned state differs from audited reference.')
            report['derived_table_variation_requires_output_replay'] = different_names
        report['state_sha256'] = hashlib.sha256(json.dumps(state, sort_keys=True).encode()).hexdigest()
        torch.cuda.synchronize()
        argument_order = ('hidden_states', 'timestep', 'encoder_hidden_states', 'encoder_hidden_states_mask',
                          'img_shapes', 'img_mask', 'attention_kwargs', 'kv_cache_mode', 'return_dict')
        # Every intact-teacher control must pass before any block is touched.
        for record in request['records']:
            path = Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt'
            if digest(path) != record['tensor_sha256']:
                raise RuntimeError('Legacy input bundle differs.')
            bundle = torch.load(path, weights_only=True, map_location='cpu')
            outputs, controls = [], []
            cache = QwenImage21KVCache(32)
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for stage in bundle['stages']:
                    full = {**bundle['shared_transformer_kwargs'], **stage['kwargs']}
                    kwargs = gpu({field: full[field] for field in argument_order})
                    kwargs['kv_cache'] = cache
                    pred = model(**kwargs)[0][:, -stage['target_image_tokens']:].detach().cpu().contiguous()
                    report['teacher_calls'] += 1
                    legacy = compare(pred, stage['teacher_velocity'])
                    entry = {'source_id': record['source_id'], 'split': bundle['row']['split'],
                             'stage': stage['stage'], 'legacy_difference': legacy}
                    outputs.append({'stage': stage['stage'], 'step_index': stage['step_index'],
                                    'scheduler_sigma': stage['scheduler_sigma'], 'velocity': pred})
                    controls.append(entry)
            filename = record['capture_id'] + '.pt'
            artifact = next(a for a in reference_report['artifacts'] if a['name'] == filename)
            if digest(reference_dir / filename) != artifact['sha256']:
                raise RuntimeError('Reference output checksum differs.')
            reference_bundle = torch.load(reference_dir / filename, weights_only=True, map_location='cpu')
            if reference_bundle['source_id'] != record['source_id'] or reference_bundle['input_sha256'] != record['tensor_sha256']:
                raise RuntimeError('Reference input provenance differs.')
            reference_outputs = {v['stage']: v['velocity'] for v in reference_bundle['outputs']}
            for entry, measured in zip(controls, outputs):
                entry['reference_difference'] = compare(measured['velocity'], reference_outputs[entry['stage']])
                entry['passed'] = entry['reference_difference']['relative_l2_error'] <= 0.001
            output_path = directory / filename
            temporary = output_path.with_suffix('.tmp')
            torch.save({'request_key': key, 'source_id': record['source_id'], 'input_sha256': record['tensor_sha256'],
                        'outputs': outputs, 'controls': controls}, temporary)
            temporary.replace(output_path)
            report['controls'].extend(controls)
            report['artifacts'].append({'name': filename, 'sha256': digest(output_path), 'bytes': output_path.stat().st_size})
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            if not all(c['passed'] for c in controls):
                raise RuntimeError('Independent-process reference replay exceeds tolerance.')
        if report['teacher_calls'] != 90:
            raise RuntimeError('Reference attempt did not cover all 90 saved stage inputs.')
        report['legacy_max_relative_l2'] = max(c['legacy_difference']['relative_l2_error'] for c in report['controls'])
        report['max_reference_relative_l2'] = max(c['reference_difference']['relative_l2_error'] for c in report['controls'])
        report['exact_equal_controls'] = sum(c['reference_difference']['exact_equal'] for c in report['controls'])
        if not all(c['passed'] for c in report['controls']):
            raise RuntimeError('Teacher controls did not pass.')
        report['status'] = 'teacher_verified'
        (directory / 'teacher_control_report.json').write_text(json.dumps(report, indent=2))
        results_volume.commit()
        print('All 90 intact-teacher controls passed; assembling frozen 31-block candidate B.', flush=True)

        import gc
        import time
        import statistics
        from types import SimpleNamespace
        
        # Inputs are independently identified; 10 gate and 50 wider development prompts.
        panels = {'gate10': [], 'wider50': []}
        for panel, records in (('gate10', [r for r in request['records'] if r['capture_id'].startswith('p5-validation-')]),
                               ('wider50', request['expanded_validation']['records'])):
            for record in records:
                path = Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt'
                if digest(path) != record['tensor_sha256']:
                    raise RuntimeError('Fixed input hash differs')
                item = torch.load(path, weights_only=True, map_location='cpu')
                if item['row']['source_id'] != record['source_id'] or item['row']['split'] != 'validation':
                    raise RuntimeError('Validation input identity differs')
                panels[panel].append(item)
        if [len(panels[p]) for p in ('gate10', 'wider50')] != [10, 50]:
            raise RuntimeError('Saved validation panel sizes differ')

        def forward(item, stage, cache):
            full = {**item['shared_transformer_kwargs'], **stage['kwargs']}
            kwargs = gpu({field: full[field] for field in argument_order})
            kwargs['kv_cache'] = cache
            return model(**kwargs)[0][:, -stage['target_image_tokens']:].detach().cpu().contiguous()

        def predictions():
            rows = []
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for panel, items in panels.items():
                    for item in items:
                        cache = QwenImage21KVCache(32)
                        for stage in item['stages']:
                            pred = forward(item, stage, cache)
                            if not torch.isfinite(pred).all() or pred.float().norm() == 0:
                                raise RuntimeError('Invalid reference velocity')
                            rows.append(dict(panel=panel, source_id=item['row']['source_id'],
                                             dimension=item['row']['dimension'],
                                             subdimension=item['row'].get('subdimension'),
                                             stage=stage['stage'], velocity=pred))
            return rows

        def save(name, value):
            path = directory / name
            temporary = path.with_suffix('.tmp')
            torch.save(value, temporary)
            temporary.replace(path)
            artifact = dict(name=name, sha256=digest(path), bytes=path.stat().st_size)
            report['artifacts'].append(artifact)
            return artifact

        teacher_predictions = predictions()
        save('teacher_predictions.pt', teacher_predictions)
        report['teacher_calls'] += len(teacher_predictions)
        # ---- Live teacher targets for TRAIN inputs (stored legacy targets differ by up to ~2.6%). ----
        def load_verified(path, sha256, size=None):
            if (size is not None and path.stat().st_size != size) or digest(path) != sha256:
                raise RuntimeError('Input tensor differs: ' + str(path))
            return torch.load(path, weights_only=True, map_location='cpu')

        train_full, early_pool = [], []
        for record in request['records']:
            if not record['capture_id'].startswith('p5-train-'):
                continue
            item = load_verified(Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt', record['tensor_sha256'])
            if item['row']['split'] != 'train' or item['row']['source_id'] != record['source_id']:
                raise RuntimeError('Training record identity differs.')
            train_full.append(item)
        fresh = request['fresh_aligned_train']
        for artifact in fresh['artifacts']:
            item = load_verified(Path(fresh['remote_directory']) / artifact['name'], artifact['sha256'], artifact['bytes'])
            if item['source_id'] != artifact['source_id'] or item['row']['split'] != 'train' or len(item['stages']) != 3:
                raise RuntimeError('Fresh aligned training provenance differs.')
            train_full.append(item)
        for example in request['early_pool']['examples']:
            item = load_verified(Path(example['remote_path']), example['sha256'], example['bytes'])
            if (item['capture_id'] != example['capture_id'] or item['row']['split'] != 'train'
                    or [s['stage'] for s in item['stages']] != ['early']):
                raise RuntimeError('Early pool example provenance differs.')
            early_pool.append(item)
        validation_ids = {r['source_id'] for r in request['records'] if r['capture_id'].startswith('p5-validation-')}
        validation_ids |= {r['source_id'] for r in request['expanded_validation']['records']}
        train_ids = {i['row']['source_id'] for i in train_full} | {i['row']['source_id'] for i in early_pool}
        if len(train_full) != 60 or train_ids & validation_ids:
            raise RuntimeError('Training set size or split differs.')
        for item in train_full + early_pool:
            cache = QwenImage21KVCache(32)
            item['live_targets'] = []
            for stage in item['stages']:
                with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                    item['live_targets'].append(forward(item, stage, cache).to(torch.bfloat16))
                report['teacher_calls'] += 1
        report['train_counts'] = {'full_prompts': len(train_full), 'early_pool': len(early_pool)}
        print('Live teacher targets for', report['train_counts'], flush=True)

        # ---- Record new TRAIN prompts with the intact teacher (steps 0 and 19 of the 40-step trajectory). ----
        import math

        class StopCapture(Exception):
            pass

        record_steps = {0: 'early', 19: 'middle'}
        capture_state = {'calls': 0, 'stages': [], 'kwargs': None}

        def cpu_tree(value):
            if isinstance(value, torch.Tensor):
                return value.detach().cpu().clone()
            if isinstance(value, dict):
                return {k: cpu_tree(v) for k, v in value.items()}
            if isinstance(value, (list, tuple)):
                return type(value)(cpu_tree(v) for v in value)
            return value

        def pre_hook(module, args, kwargs):
            if capture_state['calls'] in record_steps:
                capture_state['kwargs'] = {field: cpu_tree(kwargs.get(field)) for field in argument_order}

        def post_hook(module, args, kwargs, output):
            step = capture_state['calls']
            capture_state['calls'] += 1
            if step in record_steps:
                stage_kwargs = capture_state['kwargs']
                tokens = math.prod(int(v) for v in stage_kwargs['img_shapes'][0][-1])
                velocity = output[0][:, -tokens:].detach().to(torch.bfloat16).cpu()
                capture_state['stages'].append(dict(stage=record_steps[step], step_index=step, kwargs=stage_kwargs,
                                                    target_image_tokens=tokens, velocity=velocity))
                if step == max(record_steps):
                    raise StopCapture()

        handles = [model.register_forward_pre_hook(pre_hook, with_kwargs=True),
                   model.register_forward_hook(post_hook, with_kwargs=True)]
        new_text, new_general = [], []
        try:
            for row in request['new_prompts']:
                capture_state.update(calls=0, stages=[], kwargs=None)
                try:
                    with torch.inference_mode(), sdpa_kernel([SDPBackend.MATH]):
                        pipe(prompt=row['prompt'], width=row['width'], height=row['height'], num_inference_steps=40,
                             true_cfg_scale=teacher['true_cfg_scale'], generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=True, output_type='latent')
                except StopCapture:
                    pass
                if [s['stage'] for s in capture_state['stages']] != ['early', 'middle']:
                    raise RuntimeError('Capture did not record the expected steps for ' + row['source_id'])
                report['teacher_calls'] += capture_state['calls']
                item = {'row': {k: row[k] for k in ('source_id', 'dimension', 'subdimension')} | {'split': 'train'},
                        'shared_transformer_kwargs': {},
                        'stages': [{k: v for k, v in s.items() if k != 'velocity'} for s in capture_state['stages']],
                        'live_targets': [s['velocity'] for s in capture_state['stages']]}
                (new_text if row['subdimension'] == 'text_rendering' else new_general).append(item)
                # Keep every recording on the volume so later runs can reuse it without re-generating.
                recorded = directory / 'recorded'
                recorded.mkdir(exist_ok=True)
                temporary = recorded / (row['source_id'] + '.tmp')
                torch.save(item, temporary)
                temporary.replace(recorded / (row['source_id'] + '.pt'))
                if len(new_text) + len(new_general) in (100, 200, 300, 400):
                    results_volume.commit()
                    print('Recorded', len(new_text) + len(new_general), 'prompts', flush=True)
        finally:
            for handle in handles:
                handle.remove()
        # The replay path used for training must reproduce the recorded teacher outputs.
        probe = (new_text + new_general)[0]
        cache = QwenImage21KVCache(32)
        replay = []
        with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
            for stage, target in zip(probe['stages'], probe['live_targets']):
                replay.append(compare(forward(probe, stage, cache).to(torch.bfloat16), target)['relative_l2_error'])
        report['new_prompt_replay_relative_l2'] = replay
        if max(replay) > 0.001:
            raise RuntimeError('Recorded new-prompt inputs do not replay the teacher: ' + json.dumps(replay))
        new_ids = {i['row']['source_id'] for i in new_text + new_general}
        if new_ids & (train_ids | validation_ids):
            raise RuntimeError('New prompts overlap earlier training or validation prompts.')
        report['new_prompt_counts'] = {'text': len(new_text), 'general': len(new_general)}
        print('Recorded new prompts', report['new_prompt_counts'], 'replay', replay, flush=True)

        # Rebuild the Phase 6 selected student exactly as its verified replay did:
        # 31 teacher blocks plus the frozen pretrained rank-256 bridge at slot 5.
        namespace = {}
        exec(compile(request['core_source'], 'bridge_definition.py', 'exec'), namespace)
        exec(compile(request['wrapper_source'], 'bridge_block_definition.py', 'exec'), namespace)
        candidate = request['selected_candidate']
        path = Path('/phase3/phase5/fitting') / request['fit_key'] / Path(candidate['checkpoint_path']).name
        if digest(path) != candidate['checkpoint_sha256']:
            raise RuntimeError('Selected bridge checkpoint differs.')
        checkpoint = torch.load(path, weights_only=True, map_location='cpu')
        if (checkpoint['request_key'] != request['fit_key'] or checkpoint['layer'] != 5
                or checkpoint['step'] != candidate['selected_update']):
            raise RuntimeError('Bridge provenance differs.')
        bridge = namespace['ResidualBottleneckBridge'](4096, 256)
        bridge.load_state_dict(checkpoint['state_dict'])
        bridge.requires_grad_(False)
        namespace['insert_bridge'](model, 5, bridge, QwenImage21KVCache)
        del checkpoint
        model.requires_grad_(False).eval()
        gc.collect()
        torch.cuda.empty_cache()
        baseline = predictions()
        report['student_calls'] += len(baseline)
        save('candidate_predictions.pt', baseline)
        # The 10 gate prompts must reproduce the saved Phase 6 trained-bridge velocities.
        replay = request['selected_replay']
        replay_dir = Path('/phase3/phase6/verified_student_replay') / replay['request_key']
        reference = {}
        for record in request['records']:
            if not record['capture_id'].startswith('p5-validation-'):
                continue
            name = 'students-' + record['capture_id'] + '.pt'
            if digest(replay_dir / name) != replay['artifacts'][name]:
                raise RuntimeError('Phase 6 student replay checksum differs.')
            saved = torch.load(replay_dir / name, weights_only=True, map_location='cpu')
            for output in saved['outputs']:
                if output['candidate_id'] == candidate['candidate_id'] and output['variant'] == 'trained':
                    reference[(output['source_id'], output['stage'])] = output['velocity']
        gate_rows = [r for r in baseline if r['panel'] == 'gate10']
        if len(reference) != 30 or len(gate_rows) != 30:
            raise RuntimeError('Phase 6 replay coverage differs.')
        reproduction = [compare(r['velocity'], reference[(r['source_id'], r['stage'])]) for r in gate_rows]
        report['candidate_replay_max_relative_l2'] = max(r['relative_l2_error'] for r in reproduction)
        if report['candidate_replay_max_relative_l2'] > 0.001:
            raise RuntimeError('Selected student failed Phase 6 output replay')
        del reference

        class IdentityBlock(torch.nn.Module):
            def __init__(self, original):
                super().__init__()
                self.attn = SimpleNamespace(processor=original.attn.processor)

            def forward(self, hidden_states, *args, **kwargs):
                return hidden_states

        def metrics(rows):
            output = []
            for measured, base, teacher_row in zip(rows, baseline, teacher_predictions):
                identity = ('panel', 'source_id', 'stage')
                if any(measured[k] != base[k] or base[k] != teacher_row[k] for k in identity):
                    raise RuntimeError('Prediction alignment differs')
                teacher_error = compare(measured['velocity'], teacher_row['velocity'])['relative_l2_error']
                baseline_error = compare(base['velocity'], teacher_row['velocity'])['relative_l2_error']
                output.append({k: measured[k] for k in measured if k != 'velocity'} |
                              dict(baseline_displacement=compare(measured['velocity'], base['velocity'])['relative_l2_error'],
                                   teacher_relative_l2=teacher_error, baseline_teacher_relative_l2=baseline_error,
                                   teacher_error_change_percentage_points=100 * (teacher_error - baseline_error)))
            return output

        def aggregate(rows):
            fields = ('baseline_displacement', 'teacher_relative_l2', 'baseline_teacher_relative_l2',
                      'teacher_error_change_percentage_points')
            result = dict(comparisons=len(rows))
            for field in fields:
                values = sorted(r[field] for r in rows)
                def percentile(fraction):
                    position = fraction * (len(values) - 1)
                    low = int(position)
                    return values[low] + (values[min(low + 1, len(values) - 1)] - values[low]) * (position - low)
                result[field] = dict(mean=statistics.mean(values), maximum=max(values),
                                     p10=percentile(.1), p50=percentile(.5), p90=percentile(.9))
            result['worst_regressions'] = sorted(rows, key=lambda r:r['teacher_error_change_percentage_points'], reverse=True)[:10]
            return result

        report['baseline'] = aggregate(metrics(baseline))

        # ---- End-to-end bridge training: only the bridge at the slot learns, from the final velocity error. ----
        import random
        from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVLayerCache
        slot = request['slot']
        fit = request['fit']
        conditioned = {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned)
        components = {}
        exec(compile(request['components_source'], 'healing_components.py', 'exec'), components)
        warm_spec = request['warm_start']
        warm_path = Path(warm_spec['remote_path'])
        if digest(warm_path) != warm_spec['sha256']:
            raise RuntimeError('Warm-start checkpoint differs.')
        warm = torch.load(warm_path, weights_only=True, map_location='cpu')
        if warm['layer'] != slot or warm['variant']['name'] != warm_spec['name']:
            raise RuntimeError('Warm-start provenance differs.')
        original_blocks = list(model.transformer_blocks)
        for index in range(slot + 1, len(original_blocks)):
            model.transformer_blocks[index] = components['FunctionalCheckpointBlock'](original_blocks[index], QwenImage21KVLayerCache)
        model.requires_grad_(False)
        model.gradient_checkpointing = False
        gate_teacher = [r for r in teacher_predictions if r['panel'] == 'gate10']

        def gate_error():
            errors = []
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                position = 0
                for item in panels['gate10']:
                    cache = QwenImage21KVCache(32)
                    for stage in item['stages']:
                        reference = gate_teacher[position]
                        if reference['source_id'] != item['row']['source_id'] or reference['stage'] != stage['stage']:
                            raise RuntimeError('Gate alignment differs.')
                        errors.append(compare(forward(item, stage, cache), reference['velocity'])['relative_l2_error'])
                        position += 1
            return sum(errors) / len(errors)

        def train_loss(item):
            cache = QwenImage21KVCache(32)
            losses = []
            with model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for stage, target in zip(item['stages'], item['live_targets']):
                    full = {**item['shared_transformer_kwargs'], **stage['kwargs']}
                    kwargs = gpu({field: full[field] for field in argument_order})
                    kwargs['kv_cache'] = cache
                    prediction = model(**kwargs)[0][:, -stage['target_image_tokens']:].float()
                    target = target.to('cuda').float()
                    losses.append((prediction - target).square().mean() / target.square().mean().clamp_min(1e-12))
                    report['student_calls'] += 1
            loss = torch.stack(losses).mean()
            if not torch.isfinite(loss):
                raise RuntimeError('Nonfinite training loss.')
            return loss

        general_pool = train_full + new_general
        report['variants'] = {}
        for variant in request['variants']:
            if variant['attention']:
                bridge = conditioned['AttentionConditionedBridge'](4096, 1024, variant['heads'], 128)
            else:
                bridge = conditioned['ConditionedBridge'](4096, 1024, True)
            warm_state = dict(warm['state_dict'])
            if variant['attention']:
                # Grow the attention branch: trained heads are copied, new heads start with zero output.
                for name in ('to_q.weight', 'to_k.weight', 'to_v.weight', 'to_out.weight'):
                    target = bridge.state_dict()[name]
                    source = warm_state.get(name)
                    if source is not None and source.shape != target.shape:
                        grown = target.clone()
                        if name == 'to_out.weight':
                            grown.zero_()
                            grown[:, :source.shape[1]] = source
                        else:
                            grown[:source.shape[0]] = source
                        warm_state[name] = grown
            missing, unexpected = bridge.load_state_dict(warm_state, strict=False)
            if unexpected or (missing and not variant['attention']):
                raise RuntimeError(f'Warm start mismatch: missing {missing}, unexpected {unexpected}')
            torch.manual_seed(fit['init_seed'])
            if variant['attention'] and missing:
                for projection in (bridge.to_q, bridge.to_k, bridge.to_v):
                    torch.nn.init.normal_(projection.weight, std=projection.in_features ** -0.5)
            bridge = bridge.float()
            conditioned['insert_conditioned_bridge'](model, slot, bridge, keep_dtype=True)
            bridge = model.transformer_blocks[slot].bridge
            bridge.requires_grad_(True)
            optimizer = torch.optim.AdamW(bridge.parameters(), lr=fit['learning_rate'], weight_decay=0.0)
            rng = random.Random(fit['sampling_seed'])
            curve, best = [], None
            started = time.perf_counter()
            for step in range(fit['updates'] + 1):
                if step % fit['eval_every'] == 0:
                    bridge.eval()
                    score = gate_error()
                    curve.append(dict(step=step, gate10=score, seconds=time.perf_counter() - started))
                    if best is None or score < best[0]:
                        best = (score, step, {k: v.detach().cpu().clone() for k, v in bridge.state_dict().items()})
                    print(f'{variant["name"]} step {step}: gate10 {score:.6f} ({time.perf_counter() - started:.0f}s)', flush=True)
                    (directory / 'progress.json').write_text(json.dumps(report | {'current_curve': curve}, indent=2))
                    results_volume.commit()
                if step == fit['updates']:
                    break
                for group in optimizer.param_groups:
                    group['lr'] = fit['learning_rate'] * min(1.0, (step + 1) / fit['warmup'])
                optimizer.zero_grad(set_to_none=True)
                # Backward recomputes checkpointed blocks: keep the same attention kernel and cache context as forward.
                with model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                    pool = new_text if rng.random() < fit['text_fraction'] else general_pool
                    (train_loss(rng.choice(pool)) * (1 - fit['early_weight'])).backward()
                    (train_loss(rng.choice(early_pool)) * fit['early_weight']).backward()
                torch.nn.utils.clip_grad_norm_(bridge.parameters(), fit['gradient_clip'])
                optimizer.step()
                report['optimizer_updates'] += 1
            bridge.load_state_dict(best[2])
            bridge.requires_grad_(False)
            bridge.eval()
            checkpoint_path = directory / f'bridge_block_{slot:02d}_{variant["name"]}.pt'
            temporary = checkpoint_path.with_suffix('.tmp')
            torch.save({'request_key': key, 'layer': slot, 'variant': variant, 'step': best[1], 'state_dict': best[2]}, temporary)
            temporary.replace(checkpoint_path)
            artifact = dict(name=checkpoint_path.name, sha256=digest(checkpoint_path), bytes=checkpoint_path.stat().st_size)
            report['artifacts'].append(artifact)
            rows = predictions()
            report['student_calls'] += len(rows)
            records = metrics(rows)
            report['variants'][variant['name']] = dict(
                variant=variant, checkpoint=artifact, parameters=sum(v.numel() for v in best[2].values()),
                fit=dict(curve=curve, selected_step=best[1], selected_gate10=best[0]),
                overall=aggregate(records),
                panels={p: aggregate([r for r in records if r['panel'] == p]) for p in panels},
                stages={s: aggregate([r for r in records if r['stage'] == s]) for s in ('early', 'middle', 'late')},
                records=records)
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            print(f'{variant["name"]}: teacher-relative {report["variants"][variant["name"]]["overall"]["teacher_relative_l2"]["mean"]:.6f}', flush=True)
            del optimizer
            gc.collect()
            torch.cuda.empty_cache()
        report['status'] = 'passed'
        report['scope'] = 'End-to-end bridge training (bridge weights only) and fixed-input velocity evaluation; images are separate.'
    except Exception as exc:
        import traceback
        report['status'] = 'failed'
        report['error'] = repr(exc)
        report['traceback'] = traceback.format_exc()
    (directory / 'summary.json').write_text(json.dumps(report, indent=2))
    results_volume.commit()
    return report
'''


SOURCES['conditioned_bridge.py'] = r'''import torch
from torch import nn
from types import SimpleNamespace


class ConditionedBridge(nn.Module):
    """Residual bottleneck bridge that can read the model's shared timestep modulation.

    The transformer computes one modulation tensor per timestep, [attn scale, attn gate, MLP scale, MLP gate],
    shared by every block. A conditioned bridge mirrors a block's MLP branch:
    h + tanh(gate) * up(gelu(down(LN(h) * (1 + scale)))), with scale and gate learned per-channel mixes of the
    shared values. Unconditioned, it is the original Phase 4 bridge: h + up(gelu(down(h))).
    """

    def __init__(self, hidden_dim=4096, rank=256, conditioned=True, eps=1e-6):
        super().__init__()
        self.conditioned = conditioned
        self.down = nn.Linear(hidden_dim, rank, bias=False)
        self.up = nn.Linear(rank, hidden_dim, bias=False)
        nn.init.zeros_(self.up.weight)
        if conditioned:
            self.norm = nn.LayerNorm(hidden_dim, elementwise_affine=False, eps=eps)
            # Start from the block's own MLP modulation (weight 1 on the MLP terms, 0 on the attention terms).
            mix = torch.zeros(2, hidden_dim)
            mix[1] = 1
            self.scale_mix = nn.Parameter(mix.clone())
            self.gate_mix = nn.Parameter(mix.clone())
            self.scale_bias = nn.Parameter(torch.zeros(hidden_dim))
            self.gate_bias = nn.Parameter(torch.zeros(hidden_dim))

    def scale_gate(self, modulation_rows):
        scale1, gate1, scale2, gate2 = modulation_rows.chunk(4, dim=-1)
        scale = self.scale_mix[0] * scale1 + self.scale_mix[1] * scale2 + self.scale_bias
        gate = self.gate_mix[0] * gate1 + self.gate_mix[1] * gate2 + self.gate_bias
        return scale, gate

    def forward(self, hidden_states, modulation_rows=None):
        if not self.conditioned:
            return hidden_states + self.up(nn.functional.gelu(self.down(hidden_states)))
        scale, gate = self.scale_gate(modulation_rows)
        update = self.up(nn.functional.gelu(self.down(self.norm(hidden_states) * (1 + scale))))
        return hidden_states + torch.tanh(gate) * update


class LocalConditionedBridge(ConditionedBridge):
    """Conditioned bridge plus a depthwise 3x3 convolution over the image-token grid inside the bottleneck.

    The convolution starts at zero (identity start) and is applied to target-image tokens only; prompt tokens have
    no grid and pass through the per-token path.
    """
    local = True

    def __init__(self, hidden_dim=4096, rank=1024, eps=1e-6):
        super().__init__(hidden_dim, rank, True, eps)
        self.mix = nn.Parameter(torch.zeros(rank, 9))

    def forward(self, hidden_states, modulation_rows=None, image_mask=None, grid=None):
        scale, gate = self.scale_gate(modulation_rows)
        z = self.down(self.norm(hidden_states) * (1 + scale))
        if grid is not None:
            height, width = grid
            image = z[:, image_mask]
            batch, tokens, rank = image.shape
            planes = image.transpose(1, 2).reshape(batch, rank, height, width)
            mixed = nn.functional.conv2d(planes, self.mix.view(rank, 1, 3, 3).to(planes.dtype), padding=1, groups=rank)
            z = z.clone()
            z[:, image_mask] = image + mixed.reshape(batch, rank, tokens).transpose(1, 2)
        return hidden_states + torch.tanh(gate) * self.up(nn.functional.gelu(z))

    def forward_neighborhood(self, neighborhood, modulation_rows):
        """Training path: neighborhood is (tokens, 9, hidden) in 3x3 raster order, zeros outside the grid."""
        scale, gate = self.scale_gate(modulation_rows)
        z = self.down(self.norm(neighborhood) * (1 + scale.unsqueeze(1)))
        z = z[:, 4] + (z * self.mix.t().unsqueeze(0)).sum(1)
        return neighborhood[:, 4] + torch.tanh(gate) * self.up(nn.functional.gelu(z))


class AttentionConditionedBridge(ConditionedBridge):
    """Conditioned bridge with a small attention branch in front of its MLP, mirroring a block.

    Target-image tokens attend to the prompt prefix and all image tokens with the model's RoPE and key masks; prefix
    tokens only take the MLP path. The branch keeps its own prefix K/V in the layer cache of the slot it replaces.
    The output projection starts at zero, so a fresh branch leaves the conditioned MLP bridge unchanged.
    """
    attention = True

    def __init__(self, hidden_dim=4096, rank=1024, heads=4, head_dim=128, eps=1e-6):
        super().__init__(hidden_dim, rank, True, eps)
        inner = heads * head_dim
        self.heads, self.head_dim, self.eps = heads, head_dim, eps
        self.to_q = nn.Linear(hidden_dim, inner, bias=False)
        self.to_k = nn.Linear(hidden_dim, inner, bias=False)
        self.to_v = nn.Linear(hidden_dim, inner, bias=False)
        self.to_out = nn.Linear(inner, hidden_dim, bias=False)
        nn.init.zeros_(self.to_out.weight)
        self.norm_q = nn.Parameter(torch.ones(head_dim))
        self.norm_k = nn.Parameter(torch.ones(head_dim))
        # Attention modulation starts from the block's own attention terms (weight 1 on scale1/gate1).
        mix = torch.zeros(2, hidden_dim)
        mix[0] = 1
        self.attn_scale_mix = nn.Parameter(mix.clone())
        self.attn_gate_mix = nn.Parameter(mix.clone())
        self.attn_scale_bias = nn.Parameter(torch.zeros(hidden_dim))
        self.attn_gate_bias = nn.Parameter(torch.zeros(hidden_dim))

    def _rms(self, x, weight):
        return x * torch.rsqrt(x.float().square().mean(-1, keepdim=True) + self.eps).to(x.dtype) * weight

    @staticmethod
    def _rope(x, freqs):
        rotated = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2)) * freqs.unsqueeze(1)
        return torch.view_as_real(rotated).flatten(3).type_as(x)

    def attend(self, hidden_states, modulation_rows, rotary_emb, target_count, layer_cache, kv_cache_mode,
               cache_write_slice, key_mask):
        scale1, gate1, scale2, gate2 = modulation_rows.chunk(4, dim=-1)
        scale = self.attn_scale_mix[0] * scale1 + self.attn_scale_mix[1] * scale2 + self.attn_scale_bias
        gate = self.attn_gate_mix[0] * gate1 + self.attn_gate_mix[1] * gate2 + self.attn_gate_bias
        x = self.norm(hidden_states) * (1 + scale)
        q, k, v = (proj(x).unflatten(-1, (self.heads, self.head_dim)) for proj in (self.to_q, self.to_k, self.to_v))
        q, k = self._rms(q, self.norm_q), self._rms(k, self.norm_k)
        if rotary_emb is not None:
            q, k = self._rope(q, rotary_emb), self._rope(k, rotary_emb)
        if layer_cache is not None:
            if kv_cache_mode == 'extract' and cache_write_slice is not None:
                layer_cache.store(k[:, cache_write_slice].clone(), v[:, cache_write_slice].clone())
            elif kv_cache_mode == 'cached':
                cached_k, cached_v = layer_cache.get()
                k, v = torch.cat([cached_k, k], dim=1), torch.cat([cached_v, v], dim=1)
        prefix = hidden_states.shape[1] - target_count
        out = nn.functional.scaled_dot_product_attention(
            q[:, prefix:].transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2), attn_mask=key_mask)
        update = torch.tanh(gate[:, prefix:]) * self.to_out(out.transpose(1, 2).flatten(2))
        return torch.cat([hidden_states[:, :prefix], hidden_states[:, prefix:] + update], dim=1)


def modulation_rows(modulation, target_token_mask):
    """Per-token modulation rows, selected exactly as the transformer block does."""
    if target_token_mask is None:
        return modulation.unsqueeze(1)
    real, zero = modulation[:-1].unsqueeze(1), modulation[-1:].unsqueeze(0)
    return torch.where(target_token_mask.view(1, -1, 1), real, zero)


class ConditionedBridgeBlock(nn.Module):
    def __init__(self, bridge, processor):
        super().__init__()
        self.bridge = bridge
        # Parent forward inspects this processor to choose mask construction; no attention is performed here.
        self.attn = SimpleNamespace(processor=processor)

    def forward(self, hidden_states, modulation, rotary_emb=None, attention_mask=None, target_token_mask=None,
                layer_cache=None, kv_cache_mode=None, cache_write_slice=None, segments=None, key_valid=None):
        if kv_cache_mode not in (None, 'extract', 'cached'):
            raise ValueError('Unknown KV cache mode.')
        rows = modulation_rows(modulation, target_token_mask) if self.bridge.conditioned else None
        dtype = hidden_states.dtype
        bridge_dtype = next(self.bridge.parameters()).dtype
        if getattr(self.bridge, 'attention', False):
            # Small bridges may keep FP32 weights for training; compute in their dtype and return the model dtype.
            target_count = hidden_states.shape[1] if target_token_mask is None else int(target_token_mask.sum())
            key_mask = key_valid[:, None, None, :] if key_valid is not None else attention_mask
            h = hidden_states.to(bridge_dtype)
            r = rows.to(bridge_dtype)
            h = self.bridge.attend(h, r, rotary_emb, target_count, layer_cache, kv_cache_mode, cache_write_slice, key_mask)
            return self.bridge(h, r).to(dtype)
        if bridge_dtype != dtype:
            return self.bridge(hidden_states.to(bridge_dtype), None if rows is None else rows.to(bridge_dtype)).to(dtype)
        if getattr(self.bridge, 'local', False):
            mask = target_token_mask if target_token_mask is not None else torch.ones(
                hidden_states.shape[1], dtype=torch.bool, device=hidden_states.device)
            height, width = self.grid
            if height * width != int(mask.sum()):
                raise RuntimeError('Target image grid does not match the image-token count.')
            return self.bridge(hidden_states, rows, mask, (height, width))
        return self.bridge(hidden_states, rows)


def insert_conditioned_bridge(transformer, layer, bridge, keep_dtype=False):
    """Replace one slot; callers must use fresh KV caches afterwards. keep_dtype keeps FP32 bridge weights."""
    old = transformer.transformer_blocks[layer]
    reference = next(transformer.parameters())
    bridge = bridge.to(device=reference.device) if keep_dtype else bridge.to(device=reference.device, dtype=reference.dtype)
    wrapper = ConditionedBridgeBlock(bridge, old.attn.processor)
    wrapper.train(transformer.training)
    transformer.transformer_blocks[layer] = wrapper
    if getattr(bridge, 'local', False) and not getattr(transformer, '_bridge_grid_hook', False):
        # The block never sees img_shapes; record the target image's (height, width) in tokens on every call.
        def record_grid(module, args, kwargs):
            frames, height, width = kwargs['img_shapes'][0][-1]
            for block in module.transformer_blocks:
                if isinstance(block, ConditionedBridgeBlock):
                    block.grid = (int(height), int(width))
        transformer.register_forward_pre_hook(record_grid, with_kwargs=True)
        transformer._bridge_grid_hook = True
    return old
'''


"""Grow the attention bridge to 8 heads and continue it on 450 text prompts recorded (and saved) in the same job; reuse completed results."""
import argparse
import hashlib
import json
import sys
from pathlib import Path


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--slot', type=int, default=13)
    parser.add_argument('--old-account', action='store_true')
    parser.add_argument('--dry-run', action='store_true', help='Write the request locally without scheduling GPU work.')
    args = parser.parse_args(LAUNCH_ARGS)
    if args.old_account:
        parser.error('Old account is download-only; GPU execution must use the new account.')
    root = ROOT
    protocol_path = root / f'data/phase7/block{args.slot}_bridge_e2e_text8/protocol.json'
    protocol = json.loads(protocol_path.read_bytes())
    scan_b = json.loads(next((root / 'data/phase7/conditional_importance_b').glob('*/request.json')).read_bytes())['request']
    scan_a = json.loads((root / 'data/phase7/conditional_importance/7454da6a5c1dc33af23dd46833e698f9b759d14985ca96340bc3d40cd47fb4f5/request.json').read_bytes())['request']
    if args.slot == 5 or not 0 < args.slot < 31:
        parser.error('Slot must be a surviving interior block.')
    request = {k: v for k, v in scan_b.items() if k not in ('worker_source', 'eligible_original_block_ids', 'selection', 'scope', 'version')}
    warm = next(r for r in (json.loads(p.read_bytes()) for p in (root / 'data/phase7/block13_bridge_e2e_text').glob('*/summary.json')) if r.get('status') == 'passed')
    import random
    used = {r['source_id'] for r in scan_a['records']} | {a['source_id'] for a in scan_a['fresh_aligned_train']['artifacts']}
    used |= {'p1-' + e['capture_id'].split('p1-')[1].rsplit('-s', 1)[0] for e in scan_a['early_pool']['examples']}
    used |= {r['source_id'] for r in scan_a['expanded_validation']['records']}
    resolutions = json.loads((root / 'data/phase2/protocol.json').read_bytes())['RESOLUTIONS']
    pilot = [json.loads(line) for line in (root / 'data/phase1/pilot_20k/train.jsonl').read_text(encoding='utf-8').splitlines()]
    pilot = [r for r in pilot if r['split'] == 'train' and r['id'] not in used and r['aspect_ratio'] in resolutions]
    picker = random.Random(20261020)
    text_rows = picker.sample(sorted((r for r in pilot if r['subdimension'] == 'text_rendering'), key=lambda r: r['id']), 150)
    general_rows = picker.sample(sorted((r for r in pilot if r['subdimension'] != 'text_rendering'), key=lambda r: r['id']), 50)
    chosen = {r['id'] for r in text_rows + general_rows}
    extra_text = random.Random(20261022).sample(sorted((r for r in pilot if r['subdimension'] == 'text_rendering' and r['id'] not in chosen),
                                                       key=lambda r: r['id']), 300)
    new_prompts = [dict(source_id=r['id'], prompt=r['prompt'], seed=r['seed'], dimension=r['dimension'], subdimension=r['subdimension'],
                        width=resolutions[r['aspect_ratio']][0], height=resolutions[r['aspect_ratio']][1])
                   for r in text_rows + general_rows + extra_text]
    source = SOURCES['bridge_end_to_end_v3_worker.py']
    request.update(
        version='phase7-bridge-end-to-end-text8-v1', new_prompts=new_prompts, worker_source=source, slot=args.slot,
        protocol_sha256=hashlib.sha256(protocol_path.read_bytes()).hexdigest(),
        conditioned_bridge_source=SOURCES['conditioned_bridge.py'],
        components_source=scan_a['components_source'],
        fresh_aligned_train=scan_a['fresh_aligned_train'], early_pool=scan_a['early_pool'],
        warm_start=dict(name='e2e_attn_text', remote_path=f"/phase3/phase7/bridge_end_to_end_v2/{warm['request_key']}/{warm['variants']['e2e_attn_text']['checkpoint']['name']}",
                        sha256=warm['variants']['e2e_attn_text']['checkpoint']['sha256']),
        variants=[dict(name='e2e_attn8_text', attention=True, heads=8)],
        fit=dict(init_seed=20261017, sampling_seed=20261023, learning_rate=1e-4, warmup=20, gradient_clip=1.0,
                 updates=600, eval_every=100, early_weight=0.3, text_fraction=0.4),
        selection='Lowest gate10 velocity error at evaluations every 50 updates, including the warm start',
        scope=f'End-to-end bridges for block {args.slot} inside frozen B; TRAIN-only; whole-student velocity on scan inputs; no images or latency')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = protocol_path.parent / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2)+'\n', encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase7/bridge_end_to_end_v3/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        try:
            b''.join(results_volume.read_file(remote + '/started.json'))
        except FileNotFoundError:
            pass
        else:
            raise RuntimeError('Run already started; inspect its progress. No duplicate GPU scheduled.')
        app = modal.App('nihonga-phase7-bridge-end-to-end-text8')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(source, 'bridge_end_to_end_v3_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8,
                              memory=98304, max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['train_bridge_end_to_end'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2)+'\n')
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2)+'\n', encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'candidate_replay_max_relative_l2')}), flush=True)
    for name, v in (report.get('variants') or {}).items():
        print(name, 'step', v['fit']['selected_step'], 'teacher-relative', v['overall']['teacher_relative_l2']['mean'], flush=True)
    if report.get('status') != 'passed':
        raise SystemExit(1)

summary = reuse_or_launch(RUN, main)

print('replay', summary['new_prompt_replay_relative_l2'], summary['new_prompt_counts'])
show_variants(summary)

### What the results mean

Best velocity so far: **6.11%** overall, 6.08% wider (B: 5.40% / 5.37%).

## 13b. Text check of the 8-head bridge

Same text grid and 30 prompts.

In [ ]:
# Text check of the 8-head bridge: reuses the saved run; launches GPU work only when its result is missing and ALLOW_GPU is True.

RUN = 'data/phase7/multiseed_text/320d82f71dfdea06818ff5f50e2e4af933519b5c511111159c1bdea4a00f79aa'

LAUNCH_ARGS = ['--source', 'block13_bridge_e2e_text8', '--variant', 'e2e_attn8_text', '--with-base-prompts']

SOURCES['multiseed_text_worker.py'] = r'''def generate_multiseed_text(request, key):
    """Extra seeds for the text prompts: teacher, B, and C with the best block-13 bridge.
    Same prompt, CPU noise seed, size, scheduler and 40 steps for every model. Qualitative screen only."""
    import gc
    import hashlib
    import inspect
    import json
    import os
    import time
    from pathlib import Path

    import torch
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache, QwenImage21KVLayerCache

    directory = Path('/phase3/phase7/multiseed_text') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete image attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def table_hash(value):
        return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

    variants = ['teacher', 'B', request['c_variant']['name']]
    report = {'request_key': key, 'status': 'running', 'rows': [], 'artifacts': [],
              'calls': {v: 0 for v in variants}, 'optimizer_updates': 0}
    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('Expected H100.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
                                                   dtype=torch.bfloat16, local_files_only=True).to('cuda')
        pipe.set_progress_bar_config(disable=True)
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Transformer source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--')) /
                       'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        # Derived position/time tables: use the audited values on every host (as in steps 7-11).
        derived = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
        derived['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
        expected = request['expected_derived_tables']
        mismatched = sorted(n for n, v in derived.items() if table_hash(v) != expected[n])
        if mismatched:
            canonical = torch.load(Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt'),
                                   weights_only=True, map_location='cpu')
            if any(table_hash(canonical[n]) != expected[n] for n in expected):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [canonical[f'pos_embed.freqs.{a}'].to(device=v.device, dtype=v.dtype) for a, v in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(canonical['time_text_embed.time_proj.freqs'])
            if any(table_hash(v) != expected[n] for n, v in
                   list({f'pos_embed.freqs.{a}': v for a, v in enumerate(model.pos_embed.freqs)}.items()) +
                   [('time_text_embed.time_proj.freqs', model.time_text_embed.time_proj.freqs)]):
                raise RuntimeError('Derived tables still differ after loading the audited copy.')
            report['canonical_tables_applied'] = mismatched

        def generate(row, variant):
            path = directory / (row['source_id'] + '-' + variant + '.png')
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'],
                             num_inference_steps=40, true_cfg_scale=teacher['true_cfg_scale'],
                             generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=True, output_type='pil').images[0]
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            temporary = path.with_suffix('.tmp')
            image.save(temporary, format='PNG')
            temporary.replace(path)
            results_volume.commit()
            report['calls'][variant] += 1
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            return {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size, 'seconds': elapsed}

        namespace = {}
        exec(compile(request['core_source'], 'bridge_definition.py', 'exec'), namespace)
        exec(compile(request['wrapper_source'], 'bridge_block_definition.py', 'exec'), namespace)
        conditioned = {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned)

        def load_checkpoint(spec):
            path = Path(spec['remote_path'])
            if digest(path) != spec['sha256']:
                raise RuntimeError('Bridge checkpoint differs: ' + spec['remote_path'])
            return torch.load(path, weights_only=True, map_location='cpu')

        outputs = {v: {} for v in variants}
        references = not request.get('candidate_only', False)
        if references:
            for row in request['rows']:
                outputs['teacher'][row['source_id']] = generate(row, 'teacher')
        block5 = request['block5_bridge']
        checkpoint = load_checkpoint(block5)
        if checkpoint['layer'] != 5 or checkpoint['step'] != block5['step']:
            raise RuntimeError('Block-5 bridge provenance differs.')
        bridge = namespace['ResidualBottleneckBridge'](4096, 256)
        bridge.load_state_dict(checkpoint['state_dict'])
        bridge.requires_grad_(False)
        namespace['insert_bridge'](model, 5, bridge, QwenImage21KVCache)
        model.eval()
        if references:
            for row in request['rows']:
                outputs['B'][row['source_id']] = generate(row, 'B')
        spec = request['c_variant']
        checkpoint = load_checkpoint(spec)
        if checkpoint['layer'] != 13 or checkpoint['variant']['name'] != spec['name']:
            raise RuntimeError('Block-13 bridge provenance differs.')
        fitted = checkpoint['variant']
        keep_dtype = False
        if fitted.get('attention'):
            bridge13 = conditioned['AttentionConditionedBridge'](4096, 1024, fitted['heads'], 128)
            keep_dtype = True
        elif 'attention' in fitted:
            bridge13 = conditioned['ConditionedBridge'](4096, 1024, True)
            keep_dtype = True
        elif fitted.get('local'):
            bridge13 = conditioned['LocalConditionedBridge'](4096, fitted['rank'])
        else:
            bridge13 = conditioned['ConditionedBridge'](4096, fitted['rank'], fitted['conditioned'])
        bridge13.load_state_dict(checkpoint['state_dict'])
        bridge13.requires_grad_(False)
        conditioned['insert_conditioned_bridge'](model, 13, bridge13.float() if keep_dtype else bridge13, keep_dtype=keep_dtype)
        model.eval()
        for row in request['rows']:
            outputs[spec['name']][row['source_id']] = generate(row, spec['name'])
        rendered = [v for v in variants if outputs[v]]
        for row in request['rows']:
            report['rows'].append({**row, **{v: outputs[v][row['source_id']] for v in rendered}})
            report['artifacts'].extend(outputs[v][row['source_id']] for v in rendered)
        report['status'] = 'passed'
    except Exception as exc:
        report['status'] = 'failed'
        report['error'] = type(exc).__name__ + ': ' + str(exc)
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=True) + '\n')
    results_volume.commit()
    return report
'''


SOURCES['conditioned_bridge.py'] = r'''import torch
from torch import nn
from types import SimpleNamespace


class ConditionedBridge(nn.Module):
    """Residual bottleneck bridge that can read the model's shared timestep modulation.

    The transformer computes one modulation tensor per timestep, [attn scale, attn gate, MLP scale, MLP gate],
    shared by every block. A conditioned bridge mirrors a block's MLP branch:
    h + tanh(gate) * up(gelu(down(LN(h) * (1 + scale)))), with scale and gate learned per-channel mixes of the
    shared values. Unconditioned, it is the original Phase 4 bridge: h + up(gelu(down(h))).
    """

    def __init__(self, hidden_dim=4096, rank=256, conditioned=True, eps=1e-6):
        super().__init__()
        self.conditioned = conditioned
        self.down = nn.Linear(hidden_dim, rank, bias=False)
        self.up = nn.Linear(rank, hidden_dim, bias=False)
        nn.init.zeros_(self.up.weight)
        if conditioned:
            self.norm = nn.LayerNorm(hidden_dim, elementwise_affine=False, eps=eps)
            # Start from the block's own MLP modulation (weight 1 on the MLP terms, 0 on the attention terms).
            mix = torch.zeros(2, hidden_dim)
            mix[1] = 1
            self.scale_mix = nn.Parameter(mix.clone())
            self.gate_mix = nn.Parameter(mix.clone())
            self.scale_bias = nn.Parameter(torch.zeros(hidden_dim))
            self.gate_bias = nn.Parameter(torch.zeros(hidden_dim))

    def scale_gate(self, modulation_rows):
        scale1, gate1, scale2, gate2 = modulation_rows.chunk(4, dim=-1)
        scale = self.scale_mix[0] * scale1 + self.scale_mix[1] * scale2 + self.scale_bias
        gate = self.gate_mix[0] * gate1 + self.gate_mix[1] * gate2 + self.gate_bias
        return scale, gate

    def forward(self, hidden_states, modulation_rows=None):
        if not self.conditioned:
            return hidden_states + self.up(nn.functional.gelu(self.down(hidden_states)))
        scale, gate = self.scale_gate(modulation_rows)
        update = self.up(nn.functional.gelu(self.down(self.norm(hidden_states) * (1 + scale))))
        return hidden_states + torch.tanh(gate) * update


class LocalConditionedBridge(ConditionedBridge):
    """Conditioned bridge plus a depthwise 3x3 convolution over the image-token grid inside the bottleneck.

    The convolution starts at zero (identity start) and is applied to target-image tokens only; prompt tokens have
    no grid and pass through the per-token path.
    """
    local = True

    def __init__(self, hidden_dim=4096, rank=1024, eps=1e-6):
        super().__init__(hidden_dim, rank, True, eps)
        self.mix = nn.Parameter(torch.zeros(rank, 9))

    def forward(self, hidden_states, modulation_rows=None, image_mask=None, grid=None):
        scale, gate = self.scale_gate(modulation_rows)
        z = self.down(self.norm(hidden_states) * (1 + scale))
        if grid is not None:
            height, width = grid
            image = z[:, image_mask]
            batch, tokens, rank = image.shape
            planes = image.transpose(1, 2).reshape(batch, rank, height, width)
            mixed = nn.functional.conv2d(planes, self.mix.view(rank, 1, 3, 3).to(planes.dtype), padding=1, groups=rank)
            z = z.clone()
            z[:, image_mask] = image + mixed.reshape(batch, rank, tokens).transpose(1, 2)
        return hidden_states + torch.tanh(gate) * self.up(nn.functional.gelu(z))

    def forward_neighborhood(self, neighborhood, modulation_rows):
        """Training path: neighborhood is (tokens, 9, hidden) in 3x3 raster order, zeros outside the grid."""
        scale, gate = self.scale_gate(modulation_rows)
        z = self.down(self.norm(neighborhood) * (1 + scale.unsqueeze(1)))
        z = z[:, 4] + (z * self.mix.t().unsqueeze(0)).sum(1)
        return neighborhood[:, 4] + torch.tanh(gate) * self.up(nn.functional.gelu(z))


class AttentionConditionedBridge(ConditionedBridge):
    """Conditioned bridge with a small attention branch in front of its MLP, mirroring a block.

    Target-image tokens attend to the prompt prefix and all image tokens with the model's RoPE and key masks; prefix
    tokens only take the MLP path. The branch keeps its own prefix K/V in the layer cache of the slot it replaces.
    The output projection starts at zero, so a fresh branch leaves the conditioned MLP bridge unchanged.
    """
    attention = True

    def __init__(self, hidden_dim=4096, rank=1024, heads=4, head_dim=128, eps=1e-6):
        super().__init__(hidden_dim, rank, True, eps)
        inner = heads * head_dim
        self.heads, self.head_dim, self.eps = heads, head_dim, eps
        self.to_q = nn.Linear(hidden_dim, inner, bias=False)
        self.to_k = nn.Linear(hidden_dim, inner, bias=False)
        self.to_v = nn.Linear(hidden_dim, inner, bias=False)
        self.to_out = nn.Linear(inner, hidden_dim, bias=False)
        nn.init.zeros_(self.to_out.weight)
        self.norm_q = nn.Parameter(torch.ones(head_dim))
        self.norm_k = nn.Parameter(torch.ones(head_dim))
        # Attention modulation starts from the block's own attention terms (weight 1 on scale1/gate1).
        mix = torch.zeros(2, hidden_dim)
        mix[0] = 1
        self.attn_scale_mix = nn.Parameter(mix.clone())
        self.attn_gate_mix = nn.Parameter(mix.clone())
        self.attn_scale_bias = nn.Parameter(torch.zeros(hidden_dim))
        self.attn_gate_bias = nn.Parameter(torch.zeros(hidden_dim))

    def _rms(self, x, weight):
        return x * torch.rsqrt(x.float().square().mean(-1, keepdim=True) + self.eps).to(x.dtype) * weight

    @staticmethod
    def _rope(x, freqs):
        rotated = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2)) * freqs.unsqueeze(1)
        return torch.view_as_real(rotated).flatten(3).type_as(x)

    def attend(self, hidden_states, modulation_rows, rotary_emb, target_count, layer_cache, kv_cache_mode,
               cache_write_slice, key_mask):
        scale1, gate1, scale2, gate2 = modulation_rows.chunk(4, dim=-1)
        scale = self.attn_scale_mix[0] * scale1 + self.attn_scale_mix[1] * scale2 + self.attn_scale_bias
        gate = self.attn_gate_mix[0] * gate1 + self.attn_gate_mix[1] * gate2 + self.attn_gate_bias
        x = self.norm(hidden_states) * (1 + scale)
        q, k, v = (proj(x).unflatten(-1, (self.heads, self.head_dim)) for proj in (self.to_q, self.to_k, self.to_v))
        q, k = self._rms(q, self.norm_q), self._rms(k, self.norm_k)
        if rotary_emb is not None:
            q, k = self._rope(q, rotary_emb), self._rope(k, rotary_emb)
        if layer_cache is not None:
            if kv_cache_mode == 'extract' and cache_write_slice is not None:
                layer_cache.store(k[:, cache_write_slice].clone(), v[:, cache_write_slice].clone())
            elif kv_cache_mode == 'cached':
                cached_k, cached_v = layer_cache.get()
                k, v = torch.cat([cached_k, k], dim=1), torch.cat([cached_v, v], dim=1)
        prefix = hidden_states.shape[1] - target_count
        out = nn.functional.scaled_dot_product_attention(
            q[:, prefix:].transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2), attn_mask=key_mask)
        update = torch.tanh(gate[:, prefix:]) * self.to_out(out.transpose(1, 2).flatten(2))
        return torch.cat([hidden_states[:, :prefix], hidden_states[:, prefix:] + update], dim=1)


def modulation_rows(modulation, target_token_mask):
    """Per-token modulation rows, selected exactly as the transformer block does."""
    if target_token_mask is None:
        return modulation.unsqueeze(1)
    real, zero = modulation[:-1].unsqueeze(1), modulation[-1:].unsqueeze(0)
    return torch.where(target_token_mask.view(1, -1, 1), real, zero)


class ConditionedBridgeBlock(nn.Module):
    def __init__(self, bridge, processor):
        super().__init__()
        self.bridge = bridge
        # Parent forward inspects this processor to choose mask construction; no attention is performed here.
        self.attn = SimpleNamespace(processor=processor)

    def forward(self, hidden_states, modulation, rotary_emb=None, attention_mask=None, target_token_mask=None,
                layer_cache=None, kv_cache_mode=None, cache_write_slice=None, segments=None, key_valid=None):
        if kv_cache_mode not in (None, 'extract', 'cached'):
            raise ValueError('Unknown KV cache mode.')
        rows = modulation_rows(modulation, target_token_mask) if self.bridge.conditioned else None
        dtype = hidden_states.dtype
        bridge_dtype = next(self.bridge.parameters()).dtype
        if getattr(self.bridge, 'attention', False):
            # Small bridges may keep FP32 weights for training; compute in their dtype and return the model dtype.
            target_count = hidden_states.shape[1] if target_token_mask is None else int(target_token_mask.sum())
            key_mask = key_valid[:, None, None, :] if key_valid is not None else attention_mask
            h = hidden_states.to(bridge_dtype)
            r = rows.to(bridge_dtype)
            h = self.bridge.attend(h, r, rotary_emb, target_count, layer_cache, kv_cache_mode, cache_write_slice, key_mask)
            return self.bridge(h, r).to(dtype)
        if bridge_dtype != dtype:
            return self.bridge(hidden_states.to(bridge_dtype), None if rows is None else rows.to(bridge_dtype)).to(dtype)
        if getattr(self.bridge, 'local', False):
            mask = target_token_mask if target_token_mask is not None else torch.ones(
                hidden_states.shape[1], dtype=torch.bool, device=hidden_states.device)
            height, width = self.grid
            if height * width != int(mask.sum()):
                raise RuntimeError('Target image grid does not match the image-token count.')
            return self.bridge(hidden_states, rows, mask, (height, width))
        return self.bridge(hidden_states, rows)


def insert_conditioned_bridge(transformer, layer, bridge, keep_dtype=False):
    """Replace one slot; callers must use fresh KV caches afterwards. keep_dtype keeps FP32 bridge weights."""
    old = transformer.transformer_blocks[layer]
    reference = next(transformer.parameters())
    bridge = bridge.to(device=reference.device) if keep_dtype else bridge.to(device=reference.device, dtype=reference.dtype)
    wrapper = ConditionedBridgeBlock(bridge, old.attn.processor)
    wrapper.train(transformer.training)
    transformer.transformer_blocks[layer] = wrapper
    if getattr(bridge, 'local', False) and not getattr(transformer, '_bridge_grid_hook', False):
        # The block never sees img_shapes; record the target image's (height, width) in tokens on every call.
        def record_grid(module, args, kwargs):
            frames, height, width = kwargs['img_shapes'][0][-1]
            for block in module.transformer_blocks:
                if isinstance(block, ConditionedBridgeBlock):
                    block.grid = (int(height), int(width))
        transformer.register_forward_pre_hook(record_grid, with_kwargs=True)
        transformer._bridge_grid_hook = True
    return old
'''


"""Render the text prompts with extra seeds for teacher, B and C (best block-13 bridge); qualitative only."""
import argparse
import hashlib
import json
import sys
from pathlib import Path


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--dry-run', action='store_true')
    parser.add_argument('--variant', default='cond_r1024_long')
    parser.add_argument('--source', default='block13_bridge_v3', help='Run folder holding the bridge checkpoint.')
    parser.add_argument('--with-base-prompts', action='store_true', help='Also render the 30 base prompts at seed 0; candidate only.')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    recovered = root / 'data/recovered/modal'
    images_key = 'dda8c184e657dfb604c35fb261d83371c108c39fac033a0f57db66f7686c30ad'
    base = json.loads((recovered / 'phase7/candidate_images' / images_key / 'request.json').read_bytes())['request']
    scan_b = json.loads(next((root / 'data/phase7/conditional_importance_b').glob('*/request.json')).read_bytes())['request']
    fit = next(r for r in (json.loads(p.read_bytes()) for p in (root / 'data/phase7' / args.source).glob('*/summary.json')) if r.get('status') == 'passed')
    best = fit['variants'][args.variant]
    remote_dir = {'block13_bridge_v3': 'bridge_variants_v3', 'block13_bridge_e2e': 'bridge_end_to_end', 'block13_bridge_e2e_text': 'bridge_end_to_end_v2', 'block13_bridge_e2e_text8': 'bridge_end_to_end_v3'}[args.source]
    selected = scan_b['selected_candidate']
    text_rows = [r for r in base['rows'] if r['subdimension'] in ('text_rendering', 'posters')]
    rows = [{**r, 'source_id': f"{r['source_id']}-s{k}", 'seed': (r['seed'] + 1000003 * k) % 2**32}
            for r in text_rows for k in (1, 2, 3)]
    if args.with_base_prompts:
        rows += base['rows']
    request = {k: base[k] for k in ('teacher', 'transformer_source_sha256', 'runtime_environment', 'expected_derived_tables',
                                    'canonical_tables_key', 'core_source', 'wrapper_source')}
    request.update(
        version='phase7-multiseed-text-v1', rows=rows,
        worker_source=SOURCES['multiseed_text_worker.py'],
        conditioned_bridge_source=SOURCES['conditioned_bridge.py'],
        execution_account='new',
        block5_bridge=dict(remote_path=f"/phase3/phase5/fitting/{scan_b['fit_key']}/{Path(selected['checkpoint_path']).name}",
                           sha256=selected['checkpoint_sha256'], step=selected['selected_update']),
        c_variant=dict(name=args.variant, remote_path=f"/phase3/phase7/{remote_dir}/{fit['request_key']}/{best['checkpoint']['name']}",
                       sha256=best['checkpoint']['sha256']),
        candidate_only=args.with_base_prompts,
        scope='Extra seeds (3 per prompt) for the 5 text prompts of the Phase 7 image set' + ('; plus the 30 base prompts at seed 0, candidate only' if args.with_base_prompts else '; seed 0 images are reused'))
    from experiment_runtime import pin_reference_runtime
    pin_reference_runtime(request, recovered)
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase7/multiseed_text' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2)+'\n', encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase7/multiseed_text/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        try:
            b''.join(results_volume.read_file(remote + '/started.json'))
        except FileNotFoundError:
            pass
        else:
            raise RuntimeError('Run already started; inspect its progress. No duplicate GPU scheduled.')
        app = modal.App('nihonga-phase7-multiseed-text')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'multiseed_text_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8,
                              memory=98304, max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['generate_multiseed_text'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2)+'\n')
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2)+'\n', encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'calls')}), flush=True)
    if report.get('status') != 'passed':
        raise SystemExit(1)

summary = reuse_or_launch(RUN, main)

show_images(RUN + '/sheets', ['p1-2fa54a3c4eb8.jpg', 'p1-c9344c076abf.jpg'])

### What the results mean


| Exact signs, 3 signs x 4 seeds | Teacher | B | Per-token | Attention | + 150 text | **8 heads + 450 text** |
|---|---|---|---|---|---|---|
| Total | 11/12 | 9/12 | 2/12 | 4/12 | 5/12 | **6/12** |

**Decision (rule registered before the run): 6/12, so a bridge alone does not carry spelling.** Strategy change:
define the bridges for all four removed blocks with this recipe, then heal the whole student on text-rich data,
instead of perfecting each bridge on its own.

### Seed grids and 30-prompt sheets for any rendered bridge

Builds the text grids (rows teacher / B / bridge, columns four seeds) and the 30-prompt sheets.

In [ ]:
# Seed grids and 30-prompt sheets for any rendered bridge: local work only (no GPU).
LAUNCH_ARGS = ['--variant', 'e2e_attn8_text']

"""Build per-prompt seed grids (rows: teacher, B, C; columns: 4 seeds) for the multi-seed text check (no GPU work)."""
import hashlib
import json
import textwrap
from pathlib import Path


def main():
    import argparse
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--variant', default='cond_r1024_long')
    args = parser.parse_args(LAUNCH_ARGS)
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='download')
    import modal
    from PIL import Image, ImageDraw, ImageFont
    root = ROOT
    volume = modal.Volume.from_name('nihonga-phase3-artifacts')
    run_dir = next(p.parent for p in (root / 'data/phase7/multiseed_text').glob('*/request.json')
                   if json.loads(p.read_bytes())['request']['c_variant']['name'] == args.variant)
    request = json.loads((run_dir / 'request.json').read_bytes())
    key, c_name = request['request_key'], request['request']['c_variant']['name']
    remote = f'/phase7/multiseed_text/{key}'
    summary = json.loads(b''.join(volume.read_file(remote + '/summary.json')))
    if summary.get('status') != 'passed':
        raise RuntimeError('Multi-seed run did not pass.')
    (run_dir / 'summary.json').write_text(json.dumps(summary, indent=2) + '\n', encoding='utf-8')
    expected = {a['name']: a['sha256'] for a in summary['artifacts']}
    store = root / '.archive/multiseed_text' / key
    store.mkdir(parents=True, exist_ok=True)
    for name, sha in expected.items():
        path = store / name
        if not path.exists():
            path.write_bytes(b''.join(volume.read_file(f'{remote}/{name}')))
        if hashlib.sha256(path.read_bytes()).hexdigest() != sha:
            raise RuntimeError('Image hash mismatch: ' + name)
    # Seed 0: teacher and B from the first image run, C from the variant image run.
    first = json.loads(next((root / 'data/phase7/bridge_images').glob('*/summary.json')).read_bytes())
    first_store = root / '.archive/bridge_images' / first['request_key']
    own_seed0 = any(name == f"{r['source_id']}-{c_name}.png" for r in first['rows'] for name in expected)
    if own_seed0:
        variant_store = store
    else:
        variant_key = next(json.loads(p.read_bytes())['request_key'] for p in (root / 'data/phase7/bridge_variant_images').glob('*/request.json')
                           if any(v['name'] == c_name for v in json.loads(p.read_bytes())['request']['variants']))
        variant_store = root / '.archive/bridge_variant_images' / variant_key
    # Candidate-only runs reuse the extra-seed teacher and B images of the run that rendered them.
    reference_store = next(root / '.archive/multiseed_text' / json.loads(p.read_bytes())['request_key']
                           for p in (root / 'data/phase7/multiseed_text').glob('*/request.json')
                           if not json.loads(p.read_bytes())['request'].get('candidate_only'))
    face = next((f for f in ('C:/Windows/Fonts/msyh.ttc', 'C:/Windows/Fonts/simsun.ttc') if Path(f).exists()), 'C:/Windows/Fonts/arial.ttf')
    font, big = ImageFont.truetype(face, 20), ImageFont.truetype(face, 24)
    sheets = run_dir / 'sheets'
    sheets.mkdir(exist_ok=True)
    tile, label_width = 420, 230
    base_rows = [r for r in first['rows'] if r['subdimension'] in ('text_rendering', 'posters')]
    for row in base_rows:
        sid = row['source_id']
        canvas = Image.new('RGB', (label_width + 4 * tile, 110 + 3 * tile), 'white')
        draw = ImageDraw.Draw(canvas)
        for line, text in enumerate(textwrap.wrap(row['prompt'], 200 if row['prompt'].isascii() else 100)[:2]):
            draw.text((10, 8 + 26 * line), text, font=font, fill='black')
        for column in range(4):
            draw.text((label_width + column * tile + 10, 76), f'seed {column}', font=big, fill='black')
        models = [('teacher', 'Teacher'), ('B', 'B'), (c_name, 'C (best bridge)')]
        for line, (model, label) in enumerate(models):
            y = 110 + line * tile
            draw.text((10, y + tile // 2 - 12), label, font=big, fill='black')
            for column in range(4):
                if column == 0:
                    path = (first_store / row[model]['name']) if model in ('teacher', 'B') else variant_store / f'{sid}-{model}.png'
                elif model in ('teacher', 'B') and request['request'].get('candidate_only'):
                    path = reference_store / f'{sid}-s{column}-{model}.png'
                else:
                    path = store / f'{sid}-s{column}-{model}.png'
                with Image.open(path) as image:
                    canvas.paste(image.convert('RGB').resize((tile, tile), Image.Resampling.LANCZOS), (label_width + column * tile, y))
        canvas.save(sheets / f'{sid}.jpg', quality=88)
    if own_seed0:
        # 30-prompt sheets: teacher | B | first C | best per-token bridge | this candidate.
        long_key = next(json.loads(p.read_bytes())['request_key'] for p in (root / 'data/phase7/bridge_variant_images').glob('*/request.json')
                        if any(v['name'] == 'cond_r1024_long' for v in json.loads(p.read_bytes())['request']['variants']))
        long_store = root / '.archive/bridge_variant_images' / long_key
        columns = [('Teacher', lambda r: first_store / r['teacher']['name']), ('B', lambda r: first_store / r['B']['name']),
                   ('First C (plain r256)', lambda r: first_store / r['C']['name']),
                   ('cond r1024, 3k', lambda r: long_store / f"{r['source_id']}-cond_r1024_long.png"),
                   (c_name, lambda r: store / f"{r['source_id']}-{c_name}.png")]
        grid = run_dir / 'sheets_30'
        grid.mkdir(exist_ok=True)
        small = 480
        for index, row in enumerate(first['rows']):
            canvas = Image.new('RGB', (len(columns) * small, small + 140), 'white')
            draw = ImageDraw.Draw(canvas)
            draw.text((10, 6), f"P{index + 1:02d} | {row.get('subdimension', '')}", font=big, fill='black')
            for line, text in enumerate(textwrap.wrap(row['prompt'], 220 if row['prompt'].isascii() else 110)[:2]):
                draw.text((10, 40 + 24 * line), text, font=font, fill='black')
            for column, (label, locate) in enumerate(columns):
                draw.text((column * small + 8, 105), label, font=big, fill='black')
                with Image.open(locate(row)) as image:
                    canvas.paste(image.convert('RGB').resize((small, small), Image.Resampling.LANCZOS), (column * small, 140))
            canvas.save(grid / f'P{index + 1:02d}.jpg', quality=88)
    print(json.dumps(dict(sheets=len(base_rows), folder=str(sheets))))

if not (ROOT / 'data/phase7/multiseed_text/320d82f71dfdea06818ff5f50e2e4af933519b5c511111159c1bdea4a00f79aa/sheets_30/P30.jpg').exists():
    main()
else:
    print("Saved output present; not rebuilt.")

## 14. Healing pilot: does healing the whole student recover spelling?

The strategy now rests on one untested assumption: that healing the whole student recovers the spelling a bridge
alone cannot. The pilot tests it on the two blocks already removed, before spending on blocks #3 and #4.

* **Student C:** B plus the 8-head block-13 attention bridge (30 blocks).
* **Healing:** LoRA (rank 16) on q, k, v and out of all 30 original blocks, base weights frozen; both bridges fully
  trainable. LoRA starts at zero, so step 0 is exactly C. It is cheap, controlled and reversible, and merges into the
  weights after training at no inference cost (needed before the later speed-distillation LoRA).
* **Data:** the 60 original TRAIN prompts, 600 early examples and the **500 recordings saved by step 13** (450 text):
  nothing is generated again. Final-velocity loss against live teacher targets.
* **Decision rule** (committed first, `data/phase7/healing_pilot/protocol.json`): exact signs across four seeds
  >= 8/12 means healing recovers spelling and the plan continues; below 8/12, rethink before spending more.

In [ ]:
# Healing pilot: reuses the saved run; launches GPU work only when its result is missing and ALLOW_GPU is True.
RUN_GLOB = 'data/phase7/healing_pilot'
LAUNCH_ARGS = []

SOURCES['healing_pilot_worker.py'] = r'''def heal_student(request, key):
    """LoRA healing pilot of C = B + block-13 attention bridge, against the final velocity.

    Verifies the teacher contract and exact B replay, computes live teacher targets, loads the saved text-rich
    recordings, then trains LoRA adapters on every original block together with both bridges.
    """
    import hashlib
    import importlib.metadata
    import inspect
    import json
    import os
    import socket
    import uuid
    from pathlib import Path
    import torch
    from torch.nn.attention import sdpa_kernel, SDPBackend
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache

    directory = Path('/phase3/phase7/healing_pilot') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete healing attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()
    report = {'request_key': key, 'status': 'failed', 'mode': request['mode'], 'teacher_calls': 0,
              'student_calls': 0, 'optimizer_updates': 0, 'controls': [], 'artifacts': [],
              'process': {'pid': os.getpid(), 'host': socket.gethostname(), 'nonce': uuid.uuid4().hex,
                          'modal_task_id': os.environ.get('MODAL_TASK_ID')}}

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def gpu(value):
        if isinstance(value, torch.Tensor):
            return value.to('cuda')
        if isinstance(value, dict):
            return {k: gpu(v) for k, v in value.items()}
        if isinstance(value, (list, tuple)):
            return type(value)(gpu(v) for v in value)
        return value

    def compare(x, y):
        a, b = x.float(), y.float()
        if not torch.isfinite(a).all() or not torch.isfinite(b).all():
            raise RuntimeError('Nonfinite reference tensor.')
        d = a - b
        return {'relative_l2_error': float(d.norm()/b.norm().clamp_min(1e-12)),
                'max_abs_error': float(d.abs().max()), 'exact_equal': bool(torch.equal(x, y))}

    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('GPU differs.')
        report['packages'] = {p: importlib.metadata.version(p) for p in ('torch', 'diffusers', 'transformers', 'accelerate', 'pillow')}
        report['torch_cuda'] = torch.version.cuda
        report['cudnn_version'] = torch.backends.cudnn.version()
        report['settings'] = request['arithmetic_contract']
        if request['mode'] != 'verify':
            raise RuntimeError('Healing requires the verified teacher reference.')
        reference_dir = Path('/phase3/phase6/deterministic_reference') / request['reference_key']
        reference_report_path = reference_dir / 'summary.json'
        if hashlib.sha256(json.dumps(json.loads(reference_report_path.read_text()), sort_keys=True).encode()).hexdigest() != request['reference_report_semantic_sha256']:
            raise RuntimeError('Generation report differs.')
        reference_report = json.loads(reference_report_path.read_text())
        if reference_report['status'] != 'reference_generated':
            raise RuntimeError('Reference generation did not complete.')
        if reference_report['settings'] != request['arithmetic_contract']:
            raise RuntimeError('Arithmetic contract differs from generation.')
        report['reference_process'] = reference_report['process']
        if reference_report['process']['nonce'] == report['process']['nonce']:
            raise RuntimeError('Independent process requirement failed.')
        if reference_report['packages'] != report['packages'] or reference_report['torch_cuda'] != report['torch_cuda'] or reference_report['cudnn_version'] != report['cudnn_version']:
            raise RuntimeError('Numerical runtime differs across generation and verification.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
            dtype=torch.bfloat16, local_files_only=True).to('cuda')
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Model source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--'))
                       / 'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        state = {}
        for kind, values in (('parameter', model.named_parameters()), ('buffer', model.named_buffers())):
            for name, value in values:
                cpu = value.detach().cpu().contiguous()
                state[name] = {'kind': kind, 'dtype': str(value.dtype), 'shape': list(value.shape),
                    'stride': list(value.stride()), 'sha256': hashlib.sha256(memoryview(cpu.view(torch.uint8).numpy())).hexdigest()}
        for axis, value in enumerate(model.pos_embed.freqs):
            name = f'pos_embed.freqs.{axis}'
            cpu = value.detach().cpu().contiguous()
            state[name] = {'kind': 'unregistered_rope_table', 'dtype': str(value.dtype), 'shape': list(value.shape),
                'stride': list(value.stride()), 'sha256': hashlib.sha256(memoryview(cpu.view(torch.uint8).numpy())).hexdigest()}
        expected = request['expected_state']
        derived_names = sorted(n for n in expected if n.startswith('pos_embed.freqs.') or n == 'time_text_embed.time_proj.freqs')

        def derived_tensors():
            values = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
            values['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
            return values

        def table_hash(value):
            return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

        canonical_path = Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt')
        import time
        mismatched = [n for n in derived_names if state.get(n) != expected.get(n)]
        if not mismatched:
            if not canonical_path.exists():
                canonical_path.parent.mkdir(parents=True, exist_ok=True)
                temporary = canonical_path.with_suffix('.' + report['process']['nonce'] + '.tmp')
                torch.save({n: v.detach().cpu().contiguous() for n, v in derived_tensors().items()}, temporary)
                temporary.replace(canonical_path)
                results_volume.commit()
                report['canonical_tables_saved'] = True
        else:
            deadline = time.time() + 900
            while not canonical_path.exists():
                if time.time() > deadline:
                    raise RuntimeError('Derived tables differ on this host and no canonical copy became available.')
                time.sleep(20)
                results_volume.reload()
            saved = torch.load(canonical_path, weights_only=True, map_location='cpu')
            if sorted(saved) != derived_names or any(table_hash(saved[n]) != expected[n]['sha256'] for n in derived_names):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [saved[f'pos_embed.freqs.{axis}'].to(device=value.device, dtype=value.dtype)
                        for axis, value in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(saved['time_text_embed.time_proj.freqs'])
            for n, value in derived_tensors().items():
                state[n] = {**state[n], 'sha256': table_hash(value)}
            report['canonical_tables_applied'] = mismatched
        if state != request['expected_state']:
            expected = request['expected_state']
            different_names = sorted(name for name in expected.keys() & state.keys()
                                     if state[name] != expected[name])
            report['state_mismatch'] = {
                'actual_count': len(state), 'expected_count': len(expected),
                'missing_names': sorted(set(expected) - set(state))[:12],
                'extra_names': sorted(set(state) - set(expected))[:12],
                'different': [{'name': name, 'actual': state[name], 'expected': expected[name]}
                              for name in different_names][:12]}
            allowed_derived_tables = {'pos_embed.freqs.1', 'pos_embed.freqs.2', 'time_text_embed.time_proj.freqs'}
            if set(state) != set(expected) or not set(different_names).issubset(allowed_derived_tables):
                raise RuntimeError('Loaded teacher learned state differs from audited reference.')
            report['derived_table_variation_requires_output_replay'] = different_names
        report['state_sha256'] = hashlib.sha256(json.dumps(state, sort_keys=True).encode()).hexdigest()
        torch.cuda.synchronize()
        argument_order = ('hidden_states', 'timestep', 'encoder_hidden_states', 'encoder_hidden_states_mask',
                          'img_shapes', 'img_mask', 'attention_kwargs', 'kv_cache_mode', 'return_dict')
        # Every intact-teacher control must pass before any block is touched.
        for record in request['records']:
            path = Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt'
            if digest(path) != record['tensor_sha256']:
                raise RuntimeError('Legacy input bundle differs.')
            bundle = torch.load(path, weights_only=True, map_location='cpu')
            outputs, controls = [], []
            cache = QwenImage21KVCache(32)
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for stage in bundle['stages']:
                    full = {**bundle['shared_transformer_kwargs'], **stage['kwargs']}
                    kwargs = gpu({field: full[field] for field in argument_order})
                    kwargs['kv_cache'] = cache
                    pred = model(**kwargs)[0][:, -stage['target_image_tokens']:].detach().cpu().contiguous()
                    report['teacher_calls'] += 1
                    legacy = compare(pred, stage['teacher_velocity'])
                    entry = {'source_id': record['source_id'], 'split': bundle['row']['split'],
                             'stage': stage['stage'], 'legacy_difference': legacy}
                    outputs.append({'stage': stage['stage'], 'step_index': stage['step_index'],
                                    'scheduler_sigma': stage['scheduler_sigma'], 'velocity': pred})
                    controls.append(entry)
            filename = record['capture_id'] + '.pt'
            artifact = next(a for a in reference_report['artifacts'] if a['name'] == filename)
            if digest(reference_dir / filename) != artifact['sha256']:
                raise RuntimeError('Reference output checksum differs.')
            reference_bundle = torch.load(reference_dir / filename, weights_only=True, map_location='cpu')
            if reference_bundle['source_id'] != record['source_id'] or reference_bundle['input_sha256'] != record['tensor_sha256']:
                raise RuntimeError('Reference input provenance differs.')
            reference_outputs = {v['stage']: v['velocity'] for v in reference_bundle['outputs']}
            for entry, measured in zip(controls, outputs):
                entry['reference_difference'] = compare(measured['velocity'], reference_outputs[entry['stage']])
                entry['passed'] = entry['reference_difference']['relative_l2_error'] <= 0.001
            output_path = directory / filename
            temporary = output_path.with_suffix('.tmp')
            torch.save({'request_key': key, 'source_id': record['source_id'], 'input_sha256': record['tensor_sha256'],
                        'outputs': outputs, 'controls': controls}, temporary)
            temporary.replace(output_path)
            report['controls'].extend(controls)
            report['artifacts'].append({'name': filename, 'sha256': digest(output_path), 'bytes': output_path.stat().st_size})
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            if not all(c['passed'] for c in controls):
                raise RuntimeError('Independent-process reference replay exceeds tolerance.')
        if report['teacher_calls'] != 90:
            raise RuntimeError('Reference attempt did not cover all 90 saved stage inputs.')
        report['legacy_max_relative_l2'] = max(c['legacy_difference']['relative_l2_error'] for c in report['controls'])
        report['max_reference_relative_l2'] = max(c['reference_difference']['relative_l2_error'] for c in report['controls'])
        report['exact_equal_controls'] = sum(c['reference_difference']['exact_equal'] for c in report['controls'])
        if not all(c['passed'] for c in report['controls']):
            raise RuntimeError('Teacher controls did not pass.')
        report['status'] = 'teacher_verified'
        (directory / 'teacher_control_report.json').write_text(json.dumps(report, indent=2))
        results_volume.commit()
        print('All 90 intact-teacher controls passed; assembling frozen 31-block candidate B.', flush=True)

        import gc
        import time
        import statistics
        from types import SimpleNamespace
        
        # Inputs are independently identified; 10 gate and 50 wider development prompts.
        panels = {'gate10': [], 'wider50': []}
        for panel, records in (('gate10', [r for r in request['records'] if r['capture_id'].startswith('p5-validation-')]),
                               ('wider50', request['expanded_validation']['records'])):
            for record in records:
                path = Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt'
                if digest(path) != record['tensor_sha256']:
                    raise RuntimeError('Fixed input hash differs')
                item = torch.load(path, weights_only=True, map_location='cpu')
                if item['row']['source_id'] != record['source_id'] or item['row']['split'] != 'validation':
                    raise RuntimeError('Validation input identity differs')
                panels[panel].append(item)
        if [len(panels[p]) for p in ('gate10', 'wider50')] != [10, 50]:
            raise RuntimeError('Saved validation panel sizes differ')

        def forward(item, stage, cache):
            full = {**item['shared_transformer_kwargs'], **stage['kwargs']}
            kwargs = gpu({field: full[field] for field in argument_order})
            kwargs['kv_cache'] = cache
            return model(**kwargs)[0][:, -stage['target_image_tokens']:].detach().cpu().contiguous()

        def predictions():
            rows = []
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for panel, items in panels.items():
                    for item in items:
                        cache = QwenImage21KVCache(32)
                        for stage in item['stages']:
                            pred = forward(item, stage, cache)
                            if not torch.isfinite(pred).all() or pred.float().norm() == 0:
                                raise RuntimeError('Invalid reference velocity')
                            rows.append(dict(panel=panel, source_id=item['row']['source_id'],
                                             dimension=item['row']['dimension'],
                                             subdimension=item['row'].get('subdimension'),
                                             stage=stage['stage'], velocity=pred))
            return rows

        def save(name, value):
            path = directory / name
            temporary = path.with_suffix('.tmp')
            torch.save(value, temporary)
            temporary.replace(path)
            artifact = dict(name=name, sha256=digest(path), bytes=path.stat().st_size)
            report['artifacts'].append(artifact)
            return artifact

        teacher_predictions = predictions()
        save('teacher_predictions.pt', teacher_predictions)
        report['teacher_calls'] += len(teacher_predictions)
        # ---- Live teacher targets for TRAIN inputs (stored legacy targets differ by up to ~2.6%). ----
        def load_verified(path, sha256, size=None):
            if (size is not None and path.stat().st_size != size) or digest(path) != sha256:
                raise RuntimeError('Input tensor differs: ' + str(path))
            return torch.load(path, weights_only=True, map_location='cpu')

        train_full, early_pool = [], []
        for record in request['records']:
            if not record['capture_id'].startswith('p5-train-'):
                continue
            item = load_verified(Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt', record['tensor_sha256'])
            if item['row']['split'] != 'train' or item['row']['source_id'] != record['source_id']:
                raise RuntimeError('Training record identity differs.')
            train_full.append(item)
        fresh = request['fresh_aligned_train']
        for artifact in fresh['artifacts']:
            item = load_verified(Path(fresh['remote_directory']) / artifact['name'], artifact['sha256'], artifact['bytes'])
            if item['source_id'] != artifact['source_id'] or item['row']['split'] != 'train' or len(item['stages']) != 3:
                raise RuntimeError('Fresh aligned training provenance differs.')
            train_full.append(item)
        for example in request['early_pool']['examples']:
            item = load_verified(Path(example['remote_path']), example['sha256'], example['bytes'])
            if (item['capture_id'] != example['capture_id'] or item['row']['split'] != 'train'
                    or [s['stage'] for s in item['stages']] != ['early']):
                raise RuntimeError('Early pool example provenance differs.')
            early_pool.append(item)
        validation_ids = {r['source_id'] for r in request['records'] if r['capture_id'].startswith('p5-validation-')}
        validation_ids |= {r['source_id'] for r in request['expanded_validation']['records']}
        train_ids = {i['row']['source_id'] for i in train_full} | {i['row']['source_id'] for i in early_pool}
        if len(train_full) != 60 or train_ids & validation_ids:
            raise RuntimeError('Training set size or split differs.')
        for item in train_full + early_pool:
            cache = QwenImage21KVCache(32)
            item['live_targets'] = []
            for stage in item['stages']:
                with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                    item['live_targets'].append(forward(item, stage, cache).to(torch.bfloat16))
                report['teacher_calls'] += 1
        report['train_counts'] = {'full_prompts': len(train_full), 'early_pool': len(early_pool)}
        print('Live teacher targets for', report['train_counts'], flush=True)

        # Rebuild the Phase 6 selected student exactly as its verified replay did:
        # 31 teacher blocks plus the frozen pretrained rank-256 bridge at slot 5.
        namespace = {}
        exec(compile(request['core_source'], 'bridge_definition.py', 'exec'), namespace)
        exec(compile(request['wrapper_source'], 'bridge_block_definition.py', 'exec'), namespace)
        candidate = request['selected_candidate']
        path = Path('/phase3/phase5/fitting') / request['fit_key'] / Path(candidate['checkpoint_path']).name
        if digest(path) != candidate['checkpoint_sha256']:
            raise RuntimeError('Selected bridge checkpoint differs.')
        checkpoint = torch.load(path, weights_only=True, map_location='cpu')
        if (checkpoint['request_key'] != request['fit_key'] or checkpoint['layer'] != 5
                or checkpoint['step'] != candidate['selected_update']):
            raise RuntimeError('Bridge provenance differs.')
        bridge = namespace['ResidualBottleneckBridge'](4096, 256)
        bridge.load_state_dict(checkpoint['state_dict'])
        bridge.requires_grad_(False)
        namespace['insert_bridge'](model, 5, bridge, QwenImage21KVCache)
        del checkpoint
        model.requires_grad_(False).eval()
        gc.collect()
        torch.cuda.empty_cache()
        baseline = predictions()
        report['student_calls'] += len(baseline)
        save('candidate_predictions.pt', baseline)
        # The 10 gate prompts must reproduce the saved Phase 6 trained-bridge velocities.
        replay = request['selected_replay']
        replay_dir = Path('/phase3/phase6/verified_student_replay') / replay['request_key']
        reference = {}
        for record in request['records']:
            if not record['capture_id'].startswith('p5-validation-'):
                continue
            name = 'students-' + record['capture_id'] + '.pt'
            if digest(replay_dir / name) != replay['artifacts'][name]:
                raise RuntimeError('Phase 6 student replay checksum differs.')
            saved = torch.load(replay_dir / name, weights_only=True, map_location='cpu')
            for output in saved['outputs']:
                if output['candidate_id'] == candidate['candidate_id'] and output['variant'] == 'trained':
                    reference[(output['source_id'], output['stage'])] = output['velocity']
        gate_rows = [r for r in baseline if r['panel'] == 'gate10']
        if len(reference) != 30 or len(gate_rows) != 30:
            raise RuntimeError('Phase 6 replay coverage differs.')
        reproduction = [compare(r['velocity'], reference[(r['source_id'], r['stage'])]) for r in gate_rows]
        report['candidate_replay_max_relative_l2'] = max(r['relative_l2_error'] for r in reproduction)
        if report['candidate_replay_max_relative_l2'] > 0.001:
            raise RuntimeError('Selected student failed Phase 6 output replay')
        del reference

        class IdentityBlock(torch.nn.Module):
            def __init__(self, original):
                super().__init__()
                self.attn = SimpleNamespace(processor=original.attn.processor)

            def forward(self, hidden_states, *args, **kwargs):
                return hidden_states

        def metrics(rows):
            output = []
            for measured, base, teacher_row in zip(rows, baseline, teacher_predictions):
                identity = ('panel', 'source_id', 'stage')
                if any(measured[k] != base[k] or base[k] != teacher_row[k] for k in identity):
                    raise RuntimeError('Prediction alignment differs')
                teacher_error = compare(measured['velocity'], teacher_row['velocity'])['relative_l2_error']
                baseline_error = compare(base['velocity'], teacher_row['velocity'])['relative_l2_error']
                output.append({k: measured[k] for k in measured if k != 'velocity'} |
                              dict(baseline_displacement=compare(measured['velocity'], base['velocity'])['relative_l2_error'],
                                   teacher_relative_l2=teacher_error, baseline_teacher_relative_l2=baseline_error,
                                   teacher_error_change_percentage_points=100 * (teacher_error - baseline_error)))
            return output

        def aggregate(rows):
            fields = ('baseline_displacement', 'teacher_relative_l2', 'baseline_teacher_relative_l2',
                      'teacher_error_change_percentage_points')
            result = dict(comparisons=len(rows))
            for field in fields:
                values = sorted(r[field] for r in rows)
                def percentile(fraction):
                    position = fraction * (len(values) - 1)
                    low = int(position)
                    return values[low] + (values[min(low + 1, len(values) - 1)] - values[low]) * (position - low)
                result[field] = dict(mean=statistics.mean(values), maximum=max(values),
                                     p10=percentile(.1), p50=percentile(.5), p90=percentile(.9))
            result['worst_regressions'] = sorted(rows, key=lambda r:r['teacher_error_change_percentage_points'], reverse=True)[:10]
            return result

        report['baseline'] = aggregate(metrics(baseline))

        # ---- LoRA healing pilot: C = B + block-13 attention bridge; LoRA on every original block, both bridges trainable.
        import random
        from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVLayerCache
        fit = request['fit']
        lora = request['lora']
        conditioned = {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned)
        components = {}
        exec(compile(request['components_source'], 'healing_components.py', 'exec'), components)

        # Saved recordings from the 8-head bridge run (steps 0 and 19 with teacher targets).
        recordings = request['recordings']
        new_text, new_general = [], []
        for row in recordings['rows']:
            item = torch.load(Path(recordings['remote_directory']) / (row['source_id'] + '.pt'), weights_only=True, map_location='cpu')
            if (item['row']['source_id'] != row['source_id'] or [s['stage'] for s in item['stages']] != ['early', 'middle']
                    or len(item['live_targets']) != 2):
                raise RuntimeError('Saved recording differs: ' + row['source_id'])
            (new_text if row['subdimension'] == 'text_rendering' else new_general).append(item)
        if {i['row']['source_id'] for i in new_text + new_general} & (train_ids | validation_ids):
            raise RuntimeError('Recordings overlap earlier training or validation prompts.')
        report['recording_counts'] = {'text': len(new_text), 'general': len(new_general)}
        general_pool = train_full + new_general

        # Block 5: FP32 master copy of the same pretrained bridge (identical BF16 arithmetic, trainable).
        path5 = Path('/phase3/phase5/fitting') / request['fit_key'] / Path(candidate['checkpoint_path']).name
        if digest(path5) != candidate['checkpoint_sha256']:
            raise RuntimeError('Selected bridge checkpoint differs.')
        state5 = torch.load(path5, weights_only=True, map_location='cpu')['state_dict']
        bridge5 = components['MasterBridge'](4096, 256)
        bridge5.load_state_dict(state5)
        namespace['insert_bridge'](model, 5, bridge5, QwenImage21KVCache)
        bridge5.float()
        bridge5.load_state_dict(state5)
        # Block 13: the 8-head attention bridge, FP32.
        warm_spec = request['warm_start']
        if digest(Path(warm_spec['remote_path'])) != warm_spec['sha256']:
            raise RuntimeError('Block-13 bridge checkpoint differs.')
        warm = torch.load(Path(warm_spec['remote_path']), weights_only=True, map_location='cpu')
        spec = warm['variant']
        if warm['layer'] != 13 or spec['name'] != warm_spec['name'] or not spec.get('attention'):
            raise RuntimeError('Block-13 bridge provenance differs.')
        bridge13 = conditioned['AttentionConditionedBridge'](4096, 1024, spec['heads'], 128)
        bridge13.load_state_dict(warm['state_dict'])
        conditioned['insert_conditioned_bridge'](model, 13, bridge13.float(), keep_dtype=True)
        bridge13 = model.transformer_blocks[13].bridge
        # LoRA on q, k, v and out of every original block, then functional checkpointing for memory.
        torch.manual_seed(fit['init_seed'])
        for slot, block in enumerate(list(model.transformer_blocks)):
            if slot in (5, 13):
                continue
            for name in ('to_q', 'to_k', 'to_v'):
                setattr(block.attn, name, components['MasterLoRALinear'](getattr(block.attn, name), lora['rank'], lora['alpha']))
            block.attn.to_out[0] = components['MasterLoRALinear'](block.attn.to_out[0], lora['rank'], lora['alpha'])
            model.transformer_blocks[slot] = components['FunctionalCheckpointBlock'](block, QwenImage21KVLayerCache)
        model.requires_grad_(False)
        model.gradient_checkpointing = False
        lora_parameters = {n: p for n, p in model.named_parameters() if '.attn.' in n and ('.down.' in n or '.up.' in n)}
        bridge_parameters = {n: p for n, p in model.named_parameters() if n.startswith(('transformer_blocks.5.bridge.', 'transformer_blocks.13.bridge.'))}
        if len(lora_parameters) != 30 * 4 * 2 or not bridge_parameters:
            raise RuntimeError('Adapter structure differs.')
        trainable = {**lora_parameters, **bridge_parameters}
        for parameter in trainable.values():
            parameter.requires_grad_(True)
        report['trainable_parameters'] = {'lora': sum(p.numel() for p in lora_parameters.values()),
                                          'bridges': sum(p.numel() for p in bridge_parameters.values())}
        model.eval()
        gate_teacher = [r for r in teacher_predictions if r['panel'] == 'gate10']

        def gate_error():
            errors = []
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                position = 0
                for item in panels['gate10']:
                    cache = QwenImage21KVCache(32)
                    for stage in item['stages']:
                        reference = gate_teacher[position]
                        if reference['source_id'] != item['row']['source_id'] or reference['stage'] != stage['stage']:
                            raise RuntimeError('Gate alignment differs.')
                        errors.append(compare(forward(item, stage, cache), reference['velocity'])['relative_l2_error'])
                        position += 1
            return sum(errors) / len(errors)

        def train_loss(item):
            cache = QwenImage21KVCache(32)
            losses = []
            for stage, target in zip(item['stages'], item['live_targets']):
                full = {**item['shared_transformer_kwargs'], **stage['kwargs']}
                kwargs = gpu({field: full[field] for field in argument_order})
                kwargs['kv_cache'] = cache
                prediction = model(**kwargs)[0][:, -stage['target_image_tokens']:].float()
                target = target.to('cuda').float()
                losses.append((prediction - target).square().mean() / target.square().mean().clamp_min(1e-12))
                report['student_calls'] += 1
            loss = torch.stack(losses).mean()
            if not torch.isfinite(loss):
                raise RuntimeError('Nonfinite training loss.')
            return loss

        optimizer = torch.optim.AdamW([{'params': list(lora_parameters.values()), 'lr': lora['learning_rate'], 'base': lora['learning_rate']},
                                       {'params': list(bridge_parameters.values()), 'lr': fit['bridge_learning_rate'], 'base': fit['bridge_learning_rate']}],
                                      weight_decay=0.0)
        rng = random.Random(fit['sampling_seed'])
        curve, best = [], None
        started = time.perf_counter()
        for step in range(fit['updates'] + 1):
            if step % fit['eval_every'] == 0:
                score = gate_error()
                curve.append(dict(step=step, gate10=score, seconds=time.perf_counter() - started))
                if best is None or score < best[0]:
                    best = (score, step, {n: p.detach().cpu().clone() for n, p in trainable.items()})
                print(f'healing step {step}: gate10 {score:.6f} ({time.perf_counter() - started:.0f}s)', flush=True)
                (directory / 'progress.json').write_text(json.dumps(report | {'current_curve': curve}, indent=2))
                results_volume.commit()
            if step == fit['updates']:
                break
            for group in optimizer.param_groups:
                group['lr'] = group['base'] * min(1.0, (step + 1) / fit['warmup'])
            optimizer.zero_grad(set_to_none=True)
            # Backward recomputes checkpointed blocks: keep the forward attention kernel and cache context.
            with model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                pool = new_text if rng.random() < fit['text_fraction'] else general_pool
                (train_loss(rng.choice(pool)) * (1 - fit['early_weight'])).backward()
                (train_loss(rng.choice(early_pool)) * fit['early_weight']).backward()
            torch.nn.utils.clip_grad_norm_(list(trainable.values()), fit['gradient_clip'])
            optimizer.step()
            report['optimizer_updates'] += 1
        with torch.no_grad():
            for name, parameter in trainable.items():
                parameter.copy_(best[2][name].to(device=parameter.device, dtype=parameter.dtype))
        for parameter in trainable.values():
            parameter.requires_grad_(False)
        checkpoint_path = directory / 'healing_selected.pt'
        temporary = checkpoint_path.with_suffix('.tmp')
        torch.save({'request_key': key, 'step': best[1], 'lora': lora, 'block13_variant': spec, 'state_dict': best[2]}, temporary)
        temporary.replace(checkpoint_path)
        report['artifacts'].append(dict(name=checkpoint_path.name, sha256=digest(checkpoint_path), bytes=checkpoint_path.stat().st_size))
        rows = predictions()
        report['student_calls'] += len(rows)
        records = metrics(rows)
        report['variants'] = {'healed_c': dict(
            variant={'name': 'healed_c', 'lora': lora, 'block13': spec['name']}, checkpoint=report['artifacts'][-1],
            parameters=sum(v.numel() for v in best[2].values()),
            fit=dict(curve=curve, selected_step=best[1], selected_gate10=best[0]),
            overall=aggregate(records),
            panels={p: aggregate([r for r in records if r['panel'] == p]) for p in panels},
            stages={s: aggregate([r for r in records if r['stage'] == s]) for s in ('early', 'middle', 'late')},
            records=records)}
        report['status'] = 'passed'
        report['scope'] = 'LoRA healing pilot of C (blocks 5 and 13 bridged): LoRA r16 on all 30 original blocks, both bridges trainable; fixed-input velocity evaluation.'
    except Exception as exc:
        import traceback
        report['status'] = 'failed'
        report['error'] = repr(exc)
        report['traceback'] = traceback.format_exc()
    (directory / 'summary.json').write_text(json.dumps(report, indent=2))
    results_volume.commit()
    return report
'''

def main():
    import argparse
    import hashlib
    parser = argparse.ArgumentParser()
    parser.add_argument('--dry-run', action='store_true')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    load = lambda path: json.loads((root / path).read_text(encoding='utf-8'))
    protocol_path = root / 'data/phase7/healing_pilot/protocol.json'
    scan_b = load('data/phase7/conditional_importance_b/b0a27f0cbe0f8e7f9938044fe088576a648c7b22e63a2ddbd771ee6b8e777aae/request.json')['request']
    scan_a = load('data/phase7/conditional_importance/7454da6a5c1dc33af23dd46833e698f9b759d14985ca96340bc3d40cd47fb4f5/request.json')['request']
    text8 = load('data/phase7/block13_bridge_e2e_text8/ff9d259ae589bdf15b98076930c4d16340ef41f4a2c3db897ea3167e95397054/request.json')
    text8_summary = load('data/phase7/block13_bridge_e2e_text8/ff9d259ae589bdf15b98076930c4d16340ef41f4a2c3db897ea3167e95397054/summary.json')
    bridge = text8_summary['variants']['e2e_attn8_text']['checkpoint']
    request = {k: v for k, v in scan_b.items() if k not in ('worker_source', 'eligible_original_block_ids', 'selection', 'scope', 'version')}
    request.update(
        version='phase7-healing-pilot-v1', worker_source=SOURCES['healing_pilot_worker.py'],
        protocol_sha256=hashlib.sha256(protocol_path.read_bytes()).hexdigest(),
        conditioned_bridge_source=text8['request']['conditioned_bridge_source'], components_source=scan_a['components_source'],
        fresh_aligned_train=scan_a['fresh_aligned_train'], early_pool=scan_a['early_pool'],
        warm_start=dict(name='e2e_attn8_text', remote_path=f"/phase3/phase7/bridge_end_to_end_v3/{text8['request_key']}/{bridge['name']}",
                        sha256=bridge['sha256']),
        recordings=dict(remote_directory=f"/phase3/phase7/bridge_end_to_end_v3/{text8['request_key']}/recorded",
                        rows=[{k: r[k] for k in ('source_id', 'subdimension')} for r in text8['request']['new_prompts']]),
        lora=dict(rank=16, alpha=16, learning_rate=1e-4, targets=['to_q', 'to_k', 'to_v', 'to_out.0']),
        fit=dict(init_seed=20261018, sampling_seed=20261024, bridge_learning_rate=2e-5, warmup=20, gradient_clip=1.0,
                 updates=400, eval_every=100, early_weight=0.3, text_fraction=0.4),
        selection='Lowest gate10 velocity error at evaluations every 100 updates, including the unhealed start',
        scope='LoRA healing pilot of C (B + block-13 attention bridge): LoRA on all 30 original blocks, both bridges trainable')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase7/healing_pilot' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2) + '\n', encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return request_key
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase7/healing_pilot/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        try:
            b''.join(results_volume.read_file(remote + '/started.json'))
        except FileNotFoundError:
            pass
        else:
            raise RuntimeError('Run already started; inspect its progress. No duplicate GPU scheduled.')
        app = modal.App('nihonga-phase7-healing-pilot')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'healing_pilot_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8, memory=131072,
                              max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['heal_student'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2) + '\n')
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'recording_counts', 'trainable_parameters')}), flush=True)
    return request_key

LAUNCH_ARGS = ['--dry-run']
RUN = RUN_GLOB + '/' + main()   # writes the request and returns its key; no GPU
LAUNCH_ARGS = []
summary = reuse_or_launch(RUN, main)
if summary and summary.get('status') == 'passed':
    show_variants(summary)

## 14b. Images of the healed student

The 4-seed text grid and the 30 standard prompts for the healed C (teacher and B reused). Spelling decides.

In [ ]:
# Healed student images: reuses the saved run; launches GPU work only when its result is missing and ALLOW_GPU is True.
SOURCES['healed_images_worker.py'] = r'''def generate_multiseed_text(request, key):
    """Extra seeds for the text prompts: teacher, B, and C with the best block-13 bridge.
    Same prompt, CPU noise seed, size, scheduler and 40 steps for every model. Qualitative screen only."""
    import gc
    import hashlib
    import inspect
    import json
    import os
    import time
    from pathlib import Path

    import torch
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache, QwenImage21KVLayerCache

    directory = Path('/phase3/phase7/healed_images') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete image attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def table_hash(value):
        return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

    variants = ['teacher', 'B', request['c_variant']['name']]
    report = {'request_key': key, 'status': 'running', 'rows': [], 'artifacts': [],
              'calls': {v: 0 for v in variants}, 'optimizer_updates': 0}
    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('Expected H100.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
                                                   dtype=torch.bfloat16, local_files_only=True).to('cuda')
        pipe.set_progress_bar_config(disable=True)
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Transformer source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--')) /
                       'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        # Derived position/time tables: use the audited values on every host (as in steps 7-11).
        derived = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
        derived['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
        expected = request['expected_derived_tables']
        mismatched = sorted(n for n, v in derived.items() if table_hash(v) != expected[n])
        if mismatched:
            canonical = torch.load(Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt'),
                                   weights_only=True, map_location='cpu')
            if any(table_hash(canonical[n]) != expected[n] for n in expected):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [canonical[f'pos_embed.freqs.{a}'].to(device=v.device, dtype=v.dtype) for a, v in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(canonical['time_text_embed.time_proj.freqs'])
            if any(table_hash(v) != expected[n] for n, v in
                   list({f'pos_embed.freqs.{a}': v for a, v in enumerate(model.pos_embed.freqs)}.items()) +
                   [('time_text_embed.time_proj.freqs', model.time_text_embed.time_proj.freqs)]):
                raise RuntimeError('Derived tables still differ after loading the audited copy.')
            report['canonical_tables_applied'] = mismatched

        def generate(row, variant):
            path = directory / (row['source_id'] + '-' + variant + '.png')
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'],
                             num_inference_steps=40, true_cfg_scale=teacher['true_cfg_scale'],
                             generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=True, output_type='pil').images[0]
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            temporary = path.with_suffix('.tmp')
            image.save(temporary, format='PNG')
            temporary.replace(path)
            results_volume.commit()
            report['calls'][variant] += 1
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            return {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size, 'seconds': elapsed}

        namespace = {}
        exec(compile(request['core_source'], 'bridge_definition.py', 'exec'), namespace)
        exec(compile(request['wrapper_source'], 'bridge_block_definition.py', 'exec'), namespace)
        conditioned = {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned)

        def load_checkpoint(spec):
            path = Path(spec['remote_path'])
            if digest(path) != spec['sha256']:
                raise RuntimeError('Bridge checkpoint differs: ' + spec['remote_path'])
            return torch.load(path, weights_only=True, map_location='cpu')

        outputs = {v: {} for v in variants}
        references = not request.get('candidate_only', False)
        if references:
            for row in request['rows']:
                outputs['teacher'][row['source_id']] = generate(row, 'teacher')
        block5 = request['block5_bridge']
        checkpoint = load_checkpoint(block5)
        if checkpoint['layer'] != 5 or checkpoint['step'] != block5['step']:
            raise RuntimeError('Block-5 bridge provenance differs.')
        bridge = namespace['ResidualBottleneckBridge'](4096, 256)
        bridge.load_state_dict(checkpoint['state_dict'])
        bridge.requires_grad_(False)
        namespace['insert_bridge'](model, 5, bridge, QwenImage21KVCache)
        model.eval()
        if references:
            for row in request['rows']:
                outputs['B'][row['source_id']] = generate(row, 'B')
        spec = request['c_variant']
        healed = load_checkpoint(spec)
        state = healed['state_dict']
        components = {}
        exec(compile(request['components_source'], 'healing_components.py', 'exec'), components)
        # Block 5: the healed bridge weights in the BF16 inference bridge (same arithmetic as the FP32 master in training).
        prefix5 = 'transformer_blocks.5.bridge.'
        block5_bridge = model.transformer_blocks[5].bridge
        block5_bridge.load_state_dict({k[len(prefix5):]: v.to(torch.bfloat16) for k, v in state.items() if k.startswith(prefix5)})
        # Block 13: the healed attention bridge, FP32 as trained.
        variant13 = healed['block13_variant']
        prefix13 = 'transformer_blocks.13.bridge.'
        bridge13 = conditioned['AttentionConditionedBridge'](4096, 1024, variant13['heads'], 128)
        bridge13.load_state_dict({k[len(prefix13):]: v for k, v in state.items() if k.startswith(prefix13)})
        bridge13.requires_grad_(False)
        conditioned['insert_conditioned_bridge'](model, 13, bridge13.float(), keep_dtype=True)
        # LoRA adapters on every original block, with the healed weights.
        lora = healed['lora']
        loaded = 0
        for slot, block in enumerate(model.transformer_blocks):
            if slot in (5, 13):
                continue
            for name in ('to_q', 'to_k', 'to_v'):
                setattr(block.attn, name, components['MasterLoRALinear'](getattr(block.attn, name), lora['rank'], lora['alpha']))
            block.attn.to_out[0] = components['MasterLoRALinear'](block.attn.to_out[0], lora['rank'], lora['alpha'])
            for name, module in (('to_q', block.attn.to_q), ('to_k', block.attn.to_k), ('to_v', block.attn.to_v), ('to_out.0', block.attn.to_out[0])):
                for part in ('down', 'up'):
                    getattr(module, part).weight.data.copy_(state[f'transformer_blocks.{slot}.inner.attn.{name}.{part}.weight'])
                    loaded += 1
        if loaded != 30 * 4 * 2:
            raise RuntimeError('Healed adapter count differs.')
        model.requires_grad_(False)
        model.eval()
        for row in request['rows']:
            outputs[spec['name']][row['source_id']] = generate(row, spec['name'])
        rendered = [v for v in variants if outputs[v]]
        for row in request['rows']:
            report['rows'].append({**row, **{v: outputs[v][row['source_id']] for v in rendered}})
            report['artifacts'].extend(outputs[v][row['source_id']] for v in rendered)
        report['status'] = 'passed'
    except Exception as exc:
        report['status'] = 'failed'
        report['error'] = type(exc).__name__ + ': ' + str(exc)
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=True) + '\n')
    results_volume.commit()
    return report
'''

def main():
    import argparse
    import hashlib
    parser = argparse.ArgumentParser()
    parser.add_argument('--dry-run', action='store_true')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    load = lambda path: json.loads((root / path).read_text(encoding='utf-8'))
    base = next((root / 'data/recovered/modal/phase7/candidate_images').glob('*/request.json'))
    base = json.loads(base.read_bytes())['request']
    scan_b = load('data/phase7/conditional_importance_b/b0a27f0cbe0f8e7f9938044fe088576a648c7b22e63a2ddbd771ee6b8e777aae/request.json')['request']
    scan_a = load('data/phase7/conditional_importance/7454da6a5c1dc33af23dd46833e698f9b759d14985ca96340bc3d40cd47fb4f5/request.json')['request']
    heal_dir = next(p.parent for p in (root / 'data/phase7/healing_pilot').glob('*/summary.json')
                    if load(p.relative_to(root)).get('status') == 'passed')
    heal = load((heal_dir / 'summary.json').relative_to(root))
    heal_request = load((heal_dir / 'request.json').relative_to(root))
    selected = scan_b['selected_candidate']
    text_rows = [r for r in base['rows'] if r['subdimension'] in ('text_rendering', 'posters')]
    rows = [{**r, 'source_id': f"{r['source_id']}-s{k}", 'seed': (r['seed'] + 1000003 * k) % 2**32} for r in text_rows for k in (1, 2, 3)]
    rows += base['rows']
    checkpoint = heal['variants']['healed_c']['checkpoint']
    request = {k: base[k] for k in ('teacher', 'transformer_source_sha256', 'runtime_environment', 'expected_derived_tables',
                                    'canonical_tables_key', 'core_source', 'wrapper_source')}
    request.update(
        version='phase7-healed-images-v1', rows=rows, worker_source=SOURCES['healed_images_worker.py'],
        conditioned_bridge_source=heal_request['request']['conditioned_bridge_source'], components_source=scan_a['components_source'],
        execution_account='new', candidate_only=True,
        block5_bridge=dict(remote_path=f"/phase3/phase5/fitting/{scan_b['fit_key']}/{Path(selected['checkpoint_path']).name}",
                           sha256=selected['checkpoint_sha256'], step=selected['selected_update']),
        c_variant=dict(name='healed_c', remote_path=f"/phase3/phase7/healing_pilot/{heal['request_key']}/{checkpoint['name']}",
                       sha256=checkpoint['sha256']),
        scope='Healed student C: extra seeds for the 5 text prompts plus the 30 base prompts at seed 0; teacher and B reused')
    from experiment_runtime import pin_reference_runtime
    pin_reference_runtime(request, root / 'data/recovered/modal')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase7/multiseed_text' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2) + '\n', encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return request_key
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase7/healed_images/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        app = modal.App('nihonga-phase7-healed-images')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'healed_images_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8, memory=98304,
                              max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['generate_multiseed_text'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2) + '\n')
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'calls')}), flush=True)
    return request_key

LAUNCH_ARGS = ['--dry-run']
RUN = 'data/phase7/multiseed_text/' + main()   # writes the request and returns its key; no GPU
LAUNCH_ARGS = []
summary = reuse_or_launch(RUN, main)
if summary and summary.get('status') == 'passed' and (ROOT / RUN / 'sheets').exists():
    show_images(RUN + '/sheets', ['p1-2fa54a3c4eb8.jpg', 'p1-c9344c076abf.jpg'])

### What happened to the first healing run

The first run (LoRA learning rate 1e-4, one prompt per update) made velocity worse at every evaluation: 6.27% on the
check prompts at the start, 6.46% after 100 updates, 7.09% after 200. It also kept only the best-velocity
checkpoint, which would have been the unhealed start, so its intermediate checkpoints could never be checked for
spelling. It was stopped (`aborted.json` records the curve). Velocity can worsen while spelling improves, so every
run is now judged by **both** tests.

## 15. Healing, judged by velocity AND spelling

Same student, data and LoRA as step 14, gentler and less noisy training: LoRA learning rate 2e-5 (bridges 1e-5), two
prompts and two early examples accumulated per update, 200 updates. **Every checkpoint** (steps 0, 50, 100, 150,
200) is saved and gets two tests: the **quantitative** test (velocity on all 180 comparisons, run inside the job)
and the **qualitative** test (the 4-seed text grid, step 15b). Spelling decides; velocity is reported and breaks
ties (`data/phase7/healing_pilot_v2/protocol.json`).

In [ ]:
# Healing v2: reuses the saved run; launches GPU work only when its result is missing and ALLOW_GPU is True.
RUN_GLOB = 'data/phase7/healing_pilot_v2'
LAUNCH_ARGS = []

SOURCES['healing_pilot_v2_worker.py'] = r'''def heal_student(request, key):
    """LoRA healing pilot of C = B + block-13 attention bridge, against the final velocity.

    Verifies the teacher contract and exact B replay, computes live teacher targets, loads the saved text-rich
    recordings, then trains LoRA adapters on every original block together with both bridges.
    """
    import hashlib
    import importlib.metadata
    import inspect
    import json
    import os
    import socket
    import uuid
    from pathlib import Path
    import torch
    from torch.nn.attention import sdpa_kernel, SDPBackend
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache

    directory = Path('/phase3/phase7/healing_pilot_v2') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete healing attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()
    report = {'request_key': key, 'status': 'failed', 'mode': request['mode'], 'teacher_calls': 0,
              'student_calls': 0, 'optimizer_updates': 0, 'controls': [], 'artifacts': [],
              'process': {'pid': os.getpid(), 'host': socket.gethostname(), 'nonce': uuid.uuid4().hex,
                          'modal_task_id': os.environ.get('MODAL_TASK_ID')}}

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def gpu(value):
        if isinstance(value, torch.Tensor):
            return value.to('cuda')
        if isinstance(value, dict):
            return {k: gpu(v) for k, v in value.items()}
        if isinstance(value, (list, tuple)):
            return type(value)(gpu(v) for v in value)
        return value

    def compare(x, y):
        a, b = x.float(), y.float()
        if not torch.isfinite(a).all() or not torch.isfinite(b).all():
            raise RuntimeError('Nonfinite reference tensor.')
        d = a - b
        return {'relative_l2_error': float(d.norm()/b.norm().clamp_min(1e-12)),
                'max_abs_error': float(d.abs().max()), 'exact_equal': bool(torch.equal(x, y))}

    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('GPU differs.')
        report['packages'] = {p: importlib.metadata.version(p) for p in ('torch', 'diffusers', 'transformers', 'accelerate', 'pillow')}
        report['torch_cuda'] = torch.version.cuda
        report['cudnn_version'] = torch.backends.cudnn.version()
        report['settings'] = request['arithmetic_contract']
        if request['mode'] != 'verify':
            raise RuntimeError('Healing requires the verified teacher reference.')
        reference_dir = Path('/phase3/phase6/deterministic_reference') / request['reference_key']
        reference_report_path = reference_dir / 'summary.json'
        if hashlib.sha256(json.dumps(json.loads(reference_report_path.read_text()), sort_keys=True).encode()).hexdigest() != request['reference_report_semantic_sha256']:
            raise RuntimeError('Generation report differs.')
        reference_report = json.loads(reference_report_path.read_text())
        if reference_report['status'] != 'reference_generated':
            raise RuntimeError('Reference generation did not complete.')
        if reference_report['settings'] != request['arithmetic_contract']:
            raise RuntimeError('Arithmetic contract differs from generation.')
        report['reference_process'] = reference_report['process']
        if reference_report['process']['nonce'] == report['process']['nonce']:
            raise RuntimeError('Independent process requirement failed.')
        if reference_report['packages'] != report['packages'] or reference_report['torch_cuda'] != report['torch_cuda'] or reference_report['cudnn_version'] != report['cudnn_version']:
            raise RuntimeError('Numerical runtime differs across generation and verification.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
            dtype=torch.bfloat16, local_files_only=True).to('cuda')
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Model source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--'))
                       / 'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        state = {}
        for kind, values in (('parameter', model.named_parameters()), ('buffer', model.named_buffers())):
            for name, value in values:
                cpu = value.detach().cpu().contiguous()
                state[name] = {'kind': kind, 'dtype': str(value.dtype), 'shape': list(value.shape),
                    'stride': list(value.stride()), 'sha256': hashlib.sha256(memoryview(cpu.view(torch.uint8).numpy())).hexdigest()}
        for axis, value in enumerate(model.pos_embed.freqs):
            name = f'pos_embed.freqs.{axis}'
            cpu = value.detach().cpu().contiguous()
            state[name] = {'kind': 'unregistered_rope_table', 'dtype': str(value.dtype), 'shape': list(value.shape),
                'stride': list(value.stride()), 'sha256': hashlib.sha256(memoryview(cpu.view(torch.uint8).numpy())).hexdigest()}
        expected = request['expected_state']
        derived_names = sorted(n for n in expected if n.startswith('pos_embed.freqs.') or n == 'time_text_embed.time_proj.freqs')

        def derived_tensors():
            values = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
            values['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
            return values

        def table_hash(value):
            return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

        canonical_path = Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt')
        import time
        mismatched = [n for n in derived_names if state.get(n) != expected.get(n)]
        if not mismatched:
            if not canonical_path.exists():
                canonical_path.parent.mkdir(parents=True, exist_ok=True)
                temporary = canonical_path.with_suffix('.' + report['process']['nonce'] + '.tmp')
                torch.save({n: v.detach().cpu().contiguous() for n, v in derived_tensors().items()}, temporary)
                temporary.replace(canonical_path)
                results_volume.commit()
                report['canonical_tables_saved'] = True
        else:
            deadline = time.time() + 900
            while not canonical_path.exists():
                if time.time() > deadline:
                    raise RuntimeError('Derived tables differ on this host and no canonical copy became available.')
                time.sleep(20)
                results_volume.reload()
            saved = torch.load(canonical_path, weights_only=True, map_location='cpu')
            if sorted(saved) != derived_names or any(table_hash(saved[n]) != expected[n]['sha256'] for n in derived_names):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [saved[f'pos_embed.freqs.{axis}'].to(device=value.device, dtype=value.dtype)
                        for axis, value in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(saved['time_text_embed.time_proj.freqs'])
            for n, value in derived_tensors().items():
                state[n] = {**state[n], 'sha256': table_hash(value)}
            report['canonical_tables_applied'] = mismatched
        if state != request['expected_state']:
            expected = request['expected_state']
            different_names = sorted(name for name in expected.keys() & state.keys()
                                     if state[name] != expected[name])
            report['state_mismatch'] = {
                'actual_count': len(state), 'expected_count': len(expected),
                'missing_names': sorted(set(expected) - set(state))[:12],
                'extra_names': sorted(set(state) - set(expected))[:12],
                'different': [{'name': name, 'actual': state[name], 'expected': expected[name]}
                              for name in different_names][:12]}
            allowed_derived_tables = {'pos_embed.freqs.1', 'pos_embed.freqs.2', 'time_text_embed.time_proj.freqs'}
            if set(state) != set(expected) or not set(different_names).issubset(allowed_derived_tables):
                raise RuntimeError('Loaded teacher learned state differs from audited reference.')
            report['derived_table_variation_requires_output_replay'] = different_names
        report['state_sha256'] = hashlib.sha256(json.dumps(state, sort_keys=True).encode()).hexdigest()
        torch.cuda.synchronize()
        argument_order = ('hidden_states', 'timestep', 'encoder_hidden_states', 'encoder_hidden_states_mask',
                          'img_shapes', 'img_mask', 'attention_kwargs', 'kv_cache_mode', 'return_dict')
        # Every intact-teacher control must pass before any block is touched.
        for record in request['records']:
            path = Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt'
            if digest(path) != record['tensor_sha256']:
                raise RuntimeError('Legacy input bundle differs.')
            bundle = torch.load(path, weights_only=True, map_location='cpu')
            outputs, controls = [], []
            cache = QwenImage21KVCache(32)
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for stage in bundle['stages']:
                    full = {**bundle['shared_transformer_kwargs'], **stage['kwargs']}
                    kwargs = gpu({field: full[field] for field in argument_order})
                    kwargs['kv_cache'] = cache
                    pred = model(**kwargs)[0][:, -stage['target_image_tokens']:].detach().cpu().contiguous()
                    report['teacher_calls'] += 1
                    legacy = compare(pred, stage['teacher_velocity'])
                    entry = {'source_id': record['source_id'], 'split': bundle['row']['split'],
                             'stage': stage['stage'], 'legacy_difference': legacy}
                    outputs.append({'stage': stage['stage'], 'step_index': stage['step_index'],
                                    'scheduler_sigma': stage['scheduler_sigma'], 'velocity': pred})
                    controls.append(entry)
            filename = record['capture_id'] + '.pt'
            artifact = next(a for a in reference_report['artifacts'] if a['name'] == filename)
            if digest(reference_dir / filename) != artifact['sha256']:
                raise RuntimeError('Reference output checksum differs.')
            reference_bundle = torch.load(reference_dir / filename, weights_only=True, map_location='cpu')
            if reference_bundle['source_id'] != record['source_id'] or reference_bundle['input_sha256'] != record['tensor_sha256']:
                raise RuntimeError('Reference input provenance differs.')
            reference_outputs = {v['stage']: v['velocity'] for v in reference_bundle['outputs']}
            for entry, measured in zip(controls, outputs):
                entry['reference_difference'] = compare(measured['velocity'], reference_outputs[entry['stage']])
                entry['passed'] = entry['reference_difference']['relative_l2_error'] <= 0.001
            output_path = directory / filename
            temporary = output_path.with_suffix('.tmp')
            torch.save({'request_key': key, 'source_id': record['source_id'], 'input_sha256': record['tensor_sha256'],
                        'outputs': outputs, 'controls': controls}, temporary)
            temporary.replace(output_path)
            report['controls'].extend(controls)
            report['artifacts'].append({'name': filename, 'sha256': digest(output_path), 'bytes': output_path.stat().st_size})
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            if not all(c['passed'] for c in controls):
                raise RuntimeError('Independent-process reference replay exceeds tolerance.')
        if report['teacher_calls'] != 90:
            raise RuntimeError('Reference attempt did not cover all 90 saved stage inputs.')
        report['legacy_max_relative_l2'] = max(c['legacy_difference']['relative_l2_error'] for c in report['controls'])
        report['max_reference_relative_l2'] = max(c['reference_difference']['relative_l2_error'] for c in report['controls'])
        report['exact_equal_controls'] = sum(c['reference_difference']['exact_equal'] for c in report['controls'])
        if not all(c['passed'] for c in report['controls']):
            raise RuntimeError('Teacher controls did not pass.')
        report['status'] = 'teacher_verified'
        (directory / 'teacher_control_report.json').write_text(json.dumps(report, indent=2))
        results_volume.commit()
        print('All 90 intact-teacher controls passed; assembling frozen 31-block candidate B.', flush=True)

        import gc
        import time
        import statistics
        from types import SimpleNamespace
        
        # Inputs are independently identified; 10 gate and 50 wider development prompts.
        panels = {'gate10': [], 'wider50': []}
        for panel, records in (('gate10', [r for r in request['records'] if r['capture_id'].startswith('p5-validation-')]),
                               ('wider50', request['expanded_validation']['records'])):
            for record in records:
                path = Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt'
                if digest(path) != record['tensor_sha256']:
                    raise RuntimeError('Fixed input hash differs')
                item = torch.load(path, weights_only=True, map_location='cpu')
                if item['row']['source_id'] != record['source_id'] or item['row']['split'] != 'validation':
                    raise RuntimeError('Validation input identity differs')
                panels[panel].append(item)
        if [len(panels[p]) for p in ('gate10', 'wider50')] != [10, 50]:
            raise RuntimeError('Saved validation panel sizes differ')

        def forward(item, stage, cache):
            full = {**item['shared_transformer_kwargs'], **stage['kwargs']}
            kwargs = gpu({field: full[field] for field in argument_order})
            kwargs['kv_cache'] = cache
            return model(**kwargs)[0][:, -stage['target_image_tokens']:].detach().cpu().contiguous()

        def predictions():
            rows = []
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for panel, items in panels.items():
                    for item in items:
                        cache = QwenImage21KVCache(32)
                        for stage in item['stages']:
                            pred = forward(item, stage, cache)
                            if not torch.isfinite(pred).all() or pred.float().norm() == 0:
                                raise RuntimeError('Invalid reference velocity')
                            rows.append(dict(panel=panel, source_id=item['row']['source_id'],
                                             dimension=item['row']['dimension'],
                                             subdimension=item['row'].get('subdimension'),
                                             stage=stage['stage'], velocity=pred))
            return rows

        def save(name, value):
            path = directory / name
            temporary = path.with_suffix('.tmp')
            torch.save(value, temporary)
            temporary.replace(path)
            artifact = dict(name=name, sha256=digest(path), bytes=path.stat().st_size)
            report['artifacts'].append(artifact)
            return artifact

        teacher_predictions = predictions()
        save('teacher_predictions.pt', teacher_predictions)
        report['teacher_calls'] += len(teacher_predictions)
        # ---- Live teacher targets for TRAIN inputs (stored legacy targets differ by up to ~2.6%). ----
        def load_verified(path, sha256, size=None):
            if (size is not None and path.stat().st_size != size) or digest(path) != sha256:
                raise RuntimeError('Input tensor differs: ' + str(path))
            return torch.load(path, weights_only=True, map_location='cpu')

        train_full, early_pool = [], []
        for record in request['records']:
            if not record['capture_id'].startswith('p5-train-'):
                continue
            item = load_verified(Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt', record['tensor_sha256'])
            if item['row']['split'] != 'train' or item['row']['source_id'] != record['source_id']:
                raise RuntimeError('Training record identity differs.')
            train_full.append(item)
        fresh = request['fresh_aligned_train']
        for artifact in fresh['artifacts']:
            item = load_verified(Path(fresh['remote_directory']) / artifact['name'], artifact['sha256'], artifact['bytes'])
            if item['source_id'] != artifact['source_id'] or item['row']['split'] != 'train' or len(item['stages']) != 3:
                raise RuntimeError('Fresh aligned training provenance differs.')
            train_full.append(item)
        for example in request['early_pool']['examples']:
            item = load_verified(Path(example['remote_path']), example['sha256'], example['bytes'])
            if (item['capture_id'] != example['capture_id'] or item['row']['split'] != 'train'
                    or [s['stage'] for s in item['stages']] != ['early']):
                raise RuntimeError('Early pool example provenance differs.')
            early_pool.append(item)
        validation_ids = {r['source_id'] for r in request['records'] if r['capture_id'].startswith('p5-validation-')}
        validation_ids |= {r['source_id'] for r in request['expanded_validation']['records']}
        train_ids = {i['row']['source_id'] for i in train_full} | {i['row']['source_id'] for i in early_pool}
        if len(train_full) != 60 or train_ids & validation_ids:
            raise RuntimeError('Training set size or split differs.')
        for item in train_full + early_pool:
            cache = QwenImage21KVCache(32)
            item['live_targets'] = []
            for stage in item['stages']:
                with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                    item['live_targets'].append(forward(item, stage, cache).to(torch.bfloat16))
                report['teacher_calls'] += 1
        report['train_counts'] = {'full_prompts': len(train_full), 'early_pool': len(early_pool)}
        print('Live teacher targets for', report['train_counts'], flush=True)

        # Rebuild the Phase 6 selected student exactly as its verified replay did:
        # 31 teacher blocks plus the frozen pretrained rank-256 bridge at slot 5.
        namespace = {}
        exec(compile(request['core_source'], 'bridge_definition.py', 'exec'), namespace)
        exec(compile(request['wrapper_source'], 'bridge_block_definition.py', 'exec'), namespace)
        candidate = request['selected_candidate']
        path = Path('/phase3/phase5/fitting') / request['fit_key'] / Path(candidate['checkpoint_path']).name
        if digest(path) != candidate['checkpoint_sha256']:
            raise RuntimeError('Selected bridge checkpoint differs.')
        checkpoint = torch.load(path, weights_only=True, map_location='cpu')
        if (checkpoint['request_key'] != request['fit_key'] or checkpoint['layer'] != 5
                or checkpoint['step'] != candidate['selected_update']):
            raise RuntimeError('Bridge provenance differs.')
        bridge = namespace['ResidualBottleneckBridge'](4096, 256)
        bridge.load_state_dict(checkpoint['state_dict'])
        bridge.requires_grad_(False)
        namespace['insert_bridge'](model, 5, bridge, QwenImage21KVCache)
        del checkpoint
        model.requires_grad_(False).eval()
        gc.collect()
        torch.cuda.empty_cache()
        baseline = predictions()
        report['student_calls'] += len(baseline)
        save('candidate_predictions.pt', baseline)
        # The 10 gate prompts must reproduce the saved Phase 6 trained-bridge velocities.
        replay = request['selected_replay']
        replay_dir = Path('/phase3/phase6/verified_student_replay') / replay['request_key']
        reference = {}
        for record in request['records']:
            if not record['capture_id'].startswith('p5-validation-'):
                continue
            name = 'students-' + record['capture_id'] + '.pt'
            if digest(replay_dir / name) != replay['artifacts'][name]:
                raise RuntimeError('Phase 6 student replay checksum differs.')
            saved = torch.load(replay_dir / name, weights_only=True, map_location='cpu')
            for output in saved['outputs']:
                if output['candidate_id'] == candidate['candidate_id'] and output['variant'] == 'trained':
                    reference[(output['source_id'], output['stage'])] = output['velocity']
        gate_rows = [r for r in baseline if r['panel'] == 'gate10']
        if len(reference) != 30 or len(gate_rows) != 30:
            raise RuntimeError('Phase 6 replay coverage differs.')
        reproduction = [compare(r['velocity'], reference[(r['source_id'], r['stage'])]) for r in gate_rows]
        report['candidate_replay_max_relative_l2'] = max(r['relative_l2_error'] for r in reproduction)
        if report['candidate_replay_max_relative_l2'] > 0.001:
            raise RuntimeError('Selected student failed Phase 6 output replay')
        del reference

        class IdentityBlock(torch.nn.Module):
            def __init__(self, original):
                super().__init__()
                self.attn = SimpleNamespace(processor=original.attn.processor)

            def forward(self, hidden_states, *args, **kwargs):
                return hidden_states

        def metrics(rows):
            output = []
            for measured, base, teacher_row in zip(rows, baseline, teacher_predictions):
                identity = ('panel', 'source_id', 'stage')
                if any(measured[k] != base[k] or base[k] != teacher_row[k] for k in identity):
                    raise RuntimeError('Prediction alignment differs')
                teacher_error = compare(measured['velocity'], teacher_row['velocity'])['relative_l2_error']
                baseline_error = compare(base['velocity'], teacher_row['velocity'])['relative_l2_error']
                output.append({k: measured[k] for k in measured if k != 'velocity'} |
                              dict(baseline_displacement=compare(measured['velocity'], base['velocity'])['relative_l2_error'],
                                   teacher_relative_l2=teacher_error, baseline_teacher_relative_l2=baseline_error,
                                   teacher_error_change_percentage_points=100 * (teacher_error - baseline_error)))
            return output

        def aggregate(rows):
            fields = ('baseline_displacement', 'teacher_relative_l2', 'baseline_teacher_relative_l2',
                      'teacher_error_change_percentage_points')
            result = dict(comparisons=len(rows))
            for field in fields:
                values = sorted(r[field] for r in rows)
                def percentile(fraction):
                    position = fraction * (len(values) - 1)
                    low = int(position)
                    return values[low] + (values[min(low + 1, len(values) - 1)] - values[low]) * (position - low)
                result[field] = dict(mean=statistics.mean(values), maximum=max(values),
                                     p10=percentile(.1), p50=percentile(.5), p90=percentile(.9))
            result['worst_regressions'] = sorted(rows, key=lambda r:r['teacher_error_change_percentage_points'], reverse=True)[:10]
            return result

        report['baseline'] = aggregate(metrics(baseline))

        # ---- LoRA healing pilot: C = B + block-13 attention bridge; LoRA on every original block, both bridges trainable.
        import random
        from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVLayerCache
        fit = request['fit']
        lora = request['lora']
        conditioned = {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned)
        components = {}
        exec(compile(request['components_source'], 'healing_components.py', 'exec'), components)

        # Saved recordings from the 8-head bridge run (steps 0 and 19 with teacher targets).
        recordings = request['recordings']
        new_text, new_general = [], []
        for row in recordings['rows']:
            item = torch.load(Path(recordings['remote_directory']) / (row['source_id'] + '.pt'), weights_only=True, map_location='cpu')
            if (item['row']['source_id'] != row['source_id'] or [s['stage'] for s in item['stages']] != ['early', 'middle']
                    or len(item['live_targets']) != 2):
                raise RuntimeError('Saved recording differs: ' + row['source_id'])
            (new_text if row['subdimension'] == 'text_rendering' else new_general).append(item)
        if {i['row']['source_id'] for i in new_text + new_general} & (train_ids | validation_ids):
            raise RuntimeError('Recordings overlap earlier training or validation prompts.')
        report['recording_counts'] = {'text': len(new_text), 'general': len(new_general)}
        general_pool = train_full + new_general

        # Block 5: FP32 master copy of the same pretrained bridge (identical BF16 arithmetic, trainable).
        path5 = Path('/phase3/phase5/fitting') / request['fit_key'] / Path(candidate['checkpoint_path']).name
        if digest(path5) != candidate['checkpoint_sha256']:
            raise RuntimeError('Selected bridge checkpoint differs.')
        state5 = torch.load(path5, weights_only=True, map_location='cpu')['state_dict']
        bridge5 = components['MasterBridge'](4096, 256)
        bridge5.load_state_dict(state5)
        namespace['insert_bridge'](model, 5, bridge5, QwenImage21KVCache)
        bridge5.float()
        bridge5.load_state_dict(state5)
        # Block 13: the 8-head attention bridge, FP32.
        warm_spec = request['warm_start']
        if digest(Path(warm_spec['remote_path'])) != warm_spec['sha256']:
            raise RuntimeError('Block-13 bridge checkpoint differs.')
        warm = torch.load(Path(warm_spec['remote_path']), weights_only=True, map_location='cpu')
        spec = warm['variant']
        if warm['layer'] != 13 or spec['name'] != warm_spec['name'] or not spec.get('attention'):
            raise RuntimeError('Block-13 bridge provenance differs.')
        bridge13 = conditioned['AttentionConditionedBridge'](4096, 1024, spec['heads'], 128)
        bridge13.load_state_dict(warm['state_dict'])
        conditioned['insert_conditioned_bridge'](model, 13, bridge13.float(), keep_dtype=True)
        bridge13 = model.transformer_blocks[13].bridge
        # LoRA on q, k, v and out of every original block, then functional checkpointing for memory.
        torch.manual_seed(fit['init_seed'])
        for slot, block in enumerate(list(model.transformer_blocks)):
            if slot in (5, 13):
                continue
            for name in ('to_q', 'to_k', 'to_v'):
                setattr(block.attn, name, components['MasterLoRALinear'](getattr(block.attn, name), lora['rank'], lora['alpha']))
            block.attn.to_out[0] = components['MasterLoRALinear'](block.attn.to_out[0], lora['rank'], lora['alpha'])
            model.transformer_blocks[slot] = components['FunctionalCheckpointBlock'](block, QwenImage21KVLayerCache)
        model.requires_grad_(False)
        model.gradient_checkpointing = False
        lora_parameters = {n: p for n, p in model.named_parameters() if '.attn.' in n and ('.down.' in n or '.up.' in n)}
        bridge_parameters = {n: p for n, p in model.named_parameters() if n.startswith(('transformer_blocks.5.bridge.', 'transformer_blocks.13.bridge.'))}
        if len(lora_parameters) != 30 * 4 * 2 or not bridge_parameters:
            raise RuntimeError('Adapter structure differs.')
        trainable = {**lora_parameters, **bridge_parameters}
        for parameter in trainable.values():
            parameter.requires_grad_(True)
        report['trainable_parameters'] = {'lora': sum(p.numel() for p in lora_parameters.values()),
                                          'bridges': sum(p.numel() for p in bridge_parameters.values())}
        model.eval()
        def train_loss(item):
            cache = QwenImage21KVCache(32)
            losses = []
            for stage, target in zip(item['stages'], item['live_targets']):
                full = {**item['shared_transformer_kwargs'], **stage['kwargs']}
                kwargs = gpu({field: full[field] for field in argument_order})
                kwargs['kv_cache'] = cache
                prediction = model(**kwargs)[0][:, -stage['target_image_tokens']:].float()
                target = target.to('cuda').float()
                losses.append((prediction - target).square().mean() / target.square().mean().clamp_min(1e-12))
                report['student_calls'] += 1
            loss = torch.stack(losses).mean()
            if not torch.isfinite(loss):
                raise RuntimeError('Nonfinite training loss.')
            return loss

        optimizer = torch.optim.AdamW([{'params': list(lora_parameters.values()), 'lr': lora['learning_rate'], 'base': lora['learning_rate']},
                                       {'params': list(bridge_parameters.values()), 'lr': fit['bridge_learning_rate'], 'base': fit['bridge_learning_rate']}],
                                      weight_decay=0.0)
        rng = random.Random(fit['sampling_seed'])
        report['checkpoints'] = []
        started = time.perf_counter()

        def evaluate_and_save(step):
            # Quantitative test on all 180 comparisons, and a saved checkpoint for the qualitative (image) test.
            for parameter in trainable.values():
                parameter.requires_grad_(False)
            rows = predictions()
            for parameter in trainable.values():
                parameter.requires_grad_(True)
            report['student_calls'] += len(rows)
            records = metrics(rows)
            entry = dict(step=step, seconds=time.perf_counter() - started,
                         overall=aggregate(records),
                         panels={p: aggregate([r for r in records if r['panel'] == p]) for p in panels},
                         stages={s: aggregate([r for r in records if r['stage'] == s]) for s in ('early', 'middle', 'late')})
            if step > 0:
                path = directory / f'healing_step_{step:04d}.pt'
                temporary = path.with_suffix('.tmp')
                torch.save({'request_key': key, 'step': step, 'lora': lora, 'block13_variant': spec,
                            'state_dict': {n: p.detach().cpu().clone() for n, p in trainable.items()}}, temporary)
                temporary.replace(path)
                entry['checkpoint'] = dict(name=path.name, sha256=digest(path), bytes=path.stat().st_size)
                report['artifacts'].append(entry['checkpoint'])
            report['checkpoints'].append(entry)
            o = entry['overall']['teacher_relative_l2']
            print(f"healing step {step}: overall {o['mean']:.6f} gate10 {entry['panels']['gate10']['teacher_relative_l2']['mean']:.6f} "
                  f"wider {entry['panels']['wider50']['teacher_relative_l2']['mean']:.6f} ({entry['seconds']:.0f}s)", flush=True)
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()

        for step in range(fit['updates'] + 1):
            if step % fit['checkpoint_every'] == 0:
                evaluate_and_save(step)
            if step == fit['updates']:
                break
            for group in optimizer.param_groups:
                group['lr'] = group['base'] * min(1.0, (step + 1) / fit['warmup'])
            optimizer.zero_grad(set_to_none=True)
            # Backward recomputes checkpointed blocks: keep the forward attention kernel and cache context.
            with model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for _ in range(fit['accumulation']):
                    pool = new_text if rng.random() < fit['text_fraction'] else general_pool
                    (train_loss(rng.choice(pool)) * (1 - fit['early_weight']) / fit['accumulation']).backward()
                    (train_loss(rng.choice(early_pool)) * fit['early_weight'] / fit['accumulation']).backward()
            torch.nn.utils.clip_grad_norm_(list(trainable.values()), fit['gradient_clip'])
            optimizer.step()
            report['optimizer_updates'] += 1
        report['status'] = 'passed'
        report['scope'] = 'LoRA healing of C with every checkpoint saved and evaluated on all 180 velocity comparisons; images are rendered per checkpoint in a separate step.'
    except Exception as exc:
        import traceback
        report['status'] = 'failed'
        report['error'] = repr(exc)
        report['traceback'] = traceback.format_exc()
    (directory / 'summary.json').write_text(json.dumps(report, indent=2))
    results_volume.commit()
    return report
'''

def main():
    import argparse
    import hashlib
    parser = argparse.ArgumentParser()
    parser.add_argument('--dry-run', action='store_true')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    load = lambda path: json.loads((root / path).read_text(encoding='utf-8'))
    protocol_path = root / 'data/phase7/healing_pilot_v2/protocol.json'
    scan_b = load('data/phase7/conditional_importance_b/b0a27f0cbe0f8e7f9938044fe088576a648c7b22e63a2ddbd771ee6b8e777aae/request.json')['request']
    scan_a = load('data/phase7/conditional_importance/7454da6a5c1dc33af23dd46833e698f9b759d14985ca96340bc3d40cd47fb4f5/request.json')['request']
    text8 = load('data/phase7/block13_bridge_e2e_text8/ff9d259ae589bdf15b98076930c4d16340ef41f4a2c3db897ea3167e95397054/request.json')
    text8_summary = load('data/phase7/block13_bridge_e2e_text8/ff9d259ae589bdf15b98076930c4d16340ef41f4a2c3db897ea3167e95397054/summary.json')
    bridge = text8_summary['variants']['e2e_attn8_text']['checkpoint']
    request = {k: v for k, v in scan_b.items() if k not in ('worker_source', 'eligible_original_block_ids', 'selection', 'scope', 'version')}
    request.update(
        version='phase7-healing-pilot-v2', worker_source=SOURCES['healing_pilot_v2_worker.py'],
        protocol_sha256=hashlib.sha256(protocol_path.read_bytes()).hexdigest(),
        conditioned_bridge_source=text8['request']['conditioned_bridge_source'], components_source=scan_a['components_source'],
        fresh_aligned_train=scan_a['fresh_aligned_train'], early_pool=scan_a['early_pool'],
        warm_start=dict(name='e2e_attn8_text', remote_path=f"/phase3/phase7/bridge_end_to_end_v3/{text8['request_key']}/{bridge['name']}",
                        sha256=bridge['sha256']),
        recordings=dict(remote_directory=f"/phase3/phase7/bridge_end_to_end_v3/{text8['request_key']}/recorded",
                        rows=[{k: r[k] for k in ('source_id', 'subdimension')} for r in text8['request']['new_prompts']]),
        lora=dict(rank=16, alpha=16, learning_rate=2e-5, targets=['to_q', 'to_k', 'to_v', 'to_out.0']),
        fit=dict(init_seed=20261018, sampling_seed=20261025, bridge_learning_rate=1e-5, warmup=10, gradient_clip=1.0,
                 updates=200, checkpoint_every=50, accumulation=2, early_weight=0.3, text_fraction=0.4),
        selection='No automatic selection: every checkpoint is saved, gets the full velocity test and a text grid; spelling decides',
        scope='LoRA healing of C, every checkpoint saved for velocity and spelling tests')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase7/healing_pilot_v2' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2) + '\n', encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return request_key
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase7/healing_pilot_v2/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        try:
            b''.join(results_volume.read_file(remote + '/started.json'))
        except FileNotFoundError:
            pass
        else:
            raise RuntimeError('Run already started; inspect its progress. No duplicate GPU scheduled.')
        app = modal.App('nihonga-phase7-healing-pilot-v2')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'healing_pilot_v2_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8, memory=131072,
                              max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['heal_student'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2) + '\n')
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'recording_counts', 'trainable_parameters')}), flush=True)
    return request_key

LAUNCH_ARGS = ['--dry-run']
RUN = RUN_GLOB + '/' + main()   # writes the request and returns its key; no GPU
LAUNCH_ARGS = []
summary = reuse_or_launch(RUN, main)
if summary and summary.get('status') == 'passed':
    velocity_table([velocity_row(f"step {c['step']}", c) for c in summary['checkpoints']])

## 15b. Spelling of every healing checkpoint

The 5 text prompts x 4 seeds for every saved checkpoint. One sheet per prompt: rows teacher, B, C (step 0) and each checkpoint; columns seeds 0-3.

In [ ]:
# Healing checkpoint text grids: reuses the saved run; launches GPU work only when its result is missing and ALLOW_GPU is True.
SOURCES['healing_checkpoint_images_worker.py'] = r'''def generate_multiseed_text(request, key):
    """Extra seeds for the text prompts: teacher, B, and C with the best block-13 bridge.
    Same prompt, CPU noise seed, size, scheduler and 40 steps for every model. Qualitative screen only."""
    import gc
    import hashlib
    import inspect
    import json
    import os
    import time
    from pathlib import Path

    import torch
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache, QwenImage21KVLayerCache

    directory = Path('/phase3/phase7/healing_checkpoint_images') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete image attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def table_hash(value):
        return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

    variants = [c['name'] for c in request['checkpoints']]
    report = {'request_key': key, 'status': 'running', 'rows': [], 'artifacts': [],
              'calls': {v: 0 for v in variants}, 'optimizer_updates': 0}
    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('Expected H100.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
                                                   dtype=torch.bfloat16, local_files_only=True).to('cuda')
        pipe.set_progress_bar_config(disable=True)
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Transformer source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--')) /
                       'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        # Derived position/time tables: use the audited values on every host (as in steps 7-11).
        derived = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
        derived['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
        expected = request['expected_derived_tables']
        mismatched = sorted(n for n, v in derived.items() if table_hash(v) != expected[n])
        if mismatched:
            canonical = torch.load(Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt'),
                                   weights_only=True, map_location='cpu')
            if any(table_hash(canonical[n]) != expected[n] for n in expected):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [canonical[f'pos_embed.freqs.{a}'].to(device=v.device, dtype=v.dtype) for a, v in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(canonical['time_text_embed.time_proj.freqs'])
            if any(table_hash(v) != expected[n] for n, v in
                   list({f'pos_embed.freqs.{a}': v for a, v in enumerate(model.pos_embed.freqs)}.items()) +
                   [('time_text_embed.time_proj.freqs', model.time_text_embed.time_proj.freqs)]):
                raise RuntimeError('Derived tables still differ after loading the audited copy.')
            report['canonical_tables_applied'] = mismatched

        def generate(row, variant):
            path = directory / (row['source_id'] + '-' + variant + '.png')
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'],
                             num_inference_steps=40, true_cfg_scale=teacher['true_cfg_scale'],
                             generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=True, output_type='pil').images[0]
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            temporary = path.with_suffix('.tmp')
            image.save(temporary, format='PNG')
            temporary.replace(path)
            results_volume.commit()
            report['calls'][variant] += 1
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            return {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size, 'seconds': elapsed}

        namespace = {}
        exec(compile(request['core_source'], 'bridge_definition.py', 'exec'), namespace)
        exec(compile(request['wrapper_source'], 'bridge_block_definition.py', 'exec'), namespace)
        conditioned = {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned)

        def load_checkpoint(spec):
            path = Path(spec['remote_path'])
            if digest(path) != spec['sha256']:
                raise RuntimeError('Bridge checkpoint differs: ' + spec['remote_path'])
            return torch.load(path, weights_only=True, map_location='cpu')

        outputs = {v: {} for v in variants}
        references = False
        if references:
            for row in request['rows']:
                outputs['teacher'][row['source_id']] = generate(row, 'teacher')
        block5 = request['block5_bridge']
        checkpoint = load_checkpoint(block5)
        if checkpoint['layer'] != 5 or checkpoint['step'] != block5['step']:
            raise RuntimeError('Block-5 bridge provenance differs.')
        bridge = namespace['ResidualBottleneckBridge'](4096, 256)
        bridge.load_state_dict(checkpoint['state_dict'])
        bridge.requires_grad_(False)
        namespace['insert_bridge'](model, 5, bridge, QwenImage21KVCache)
        model.eval()
        if references:
            for row in request['rows']:
                outputs['B'][row['source_id']] = generate(row, 'B')
        components = {}
        exec(compile(request['components_source'], 'healing_components.py', 'exec'), components)
        bridge13 = None
        lora_modules = {}
        for checkpoint_spec in request['checkpoints']:
            healed = load_checkpoint(checkpoint_spec)
            state = healed['state_dict']
            lora = healed['lora']
            prefix5, prefix13 = 'transformer_blocks.5.bridge.', 'transformer_blocks.13.bridge.'
            model.transformer_blocks[5].bridge.load_state_dict(
                {k[len(prefix5):]: v.to(torch.bfloat16) for k, v in state.items() if k.startswith(prefix5)})
            if bridge13 is None:
                variant13 = healed['block13_variant']
                bridge13 = conditioned['AttentionConditionedBridge'](4096, 1024, variant13['heads'], 128)
                bridge13.requires_grad_(False)
                conditioned['insert_conditioned_bridge'](model, 13, bridge13.float(), keep_dtype=True)
                bridge13 = model.transformer_blocks[13].bridge
                for slot, block in enumerate(model.transformer_blocks):
                    if slot in (5, 13):
                        continue
                    for name in ('to_q', 'to_k', 'to_v'):
                        setattr(block.attn, name, components['MasterLoRALinear'](getattr(block.attn, name), lora['rank'], lora['alpha']))
                    block.attn.to_out[0] = components['MasterLoRALinear'](block.attn.to_out[0], lora['rank'], lora['alpha'])
                    for name, module in (('to_q', block.attn.to_q), ('to_k', block.attn.to_k), ('to_v', block.attn.to_v), ('to_out.0', block.attn.to_out[0])):
                        lora_modules[(slot, name)] = module
                model.requires_grad_(False)
            bridge13.load_state_dict({k[len(prefix13):]: v for k, v in state.items() if k.startswith(prefix13)})
            loaded = 0
            for (slot, name), module in lora_modules.items():
                for part in ('down', 'up'):
                    getattr(module, part).weight.data.copy_(state[f'transformer_blocks.{slot}.inner.attn.{name}.{part}.weight'])
                    loaded += 1
            if loaded != 30 * 4 * 2:
                raise RuntimeError('Healed adapter count differs.')
            model.eval()
            for row in request['rows']:
                outputs[checkpoint_spec['name']][row['source_id']] = generate(row, checkpoint_spec['name'])
        rendered = [v for v in variants if outputs[v]]
        for row in request['rows']:
            report['rows'].append({**row, **{v: outputs[v][row['source_id']] for v in rendered}})
            report['artifacts'].extend(outputs[v][row['source_id']] for v in rendered)
        report['status'] = 'passed'
    except Exception as exc:
        report['status'] = 'failed'
        report['error'] = type(exc).__name__ + ': ' + str(exc)
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=True) + '\n')
    results_volume.commit()
    return report
'''

def main():
    import argparse
    import hashlib
    parser = argparse.ArgumentParser()
    parser.add_argument('--dry-run', action='store_true')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    load = lambda path: json.loads((root / path).read_text(encoding='utf-8'))
    base = json.loads(next((root / 'data/recovered/modal/phase7/candidate_images').glob('*/request.json')).read_bytes())['request']
    scan_b = load('data/phase7/conditional_importance_b/b0a27f0cbe0f8e7f9938044fe088576a648c7b22e63a2ddbd771ee6b8e777aae/request.json')['request']
    scan_a = load('data/phase7/conditional_importance/7454da6a5c1dc33af23dd46833e698f9b759d14985ca96340bc3d40cd47fb4f5/request.json')['request']
    heal_dir = next(p.parent for p in (root / 'data/phase7/healing_pilot_v2').glob('*/summary.json')
                    if load(p.relative_to(root)).get('status') == 'passed')
    heal = load((heal_dir / 'summary.json').relative_to(root))
    heal_request = load((heal_dir / 'request.json').relative_to(root))
    selected = scan_b['selected_candidate']
    text_rows = [r for r in base['rows'] if r['subdimension'] in ('text_rendering', 'posters')]
    rows = list(text_rows) + [{**r, 'source_id': f"{r['source_id']}-s{k}", 'seed': (r['seed'] + 1000003 * k) % 2**32}
                              for r in text_rows for k in (1, 2, 3)]
    checkpoints = [dict(name=f"healed_s{c['step']:03d}", step=c['step'],
                        remote_path=f"/phase3/phase7/healing_pilot_v2/{heal['request_key']}/{c['checkpoint']['name']}",
                        sha256=c['checkpoint']['sha256']) for c in heal['checkpoints'] if c['step'] > 0]
    request = {k: base[k] for k in ('teacher', 'transformer_source_sha256', 'runtime_environment', 'expected_derived_tables',
                                    'canonical_tables_key', 'core_source', 'wrapper_source')}
    request.update(
        version='phase7-healing-checkpoint-images-v1', rows=rows, worker_source=SOURCES['healing_checkpoint_images_worker.py'],
        conditioned_bridge_source=heal_request['request']['conditioned_bridge_source'], components_source=scan_a['components_source'],
        execution_account='new', candidate_only=True, checkpoints=checkpoints,
        block5_bridge=dict(remote_path=f"/phase3/phase5/fitting/{scan_b['fit_key']}/{Path(selected['checkpoint_path']).name}",
                           sha256=selected['checkpoint_sha256'], step=selected['selected_update']),
        c_variant=dict(name='unused'),
        scope='4-seed text grid for every saved healing checkpoint (teacher, B and C reused)')
    from experiment_runtime import pin_reference_runtime
    pin_reference_runtime(request, root / 'data/recovered/modal')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase7/healing_checkpoint_images' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2) + chr(10), encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return request_key
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase7/healing_checkpoint_images/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        app = modal.App('nihonga-phase7-healing-checkpoint-images')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'healing_checkpoint_images_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8, memory=98304,
                              max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['generate_multiseed_text'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2) + chr(10))
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2) + chr(10), encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'calls')}), flush=True)
    return request_key

def build_checkpoint_grids(run):
    """One sheet per text prompt: rows teacher, B, C (step 0) and every healing checkpoint; columns seeds 0-3."""
    import hashlib
    import textwrap
    from PIL import Image as PILImage, ImageDraw, ImageFont
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='download')
    import modal
    volume = modal.Volume.from_name('nihonga-phase3-artifacts')
    key = run.rstrip('/').split('/')[-1]
    summary = json.loads((ROOT / run / 'summary.json').read_text(encoding='utf-8'))
    store = ROOT / '.archive/healing_checkpoint_images' / key
    store.mkdir(parents=True, exist_ok=True)
    for artifact in summary['artifacts']:
        path = store / artifact['name']
        if not path.exists():
            path.write_bytes(b''.join(volume.read_file(f"/phase7/healing_checkpoint_images/{key}/{artifact['name']}")))
        if hashlib.sha256(path.read_bytes()).hexdigest() != artifact['sha256']:
            raise RuntimeError('Image hash mismatch: ' + artifact['name'])
    first = json.loads(next((ROOT / 'data/phase7/bridge_images').glob('*/summary.json')).read_bytes())
    first_store = ROOT / '.archive/bridge_images' / first['request_key']
    reference = ROOT / '.archive/multiseed_text/af4a34ba1a0c739121706991b89e79ec92bff3debaf1b1ddbf5b6188c182f514'
    c_store = ROOT / '.archive/multiseed_text/320d82f71dfdea06818ff5f50e2e4af933519b5c511111159c1bdea4a00f79aa'
    names = [c['name'] for c in json.loads((ROOT / run / 'request.json').read_bytes())['request']['checkpoints']]
    face = next((f for f in ('C:/Windows/Fonts/msyh.ttc', 'C:/Windows/Fonts/arial.ttf') if Path(f).exists()))
    font, big = ImageFont.truetype(face, 18), ImageFont.truetype(face, 22)
    sheets = ROOT / run / 'sheets'
    sheets.mkdir(exist_ok=True)
    tile, label = 330, 230
    for row in [r for r in first['rows'] if r['subdimension'] in ('text_rendering', 'posters')]:
        sid = row['source_id']
        models = [('Teacher', lambda k: first_store / row['teacher']['name'] if k == 0 else reference / f'{sid}-s{k}-teacher.png'),
                  ('B', lambda k: first_store / row['B']['name'] if k == 0 else reference / f'{sid}-s{k}-B.png'),
                  ('C (step 0)', lambda k: c_store / (f'{sid}-e2e_attn8_text.png' if k == 0 else f'{sid}-s{k}-e2e_attn8_text.png'))]
        models += [(f"healed step {int(n.split('_s')[1])}", lambda k, n=n: store / (f'{sid}-{n}.png' if k == 0 else f'{sid}-s{k}-{n}.png'))
                   for n in names]
        canvas = PILImage.new('RGB', (label + 4 * tile, 80 + len(models) * tile), 'white')
        draw = ImageDraw.Draw(canvas)
        draw.text((10, 8), textwrap.shorten(row['prompt'], 190), font=font, fill='black')
        for column in range(4):
            draw.text((label + column * tile + 10, 48), f'seed {column}', font=big, fill='black')
        for line, (name, locate) in enumerate(models):
            y = 80 + line * tile
            draw.text((10, y + tile // 2 - 12), name, font=big, fill='black')
            for column in range(4):
                with PILImage.open(locate(column)) as image:
                    canvas.paste(image.convert('RGB').resize((tile, tile), PILImage.Resampling.LANCZOS), (label + column * tile, y))
        canvas.save(sheets / f'{sid}.jpg', quality=88)
    return sheets

LAUNCH_ARGS = ['--dry-run']
RUN = 'data/phase7/healing_checkpoint_images/' + main()   # writes the request and returns its key; no GPU
LAUNCH_ARGS = []
summary = reuse_or_launch(RUN, main)
if summary and summary.get('status') == 'passed':
    sheets = build_checkpoint_grids(RUN)
    show_images(str(sheets.relative_to(ROOT)), ['p1-2fa54a3c4eb8.jpg', 'p1-c9344c076abf.jpg'])

### What the healing results mean

| Checkpoint | Overall velocity | Wider (50) | Exact signs / 12 |
|---|---|---|---|
| step 0 (C) | 6.11% | 6.08% | 6 |
| step 50 | 6.11% | 6.06% | 5-6 |
| step 100 | 6.06% | 6.00% | 6 |
| step 150 | **6.04%** | **5.98%** | **~7** |
| step 200 | 6.19% | 6.13% | 6-7 |

Teacher 11/12, B 9/12. Gentle LoRA healing improves velocity slightly up to step 150 and then degrades it; spelling
barely moves. Seeds 1 and 2 fail the same way at every step ("Achar", "Acher"; "Cnna", "Enna"). **By the
preregistered rule (>= 8/12), healing does not recover spelling.** Removing block 13 takes something the model
needs for spelling that neither a bridge nor light healing replaces, so the next step changes how blocks are chosen.

## 16. Text-aware block selection

Until now blocks were ranked by average velocity, and block 13 came first; but it is the block whose removal
breaks spelling. Here each candidate block of B is skipped and measured **on saved text prompts** (80) and, as a
control, on general prompts (50), against the recorded teacher outputs. Blocks with a low text cost are the
candidates for removals #2-#4. The quantitative scan is followed by a qualitative check: the 4-seed text grid with
each shortlisted block skipped (`data/phase7/text_importance/protocol.json`).

In [ ]:
# Text-aware importance: reuses the saved run; launches GPU work only when its result is missing and ALLOW_GPU is True.
SOURCES['text_importance_worker.py'] = r'''def text_importance(request, key):
    """Bypass one block of B at a time and measure velocity error on saved text and general recordings.

    Verifies the teacher contract and exact B replay first; no training, no block accepted.
    """
    import hashlib
    import importlib.metadata
    import inspect
    import json
    import os
    import socket
    import uuid
    from pathlib import Path
    import torch
    from torch.nn.attention import sdpa_kernel, SDPBackend
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache

    directory = Path('/phase3/phase7/text_importance') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete healing attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()
    report = {'request_key': key, 'status': 'failed', 'mode': request['mode'], 'teacher_calls': 0,
              'student_calls': 0, 'optimizer_updates': 0, 'controls': [], 'artifacts': [],
              'process': {'pid': os.getpid(), 'host': socket.gethostname(), 'nonce': uuid.uuid4().hex,
                          'modal_task_id': os.environ.get('MODAL_TASK_ID')}}

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def gpu(value):
        if isinstance(value, torch.Tensor):
            return value.to('cuda')
        if isinstance(value, dict):
            return {k: gpu(v) for k, v in value.items()}
        if isinstance(value, (list, tuple)):
            return type(value)(gpu(v) for v in value)
        return value

    def compare(x, y):
        a, b = x.float(), y.float()
        if not torch.isfinite(a).all() or not torch.isfinite(b).all():
            raise RuntimeError('Nonfinite reference tensor.')
        d = a - b
        return {'relative_l2_error': float(d.norm()/b.norm().clamp_min(1e-12)),
                'max_abs_error': float(d.abs().max()), 'exact_equal': bool(torch.equal(x, y))}

    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('GPU differs.')
        report['packages'] = {p: importlib.metadata.version(p) for p in ('torch', 'diffusers', 'transformers', 'accelerate', 'pillow')}
        report['torch_cuda'] = torch.version.cuda
        report['cudnn_version'] = torch.backends.cudnn.version()
        report['settings'] = request['arithmetic_contract']
        if request['mode'] != 'verify':
            raise RuntimeError('Healing requires the verified teacher reference.')
        reference_dir = Path('/phase3/phase6/deterministic_reference') / request['reference_key']
        reference_report_path = reference_dir / 'summary.json'
        if hashlib.sha256(json.dumps(json.loads(reference_report_path.read_text()), sort_keys=True).encode()).hexdigest() != request['reference_report_semantic_sha256']:
            raise RuntimeError('Generation report differs.')
        reference_report = json.loads(reference_report_path.read_text())
        if reference_report['status'] != 'reference_generated':
            raise RuntimeError('Reference generation did not complete.')
        if reference_report['settings'] != request['arithmetic_contract']:
            raise RuntimeError('Arithmetic contract differs from generation.')
        report['reference_process'] = reference_report['process']
        if reference_report['process']['nonce'] == report['process']['nonce']:
            raise RuntimeError('Independent process requirement failed.')
        if reference_report['packages'] != report['packages'] or reference_report['torch_cuda'] != report['torch_cuda'] or reference_report['cudnn_version'] != report['cudnn_version']:
            raise RuntimeError('Numerical runtime differs across generation and verification.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
            dtype=torch.bfloat16, local_files_only=True).to('cuda')
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Model source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--'))
                       / 'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        state = {}
        for kind, values in (('parameter', model.named_parameters()), ('buffer', model.named_buffers())):
            for name, value in values:
                cpu = value.detach().cpu().contiguous()
                state[name] = {'kind': kind, 'dtype': str(value.dtype), 'shape': list(value.shape),
                    'stride': list(value.stride()), 'sha256': hashlib.sha256(memoryview(cpu.view(torch.uint8).numpy())).hexdigest()}
        for axis, value in enumerate(model.pos_embed.freqs):
            name = f'pos_embed.freqs.{axis}'
            cpu = value.detach().cpu().contiguous()
            state[name] = {'kind': 'unregistered_rope_table', 'dtype': str(value.dtype), 'shape': list(value.shape),
                'stride': list(value.stride()), 'sha256': hashlib.sha256(memoryview(cpu.view(torch.uint8).numpy())).hexdigest()}
        expected = request['expected_state']
        derived_names = sorted(n for n in expected if n.startswith('pos_embed.freqs.') or n == 'time_text_embed.time_proj.freqs')

        def derived_tensors():
            values = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
            values['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
            return values

        def table_hash(value):
            return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

        canonical_path = Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt')
        import time
        mismatched = [n for n in derived_names if state.get(n) != expected.get(n)]
        if not mismatched:
            if not canonical_path.exists():
                canonical_path.parent.mkdir(parents=True, exist_ok=True)
                temporary = canonical_path.with_suffix('.' + report['process']['nonce'] + '.tmp')
                torch.save({n: v.detach().cpu().contiguous() for n, v in derived_tensors().items()}, temporary)
                temporary.replace(canonical_path)
                results_volume.commit()
                report['canonical_tables_saved'] = True
        else:
            deadline = time.time() + 900
            while not canonical_path.exists():
                if time.time() > deadline:
                    raise RuntimeError('Derived tables differ on this host and no canonical copy became available.')
                time.sleep(20)
                results_volume.reload()
            saved = torch.load(canonical_path, weights_only=True, map_location='cpu')
            if sorted(saved) != derived_names or any(table_hash(saved[n]) != expected[n]['sha256'] for n in derived_names):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [saved[f'pos_embed.freqs.{axis}'].to(device=value.device, dtype=value.dtype)
                        for axis, value in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(saved['time_text_embed.time_proj.freqs'])
            for n, value in derived_tensors().items():
                state[n] = {**state[n], 'sha256': table_hash(value)}
            report['canonical_tables_applied'] = mismatched
        if state != request['expected_state']:
            expected = request['expected_state']
            different_names = sorted(name for name in expected.keys() & state.keys()
                                     if state[name] != expected[name])
            report['state_mismatch'] = {
                'actual_count': len(state), 'expected_count': len(expected),
                'missing_names': sorted(set(expected) - set(state))[:12],
                'extra_names': sorted(set(state) - set(expected))[:12],
                'different': [{'name': name, 'actual': state[name], 'expected': expected[name]}
                              for name in different_names][:12]}
            allowed_derived_tables = {'pos_embed.freqs.1', 'pos_embed.freqs.2', 'time_text_embed.time_proj.freqs'}
            if set(state) != set(expected) or not set(different_names).issubset(allowed_derived_tables):
                raise RuntimeError('Loaded teacher learned state differs from audited reference.')
            report['derived_table_variation_requires_output_replay'] = different_names
        report['state_sha256'] = hashlib.sha256(json.dumps(state, sort_keys=True).encode()).hexdigest()
        torch.cuda.synchronize()
        argument_order = ('hidden_states', 'timestep', 'encoder_hidden_states', 'encoder_hidden_states_mask',
                          'img_shapes', 'img_mask', 'attention_kwargs', 'kv_cache_mode', 'return_dict')
        # Every intact-teacher control must pass before any block is touched.
        for record in request['records']:
            path = Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt'
            if digest(path) != record['tensor_sha256']:
                raise RuntimeError('Legacy input bundle differs.')
            bundle = torch.load(path, weights_only=True, map_location='cpu')
            outputs, controls = [], []
            cache = QwenImage21KVCache(32)
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for stage in bundle['stages']:
                    full = {**bundle['shared_transformer_kwargs'], **stage['kwargs']}
                    kwargs = gpu({field: full[field] for field in argument_order})
                    kwargs['kv_cache'] = cache
                    pred = model(**kwargs)[0][:, -stage['target_image_tokens']:].detach().cpu().contiguous()
                    report['teacher_calls'] += 1
                    legacy = compare(pred, stage['teacher_velocity'])
                    entry = {'source_id': record['source_id'], 'split': bundle['row']['split'],
                             'stage': stage['stage'], 'legacy_difference': legacy}
                    outputs.append({'stage': stage['stage'], 'step_index': stage['step_index'],
                                    'scheduler_sigma': stage['scheduler_sigma'], 'velocity': pred})
                    controls.append(entry)
            filename = record['capture_id'] + '.pt'
            artifact = next(a for a in reference_report['artifacts'] if a['name'] == filename)
            if digest(reference_dir / filename) != artifact['sha256']:
                raise RuntimeError('Reference output checksum differs.')
            reference_bundle = torch.load(reference_dir / filename, weights_only=True, map_location='cpu')
            if reference_bundle['source_id'] != record['source_id'] or reference_bundle['input_sha256'] != record['tensor_sha256']:
                raise RuntimeError('Reference input provenance differs.')
            reference_outputs = {v['stage']: v['velocity'] for v in reference_bundle['outputs']}
            for entry, measured in zip(controls, outputs):
                entry['reference_difference'] = compare(measured['velocity'], reference_outputs[entry['stage']])
                entry['passed'] = entry['reference_difference']['relative_l2_error'] <= 0.001
            output_path = directory / filename
            temporary = output_path.with_suffix('.tmp')
            torch.save({'request_key': key, 'source_id': record['source_id'], 'input_sha256': record['tensor_sha256'],
                        'outputs': outputs, 'controls': controls}, temporary)
            temporary.replace(output_path)
            report['controls'].extend(controls)
            report['artifacts'].append({'name': filename, 'sha256': digest(output_path), 'bytes': output_path.stat().st_size})
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            if not all(c['passed'] for c in controls):
                raise RuntimeError('Independent-process reference replay exceeds tolerance.')
        if report['teacher_calls'] != 90:
            raise RuntimeError('Reference attempt did not cover all 90 saved stage inputs.')
        report['legacy_max_relative_l2'] = max(c['legacy_difference']['relative_l2_error'] for c in report['controls'])
        report['max_reference_relative_l2'] = max(c['reference_difference']['relative_l2_error'] for c in report['controls'])
        report['exact_equal_controls'] = sum(c['reference_difference']['exact_equal'] for c in report['controls'])
        if not all(c['passed'] for c in report['controls']):
            raise RuntimeError('Teacher controls did not pass.')
        report['status'] = 'teacher_verified'
        (directory / 'teacher_control_report.json').write_text(json.dumps(report, indent=2))
        results_volume.commit()
        print('All 90 intact-teacher controls passed; assembling frozen 31-block candidate B.', flush=True)

        import gc
        import time
        import statistics
        from types import SimpleNamespace
        
        # Inputs are independently identified; 10 gate and 50 wider development prompts.
        panels = {'gate10': [], 'wider50': []}
        for panel, records in (('gate10', [r for r in request['records'] if r['capture_id'].startswith('p5-validation-')]),
                               ('wider50', request['expanded_validation']['records'])):
            for record in records:
                path = Path('/phase3/phase5/capture') / record['request_key'] / 'targets.pt'
                if digest(path) != record['tensor_sha256']:
                    raise RuntimeError('Fixed input hash differs')
                item = torch.load(path, weights_only=True, map_location='cpu')
                if item['row']['source_id'] != record['source_id'] or item['row']['split'] != 'validation':
                    raise RuntimeError('Validation input identity differs')
                panels[panel].append(item)
        if [len(panels[p]) for p in ('gate10', 'wider50')] != [10, 50]:
            raise RuntimeError('Saved validation panel sizes differ')

        def forward(item, stage, cache):
            full = {**item['shared_transformer_kwargs'], **stage['kwargs']}
            kwargs = gpu({field: full[field] for field in argument_order})
            kwargs['kv_cache'] = cache
            return model(**kwargs)[0][:, -stage['target_image_tokens']:].detach().cpu().contiguous()

        def predictions():
            rows = []
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for panel, items in panels.items():
                    for item in items:
                        cache = QwenImage21KVCache(32)
                        for stage in item['stages']:
                            pred = forward(item, stage, cache)
                            if not torch.isfinite(pred).all() or pred.float().norm() == 0:
                                raise RuntimeError('Invalid reference velocity')
                            rows.append(dict(panel=panel, source_id=item['row']['source_id'],
                                             dimension=item['row']['dimension'],
                                             subdimension=item['row'].get('subdimension'),
                                             stage=stage['stage'], velocity=pred))
            return rows

        def save(name, value):
            path = directory / name
            temporary = path.with_suffix('.tmp')
            torch.save(value, temporary)
            temporary.replace(path)
            artifact = dict(name=name, sha256=digest(path), bytes=path.stat().st_size)
            report['artifacts'].append(artifact)
            return artifact

        teacher_predictions = predictions()
        save('teacher_predictions.pt', teacher_predictions)
        report['teacher_calls'] += len(teacher_predictions)
        # Rebuild the Phase 6 selected student exactly as its verified replay did:
        # 31 teacher blocks plus the frozen pretrained rank-256 bridge at slot 5.
        namespace = {}
        exec(compile(request['core_source'], 'bridge_definition.py', 'exec'), namespace)
        exec(compile(request['wrapper_source'], 'bridge_block_definition.py', 'exec'), namespace)
        candidate = request['selected_candidate']
        path = Path('/phase3/phase5/fitting') / request['fit_key'] / Path(candidate['checkpoint_path']).name
        if digest(path) != candidate['checkpoint_sha256']:
            raise RuntimeError('Selected bridge checkpoint differs.')
        checkpoint = torch.load(path, weights_only=True, map_location='cpu')
        if (checkpoint['request_key'] != request['fit_key'] or checkpoint['layer'] != 5
                or checkpoint['step'] != candidate['selected_update']):
            raise RuntimeError('Bridge provenance differs.')
        bridge = namespace['ResidualBottleneckBridge'](4096, 256)
        bridge.load_state_dict(checkpoint['state_dict'])
        bridge.requires_grad_(False)
        namespace['insert_bridge'](model, 5, bridge, QwenImage21KVCache)
        del checkpoint
        model.requires_grad_(False).eval()
        gc.collect()
        torch.cuda.empty_cache()
        baseline = predictions()
        report['student_calls'] += len(baseline)
        save('candidate_predictions.pt', baseline)
        # The 10 gate prompts must reproduce the saved Phase 6 trained-bridge velocities.
        replay = request['selected_replay']
        replay_dir = Path('/phase3/phase6/verified_student_replay') / replay['request_key']
        reference = {}
        for record in request['records']:
            if not record['capture_id'].startswith('p5-validation-'):
                continue
            name = 'students-' + record['capture_id'] + '.pt'
            if digest(replay_dir / name) != replay['artifacts'][name]:
                raise RuntimeError('Phase 6 student replay checksum differs.')
            saved = torch.load(replay_dir / name, weights_only=True, map_location='cpu')
            for output in saved['outputs']:
                if output['candidate_id'] == candidate['candidate_id'] and output['variant'] == 'trained':
                    reference[(output['source_id'], output['stage'])] = output['velocity']
        gate_rows = [r for r in baseline if r['panel'] == 'gate10']
        if len(reference) != 30 or len(gate_rows) != 30:
            raise RuntimeError('Phase 6 replay coverage differs.')
        reproduction = [compare(r['velocity'], reference[(r['source_id'], r['stage'])]) for r in gate_rows]
        report['candidate_replay_max_relative_l2'] = max(r['relative_l2_error'] for r in reproduction)
        if report['candidate_replay_max_relative_l2'] > 0.001:
            raise RuntimeError('Selected student failed Phase 6 output replay')
        del reference


        # ---- Text-aware importance: bypass one block of B at a time, measured on saved text and general prompts. ----
        recordings = request['recordings']
        groups = {'text': [], 'general': []}
        for row in recordings['rows']:
            item = torch.load(Path(recordings['remote_directory']) / (row['source_id'] + '.pt'), weights_only=True, map_location='cpu')
            if (item['row']['source_id'] != row['source_id'] or [s['stage'] for s in item['stages']] != ['early', 'middle']
                    or len(item['live_targets']) != 2):
                raise RuntimeError('Saved recording differs: ' + row['source_id'])
            groups[row['group']].append(item)
        report['recording_counts'] = {g: len(v) for g, v in groups.items()}

        class IdentityBlock(torch.nn.Module):
            def __init__(self, original):
                super().__init__()
                self.attn = SimpleNamespace(processor=original.attn.processor)

            def forward(self, hidden_states, *args, **kwargs):
                return hidden_states

        def errors():
            values = []
            with torch.inference_mode(), model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for group, items in groups.items():
                    for item in items:
                        cache = QwenImage21KVCache(32)
                        for stage, target in zip(item['stages'], item['live_targets']):
                            relative = compare(forward(item, stage, cache), target)['relative_l2_error']
                            values.append(dict(group=group, source_id=item['row']['source_id'], stage=stage['stage'], relative_l2=relative))
            return values

        def summarise(values):
            out = {}
            for group in groups:
                rows = [v['relative_l2'] for v in values if v['group'] == group]
                out[group] = dict(mean=statistics.mean(rows), p90=sorted(rows)[int(0.9 * (len(rows) - 1))], maximum=max(rows),
                                  stages={s: statistics.mean(v['relative_l2'] for v in values if v['group'] == group and v['stage'] == s)
                                          for s in ('early', 'middle')})
            return out

        started = time.perf_counter()
        base_values = errors()
        report['student_calls'] += len(base_values)
        report['baseline'] = summarise(base_values)
        print('B on saved prompts:', json.dumps({g: round(v['mean'], 6) for g, v in report['baseline'].items()}), flush=True)
        report['blocks'] = []
        original_blocks = list(model.transformer_blocks)
        for slot in request['eligible_original_block_ids']:
            model.transformer_blocks[slot] = IdentityBlock(original_blocks[slot])
            try:
                values = errors()
            finally:
                model.transformer_blocks[slot] = original_blocks[slot]
            report['student_calls'] += len(values)
            summary = summarise(values)
            report['blocks'].append(dict(original_block_id=slot, summary=summary,
                                         text_cost=summary['text']['mean'] - report['baseline']['text']['mean'],
                                         general_cost=summary['general']['mean'] - report['baseline']['general']['mean'],
                                         records=values, seconds=time.perf_counter() - started))
            print(f"block {slot}: text {summary['text']['mean']:.6f} general {summary['general']['mean']:.6f} "
                  f"({time.perf_counter() - started:.0f}s)", flush=True)
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
        report['ranking_by_text_cost'] = [b['original_block_id'] for b in sorted(report['blocks'], key=lambda b: b['text_cost'])]
        report['status'] = 'passed'
        report['optimizer_updates'] = 0
        report['scope'] = 'Text-aware identity-bypass screen inside B on saved text and general recordings; no block accepted.'
    except Exception as exc:
        import traceback
        report['status'] = 'failed'
        report['error'] = repr(exc)
        report['traceback'] = traceback.format_exc()
    (directory / 'summary.json').write_text(json.dumps(report, indent=2))
    results_volume.commit()
    return report
'''

def main():
    import argparse
    import hashlib
    import random
    parser = argparse.ArgumentParser()
    parser.add_argument('--dry-run', action='store_true')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    load = lambda path: json.loads((root / path).read_text(encoding='utf-8'))
    protocol_path = root / 'data/phase7/text_importance/protocol.json'
    scan_b = load('data/phase7/conditional_importance_b/b0a27f0cbe0f8e7f9938044fe088576a648c7b22e63a2ddbd771ee6b8e777aae/request.json')['request']
    text8 = load('data/phase7/block13_bridge_e2e_text8/ff9d259ae589bdf15b98076930c4d16340ef41f4a2c3db897ea3167e95397054/request.json')
    rows = text8['request']['new_prompts']
    picker = random.Random(20261026)
    text_rows = picker.sample(sorted((r for r in rows if r['subdimension'] == 'text_rendering'), key=lambda r: r['source_id']), 80)
    general_rows = sorted((r for r in rows if r['subdimension'] != 'text_rendering'), key=lambda r: r['source_id'])
    request = {k: v for k, v in scan_b.items() if k not in ('worker_source', 'eligible_original_block_ids', 'selection', 'scope', 'version')}
    request.update(
        version='phase7-text-importance-v1', worker_source=SOURCES['text_importance_worker.py'],
        protocol_sha256=hashlib.sha256(protocol_path.read_bytes()).hexdigest(),
        recordings=dict(remote_directory=f"/phase3/phase7/bridge_end_to_end_v3/{text8['request_key']}/recorded",
                        rows=[dict(source_id=r['source_id'], group='text') for r in text_rows] +
                             [dict(source_id=r['source_id'], group='general') for r in general_rows]),
        eligible_original_block_ids=[i for i in range(2, 30) if i != 5],
        selection='Screening only: rank blocks by the velocity cost of bypassing them on text prompts; images decide',
        scope='Text-aware identity-bypass screen inside B on 80 saved text and 50 general recordings (early and middle moments)')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase7/text_importance' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2) + chr(10), encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return request_key
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase7/text_importance/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        try:
            b''.join(results_volume.read_file(remote + '/started.json'))
        except FileNotFoundError:
            pass
        else:
            raise RuntimeError('Run already started; inspect its progress. No duplicate GPU scheduled.')
        app = modal.App('nihonga-phase7-text-importance')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'text_importance_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8, memory=98304,
                              max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['text_importance'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2) + chr(10))
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2) + chr(10), encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'recording_counts', 'ranking_by_text_cost')}), flush=True)
    return request_key

LAUNCH_ARGS = ['--dry-run']
RUN = 'data/phase7/text_importance/' + main()   # writes the request and returns its key; no GPU
LAUNCH_ARGS = []
summary = reuse_or_launch(RUN, main)
if summary and summary.get('status') == 'passed':
    rows = sorted(summary['blocks'], key=lambda b: b['text_cost'])
    head = '| Block | Text error | Text cost | General error | General cost |' + chr(10) + '|---|---|---|---|---|' + chr(10)
    display(Markdown(head + chr(10).join(f"| {b['original_block_id']} | {pct(b['summary']['text']['mean'])} | +{pct(b['text_cost'])} | "
                                         f"{pct(b['summary']['general']['mean'])} | +{pct(b['general_cost'])} |" for b in rows)))

## 16b. Is it block 13, or block 13 on top of a weak block-5 bridge?

B already loses spelling against the teacher (9/12 against 11/12): its block-5 bridge is the simplest one, without
timestep or attention, and was never healed. Errors may compound: when block 13 is also removed, the rest of the
model has less room to compensate, and the block-13 bridge was trained on top of B's drift. This test removes block
13 **from the intact teacher** (block 5 kept), once as a plain skip and once with the best block-13 bridge, and
renders the 4-seed text grid. If spelling stays near 9-11/12, the combination with the weak block-5 bridge is the
problem and block 5 should be upgraded; if it drops near 6/12, block 13 matters for spelling on its own.

In [ ]:
# Teacher without block 13: reuses the saved run; launches GPU work only when its result is missing and ALLOW_GPU is True.
SOURCES['teacher_minus13_images_worker.py'] = r'''def generate_multiseed_text(request, key):
    """Extra seeds for the text prompts: teacher, B, and C with the best block-13 bridge.
    Same prompt, CPU noise seed, size, scheduler and 40 steps for every model. Qualitative screen only."""
    import gc
    import hashlib
    import inspect
    import json
    import os
    import time
    from pathlib import Path

    import torch
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache, QwenImage21KVLayerCache

    directory = Path('/phase3/phase7/teacher_minus13_images') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete image attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def table_hash(value):
        return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

    variants = ['teacher_skip13', 'teacher_bridge13']
    report = {'request_key': key, 'status': 'running', 'rows': [], 'artifacts': [],
              'calls': {v: 0 for v in variants}, 'optimizer_updates': 0}
    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('Expected H100.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
                                                   dtype=torch.bfloat16, local_files_only=True).to('cuda')
        pipe.set_progress_bar_config(disable=True)
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Transformer source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--')) /
                       'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        # Derived position/time tables: use the audited values on every host (as in steps 7-11).
        derived = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
        derived['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
        expected = request['expected_derived_tables']
        mismatched = sorted(n for n, v in derived.items() if table_hash(v) != expected[n])
        if mismatched:
            canonical = torch.load(Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt'),
                                   weights_only=True, map_location='cpu')
            if any(table_hash(canonical[n]) != expected[n] for n in expected):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [canonical[f'pos_embed.freqs.{a}'].to(device=v.device, dtype=v.dtype) for a, v in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(canonical['time_text_embed.time_proj.freqs'])
            if any(table_hash(v) != expected[n] for n, v in
                   list({f'pos_embed.freqs.{a}': v for a, v in enumerate(model.pos_embed.freqs)}.items()) +
                   [('time_text_embed.time_proj.freqs', model.time_text_embed.time_proj.freqs)]):
                raise RuntimeError('Derived tables still differ after loading the audited copy.')
            report['canonical_tables_applied'] = mismatched

        def generate(row, variant):
            path = directory / (row['source_id'] + '-' + variant + '.png')
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'],
                             num_inference_steps=40, true_cfg_scale=teacher['true_cfg_scale'],
                             generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=True, output_type='pil').images[0]
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            temporary = path.with_suffix('.tmp')
            image.save(temporary, format='PNG')
            temporary.replace(path)
            results_volume.commit()
            report['calls'][variant] += 1
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            return {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size, 'seconds': elapsed}

        namespace = {}
        exec(compile(request['core_source'], 'bridge_definition.py', 'exec'), namespace)
        exec(compile(request['wrapper_source'], 'bridge_block_definition.py', 'exec'), namespace)
        conditioned = {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned)

        def load_checkpoint(spec):
            path = Path(spec['remote_path'])
            if digest(path) != spec['sha256']:
                raise RuntimeError('Bridge checkpoint differs: ' + spec['remote_path'])
            return torch.load(path, weights_only=True, map_location='cpu')

        outputs = {v: {} for v in variants}
        references = False
        if references:
            for row in request['rows']:
                outputs['teacher'][row['source_id']] = generate(row, 'teacher')
        from types import SimpleNamespace

        class IdentityBlock(torch.nn.Module):
            def __init__(self, original):
                super().__init__()
                self.attn = SimpleNamespace(processor=original.attn.processor)

            def forward(self, hidden_states, *args, **kwargs):
                return hidden_states

        original13 = model.transformer_blocks[13]
        model.transformer_blocks[13] = IdentityBlock(original13)
        model.eval()
        for row in request['rows']:
            outputs['teacher_skip13'][row['source_id']] = generate(row, 'teacher_skip13')
        model.transformer_blocks[13] = original13
        spec = request['c_variant']
        checkpoint = load_checkpoint(spec)
        fitted = checkpoint['variant']
        if checkpoint['layer'] != 13 or fitted['name'] != spec['name'] or not fitted.get('attention'):
            raise RuntimeError('Block-13 bridge provenance differs.')
        bridge13 = conditioned['AttentionConditionedBridge'](4096, 1024, fitted['heads'], 128)
        bridge13.load_state_dict(checkpoint['state_dict'])
        bridge13.requires_grad_(False)
        conditioned['insert_conditioned_bridge'](model, 13, bridge13.float(), keep_dtype=True)
        model.eval()
        for row in request['rows']:
            outputs['teacher_bridge13'][row['source_id']] = generate(row, 'teacher_bridge13')
        rendered = [v for v in variants if outputs[v]]
        for row in request['rows']:
            report['rows'].append({**row, **{v: outputs[v][row['source_id']] for v in rendered}})
            report['artifacts'].extend(outputs[v][row['source_id']] for v in rendered)
        report['status'] = 'passed'
    except Exception as exc:
        report['status'] = 'failed'
        report['error'] = type(exc).__name__ + ': ' + str(exc)
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=True) + '\n')
    results_volume.commit()
    return report
'''

def main():
    import argparse
    import hashlib
    parser = argparse.ArgumentParser()
    parser.add_argument('--dry-run', action='store_true')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    load = lambda path: json.loads((root / path).read_text(encoding='utf-8'))
    base = json.loads(next((root / 'data/recovered/modal/phase7/candidate_images').glob('*/request.json')).read_bytes())['request']
    text8 = load('data/phase7/block13_bridge_e2e_text8/ff9d259ae589bdf15b98076930c4d16340ef41f4a2c3db897ea3167e95397054/request.json')
    text8_summary = load('data/phase7/block13_bridge_e2e_text8/ff9d259ae589bdf15b98076930c4d16340ef41f4a2c3db897ea3167e95397054/summary.json')
    bridge = text8_summary['variants']['e2e_attn8_text']['checkpoint']
    text_rows = [r for r in base['rows'] if r['subdimension'] in ('text_rendering', 'posters')]
    rows = list(text_rows) + [{**r, 'source_id': f"{r['source_id']}-s{k}", 'seed': (r['seed'] + 1000003 * k) % 2**32}
                              for r in text_rows for k in (1, 2, 3)]
    request = {k: base[k] for k in ('teacher', 'transformer_source_sha256', 'runtime_environment', 'expected_derived_tables',
                                    'canonical_tables_key', 'core_source', 'wrapper_source')}
    request.update(
        version='phase7-teacher-minus13-images-v1', rows=rows, worker_source=SOURCES['teacher_minus13_images_worker.py'],
        conditioned_bridge_source=text8['request']['conditioned_bridge_source'], execution_account='new', candidate_only=True,
        c_variant=dict(name='e2e_attn8_text', remote_path=f"/phase3/phase7/bridge_end_to_end_v3/{text8['request_key']}/{bridge['name']}",
                       sha256=bridge['sha256']),
        scope='Block 5 intact: block 13 skipped, and block 13 replaced by the 8-head attention bridge; 4-seed text grid')
    from experiment_runtime import pin_reference_runtime
    pin_reference_runtime(request, root / 'data/recovered/modal')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase7/teacher_minus13_images' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2) + chr(10), encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return request_key
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase7/teacher_minus13_images/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        app = modal.App('nihonga-phase7-teacher-minus13-images')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'teacher_minus13_images_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8, memory=98304,
                              max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['generate_multiseed_text'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2) + chr(10))
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2) + chr(10), encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'calls')}), flush=True)
    return request_key

def build_teacher_minus13_grids(run):
    """One sheet per text prompt: Teacher, B, C, teacher without block 13 (skip) and with the bridge; seeds 0-3."""
    import hashlib
    import textwrap
    from PIL import Image as PILImage, ImageDraw, ImageFont
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='download')
    import modal
    volume = modal.Volume.from_name('nihonga-phase3-artifacts')
    key = run.rstrip('/').split('/')[-1]
    summary = json.loads((ROOT / run / 'summary.json').read_text(encoding='utf-8'))
    store = ROOT / '.archive/teacher_minus13_images' / key
    store.mkdir(parents=True, exist_ok=True)
    for artifact in summary['artifacts']:
        path = store / artifact['name']
        if not path.exists():
            path.write_bytes(b''.join(volume.read_file(f"/phase7/teacher_minus13_images/{key}/{artifact['name']}")))
        if hashlib.sha256(path.read_bytes()).hexdigest() != artifact['sha256']:
            raise RuntimeError('Image hash mismatch: ' + artifact['name'])
    first = json.loads(next((ROOT / 'data/phase7/bridge_images').glob('*/summary.json')).read_bytes())
    first_store = ROOT / '.archive/bridge_images' / first['request_key']
    reference = ROOT / '.archive/multiseed_text/af4a34ba1a0c739121706991b89e79ec92bff3debaf1b1ddbf5b6188c182f514'
    c_store = ROOT / '.archive/multiseed_text/320d82f71dfdea06818ff5f50e2e4af933519b5c511111159c1bdea4a00f79aa'
    face = next((f for f in ('C:/Windows/Fonts/msyh.ttc', 'C:/Windows/Fonts/arial.ttf') if Path(f).exists()))
    font, big = ImageFont.truetype(face, 18), ImageFont.truetype(face, 21)
    sheets = ROOT / run / 'sheets'
    sheets.mkdir(exist_ok=True)
    tile, label = 330, 260
    for row in [r for r in first['rows'] if r['subdimension'] in ('text_rendering', 'posters')]:
        sid = row['source_id']
        models = [('Teacher (32)', lambda k: first_store / row['teacher']['name'] if k == 0 else reference / f'{sid}-s{k}-teacher.png'),
                  ('B: 5 bridged', lambda k: first_store / row['B']['name'] if k == 0 else reference / f'{sid}-s{k}-B.png'),
                  ('C: 5 + 13 bridged', lambda k: c_store / (f'{sid}-e2e_attn8_text.png' if k == 0 else f'{sid}-s{k}-e2e_attn8_text.png')),
                  ('Teacher − 13 (skip)', lambda k: store / (f'{sid}-teacher_skip13.png' if k == 0 else f'{sid}-s{k}-teacher_skip13.png')),
                  ('Teacher − 13 (bridge)', lambda k: store / (f'{sid}-teacher_bridge13.png' if k == 0 else f'{sid}-s{k}-teacher_bridge13.png'))]
        canvas = PILImage.new('RGB', (label + 4 * tile, 80 + len(models) * tile), 'white')
        draw = ImageDraw.Draw(canvas)
        draw.text((10, 8), textwrap.shorten(row['prompt'], 190), font=font, fill='black')
        for column in range(4):
            draw.text((label + column * tile + 10, 48), f'seed {column}', font=big, fill='black')
        for line, (name, locate) in enumerate(models):
            y = 80 + line * tile
            draw.text((10, y + tile // 2 - 12), name, font=big, fill='black')
            for column in range(4):
                with PILImage.open(locate(column)) as image:
                    canvas.paste(image.convert('RGB').resize((tile, tile), PILImage.Resampling.LANCZOS), (label + column * tile, y))
        canvas.save(sheets / f'{sid}.jpg', quality=88)
    return sheets

LAUNCH_ARGS = ['--dry-run']
RUN = 'data/phase7/teacher_minus13_images/' + main()   # writes the request and returns its key; no GPU
LAUNCH_ARGS = []
summary = reuse_or_launch(RUN, main)
if summary and summary.get('status') == 'passed':
    sheets = build_teacher_minus13_grids(RUN)
    show_images(str(sheets.relative_to(ROOT)), ['p1-2fa54a3c4eb8.jpg', 'p1-c9344c076abf.jpg'])

### What the test means

| Exact signs (4 seeds) | Teacher | B | C (5 + 13) | Teacher - 13 (skip) | Teacher - 13 (bridge) |
|---|---|---|---|---|---|
| The Rusty Anchor | 4 | 3 | 1 | 2 | 1 |
| Happy Birthday Emma | 4 | 3 | 2 | 2 | 2 |
| Moonlight Motel | 3 | 3 | 3 | ~1 | ~1-2 |
| **Total / 12** | **11** | **9** | **6** | **~5** | **~4-5** |

**Block 13 is critical for spelling on its own.** With block 5 intact, removing block 13 drops spelling to about
5/12, the same as C. The same errors appear without block 5 involved ("Achor", "Enna"/"Emna"). The block-5 bridge
costs something (teacher 11 -> B 9), but much less. The reused bridge does not help here either, although it was
trained inside B and sees slightly different inputs. Choosing blocks by their text cost (step 16) is the right path.

### What the text-aware scan means

B alone measures 6.01% on the 80 text prompts and 6.04% on the 50
general prompts (early and middle moments, against the recorded teacher outputs). Skipping one block at a time:

| Rank | Block | Text error | Text cost | General error | General cost |
|---|---|---|---|---|---|
| 1 | 17 | 7.93% | +1.92 | 8.32% | +2.27 |
| 2 | 25 | 8.33% | +2.32 | 8.48% | +2.44 |
| 3 | 20 | 8.39% | +2.39 | 8.78% | +2.74 |
| 4 | 18 | 8.59% | +2.59 | 8.65% | +2.60 |
| 5 | 4 | 8.84% | +2.83 | 9.34% | +3.30 |
| 6 | 16 | 8.89% | +2.89 | 8.98% | +2.94 |
| 7 | 14 | 9.03% | +3.02 | 9.30% | +3.25 |
| 8 | 15 | 9.04% | +3.03 | 9.41% | +3.37 |
| 9 | 13 | 9.11% | +3.10 | 9.70% | +3.66 |
| 10 | 12 | 9.12% | +3.11 | 9.73% | +3.69 |
| 11 | 6 | 9.16% | +3.15 | 9.90% | +3.86 |
| 12 | 26 | 9.17% | +3.17 | 9.37% | +3.33 |
| 13 | 7 | 9.17% | +3.17 | 10.23% | +4.19 |
| 14 | 8 | 9.39% | +3.39 | 10.79% | +4.75 |
| 15 | 11 | 9.41% | +3.41 | 10.01% | +3.96 |
| 16 | 9 | 9.60% | +3.60 | 10.38% | +4.34 |
| 17 | 10 | 9.75% | +3.75 | 11.11% | +5.07 |
| 18 | 23 | 10.08% | +4.07 | 10.39% | +4.35 |
| 19 | 21 | 10.13% | +4.13 | 10.44% | +4.40 |
| 20 | 24 | 10.17% | +4.16 | 10.48% | +4.44 |
| 21 | 3 | 10.66% | +4.65 | 12.55% | +6.51 |
| 22 | 22 | 11.10% | +5.09 | 11.50% | +5.46 |
| 23 | 27 | 11.19% | +5.18 | 11.64% | +5.60 |
| 24 | 19 | 11.31% | +5.31 | 11.87% | +5.83 |
| 25 | 2 | 12.02% | +6.02 | 13.94% | +7.90 |
| 26 | 28 | 14.15% | +8.15 | 15.01% | +8.96 |
| 27 | 29 | 18.08% | +12.07 | 18.99% | +12.95 |

**The check fails.** Block 13, whose removal breaks spelling in the images, sits in the middle of the ranking: its
text cost (+3.10 points) is no higher than its general cost (+3.66). Letters are a tiny fraction of the pixels, so
a misspelled sign barely moves the average velocity error. Velocity on text prompts therefore filters overall damage
but cannot judge spelling; the qualitative test decides.

**Registered addendum** (`data/phase7/text_importance/protocol_addendum.json`): the image test (16c) was launched
for blocks 17, 20, 18 and 16 while the scan was still running blocks 23-29. Block 25 then ranked second (8.33%) and
had been left out, so it is added as step 16d with the same decision rule.

## 16c. Qualitative test of the text-aware scan: spelling with each candidate skipped

The velocity scan cannot see spelling: block 13, which breaks it, does not stand out (its text cost, +3.1 points, is
no higher than its general cost). Letters are a tiny fraction of the pixels, so a misspelled sign barely moves the
average error. The scan is kept as a filter for overall damage, and the decision is made on images: inside B, each
candidate with a low velocity cost and not next to block 5 (17, 20, 18, 16) is skipped in turn
and the 4-seed text grid is rendered, beside the teacher, B and the intact teacher without block 13 (about 5/12).

In [ ]:
# Skip-candidate text grids: reuses the saved run; launches GPU work only when its result is missing and ALLOW_GPU is True.
SKIP_BLOCKS = [17, 20, 18, 16]
SOURCES['skip_candidate_images_worker.py'] = r'''def generate_multiseed_text(request, key):
    """Extra seeds for the text prompts: teacher, B, and C with the best block-13 bridge.
    Same prompt, CPU noise seed, size, scheduler and 40 steps for every model. Qualitative screen only."""
    import gc
    import hashlib
    import inspect
    import json
    import os
    import time
    from pathlib import Path

    import torch
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache, QwenImage21KVLayerCache

    directory = Path('/phase3/phase7/skip_candidate_images') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete image attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def table_hash(value):
        return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

    variants = [f'B_skip{b}' for b in request['skip_blocks']]
    report = {'request_key': key, 'status': 'running', 'rows': [], 'artifacts': [],
              'calls': {v: 0 for v in variants}, 'optimizer_updates': 0}
    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('Expected H100.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
                                                   dtype=torch.bfloat16, local_files_only=True).to('cuda')
        pipe.set_progress_bar_config(disable=True)
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Transformer source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--')) /
                       'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        # Derived position/time tables: use the audited values on every host (as in steps 7-11).
        derived = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
        derived['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
        expected = request['expected_derived_tables']
        mismatched = sorted(n for n, v in derived.items() if table_hash(v) != expected[n])
        if mismatched:
            canonical = torch.load(Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt'),
                                   weights_only=True, map_location='cpu')
            if any(table_hash(canonical[n]) != expected[n] for n in expected):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [canonical[f'pos_embed.freqs.{a}'].to(device=v.device, dtype=v.dtype) for a, v in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(canonical['time_text_embed.time_proj.freqs'])
            if any(table_hash(v) != expected[n] for n, v in
                   list({f'pos_embed.freqs.{a}': v for a, v in enumerate(model.pos_embed.freqs)}.items()) +
                   [('time_text_embed.time_proj.freqs', model.time_text_embed.time_proj.freqs)]):
                raise RuntimeError('Derived tables still differ after loading the audited copy.')
            report['canonical_tables_applied'] = mismatched

        def generate(row, variant):
            path = directory / (row['source_id'] + '-' + variant + '.png')
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'],
                             num_inference_steps=40, true_cfg_scale=teacher['true_cfg_scale'],
                             generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=True, output_type='pil').images[0]
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            temporary = path.with_suffix('.tmp')
            image.save(temporary, format='PNG')
            temporary.replace(path)
            results_volume.commit()
            report['calls'][variant] += 1
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            return {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size, 'seconds': elapsed}

        namespace = {}
        exec(compile(request['core_source'], 'bridge_definition.py', 'exec'), namespace)
        exec(compile(request['wrapper_source'], 'bridge_block_definition.py', 'exec'), namespace)
        conditioned = {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned)

        def load_checkpoint(spec):
            path = Path(spec['remote_path'])
            if digest(path) != spec['sha256']:
                raise RuntimeError('Bridge checkpoint differs: ' + spec['remote_path'])
            return torch.load(path, weights_only=True, map_location='cpu')

        outputs = {v: {} for v in variants}
        references = False
        if references:
            for row in request['rows']:
                outputs['teacher'][row['source_id']] = generate(row, 'teacher')
        block5 = request['block5_bridge']
        checkpoint = load_checkpoint(block5)
        if checkpoint['layer'] != 5 or checkpoint['step'] != block5['step']:
            raise RuntimeError('Block-5 bridge provenance differs.')
        bridge = namespace['ResidualBottleneckBridge'](4096, 256)
        bridge.load_state_dict(checkpoint['state_dict'])
        bridge.requires_grad_(False)
        namespace['insert_bridge'](model, 5, bridge, QwenImage21KVCache)
        model.eval()
        if references:
            for row in request['rows']:
                outputs['B'][row['source_id']] = generate(row, 'B')
        from types import SimpleNamespace

        class IdentityBlock(torch.nn.Module):
            def __init__(self, original):
                super().__init__()
                self.attn = SimpleNamespace(processor=original.attn.processor)

            def forward(self, hidden_states, *args, **kwargs):
                return hidden_states

        for slot in request['skip_blocks']:
            original = model.transformer_blocks[slot]
            model.transformer_blocks[slot] = IdentityBlock(original)
            model.eval()
            try:
                for row in request['rows']:
                    outputs[f'B_skip{slot}'][row['source_id']] = generate(row, f'B_skip{slot}')
            finally:
                model.transformer_blocks[slot] = original
        rendered = [v for v in variants if outputs[v]]
        for row in request['rows']:
            report['rows'].append({**row, **{v: outputs[v][row['source_id']] for v in rendered}})
            report['artifacts'].extend(outputs[v][row['source_id']] for v in rendered)
        report['status'] = 'passed'
    except Exception as exc:
        report['status'] = 'failed'
        report['error'] = type(exc).__name__ + ': ' + str(exc)
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=True) + '\n')
    results_volume.commit()
    return report
'''

def main():
    import argparse
    import hashlib
    parser = argparse.ArgumentParser()
    parser.add_argument('--dry-run', action='store_true')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    load = lambda path: json.loads((root / path).read_text(encoding='utf-8'))
    base = json.loads(next((root / 'data/recovered/modal/phase7/candidate_images').glob('*/request.json')).read_bytes())['request']
    scan_b = load('data/phase7/conditional_importance_b/b0a27f0cbe0f8e7f9938044fe088576a648c7b22e63a2ddbd771ee6b8e777aae/request.json')['request']
    text8 = load('data/phase7/block13_bridge_e2e_text8/ff9d259ae589bdf15b98076930c4d16340ef41f4a2c3db897ea3167e95397054/request.json')
    selected = scan_b['selected_candidate']
    text_rows = [r for r in base['rows'] if r['subdimension'] in ('text_rendering', 'posters')]
    rows = list(text_rows) + [{**r, 'source_id': f"{r['source_id']}-s{k}", 'seed': (r['seed'] + 1000003 * k) % 2**32}
                              for r in text_rows for k in (1, 2, 3)]
    request = {k: base[k] for k in ('teacher', 'transformer_source_sha256', 'runtime_environment', 'expected_derived_tables',
                                    'canonical_tables_key', 'core_source', 'wrapper_source')}
    request.update(
        version='phase7-skip-candidate-images-v1', rows=rows, worker_source=SOURCES['skip_candidate_images_worker.py'],
        conditioned_bridge_source=text8['request']['conditioned_bridge_source'], execution_account='new', candidate_only=True,
        skip_blocks=SKIP_BLOCKS,
        block5_bridge=dict(remote_path=f"/phase3/phase5/fitting/{scan_b['fit_key']}/{Path(selected['checkpoint_path']).name}",
                           sha256=selected['checkpoint_sha256'], step=selected['selected_update']),
        c_variant=dict(name='unused'),
        scope='Inside B: each shortlisted block skipped in turn; 4-seed text grid for the qualitative test of the text-aware scan')
    from experiment_runtime import pin_reference_runtime
    pin_reference_runtime(request, root / 'data/recovered/modal')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase7/skip_candidate_images' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2) + chr(10), encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return request_key
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase7/skip_candidate_images/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        app = modal.App('nihonga-phase7-skip-candidate-images')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'skip_candidate_images_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8, memory=98304,
                              max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['generate_multiseed_text'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2) + chr(10))
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2) + chr(10), encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'calls')}), flush=True)
    return request_key

def build_skip_grids(run):
    """One sheet per text prompt: Teacher, B, B without block 13 (reference), then B without each candidate; seeds 0-3."""
    import hashlib
    import textwrap
    from PIL import Image as PILImage, ImageDraw, ImageFont
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='download')
    import modal
    volume = modal.Volume.from_name('nihonga-phase3-artifacts')
    key = run.rstrip('/').split('/')[-1]
    summary = json.loads((ROOT / run / 'summary.json').read_text(encoding='utf-8'))
    store = ROOT / '.archive/skip_candidate_images' / key
    store.mkdir(parents=True, exist_ok=True)
    for artifact in summary['artifacts']:
        path = store / artifact['name']
        if not path.exists():
            path.write_bytes(b''.join(volume.read_file(f"/phase7/skip_candidate_images/{key}/{artifact['name']}")))
        if hashlib.sha256(path.read_bytes()).hexdigest() != artifact['sha256']:
            raise RuntimeError('Image hash mismatch: ' + artifact['name'])
    first = json.loads(next((ROOT / 'data/phase7/bridge_images').glob('*/summary.json')).read_bytes())
    first_store = ROOT / '.archive/bridge_images' / first['request_key']
    reference = ROOT / '.archive/multiseed_text/af4a34ba1a0c739121706991b89e79ec92bff3debaf1b1ddbf5b6188c182f514'
    minus13 = ROOT / '.archive/teacher_minus13_images/3145e19e91dd8475d3594807e3be3fbe7d08d449edd0960be295726699bec86e'
    blocks = json.loads((ROOT / run / 'request.json').read_bytes())['request']['skip_blocks']
    face = next((f for f in ('C:/Windows/Fonts/msyh.ttc', 'C:/Windows/Fonts/arial.ttf') if Path(f).exists()))
    font, big = ImageFont.truetype(face, 18), ImageFont.truetype(face, 21)
    sheets = ROOT / run / 'sheets'
    sheets.mkdir(exist_ok=True)
    tile, label = 300, 260
    for row in [r for r in first['rows'] if r['subdimension'] in ('text_rendering', 'posters')]:
        sid = row['source_id']
        name = lambda k, suffix: f'{sid}-{suffix}.png' if k == 0 else f'{sid}-s{k}-{suffix}.png'
        models = [('Teacher (32)', lambda k: first_store / row['teacher']['name'] if k == 0 else reference / name(k, 'teacher')),
                  ('B (block 5 bridged)', lambda k: first_store / row['B']['name'] if k == 0 else reference / name(k, 'B')),
                  ('Teacher − 13 (ref.)', lambda k: minus13 / name(k, 'teacher_skip13'))]
        models += [(f'B − {b} (skip)', lambda k, b=b: store / name(k, f'B_skip{b}')) for b in blocks]
        canvas = PILImage.new('RGB', (label + 4 * tile, 80 + len(models) * tile), 'white')
        draw = ImageDraw.Draw(canvas)
        draw.text((10, 8), textwrap.shorten(row['prompt'], 190), font=font, fill='black')
        for column in range(4):
            draw.text((label + column * tile + 10, 48), f'seed {column}', font=big, fill='black')
        for line, (title, locate) in enumerate(models):
            y = 80 + line * tile
            draw.text((10, y + tile // 2 - 12), title, font=big, fill='black')
            for column in range(4):
                with PILImage.open(locate(column)) as image:
                    canvas.paste(image.convert('RGB').resize((tile, tile), PILImage.Resampling.LANCZOS), (label + column * tile, y))
        canvas.save(sheets / f'{sid}.jpg', quality=88)
    return sheets

LAUNCH_ARGS = ['--dry-run']
RUN = 'data/phase7/skip_candidate_images/' + main()   # writes the request and returns its key; no GPU
LAUNCH_ARGS = []
summary = reuse_or_launch(RUN, main)
if summary and summary.get('status') == 'passed':
    sheets = build_skip_grids(RUN)
    show_images(str(sheets.relative_to(ROOT)), ['p1-2fa54a3c4eb8.jpg', 'p1-c9344c076abf.jpg'])

## 16d. Block 25 added to the qualitative test

Same renderer and settings as step 16c (inside B, block skipped, 5 text prompts x 4 seeds); the combined sheet shows
every candidate (17, 20, 18, 16 and 25) beside the teacher, B and the teacher without block 13.

In [ ]:
# Block 25 skip grid: reuses the 16c renderer (run step 16c first); GPU only when the result is missing and ALLOW_GPU is True.
SKIP_BLOCKS = [25]
LAUNCH_ARGS = ['--dry-run']
RUN_25 = 'data/phase7/skip_candidate_images/' + main()   # writes the request and returns its key; no GPU
LAUNCH_ARGS = []
summary_25 = reuse_or_launch(RUN_25, main)

def build_combined_skip_grids(runs, out):
    """Teacher, B, teacher without 13, then every skipped candidate from the given runs; one sheet per text prompt."""
    import textwrap
    from PIL import Image as PILImage, ImageDraw, ImageFont
    for run in runs:
        build_skip_grids(run)          # downloads and hash-checks each run's images
    first = json.loads(next((ROOT / 'data/phase7/bridge_images').glob('*/summary.json')).read_bytes())
    first_store = ROOT / '.archive/bridge_images' / first['request_key']
    reference = ROOT / '.archive/multiseed_text/af4a34ba1a0c739121706991b89e79ec92bff3debaf1b1ddbf5b6188c182f514'
    minus13 = ROOT / '.archive/teacher_minus13_images/3145e19e91dd8475d3594807e3be3fbe7d08d449edd0960be295726699bec86e'
    candidates = [(b, ROOT / '.archive/skip_candidate_images' / run.rstrip('/').split('/')[-1]) for run in runs
                  for b in json.loads((ROOT / run / 'request.json').read_bytes())['request']['skip_blocks']]
    face = next((f for f in ('C:/Windows/Fonts/msyh.ttc', 'C:/Windows/Fonts/arial.ttf') if Path(f).exists()))
    font, big = ImageFont.truetype(face, 18), ImageFont.truetype(face, 21)
    out = ROOT / out
    out.mkdir(parents=True, exist_ok=True)
    tile, label = 280, 250
    for row in [r for r in first['rows'] if r['subdimension'] in ('text_rendering', 'posters')]:
        sid = row['source_id']
        name = lambda k, suffix: f'{sid}-{suffix}.png' if k == 0 else f'{sid}-s{k}-{suffix}.png'
        models = [('Teacher (32)', lambda k: first_store / row['teacher']['name'] if k == 0 else reference / name(k, 'teacher')),
                  ('B (block 5 bridged)', lambda k: first_store / row['B']['name'] if k == 0 else reference / name(k, 'B')),
                  ('Teacher − 13 (ref.)', lambda k: minus13 / name(k, 'teacher_skip13'))]
        models += [(f'B − {b} (skip)', lambda k, b=b, s=s: s / name(k, f'B_skip{b}')) for b, s in candidates]
        canvas = PILImage.new('RGB', (label + 4 * tile, 80 + len(models) * tile), 'white')
        draw = ImageDraw.Draw(canvas)
        draw.text((10, 8), textwrap.shorten(row['prompt'], 190), font=font, fill='black')
        for column in range(4):
            draw.text((label + column * tile + 10, 48), f'seed {column}', font=big, fill='black')
        for line, (title, locate) in enumerate(models):
            y = 80 + line * tile
            draw.text((10, y + tile // 2 - 12), title, font=big, fill='black')
            for column in range(4):
                with PILImage.open(locate(column)) as image:
                    canvas.paste(image.convert('RGB').resize((tile, tile), PILImage.Resampling.LANCZOS), (label + column * tile, y))
        canvas.save(out / f'{sid}.jpg', quality=88)
    return out

if summary_25 and summary_25.get('status') == 'passed' and summary and summary.get('status') == 'passed':
    combined = build_combined_skip_grids([RUN, RUN_25], 'data/phase7/skip_candidate_images/combined_sheets')
    show_images(str(combined.relative_to(ROOT)), ['p1-2fa54a3c4eb8.jpg', 'p1-c9344c076abf.jpg'])